# CADET v2 -- Final improvement phase (video only; EfficientNet-B1 + LSTM; FGSM is the only attack)

Built on the executed v1 notebook (not rebuilt). Attach the **output of the v1 run as an input dataset**: valid checkpoints, histories and face crops are verified (provenance) and re-used, never silently replaced.

| Priority | Cell | What it does |
|---|---|---|
| P5 reproducibility | 24b | checkpoint provenance (sha256), per-epoch gate decomposition, seed-2 gate reconciliation, valid-seed list |
| P1 threshold collapse | 24c | thresholds A (clean-val EER) / B (attack-aware, eps 0.5/1/2) frozen per seed on validation BEFORE test; both models, 3 seeds, eps 0-8 |
| P2 unseen datasets | 25a | leave-one-dataset-out (FF++ and Celeb-DF held out), leakage checks, registry; UADFV exploratory/pending |
| P3 clean performance | 25b | controlled one-change-at-a-time pilots (validation only), predefined score, guarded candidate run |
| P4 Grad-CAM | 21b | >=100 videos, stability vs faithfulness (localisation NOT evaluated), layer ablation |
| report | 26 | final report A-F from artifacts on disk |

FGSM only: no PGD/BIM/AutoAttack. Anything not executed is recorded as **PENDING**; no result is claimed for it. Outputs are stripped: this notebook has NOT been run on real data.


In [ ]:
# ============================================================
# CELL 1: Configuration, imports and shared utilities
# ============================================================
import os
# Reduce CUDA memory fragmentation (must be set before torch initializes its
# CUDA caching allocator, so this line stays first, before `import torch`).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import copy
import gc
import hashlib
import json
import math
import random
import time
from contextlib import nullcontext
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch._utils  # noqa: F401
# Kaggle-environment fix (preserved): some torch/torchvision builds don't have
# `torch._utils` bound as an attribute of `torch` yet at this point, which
# makes `from torchvision import transforms` fail with
# "AttributeError: module 'torch' has no attribute '_utils'". Must run before
# any torchvision import.
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
import torchvision.transforms.functional as TF

import torch.multiprocessing as _mp
try:
    # Prefer "fork" so DataLoader worker processes inherit this process's
    # already-fixed `torch` module instead of re-importing it from scratch.
    _mp.set_start_method("fork", force=True)
except RuntimeError:
    pass  # start method already set for this kernel session; safe to ignore

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    classification_report, roc_curve,
)
from tqdm import tqdm

try:
    from scipy.stats import binomtest as _binomtest
except Exception:          # very old scipy: fall back to a manual exact test below
    _binomtest = None


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

NOTEBOOK_START = time.time()   # used by the time guards (multi-seed loop, runtime estimate)
OUT_DIR = os.environ.get("V8_OUT_DIR", "/kaggle/working")      # every output of this notebook lives here
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# ============================================================
# CONFIGURATION  (v8: FGSM is the ONLY adversarial attack; there is no multi-step attack anywhere in this notebook)
# ============================================================
CONFIG = {
    # ---- Data ----
    "datasets": {
        "ffpp": {
            "root": "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23",
            "parser": "parse_ff_c23",
            "weight": 0.45
        },
        "celeb_df": {
            "root": "/kaggle/input/datasets/reubensuju/celeb-df-v2",
            "parser": "parse_celeb_df",
            "weight": 0.45  # Set > 0 to enable
        },
        "uadfv": {
            "root": "/kaggle/input/datasets/adityakeshri9234/uadfv-dataset/UADFV",
            "parser": "parse_uadfv",
            "weight": 0.10           
        }
    },
    "max_videos_to_use": 3000, # v10: budget of SELECTED videos (all splits), applied AFTER the identity-disjoint split (v9 truncated before it: 8000)
    "seed": 42,                       # TRAINING seed of the main run
    "split_seed": 42,                 # identity split seed (shared by all training seeds)
    "extra_seeds": [1, 2],            # extra TRAINING seeds (Cell 18). Each starts ONLY if the time guard says it fits; mode = full
                                      # retrain (clean + FGSM) if that fits, else FGSM-AT only from the shared clean checkpoint (labelled).
    "time_budget_hours": 9.0,         # planning budget for the WHOLE notebook (user limit: 8-9 h on Kaggle T4)
    "seed_reserve_hours": 0.5,        # an extra seed must be predicted to finish this long before the budget ends

    # ---- Identity-disjoint split ----
    "val_id_fraction": 0.15,
    "test_id_fraction": 0.20,
    "split_search_trials": 300,
    "train_per_class_cap": 2500,
    "eval_per_class_cap": 1000,
    "train_fake_to_real_ratio": 1.0,  # if is 2.0, then 2 fakes per real in TRAIN (class-balanced sampler is enabled automatically)

    # ---- Temporal sampling ----
    "seq_len": 16,                    # model input: 16 frames = one of the two frames of each of 16 equal bins of the 32-frame pool (train: random; eval: first of bin)
    "frame_pool_size": 32,
    "img_size": 224,

    # ---- Face cropping ----
    "face_margin": 1.3,
    "face_crop_size": 256,
    "face_detect_every": 4,
    "face_min_detect_rate": 0.25,
    "face_cache_input_dir": None,     # e.g. "/kaggle/input/<crops-dataset>/face_frames"; provenance is CHECKED before reuse
    "face_frames_dir": os.path.join(OUT_DIR, "face_frames"),

    # ---- Batching ----
    "batch_size": 8,
    "fgsm_batch_size": 8,             # FGSM-AT batch; `adv_ratio` of it is perturbed, the rest stays clean
    "eval_batch_size": 8,
    "eval_attack_batch_size": 4,      # fp32 attacks need more memory than AMP ones
    "num_workers": 2,

    # ---- Social-media DEGRADATION (random training augmentation) ----
    "compression_prob": 0.3,
    "compression_downscale_range": (0.7, 1.0),
    "compression_jpeg_quality_range": (30, 70),
    "compression_blur_prob": 0.3,
    "compression_blur_kernel": 3,

    # ---- Spatial AUGMENTATION ----
    "aug_flip_prob": 0.5,
    "aug_rotation_range": (-5.0, 5.0),
    "aug_brightness_range": (0.9, 1.1),
    "aug_contrast_range": (0.9, 1.1),
    "aug_saturation_range": (0.95, 1.05),
    "aug_hue_range": (0.0, 0.0),
    "aug_crop_padding": 4,
    "aug_noise_prob": 0.2,
    "aug_noise_std": 0.01,

    # ---- Model architecture (unchanged) ----
    "lstm_hidden_size": 512,
    "lstm_num_layers": 2,
    "lstm_bidirectional": False,
    "dropout": 0.3,
    "use_temporal_attention": False,

    # ---- Clean (control) training: identical to v6/v7 ----
    "phase1_epochs": 0,
    "phase2_epochs": 16,
    "unfreeze_last_n_blocks": 5,
    "warmup_epochs": 1.0,
    "min_lr_ratio": 0.02,
    "backbone_lr": 7e-5,
    "head_lr": 2e-4,
    "weight_decay": 2e-2,
    "grad_clip_norm": 5.0,
    "early_stopping_patience": 4,
    "label_smoothing": 0.05,
    "ema_decay": 0.998,

    # ---- FGSM adversarial training (the ONLY adversarial component) ----
    # v9 recipe, every change below is motivated by a measured v8 failure (see Cell 7b and the notes in Cell 6):
    #  * v8 collapsed to the trivial hedge (train loss = ln 2 = 0.693, logit std 0.04, clean val acc 0.50) in epochs 1-2 because the
    #    clean-trained network is broken by eps >= 0.25/255, so the adversarial half could only be 'solved' by outputting ~0.
    #    -> slow eps ramp, clean-heavy start (adversarial ratio ramps), clean-heavy loss weights, adversarial logit pairing (ALP).
    #  * v8 stopped after 3 epochs although validation was still improving (FGSM AUC 0.045 -> 0.167 -> 0.375).
    #    -> the early-stop rule needs >= `early_stop_min_epochs` epochs and no recent improvement.
    "fgsm_epochs": 14,
    "fgsm_steps_per_epoch": 300,      # batches per epoch (300 x 8 = 2400 videos)
    "fgsm_lr_scale": 0.25,            # LRs = 0.25 x clean-training LRs (fine-tuning from the clean checkpoint)
    "fgsm_warmup_steps": 150,
    "fgsm_train_eps": 2.0 / 255.0,    # v2: FIXED to the recipe the v1 pilot selected on VALIDATION (random U(0,2/255), no ALP); the v1 pilot (Cell 8) is therefore skipped. Not tuned on test.
    "fgsm_eps_ramp_epochs": 4.0,      # epsilon ramps 0 -> train eps over this many epochs (v8: 2)
    "adv_ratio": 0.5,                 # final fraction of every batch replaced by its FGSM version
    "adv_ratio_start": 0.25,          # ... starting here and ramping linearly over `adv_ratio_ramp_epochs`
    "adv_ratio_ramp_epochs": 4.0,
    "adv_label_source": "true",       # labels used to CRAFT the training perturbation: "true" or "pred" (model's own prediction)
    "adv_random_eps": True,           # v2: = the v1 pilot choice (validation). per-sample eps ~ U(0, eps_t)
    "fgsm_bn_eval": True,             # BatchNorm statistics frozen during adversarial training
    "fgsm_clean_loss_weight": 0.6,    # v8: 0.5 / 0.5 -> clean-heavy to protect clean accuracy
    "fgsm_adv_loss_weight": 0.4,
    "alp_weight": 0.0,                # v2: = the v1 pilot choice (no ALP); adversarial logit pairing (disabled): Huber(z_adv, stopgrad(z_clean of the SAME video)); counters the hedge
    "alp_huber_delta": 1.0,           # (a regulariser on the loss, not an attack: FGSM remains the only adversarial example generator)
    "fgsm_grad_clip_mode": "adaptive",# "adaptive": clip at mult x running mean of the (clipped) norm, never below clip_min; "fixed": clip_norm
    "fgsm_grad_clip_norm": 1.0,       # used by "fixed" and as the floor of "adaptive"
    "fgsm_grad_clip_mult": 2.5,
    "ema_decay_fgsm": 0.995,
    "ga_check_every": 50,             # catastrophic-overfitting heuristic: gradient-alignment cosine every N steps
    "train_attack_amp": None,         # None -> decided by the AMP-vs-fp32 attack audit (Cell 7b); True/False forces it
    "nan_fp32_retry": True,           # a non-finite fp16 step is re-computed in fp32 once before it is skipped

    # ---- Epsilon grids (pixel scale / 255) ----
    "eval_eps_grid": [0, 0.25, 0.5, 1, 1.5, 2, 4, 8, 12, 16],       # EVALUATION epsilons (x/255)
    "val_eps_grid": [0.5, 1, 2],      # validation epsilons used every epoch for selection (mean / min over them)
    "target_eval_eps": 1.0,           # target epsilon (x/255) for the success criteria and the attack-aware threshold
    "multiseed_eps_grid": [0, 0.5, 1, 2, 4, 8],

    # ---- Pilot (validation only): recipes are compared, none is assumed ----
    "pilot_epochs": 3,
    "pilot_steps": 150,
    "pilot_ramp_epochs": 1.5,
    "pilot_variants": [               # (name, train eps /255, random per-sample eps, ALP weight)
        ("fixed 1/255 + ALP", 1.0, False, 0.2),
        ("random U(0,1/255) + ALP", 1.0, True, 0.2),
        ("random U(0,2/255) + ALP", 2.0, True, 0.2),
        ("random U(0,2/255), no ALP", 2.0, True, 0.0),
    ],
    "pilot_min_clean_auc_drop": 0.06, # a variant is 'healthy' only if it did not hedge and kept its clean AUC within this of the baseline
    "pilot_min_clean_logit_std": 1.0, # hedging detector: clean logit std below this = collapsed (clean-trained model: ~3)

    # ---- Validation-only checkpoint selection ----
    "val_adv_n": 160,                 # fixed balanced validation subset on which FGSM metrics are computed every epoch
    "select_smooth_epochs": 2,
    "select_weights": {"clean_auc": 0.4, "fgsm_mean_auc": 0.4, "fgsm_min_auc": 0.2},
    "select_stability_penalty": 0.5,
    "select_max_clean_auc_drop": 0.03,    # gate: val clean AUC >= baseline - this
    "select_min_fgsm_auc": 0.55,      # gate: val FGSM AUC (target eps) must exceed this
    "select_min_logit_std": 0.8,      # gate: clean logit std on validation must not collapse (hedging)
    "early_stop_min_epochs": 7,       # never stop before this many epochs (v8 stopped at 3, while still improving)
    "early_stop_bad_epochs": 4,       # ... then stop after this many consecutive failed gates ...
    "early_stop_no_gain": 0.01,       # ... AND the best raw score of the last 3 epochs did not beat the earlier best by this much

    # ---- Acceptance criteria (frozen to acceptance_criteria.json BEFORE the test set is touched) ----
    "accept_min_test_clean_auc_ratio": 0.97,
    "accept_min_val_fgsm_auc_target": 0.75,
    "accept_min_val_fgsm_auc_gain": 0.10,
    "accept_max_label_leak_margin": 0.02,
    "accept_min_class_recall": 0.35,           # C7: at the target eps, with the attack-aware threshold, BOTH real specificity and fake recall >= this
    "accept_min_seeds_passing": 2,             # C8: FGSM validation gate must pass in at least this many seeds

    # ---- Evaluation ----
    "eval_use_amp": False,            # evaluation + attacks in fp32
    "threshold_method": "eer",
    "threshold_search_min": 0.05,
    "threshold_search_max": 0.95,
    "threshold_search_step": 0.01,
    "bootstrap_n": 1000,
    "diag_n_per_class": 40,
    "audit_n_videos": 24,             # validation videos used by the attack audit (Cell 7b)           # test videos per class for the (cheap, FGSM-only) diagnostics of Cell 15

    # ---- Checkpoints / artifacts ----
    "save_clean_checkpoint": os.path.join(OUT_DIR, "best_clean_model.pth"),
    "save_fgsm_checkpoint": os.path.join(OUT_DIR, "best_fgsm_model.pth"),
    "resume_clean_path": os.path.join(OUT_DIR, "resume_clean.pth"),
    "resume_fgsm_path": os.path.join(OUT_DIR, "resume_fgsm.pth"),
    "clean_checkpoint_input": None,   # re-use a clean checkpoint from an earlier session; PROVENANCE IS CHECKED (split + architecture)
    "auto_discover_inputs": True,     # look in /kaggle/input for a face-crop cache and best_clean_model.pth (provenance-checked before use)
    "metrics_csv": os.path.join(OUT_DIR, "fgsm_v9_metrics.csv"),
    "acceptance_json": os.path.join(OUT_DIR, "acceptance_criteria.json"),
    "run_config_json": os.path.join(OUT_DIR, "run_config.json"),


    # ---- v10: data selection / audit / frame policy / held-out-dataset evaluation / Grad-CAM (all new keys) ----
    "selection_mode": "stratified_random_v10",   # "legacy_v9" reproduces the v9 selection EXACTLY (alphabetical first-N + character-level ids);
                                                 # use it ONLY to re-use v9 checkpoints / face caches (their provenance check needs the v9 split)
    "selection_seed": 42,                        # seed of the reproducible stratified sampling (dataset x label x manipulation method)
    "legacy_max_videos_to_use": 8000,            # the v9 value of max_videos_to_use (used only by legacy_v9)
    "split_refine_passes": 3,                    # local-search passes that move single identities to cut the number of dropped videos
    "split_quota_slack": 0.15,                   # per-dataset split quotas may deviate by this fraction during the refinement
    "split_budget_fractions": {"train": 0.70, "val": 0.12, "test": 0.18},   # share of max_videos_to_use per split (v9 realised 0.71/0.12/0.17)
    "dedupe_videos": True,                       # drop byte-identical videos (size + first/last 64 KiB); conflicting labels -> all copies dropped
    "min_usable_frames": 16,                     # >= seq_len. Videos with fewer usable frames are DROPPED and counted (v9 repeated the last frame)
    "short_video_policy": "drop",                # "drop" (v10) | "pad_repeat_last" (v9 behaviour; padded sequences contain repeated frames)
    "extract_sec_per_video": 2.5,                # planning constant: v9 measured 2.50 s/video (1 h 38 min for 2356 videos) on a Kaggle T4
    "min_dataset_videos_per_class_eval": 20,     # a dataset with fewer val/test videos per class is flagged NOT EVALUABLE in that split
    "face_fallback_warn_rate": 0.10,             # warn when a dataset x class has a larger centre-crop fallback rate
    # held-out-dataset (leave-one-dataset-out) evaluation, Cell 19b. Off by default: it retrains both models per fold (about 1.2 h per fold on a T4).
    "lodo_enabled": True,                         # v2: Priority 2 (guarded by the time budget; every fold is resumable)
    "lodo_datasets": None,                       # None -> every dataset with >= lodo_min_videos_per_class real AND fake eligible videos
    "lodo_min_videos_per_class": 100,
    "lodo_seeds": [42],
    "lodo_repilot": False,                       # False: re-use the recipe picked by the MIXED pilot (disclosed leak: that validation set contained the held-out dataset)
    "lodo_train_per_class_cap": 600,
    "run_pilots_v10": True,                      # controlled one-change-at-a-time FGSM-AT pilots (Cell 19b); each is ~0.4 h on a T4
    "pilot_v10_epochs": 6, "pilot_v10_steps": 150,
    # Grad-CAM v2 and explanation tests (Cell 16b)
    "gradcam_ref_percentile": 99.0,              # shared display scale = this percentile of the RAW maps of all panels of one figure
    "gradcam_n_videos": 16,                      # videos used by the quantitative explanation (deletion) tests
    "gradcam_occlusion_fracs": [0.1, 0.2, 0.3, 0.5],
    "gradcam_random_repeats": 5,

    # ---------------- v2 additions (Priorities 1-5) ----------------
    "v1_output_dir": None,                       # None -> auto-discovered under /kaggle/input (a folder holding best_fgsm_model.pth + best_clean_model.pth)
    "reuse_v1_artifacts": True,                  # copy v1 checkpoints / histories into the working dir and verify their provenance before use
    "v2_eval_eps_grid": [0, 0.5, 1, 2, 4, 8],    # /255, Priority 1 and 2 evaluation epsilons
    "v2_thr_val_eps": [0.5, 1, 2],               # /255, validation FGSM conditions that define the attack-aware threshold (predefined)
    "v2_thr_tol": 0.01,                          # near-optimal tolerance of the attack-aware objective; the median near-optimal threshold is used
    "v2_label_free_eval": True,                  # also evaluate FGSM built from the PREDICTED label (no label information in the attack)
    "v2_bootstrap_n": 2000,
    "v2_strict_class_mapping_check": True,        # asserts FAKE logits > REAL logits and AUC > 0.5 on clean validation (class mapping Real=0 / Fake=1); only synthetic smoke tests switch it off
    "v2_seeds": [42, 1, 2],
    "lodo_heldout_per_class": 250,               # held-out videos per class (seeded, stratified by method); bounded by face-extraction time (~2.5 s/video)
    "lodo_clean_epochs": 12, "lodo_fgsm_epochs": 10, "lodo_fgsm_steps": 200,   # compute-limited fold recipe, identical for both folds
    "lodo_run_uadfv_exploratory": False,         # UADFV has 49 real + 49 fake videos: exploratory only; off by default
    "lodo_est_hours_per_fold": 1.4,              # planning figure for the time guard, not a measurement
    "v2_candidate_train": True,                  # full run of the best validated pilot configuration (only if it beats the baseline pilot by v2_candidate_min_gain)
    "v2_candidate_min_gain": 0.01,
    "v2_candidate_extra_seeds": [1, 2],          # time-guarded
    "v2_phase_reserve_hours": 0.4,
    "gradcam_v2_n_videos": 120,                  # >= 100, stratified by dataset x class x method, includes misclassified videos
    "gradcam_v2_eps": [0.5, 1, 2],               # /255
    "gradcam_v2_delete_fracs": [0.1, 0.3, 0.5],
    "gradcam_v2_random_repeats": 3,
    "gradcam_v2_layer_ablation_n": 30,

    # ---- System ----
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}
CONFIG["fgsm_epochs_effective"] = CONFIG["fgsm_epochs"]
USE_AMP = (CONFIG["device"].type == "cuda")   # AMP is used for TRAINING only; evaluation and evaluation attacks are fp32
EPS_UNIT = 1.0 / 255.0


# ---- Shared utility functions ----
def free_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def fmt_eps(eps_pixel):
    """Pixel epsilon -> '/255' label without rounding away fractions (0.25/255 -> '0.25')."""
    return f"{eps_pixel * 255:g}"


def atomic_torch_save(obj, path):
    """Write to a temp file in the same directory, flush to disk, then os.replace: a kill mid-write can never leave a
    corrupt checkpoint behind."""
    path = str(path)
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        torch.save(obj, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_json_save(obj, path):
    path = str(path)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path


def _config_fingerprint(keys):
    d = {k: (list(CONFIG[k]) if isinstance(CONFIG[k], tuple) else CONFIG[k]) for k in keys}
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]


ARCH_KEYS = ["lstm_hidden_size", "lstm_num_layers", "lstm_bidirectional", "dropout", "use_temporal_attention",
             "seq_len", "img_size"]
ARCH_FINGERPRINT = _config_fingerprint(ARCH_KEYS)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(CONFIG["seed"])


def unnormalize_frame(tensor_frame):
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    img = tensor_frame.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(std * img + mean, 0, 1)


# ---- Metrics. POSITIVE CLASS = FAKE (label 1); probs = P(fake) = sigmoid(logit). ----
def safe_roc_auc(labels, scores):
    """ROC-AUC from CONTINUOUS scores. NaN (not 0.0) when undefined, so an undefined AUC can never be mistaken for AUC=0."""
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.min() == labels.max() or not np.all(np.isfinite(scores)):
        return float("nan")
    return float(roc_auc_score(labels, scores))


def compute_metrics(labels, probs, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs, dtype=np.float64)
    preds = (probs >= threshold).astype(int)
    try:
        pr_auc = float(average_precision_score(labels, probs))
    except ValueError:
        pr_auc = float("nan")
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = (int(v) for v in cm.ravel())
    sens = tp / (tp + fn) if (tp + fn) else float("nan")      # recall of FAKE = true-positive rate
    spec = tn / (tn + fp) if (tn + fp) else float("nan")      # recall of REAL = true-negative rate
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "precision_fake": float(precision_score(labels, preds, pos_label=1, zero_division=0)),
        "recall_fake": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
        "f1_fake": float(f1_score(labels, preds, pos_label=1, zero_division=0)),
        "precision_real": float(precision_score(labels, preds, pos_label=0, zero_division=0)),
        "recall_real": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "f1_real": float(f1_score(labels, preds, pos_label=0, zero_division=0)),
        "roc_auc": safe_roc_auc(labels, probs),
        "pr_auc": pr_auc,
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)),
        "sensitivity": sens, "specificity": spec,
        "fpr": (1.0 - spec) if spec == spec else float("nan"),
        "fnr": (1.0 - sens) if sens == sens else float("nan"),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def print_metrics(tag, m):
    print(f"[{tag}] thr={m['threshold']:.3f}  Acc={m['accuracy']:.4f}  F1(macro)={m['f1']:.4f}  "
          f"ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}  BalAcc={m['balanced_accuracy']:.4f}  "
          f"| TP={m['tp']} TN={m['tn']} FP={m['fp']} FN={m['fn']}")


# ---- Statistics helpers ----
def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval for a binomial proportion k/n."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (centre - half, centre + half)


def bootstrap_ci(labels, probs, threshold, n_boot=None, seed=0):
    """Percentile bootstrap (over videos) for accuracy and ROC-AUC. Returns dict of (low, high)."""
    n_boot = n_boot or CONFIG["bootstrap_n"]
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs)
    rng = np.random.default_rng(seed)
    n = len(labels)
    accs, aucs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        l, p = labels[idx], probs[idx]
        accs.append(float(np.mean((p >= threshold).astype(int) == l)))
        if l.min() != l.max():
            aucs.append(roc_auc_score(l, p))
    q = lambda a: (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))) if len(a) else (float("nan"),) * 2
    return {"acc": q(accs), "auc": q(aucs)}


def paired_bootstrap_auc_delta(labels, scores_a, scores_b, n_boot=None, seed=0):
    """Paired bootstrap of AUC(B) - AUC(A) over the SAME videos. Returns (delta, lo, hi)."""
    n_boot = n_boot or CONFIG["bootstrap_n"]
    labels = np.asarray(labels).astype(int)
    a, b = np.asarray(scores_a), np.asarray(scores_b)
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(labels), len(labels))
        if labels[idx].min() == labels[idx].max():
            continue
        deltas.append(roc_auc_score(labels[idx], b[idx]) - roc_auc_score(labels[idx], a[idx]))
    d0 = safe_roc_auc(labels, b) - safe_roc_auc(labels, a)
    if not deltas:
        return d0, float("nan"), float("nan")
    return d0, float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))


def mcnemar_exact(correct_a, correct_b):
    """Exact McNemar test on paired per-video correctness. Returns (b, c, p): b = A right & B wrong, c = A wrong & B right."""
    a = np.asarray(correct_a).astype(bool)
    b_ = np.asarray(correct_b).astype(bool)
    b = int((a & ~b_).sum())
    c = int((~a & b_).sum())
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    if _binomtest is not None:
        return b, c, float(_binomtest(k, n, 0.5).pvalue)
    p = 2.0 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return b, c, float(min(1.0, p))


def mean_std_str(values, fmt="{:.4f}"):
    """mean +/- std over seeds. A std from a single run is undefined and is shown as 'n/a (n=1)', never as NaN."""
    v = np.asarray([x for x in values if x == x], dtype=float)
    if len(v) == 0:
        return "n/a"
    if len(v) == 1:
        return fmt.format(v[0]) + " (n=1, std n/a)"
    return fmt.format(v.mean()) + " +/- " + fmt.format(v.std(ddof=1)) + f" (n={len(v)})"


def expected_calibration_error(labels, probs, n_bins=10):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (probs >= lo) & (probs < hi if hi < 1.0 else probs <= hi)
        if m.sum() == 0:
            continue
        ece += m.mean() * abs(probs[m].mean() - labels[m].mean())
    return float(ece)


def fit_temperature(logits, labels):
    """Temperature scaling: one scalar T>0 minimising validation NLL of sigmoid(logit / T)."""
    z = torch.tensor(np.asarray(logits), dtype=torch.float32)
    y = torch.tensor(np.asarray(labels), dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = Fn.binary_cross_entropy_with_logits(z / torch.exp(log_t), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05, 20.0).item())


def sigmoid_np(z, T=1.0):
    return 1.0 / (1.0 + np.exp(-np.asarray(z, dtype=np.float64) / T))


RUNTIME_LOG = {}      # name -> {"seconds": ..., "peak_gpu_mb": ...}; printed in Cell 18


class timed:
    """Context manager recording wall-clock and peak GPU memory of a section into RUNTIME_LOG."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        peak = torch.cuda.max_memory_allocated() / 2 ** 20 if torch.cuda.is_available() else float("nan")
        prev = RUNTIME_LOG.get(self.name, {"seconds": 0.0, "peak_gpu_mb": 0.0})
        RUNTIME_LOG[self.name] = {"seconds": prev["seconds"] + time.time() - self.t0,
                                  "peak_gpu_mb": max(prev["peak_gpu_mb"], peak if peak == peak else 0.0)}
        return False


print("=" * 70)
print("CONFIGURATION SUMMARY  (v9: FGSM-only adversarial training; no multi-step attack exists in this notebook)")
print("=" * 70)
print(f"Device                    : {CONFIG['device']}   (AMP for training only: {USE_AMP})")
print(f"Training seed / split seed: {CONFIG['seed']} / {CONFIG['split_seed']}   extra seeds (time-guarded): {CONFIG['extra_seeds']}")
print(f"seq_len / img_size / batch: {CONFIG['seq_len']} / {CONFIG['img_size']} / {CONFIG['batch_size']}")
print(f"Clean control             : {CONFIG['phase2_epochs']} epochs, LR {CONFIG['backbone_lr']}/{CONFIG['head_lr']}, EMA {CONFIG['ema_decay']}")
print(f"FGSM-AT (v9 recipe)       : up to {CONFIG['fgsm_epochs']} epochs x {CONFIG['fgsm_steps_per_epoch']} batches of {CONFIG['fgsm_batch_size']}; adv ratio {CONFIG['adv_ratio_start']}->{CONFIG['adv_ratio']} "
      f"over {CONFIG['adv_ratio_ramp_epochs']} ep, eps ramp {CONFIG['fgsm_eps_ramp_epochs']} ep, loss weights clean/adv/ALP = {CONFIG['fgsm_clean_loss_weight']}/{CONFIG['fgsm_adv_loss_weight']}/{CONFIG['alp_weight']}, "
      f"BN frozen={CONFIG['fgsm_bn_eval']}, clip={CONFIG['fgsm_grad_clip_mode']}")
print(f"Pilot variants            : " + "; ".join(v[0] for v in CONFIG['pilot_variants']) + "  (training eps chosen by validation)")
print(f"Evaluation epsilons (/255): {CONFIG['eval_eps_grid']}   target eps for selection: {CONFIG['target_eval_eps']}/255")
print(f"Threshold                 : {CONFIG['threshold_method']} on validation (calibrated); metrics at 0.5, at it, and at an ATTACK-AWARE validation threshold")
print(f"Outputs                   : {OUT_DIR}")
print("=" * 70)
print("NOTE on reproducibility: cuDNN's fused LSTM backward is not bit-exact deterministic even with cudnn.deterministic=True.")

In [ ]:
# ============================================================
# CELL 1b (v2): Re-use the artifacts of the v1 run (checkpoints, histories, face crops) -- nothing is retrained that is already valid
# ============================================================
# v2 does NOT change data selection, the split, preprocessing, the model or the training recipe of the main seeds. It therefore re-uses the v1 outputs:
#   * face crops          -> CONFIG['face_cache_input_dir'] (provenance is checked by Cell 2: margin / crop size / pool size)
#   * clean checkpoints   -> best_clean_model.pth, seed{1,2}_clean.pth      (provenance: split + architecture fingerprint, checked again when loaded)
#   * FGSM checkpoints    -> best_fgsm_model.pth, seed{1,2}_fgsm.pth (+ soft-best and resume files, which hold the per-epoch validation histories)
# Attach the OUTPUT of the v1 notebook as an input dataset of this notebook (Add data -> Notebook output files). Nothing is overwritten that already
# exists in the working directory, and every copy is logged to v1_reuse_log_v2.json. If nothing is found, v2 simply trains from scratch (slow).
import shutil, glob

V1_FILES = ["best_clean_model.pth", "best_clean_model_history.json", "best_fgsm_model.pth", "soft_best_fgsm_model.pth", "fgsm_training_history.json",
            "fgsm_pilot.csv", "fgsm_v9_multiseed.csv", "fgsm_v9_metrics.csv", "run_config.json", "_video_fingerprints.json"] + \
           [f"seed{sd}_{k}.pth" for sd in (1, 2) for k in ("clean", "fgsm", "soft", "resume_fgsm", "resume_clean")] + ["resume_fgsm.pth"]


def find_v1_output_dir():
    if CONFIG.get("v1_output_dir"):
        return CONFIG["v1_output_dir"] if os.path.isdir(CONFIG["v1_output_dir"]) else None
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return None
    for root, dirs, files in os.walk(base):
        if root[len(base):].count(os.sep) >= 4:
            dirs[:] = []
        if "best_fgsm_model.pth" in files and "best_clean_model.pth" in files:
            return root
    return None


V1_DIR = find_v1_output_dir()
V1_REUSE_LOG = {"v1_dir": V1_DIR, "copied": [], "already_present": [], "missing": [], "face_frames": None}
if not CONFIG.get("reuse_v1_artifacts", True):
    print("reuse_v1_artifacts=False: nothing is copied; every checkpoint is trained from scratch.")
elif V1_DIR is None:
    print("No v1 output directory found under /kaggle/input (set CONFIG['v1_output_dir']). v2 will train every model from scratch (several hours).")
else:
    print(f"v1 output directory: {V1_DIR}")
    for fn in V1_FILES:
        src, dst = os.path.join(V1_DIR, fn), os.path.join(OUT_DIR, fn)
        if os.path.exists(dst):
            V1_REUSE_LOG["already_present"].append(fn)
        elif os.path.exists(src):
            shutil.copy(src, dst)
            V1_REUSE_LOG["copied"].append(fn)
        else:
            V1_REUSE_LOG["missing"].append(fn)
    fr = os.path.join(V1_DIR, "face_frames")
    if os.path.isdir(fr) and not CONFIG.get("face_cache_input_dir"):
        CONFIG["face_cache_input_dir"] = fr                    # Cell 2 checks the cache manifest before reusing any crop
        V1_REUSE_LOG["face_frames"] = fr
    print(f"  copied {len(V1_REUSE_LOG['copied'])} files, {len(V1_REUSE_LOG['already_present'])} already present, {len(V1_REUSE_LOG['missing'])} absent in v1 output; "
          f"face crops: {V1_REUSE_LOG['face_frames'] or CONFIG.get('face_cache_input_dir') or 'not found'}")
    if V1_REUSE_LOG["missing"]:
        print("  absent (will be recomputed or skipped):", V1_REUSE_LOG["missing"])
atomic_json_save(V1_REUSE_LOG, os.path.join(OUT_DIR, "v1_reuse_log_v2.json"))


def v2_reuse_fgsm_main():
    """Re-use the v1 main-seed FGSM checkpoint instead of retraining. Returns (history, best) like run_fit, or None when it cannot be verified.
    Verified: file + history exist, provenance (kind/split/architecture), training epsilon and random-eps flag equal the CURRENT recipe, and the
    gate had passed (a soft-best 'gate never passed' checkpoint is never silently re-used)."""
    if not CONFIG.get("reuse_v1_artifacts", True):
        return None
    ck_path, hist_path = CONFIG["save_fgsm_checkpoint"], os.path.join(OUT_DIR, "fgsm_training_history.json")
    if not (os.path.exists(ck_path) and os.path.exists(hist_path)):
        return None
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    ok, why = check_provenance(ck, "fgsm")
    meta = ck.get("meta", {})
    if not ok:
        print(f"FGSM checkpoint NOT re-used: {why}")
        return None
    if meta.get("gate_never_passed") or ck.get("epoch") is None:
        print("FGSM checkpoint NOT re-used: it is a soft-best (gate never passed) file.")
        return None
    if abs(float(meta.get("train_eps", -1)) - float(FGSM_EPS)) > 1e-9 or bool(meta.get("random_eps")) != bool(CONFIG["adv_random_eps"]):
        print(f"FGSM checkpoint NOT re-used: trained with eps {meta.get('train_eps')} random={meta.get('random_eps')}, current recipe eps {FGSM_EPS} random={CONFIG['adv_random_eps']}")
        return None
    history = json.load(open(hist_path))
    best = {"score": float(ck["val_score"]), "epoch": int(ck["epoch"]), "soft_raw": float("nan"), "soft_epoch": None, "reused_from_v1": True,
            "stopped_early_bad_gate": None}
    print(f"FGSM-AT main run SKIPPED: verified v1 checkpoint re-used (epoch {best['epoch']} of {len(history)} run, selection score {best['score']:.4f}).")
    return history, best


def v2_reuse_seed(sd):
    """Verified re-use of a v1 extra-seed pair (clean + FGSM). Returns dict(history, passed, best_epoch) or None."""
    if not CONFIG.get("reuse_v1_artifacts", True):
        return None
    p = {k: os.path.join(OUT_DIR, f"seed{sd}_{k}.pth") for k in ("clean", "fgsm", "soft", "resume_fgsm")}
    if not (os.path.exists(p["clean"]) and os.path.exists(p["resume_fgsm"])):
        return None
    cc = torch.load(p["clean"], map_location="cpu", weights_only=False)
    ok_c, why_c = check_provenance(cc, "clean")
    rf = torch.load(p["resume_fgsm"], map_location="cpu", weights_only=False)
    ok_r, why_r = check_provenance(rf, "fgsm_resume")
    if not (ok_c and ok_r):
        print(f"seed {sd}: v1 artifacts NOT re-used ({why_c if not ok_c else why_r}) -> retraining.")
        return None
    hist, best = rf["history"], rf["best"]
    passed = best.get("epoch") is not None
    use = p["fgsm"] if passed else p["soft"]
    if not os.path.exists(use):
        return None
    fk = torch.load(use, map_location="cpu", weights_only=False)
    if not check_provenance(fk, "fgsm")[0] or int(fk.get("meta", {}).get("seed", -1)) != int(sd):
        print(f"seed {sd}: FGSM checkpoint provenance/seed mismatch -> retraining.")
        return None
    return {"history": hist, "passed": bool(passed), "best_epoch": best.get("epoch"), "soft_epoch": best.get("soft_epoch"), "fgsm_path": use, "clean_path": p["clean"],
            "stopped_early": best.get("stopped_early_bad_gate")}


In [ ]:
# ============================================================
# CELL 2 (v10): Data -- discovery, dedupe, identity-disjoint split, stratified selection, face crops, dataset, loaders
# ============================================================
# Review item 3: v4 grouped videos by the FIRST id of the folder name only. Fake videos are named
# TARGET_SOURCE, so the SOURCE person could sit in train while the same person appeared in a test
# video. Here every identity (both ids) is assigned to exactly ONE split and a video is kept only
# if all of its ids live in that split. (A connected-components grouping is NOT used: in FF++ the
# target/source pairs chain almost all identities into one giant component, which cannot be split.)
SPLITS = ("train", "val", "test")
import re
from collections import Counter, defaultdict

# ======================================================================================================================
# v10 DATA PIPELINE (replaces the v9 discovery / truncation / split code; the v9 code is kept verbatim below as `legacy_*`
# and is selectable with CONFIG["selection_mode"] = "legacy_v9" so that v9 checkpoints / face caches stay reproducible).
#
# Defects of the v9 pipeline that this block fixes (each one verified in the v9 outputs, see AUDIT_REPORT_v10.md):
#   D-1  `sorted(paths)[:half_budget]` BEFORE the split kept the first N files in alphabetical order: FF++ fakes -> Deepfakes (1000) +
#        Face2Face (800) only, so FaceShifter / FaceSwap / NeuralTextures were never used; Celeb-DF fakes -> only the first targets
#        in lexicographic order.
#   D-2  `ids = _video_ids(p)[0]; tuple(f"..{t}" for t in ids)` iterated over the CHARACTERS of the id string ("id0" -> i, d, 0) for
#        Celeb-real and for both UADFV classes, so the "identities" were characters that occur in almost every video. 2990 of
#        5588 videos were dropped as "straddling" and UADFV vanished from val/test.
#   D-3  No duplicate detection; no per-stage accounting; cache folder names `parent__stem` were not dataset-qualified.
# ======================================================================================================================

EXPECTED_METHODS = {"ffpp": ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"],
                    "celeb_df": ["Celeb-synthesis"], "uadfv": ["fake"]}
PARSE_WARNINGS = Counter()


def _stem_tokens(path):
    return tuple(t for t in Path(path).stem.split("_") if t)


# kept for backward compatibility with any code that still calls the v9 helper
_video_ids = _stem_tokens


def _list_mp4(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    return sorted(set(folder.glob("*.mp4")) | set(folder.glob("*/*.mp4")))


def _mkvid(path, label, method, ids, dataset):
    return {"path": str(path), "label": int(label), "method": method, "ids": tuple(ids), "dataset": dataset}


# ---------------- parsers (identity metadata = the strongest reliable information in the file names) ----------------
# FF++      real  NNN.mp4          -> {NNN}                 fake  TARGET_SOURCE.mp4 -> {TARGET, SOURCE}   (both persons/source videos)
# Celeb-DF  real  idK_NNNN.mp4     -> {idK}                 fake  idT_idS_NNNN.mp4   -> {idT, idS}
#           YouTube-real NNNNN.mp4 -> NO identity metadata: each video is its own group (an identity is NOT invented)
# UADFV     real  NNNN.mp4         -> {NNNN}                fake  NNNN_fake.mp4      -> {NNNN}  (same source video number; assumption
#                                                           from the naming convention, errs on the safe, stricter side)
def parse_ff_c23_v10(root_path):
    root = Path(root_path)
    out = []
    for rf in ["original", "Original", "real"]:
        for p in _list_mp4(root / rf):
            out.append(_mkvid(p, 0, "real", [f"ffpp_{t}" for t in _stem_tokens(p)], "ffpp"))
    for ff in EXPECTED_METHODS["ffpp"]:
        for p in _list_mp4(root / ff):
            out.append(_mkvid(p, 1, ff, [f"ffpp_{t}" for t in _stem_tokens(p)], "ffpp"))
    return out


def parse_celeb_df_v10(root_path):
    root = Path(root_path)
    out = []
    for p in _list_mp4(root / "Celeb-real"):
        t = _stem_tokens(p)
        if t and re.fullmatch(r"id\d+", t[0]):
            ids = [f"celebdf_{t[0]}"]
        else:
            PARSE_WARNINGS["celebdf_real_name_not_idK_NNNN"] += 1
            ids = [f"celebdf_unparsed_{p.stem}"]
        out.append(_mkvid(p, 0, "real", ids, "celeb_df"))
    for p in _list_mp4(root / "YouTube-real"):
        out.append(_mkvid(p, 0, "real", [f"celebdf_yt_{p.stem}"], "celeb_df"))
    for p in _list_mp4(root / "Celeb-synthesis"):
        t = _stem_tokens(p)
        if len(t) >= 2 and re.fullmatch(r"id\d+", t[0]) and re.fullmatch(r"id\d+", t[1]):
            ids = [f"celebdf_{t[0]}", f"celebdf_{t[1]}"]
        else:
            PARSE_WARNINGS["celebdf_fake_name_not_idT_idS_NNNN"] += 1
            ids = [f"celebdf_unparsed_{p.stem}"]
        out.append(_mkvid(p, 1, "Celeb-synthesis", ids, "celeb_df"))
    return out


def parse_uadfv_v10(root_path):
    root = Path(root_path)
    out = []
    for p in _list_mp4(root / "real"):
        t = _stem_tokens(p)
        out.append(_mkvid(p, 0, "real", [f"uadfv_{t[0] if t else p.stem}"], "uadfv"))
    for p in _list_mp4(root / "fake"):
        t = _stem_tokens(p)
        out.append(_mkvid(p, 1, "fake", [f"uadfv_{t[0] if t else p.stem}"], "uadfv"))
    return out


PARSER_REGISTRY_V10 = {"parse_ff_c23": parse_ff_c23_v10, "parse_celeb_df": parse_celeb_df_v10, "parse_uadfv": parse_uadfv_v10}


def discover_all_v10(config):
    """List EVERY candidate video of every active dataset (no truncation, no decoding). Returns (videos, discovery_report)."""
    active = {k: v for k, v in config.get("datasets", {}).items() if v.get("weight", 0) > 0}
    if not active:
        raise ValueError("No active datasets selected in CONFIG['datasets']!")
    videos, report = [], []
    seen = set()
    for name, info in active.items():
        root = info["root"]
        exists = os.path.isdir(root)
        by = Counter()
        if exists:
            for v in PARSER_REGISTRY_V10[info["parser"]](root):
                key = os.path.realpath(v["path"])
                if key in seen:                       # same file reached twice (e.g. case-insensitive folders) -> count once
                    PARSE_WARNINGS["same_path_listed_twice"] += 1
                    continue
                seen.add(key)
                videos.append(v)
                by[("fake" if v["label"] else "real", v["method"])] += 1
        else:
            print(f"WARNING: root for dataset '{name}' not found: {root} -> dataset SKIPPED")
        found_methods = {m for (l, m) in by if l == "fake"}
        report.append({"dataset": name, "root": root, "root_exists": bool(exists), "n_discovered": int(sum(by.values())),
                       "n_real": int(sum(c for (l, m), c in by.items() if l == "real")),
                       "n_fake": int(sum(c for (l, m), c in by.items() if l == "fake")),
                       "fake_methods_found": sorted(found_methods),
                       "expected_fake_methods_missing": sorted(set(EXPECTED_METHODS.get(name, [])) - found_methods) if exists else "n/a (root missing)",
                       "by_label_method": {f"{l}/{m}": int(c) for (l, m), c in sorted(by.items())}})
    return videos, report


# ---------------- duplicate detection (content fingerprint: size + first/last 64 KiB) ----------------
def video_fingerprint(path, nbytes=65536):
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(nbytes)
        f.seek(max(0, size - nbytes))
        tail = f.read(nbytes)
    return hashlib.md5(head + tail + str(size).encode()).hexdigest()


def dedupe_videos(videos, priority, cache_path=None):
    """Returns (kept, dropped) where dropped = [(video, reason)]. reason: 'duplicate' (same bytes, same label; the copy from the
    highest-priority dataset is kept) or 'label_conflict_duplicate' (same bytes but different labels: ALL copies are dropped)."""
    cache = {}
    if cache_path and Path(cache_path).exists():
        try:
            cache = json.load(open(cache_path))
        except Exception:
            cache = {}
    groups = defaultdict(list)
    for v in videos:
        st = os.stat(v["path"])
        c = cache.get(v["path"])
        if c and c[0] == st.st_size and c[1] == st.st_mtime_ns:
            fp = c[2]
        else:
            fp = video_fingerprint(v["path"])
            cache[v["path"]] = [st.st_size, st.st_mtime_ns, fp]
        groups[fp].append(v)
    if cache_path:
        try:
            json.dump(cache, open(cache_path, "w"))
        except Exception as e:
            print(f"(fingerprint cache not written: {e})")
    kept, dropped = [], []
    for fp, vs in groups.items():
        if len(vs) == 1:
            kept.append(vs[0])
            continue
        if len({v["label"] for v in vs}) > 1:
            dropped.extend((v, "label_conflict_duplicate") for v in vs)
            continue
        vs = sorted(vs, key=lambda v: (priority.get(v["dataset"], 99), v["path"]))
        kept.append(vs[0])
        dropped.extend((v, "duplicate") for v in vs[1:])
    kept.sort(key=lambda v: v["path"])
    return kept, dropped


# ---------------- identity-disjoint split (stratified by dataset, refined by local search) ----------------
# Grouping = the identity / source-video ids parsed above. A video is kept only if ALL its ids live in one split (so no person /
# source video is shared between splits). Connected components are NOT used (FF++ pairs chain ~all ids into one component).
# Each dataset's ids are split with the same fractions (so every dataset is present in val/test when it has enough ids); a greedy
# local search then moves single ids to the split that keeps more multi-id videos, within +-quota slack. Deterministic per seed.
def _split_objective(videos, split_of, datasets, val_frac, test_frac):
    c = defaultdict(int)
    for v in videos:
        s = {split_of[i] for i in v["ids"]}
        if len(s) == 1:
            c[(next(iter(s)), v["dataset"], v["label"])] += 1
    obj = 0.0
    for d in datasets:
        bal = {s: min(c[(s, d, 0)], c[(s, d, 1)]) for s in SPLITS}
        obj += min(bal["test"] / max(test_frac, 1e-9), bal["val"] / max(val_frac, 1e-9)) + 0.1 * bal["train"]
    return obj


def _initial_assignment(ids_by_ds, seed, val_frac, test_frac):
    split_of, quota = {}, {}
    for d, ids in ids_by_ds.items():
        ids = sorted(ids)
        random.Random(f"{seed}|{d}").shuffle(ids)
        n = len(ids)
        n_te, n_va = int(round(n * test_frac)), int(round(n * val_frac))
        if n >= 6:
            n_te, n_va = max(n_te, 1), max(n_va, 1)
        quota[d] = {"test": n_te, "val": n_va, "train": n - n_te - n_va}
        for k, t in enumerate(ids):
            split_of[t] = "test" if k < n_te else ("val" if k < n_te + n_va else "train")
    return split_of, quota


def _refine_assignment(videos, vids_of, ds_of, split_of, quota, slack, passes, rng):
    counts = {d: Counter() for d in quota}
    for t, s in split_of.items():
        counts[ds_of[t]][s] += 1
    lo = {d: {s: int(math.floor(q[s] * (1 - slack))) for s in SPLITS} for d, q in quota.items()}
    hi = {d: {s: max(int(math.ceil(q[s] * (1 + slack))), q[s] + (1 if q[s] > 0 else 0)) for s in SPLITS} for d, q in quota.items()}
    multi = [t for t, vi in vids_of.items() if any(len(videos[i]["ids"]) > 1 for i in vi)]
    for _ in range(passes):
        rng.shuffle(multi)
        moved = 0
        for t in multi:
            s0, d = split_of[t], ds_of[t]
            kept = Counter()
            for i in vids_of[t]:
                others = {split_of[o] for o in videos[i]["ids"] if o != t}
                if len(videos[i]["ids"]) > 1 and len(others) == 1:
                    kept[next(iter(others))] += 1
            best_s, best_gain = None, 0
            for s1 in SPLITS:
                if s1 == s0:
                    continue
                gain = kept[s1] - kept[s0]
                if gain > best_gain and counts[d][s1] + 1 <= hi[d][s1] and counts[d][s0] - 1 >= lo[d][s0]:
                    best_s, best_gain = s1, gain
            if best_s is not None:
                counts[d][s0] -= 1
                counts[d][best_s] += 1
                split_of[t] = best_s
                moved += 1
        if not moved:
            break
    return split_of


def identity_disjoint_split_v10(videos, split_seed, val_frac, test_frac, n_trials, passes=3, slack=0.15):
    vids_of, ds_of = defaultdict(list), {}
    for i, v in enumerate(videos):
        for t in v["ids"]:
            vids_of[t].append(i)
            ds_of[t] = v["dataset"]
    ids_by_ds = defaultdict(list)
    for t, d in ds_of.items():
        ids_by_ds[d].append(t)
    datasets = sorted(ids_by_ds)
    best = None
    random_baseline = None
    for t in range(n_trials):
        split_of, quota = _initial_assignment(ids_by_ds, f"{split_seed}|{t}", val_frac, test_frac)
        if t == 0:
            random_baseline = _count_straddle(videos, split_of)
        split_of = _refine_assignment(videos, vids_of, ds_of, split_of, quota, slack, passes, random.Random(f"{split_seed}|ref|{t}"))
        obj = _split_objective(videos, split_of, datasets, val_frac, test_frac)
        if best is None or obj > best[0]:
            best = (obj, t, dict(split_of))
    _, t_best, split_of = best
    assigned, straddle = {s: [] for s in SPLITS}, []
    for v in videos:
        s = {split_of[i] for i in v["ids"]}
        if len(s) == 1:
            assigned[next(iter(s))].append(v)
        else:
            straddle.append(v)
    info = {"best_trial": t_best, "n_trials": n_trials, "objective": best[0], "straddle_dropped": len(straddle),
            "straddle_dropped_random_assignment_trial0": random_baseline,
            "ids_per_split": {s: sum(1 for x in split_of.values() if x == s) for s in SPLITS}}
    return split_of, assigned, straddle, info


def _count_straddle(videos, split_of):
    return sum(1 for v in videos if len({split_of[i] for i in v["ids"]}) > 1)


# ---------------- reproducible stratified selection (dataset x label x manipulation method) ----------------
def waterfill(maxes, weights, total):
    """Integer allocation of `total` among keys proportional to `weights`, never above `maxes[key]`."""
    alloc = {k: 0 for k in maxes}
    active = [k for k in maxes if maxes[k] > 0 and weights.get(k, 0) > 0]
    remaining = int(total)
    while remaining > 0 and active:
        wsum = sum(weights[k] for k in active)
        shares = {k: remaining * weights[k] / wsum for k in active}
        capped = [k for k in active if maxes[k] - alloc[k] <= shares[k] + 1e-9]
        if capped:
            for k in capped:
                remaining -= maxes[k] - alloc[k]
                alloc[k] = maxes[k]
                active.remove(k)
            continue
        base = {k: int(math.floor(shares[k])) for k in active}
        for k in active:
            alloc[k] += base[k]
        remaining -= sum(base.values())
        for k in sorted(active, key=lambda k: shares[k] - base[k], reverse=True)[:remaining]:
            alloc[k] += 1
        remaining = 0
    return alloc


def stratified_select_v10(assigned, config, seed):
    """assigned[split] = eligible videos. Per split and per dataset: real count n_d = min(#real, #fake/ratio) so each dataset is class-
    balanced (no dataset<->label shortcut); fakes are drawn ROUND-ROBIN over the manipulation methods from randomly shuffled lists (seeded);
    dataset shares follow CONFIG weights (water-filling: a small dataset keeps everything it has, the surplus goes to the others);
    per-split budget = split_budget_fractions x max_videos_to_use (None -> only the per-class caps)."""
    weights = {k: v["weight"] for k, v in config["datasets"].items() if v.get("weight", 0) > 0}
    budget = config.get("max_videos_to_use")
    fr = config["split_budget_fractions"]
    selected = {}
    for s in SPLITS:
        ratio = config["train_fake_to_real_ratio"] if s == "train" else 1.0
        cap = config["train_per_class_cap"] if s == "train" else config["eval_per_class_cap"]
        if budget:
            cap = min(cap, int(budget * fr[s] / (1 + ratio)))
        pools = defaultdict(lambda: {"real": [], "fake": defaultdict(list)})
        for v in assigned[s]:
            (pools[v["dataset"]]["real"] if v["label"] == 0 else pools[v["dataset"]]["fake"][v["method"]]).append(v)
        maxes = {d: min(len(p["real"]), int(sum(len(x) for x in p["fake"].values()) / ratio)) for d, p in pools.items()}
        alloc = waterfill(maxes, weights, cap)
        chosen = []
        for d in sorted(pools):
            n = alloc.get(d, 0)
            if n <= 0:
                continue
            rng = random.Random(f"{seed}|sel|{s}|{d}")
            reals = sorted(pools[d]["real"], key=lambda v: v["path"])
            rng.shuffle(reals)
            chosen.extend(reals[:n])
            methods = sorted(pools[d]["fake"])
            fk = {m: sorted(pools[d]["fake"][m], key=lambda v: v["path"]) for m in methods}
            for m in methods:
                rng.shuffle(fk[m])
            n_fake, picked = int(round(n * ratio)), []
            while len(picked) < n_fake:
                progressed = False
                for m in methods:
                    if fk[m] and len(picked) < n_fake:
                        picked.append(fk[m].pop())
                        progressed = True
                if not progressed:
                    break
            chosen.extend(picked)
        selected[s] = chosen
    return selected


# ---------------- v9 code, UNCHANGED (selectable via CONFIG['selection_mode'] = 'legacy_v9') ----------------
def legacy_parse_ff_c23(root_path):
    root = Path(root_path)
    real_folders = ["original", "Original", "real"]
    fake_folders = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
    videos = []
    for rf in real_folders:
        r_path = root / rf
        if r_path.exists():
            for p in sorted(list(r_path.glob("*.mp4")) + list(r_path.glob("*/*.mp4"))):
                videos.append({"path": str(p), "label": 0, "method": "real", "ids": tuple(f"ffpp_{t}" for t in _video_ids(p))})
    for ff in fake_folders:
        f_path = root / ff
        if f_path.exists():
            for p in sorted(list(f_path.glob("*.mp4")) + list(f_path.glob("*/*.mp4"))):
                videos.append({"path": str(p), "label": 1, "method": ff, "ids": tuple(f"ffpp_{t}" for t in _video_ids(p))})
    return videos


def legacy_parse_celeb_df(root_path):
    root = Path(root_path)
    real_vids = list((root / "Celeb-real").glob("*.mp4")) + list((root / "YouTube-real").glob("*.mp4"))
    fake_vids = list((root / "Celeb-synthesis").glob("*.mp4"))
    videos = []
    for p in real_vids:
        ids = _video_ids(p)[0]                       # BUG kept for reproducibility: a str -> iterated character by character
        videos.append({"path": str(p), "label": 0, "method": "real", "ids": tuple(f"celebdf_{t}" for t in ids)})
    for p in fake_vids:
        ids = _video_ids(p)[:2]
        videos.append({"path": str(p), "label": 1, "method": "Celeb-synthesis", "ids": tuple(f"celebdf_{t}" for t in ids)})
    return videos


def legacy_parse_uadfv(root_path):
    root = Path(root_path)
    real_vids = list((root / "real").glob("*.mp4"))
    fake_vids = list((root / "fake").glob("*.mp4"))
    videos = []
    for p in real_vids:
        ids = _video_ids(p)[0]                       # BUG kept for reproducibility (character-level ids)
        videos.append({"path": str(p), "label": 0, "method": "real", "ids": tuple(f"uadfv_{t}" for t in ids)})
    for p in fake_vids:
        ids = _video_ids(p)[0]
        videos.append({"path": str(p), "label": 1, "method": "fake", "ids": tuple(f"uadfv_{t}" for t in ids)})
    return videos


LEGACY_PARSER_REGISTRY = {"parse_ff_c23": legacy_parse_ff_c23, "parse_celeb_df": legacy_parse_celeb_df, "parse_uadfv": legacy_parse_uadfv}


def legacy_gather_combined_videos(config):
    dataset_configs = config.get("datasets", {})
    total_max = config.get("legacy_max_videos_to_use", 8000)
    active_configs = {k: v for k, v in dataset_configs.items() if v.get("weight", 0) > 0}
    total_weight = sum(v["weight"] for v in active_configs.values())
    collected = []
    for ds_name, ds_info in active_configs.items():
        root = ds_info["root"]
        if not os.path.exists(root):
            print(f"Warning: Root directory for {ds_name} ('{root}') not found. Skipping...")
            continue
        all_ds = LEGACY_PARSER_REGISTRY[ds_info["parser"]](root)
        reals = [v for v in all_ds if v["label"] == 0]
        fakes = [v for v in all_ds if v["label"] == 1]
        if total_max is not None:
            half = int(total_max * ds_info["weight"] / total_weight) // 2
            reals = sorted(reals, key=lambda x: x["path"])[:half]
            fakes = sorted(fakes, key=lambda x: x["path"])[:half]
        else:
            reals, fakes = sorted(reals, key=lambda x: x["path"]), sorted(fakes, key=lambda x: x["path"])
        for v in reals + fakes:
            v["dataset"] = ds_name
        print(f"[{ds_name}] LEGACY selection: {len(reals) + len(fakes)} videos ({len(reals)} real, {len(fakes)} fake)")
        collected.extend(reals + fakes)
    return collected


def legacy_assign_identities(all_ids, seed, val_frac, test_frac):
    ids = sorted(all_ids)
    random.Random(seed).shuffle(ids)
    n = len(ids)
    n_test, n_val = int(round(n * test_frac)), int(round(n * val_frac))
    return {ident: ("test" if i < n_test else ("val" if i < n_test + n_val else "train")) for i, ident in enumerate(ids)}


def legacy_identity_disjoint_split(videos, split_seed, val_frac, test_frac, n_trials):
    all_ids = {i for v in videos for i in v["ids"]}
    best = None
    for t in range(n_trials):
        id2split = legacy_assign_identities(all_ids, split_seed + t, val_frac, test_frac)
        counts = {s: {"real": 0, "fake": 0} for s in SPLITS}
        for v in videos:
            ss = {id2split[i] for i in v["ids"]}
            if len(ss) == 1:
                counts[next(iter(ss))]["real" if v["label"] == 0 else "fake"] += 1
        bal = {s: min(counts[s]["real"], counts[s]["fake"]) for s in SPLITS}
        objective = min(bal["test"] / max(test_frac, 1e-9), bal["val"] / max(val_frac, 1e-9)) + 0.1 * bal["train"]
        if best is None or objective > best[0]:
            best = (objective, t, id2split, counts)
    _, t_best, id2split, counts = best
    pools = {s: {"real": [], "fake": {}} for s in SPLITS}
    n_dropped = 0
    for v in videos:
        ss = {id2split[i] for i in v["ids"]}
        if len(ss) != 1:
            n_dropped += 1
            continue
        s = next(iter(ss))
        if v["label"] == 0:
            pools[s]["real"].append(v)
        else:
            pools[s]["fake"].setdefault(v["method"], []).append(v)
    return id2split, pools, counts, n_dropped, t_best


def legacy_balanced_sample(real_pool, fake_pools, n_real_cap, ratio, rng):
    reals = list(real_pool)
    rng.shuffle(reals)
    methods = sorted(fake_pools)
    fakes = {m: rng.sample(fake_pools[m], len(fake_pools[m])) for m in methods}
    n_fake_avail = sum(len(x) for x in fakes.values())
    n_real = min(len(reals), n_real_cap, int(n_fake_avail / ratio))
    n_fake = int(round(n_real * ratio))
    chosen_fake = []
    while len(chosen_fake) < n_fake:
        progressed = False
        for m in methods:
            if fakes[m] and len(chosen_fake) < n_fake:
                chosen_fake.append(fakes[m].pop())
                progressed = True
        if not progressed:
            break
    return reals[:n_real], chosen_fake


# ======================================================================================================================
# v10 MAIN FLOW: discover -> dedupe -> identity-disjoint stratified split -> stratified random selection, with a per-video FATE
# registry (exclusive outcome per video => no double counting in the audit table).
# ======================================================================================================================
VIDEO_FATE = {}      # path -> {"dataset","label","method","ids","fate","split"}; fates: see FATE_ORDER
FATE_ORDER = ["duplicate", "label_conflict_duplicate", "identity_straddle", "not_selected_budget_balance", "undecodable",
              "insufficient_frames", "trimmed_after_failures", "included"]


def _set_fate(v, fate, split=None):
    r = VIDEO_FATE.setdefault(v["path"], {"dataset": v["dataset"], "label": v["label"], "method": v["method"], "ids": v["ids"],
                                          "fate": "discovered", "split_assigned": None, "split_selected": None})
    r["fate"] = fate
    if split is not None:
        r["split_selected" if fate in ("selected", "included", "undecodable", "insufficient_frames", "trimmed_after_failures") else "split_assigned"] = split


MODE = CONFIG.get("selection_mode", "stratified_random_v10")
DISCOVERED, DISCOVERY_REPORT = discover_all_v10(CONFIG)
if not DISCOVERED:
    raise ValueError("No videos found. Check your dataset paths and roots in CONFIG['datasets'].")
for v in DISCOVERED:
    _set_fate(v, "discovered")
print(f"Selection mode: {MODE}   (v10 = stratified random, identity-correct;  legacy_v9 = v9 behaviour, kept only for provenance)")
for r in DISCOVERY_REPORT:
    print(f"[{r['dataset']}] root exists={r['root_exists']}  discovered={r['n_discovered']} (real {r['n_real']}, fake {r['n_fake']})  "
          f"fake methods={r['fake_methods_found']}  missing expected methods={r['expected_fake_methods_missing']}")
if PARSE_WARNINGS:
    print("Parser warnings:", dict(PARSE_WARNINGS))
print(f"Total discovered across active datasets: {sum(1 for v in DISCOVERED if v['label'] == 0)} real and "
      f"{sum(1 for v in DISCOVERED if v['label'] == 1)} fake videos (Total: {len(DISCOVERED)})")

n_dup = 0
if MODE == "legacy_v9":
    all_videos = legacy_gather_combined_videos(CONFIG)
    n_all_real = sum(1 for v in all_videos if v["label"] == 0)
    print(f"LEGACY: {n_all_real} real and {len(all_videos) - n_all_real} fake videos after the v9 alphabetical truncation.")
    id2split, _legacy_pools, elig_counts, n_dropped, best_trial = legacy_identity_disjoint_split(
        all_videos, CONFIG["split_seed"], CONFIG["val_id_fraction"], CONFIG["test_id_fraction"], CONFIG["split_search_trials"])
    print(f"Identity partition (legacy): trial {best_trial}; videos dropped because their ids straddle two splits: {n_dropped}/{len(all_videos)}")
    _rng = random.Random(CONFIG["split_seed"])
    selected = {}
    for s in SPLITS:
        cap = CONFIG["train_per_class_cap"] if s == "train" else CONFIG["eval_per_class_cap"]
        ratio = CONFIG["train_fake_to_real_ratio"] if s == "train" else 1.0
        r, f = legacy_balanced_sample(_legacy_pools[s]["real"], _legacy_pools[s]["fake"], cap, ratio, _rng)
        selected[s] = r + f
        print(f"  {s:<5}: eligible real={elig_counts[s]['real']:5d} fake={elig_counts[s]['fake']:5d}  ->  selected real={len(r):4d} fake={len(f):4d}")
    _in_legacy = {v["path"] for v in all_videos}
    _sel_paths = {v["path"]: s for s in SPLITS for v in selected[s]}
    _eligible_paths = {}
    for s in SPLITS:
        for v in list(_legacy_pools[s]["real"]) + [x for m in _legacy_pools[s]["fake"].values() for x in m]:
            _eligible_paths[v["path"]] = s
    for v in DISCOVERED:
        if v["path"] in _sel_paths:
            _set_fate(v, "selected", _sel_paths[v["path"]])
            VIDEO_FATE[v["path"]]["split_assigned"] = _sel_paths[v["path"]]
        elif v["path"] in _eligible_paths:
            VIDEO_FATE[v["path"]]["split_assigned"] = _eligible_paths[v["path"]]
            _set_fate(v, "not_selected_budget_balance")
        elif v["path"] in _in_legacy:
            _set_fate(v, "identity_straddle")
        else:
            _set_fate(v, "not_selected_budget_balance")          # removed by the v9 alphabetical truncation
    SPLIT_INFO = {"mode": MODE, "straddle_dropped": n_dropped}
else:
    kept = DISCOVERED
    dropped_dups = []
    if CONFIG.get("dedupe_videos", True):
        _prio = {k: i for i, k in enumerate(CONFIG["datasets"])}
        kept, dropped_dups = dedupe_videos(DISCOVERED, _prio, os.path.join(OUT_DIR, "_video_fingerprints.json"))
        for v, why in dropped_dups:
            _set_fate(v, why)
        n_dup = len(dropped_dups)
    print(f"Duplicate videos (identical bytes) dropped: {n_dup}  ({Counter(w for _, w in dropped_dups)})")
    id2split, ASSIGNED, STRADDLE, SPLIT_INFO = identity_disjoint_split_v10(
        kept, CONFIG["split_seed"], CONFIG["val_id_fraction"], CONFIG["test_id_fraction"], CONFIG["split_search_trials"],
        CONFIG["split_refine_passes"], CONFIG["split_quota_slack"])
    SPLIT_INFO["mode"] = MODE
    for v in STRADDLE:
        _set_fate(v, "identity_straddle")
    for s in SPLITS:
        for v in ASSIGNED[s]:
            _set_fate(v, "assigned", s)
    n_dropped = len(STRADDLE)
    print(f"Identity partition: best of {SPLIT_INFO['n_trials']} trials = #{SPLIT_INFO['best_trial']}; ids train/val/test = "
          f"{SPLIT_INFO['ids_per_split']['train']} / {SPLIT_INFO['ids_per_split']['val']} / {SPLIT_INFO['ids_per_split']['test']}")
    print(f"Videos dropped because their ids straddle two splits: {n_dropped}/{len(kept)} "
          f"(random assignment without refinement would drop {SPLIT_INFO['straddle_dropped_random_assignment_trial0']} in trial 0)")
    selected = stratified_select_v10(ASSIGNED, CONFIG, CONFIG["selection_seed"])
    _sel_paths = {v["path"]: s for s in SPLITS for v in selected[s]}
    for s in SPLITS:
        for v in ASSIGNED[s]:
            if v["path"] in _sel_paths:
                _set_fate(v, "selected", s)
            else:
                _set_fate(v, "not_selected_budget_balance")
                VIDEO_FATE[v["path"]]["split_assigned"] = s
    for s in SPLITS:
        _n = lambda lab: sum(1 for v in ASSIGNED[s] if v["label"] == lab)
        print(f"  {s:<5}: eligible real={_n(0):5d} fake={_n(1):5d}  ->  selected real={sum(1 for v in selected[s] if v['label'] == 0):4d} "
              f"fake={sum(1 for v in selected[s] if v['label'] == 1):4d}   per dataset: "
              + ", ".join(f"{d}={sum(1 for v in selected[s] if v['dataset'] == d)}" for d in CONFIG['datasets']))
    elig_counts = {s: {"real": sum(1 for v in ASSIGNED[s] if v["label"] == 0), "fake": sum(1 for v in ASSIGNED[s] if v["label"] == 1)} for s in SPLITS}

n_sel_total = sum(len(selected[s]) for s in SPLITS)
print(f"Selected {n_sel_total} videos for face-crop extraction; estimated extraction time without cache: "
      f"{n_sel_total * CONFIG['extract_sec_per_video'] / 3600:.2f} h (v9 measured {CONFIG['extract_sec_per_video']} s/video on a Kaggle T4; cached crops are reused).")
for s in SPLITS:
    _fp = hashlib.md5("|".join(sorted(v["path"] for v in selected[s])).encode()).hexdigest()[:10]
    print(f"  {s} set fingerprint: {_fp}  (identical across runs with the same selection_mode / seeds / caps)")

# Hard guarantee, over ALL ids of every selected video:
ids_by_split = {s: {i for v in selected[s] for i in v["ids"]} for s in SPLITS}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = ids_by_split[a] & ids_by_split[b]
    assert not inter, f"Identity leakage between {a} and {b}: {sorted(inter)[:10]}"
print("Identity-overlap check (ALL ids of every video): PASSED -- no person / source video appears in more than one split.")


# ---------------- Face detection / cropping (review item 8) ----------------
_CASCADE_FILES = ["haarcascade_frontalface_default.xml", "haarcascade_frontalface_alt2.xml"]
_FACE_CASCADES = [cv2.CascadeClassifier(cv2.data.haarcascades + f) for f in _CASCADE_FILES]
_FACE_CASCADES = [c for c in _FACE_CASCADES if not c.empty()]
if not _FACE_CASCADES:
    raise RuntimeError("No Haar cascade could be loaded from cv2.data.haarcascades")


def _detect_face_box(frame_bgr):
    h, w = frame_bgr.shape[:2]
    scale = 640.0 / max(h, w) if max(h, w) > 640 else 1.0
    small = cv2.resize(frame_bgr, None, fx=scale, fy=scale) if scale < 1.0 else frame_bgr
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    for cascade, min_neighbors in zip(_FACE_CASCADES, (5, 4)):   # 2nd cascade only if the 1st finds nothing
        faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=min_neighbors, minSize=(40, 40))
        if len(faces) > 0:
            x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
            return x / scale, y / scale, fw / scale, fh / scale
    return None


def _video_face_square(frames_bgr, margin):
    """One stable square crop per clip (median centre, largest accepted size); outlier boxes (false positives) are rejected first.
    v10: IDENTICAL crop geometry to v9 (so cached v9 crops stay valid) plus stability / failure diagnostics.
    Returns (x0, y0, x1, y1, detected_flag, detect_rate, stats)."""
    h, w = frames_bgr[0].shape[:2]
    tried = frames_bgr[::max(1, CONFIG["face_detect_every"])]
    raw_boxes = [bx for bx in (_detect_face_box(f) for f in tried) if bx is not None]
    boxes = raw_boxes
    if len(boxes) >= 3:
        cxs = np.array([x + fw / 2 for x, y, fw, fh in boxes])
        cys = np.array([y + fh / 2 for x, y, fw, fh in boxes])
        szs = np.array([max(fw, fh) for x, y, fw, fh in boxes])
        mcx, mcy, msz = np.median(cxs), np.median(cys), np.median(szs)
        keep = (np.abs(cxs - mcx) < 0.5 * msz) & (np.abs(cys - mcy) < 0.5 * msz) & (szs > 0.5 * msz) & (szs < 2.0 * msz)
        boxes = [b for b, k in zip(boxes, keep) if k]
    detect_rate = len(boxes) / max(len(tried), 1)
    jitter = outside = float("nan")
    if boxes and detect_rate >= CONFIG["face_min_detect_rate"]:
        cx = float(np.median([x + fw / 2 for x, y, fw, fh in boxes]))
        cy = float(np.median([y + fh / 2 for x, y, fw, fh in boxes]))
        size = max(max(fw, fh) for x, y, fw, fh in boxes) * margin
        detected = True
    else:                                      # fallback: central square (flagged in the QA stats)
        cx, cy, size, detected = w / 2, h / 2, 0.7 * min(h, w), False
    size = min(size, h, w)
    x0 = int(np.clip(cx - size / 2, 0, w - size))
    y0 = int(np.clip(cy - size / 2, 0, h - size))
    x1, y1 = int(x0 + size), int(y0 + size)
    if detected and boxes:
        side = max(x1 - x0, 1)
        jitter = float(np.mean([np.hypot((x + fw / 2) - cx, (y + fh / 2) - cy) for x, y, fw, fh in boxes]) / side)   # mean centre drift / crop side
        outside = float(np.mean([not (x >= x0 and y >= y0 and x + fw <= x1 and y + fh <= y1) for x, y, fw, fh in boxes]))  # detected faces not fully inside the single static crop
    stats = {"n_tried": len(tried), "n_raw_detections": len(raw_boxes), "n_boxes_used": len(boxes), "any_detection": bool(raw_boxes),
             "n_outliers_rejected": len(raw_boxes) - len(boxes), "centre_drift_over_side": jitter, "faces_outside_static_crop_rate": outside}
    return x0, y0, x1, y1, detected, detect_rate, stats


def _read_indices(v_path, targets):
    """Decode sequentially and keep exactly the frames whose index is in `targets` (grab/retrieve: unneeded frames are not converted).
    Frames that cannot be retrieved are skipped and are NOT replaced. Returns (frames_bgr, kept_indices)."""
    cap = cv2.VideoCapture(str(v_path))
    tset, last = set(targets), max(targets)
    frames, kept, cur = [], [], 0
    while cap.isOpened():
        if not cap.grab():
            break
        if cur in tset:
            ok, fr = cap.retrieve()
            if ok and fr is not None:
                frames.append(fr)
                kept.append(cur)
        cur += 1
        if cur > last:
            break
    cap.release()
    return frames, kept


def _count_decodable(v_path, limit=200000):
    cap = cv2.VideoCapture(str(v_path))
    n = 0
    while cap.grab():
        n += 1
        if n >= limit:
            break
    cap.release()
    return n


def _plan_pool_indices(total, pool):
    if total <= 0:
        return []
    idx = np.linspace(0, total - 1, pool, dtype=int) if total >= pool else np.arange(total)
    return sorted(set(int(i) for i in idx))


def _read_pool_frames(v_path, frame_pool_size):
    """v10. Same sampling positions as v9 when the container's frame count is right: np.linspace(0, total-1, pool) (every video with >=pool
    frames) or all frames (shorter videos). If decoding stops before all planned frames were read (inflated CAP_PROP_FRAME_COUNT or a corrupt
    tail) the video is RE-COUNTED once by decoding it and the positions are re-planned on the true length. No padding happens here.
    Returns (frames_bgr, kept_frame_indices, meta)."""
    cap = cv2.VideoCapture(str(v_path))
    opened = bool(cap.isOpened())
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else 0
    cap.release()
    meta = {"opened": opened, "reported_total": total, "recounted": False, "true_total": None, "n_planned": 0}
    if not opened:
        return [], [], {**meta, "status": "undecodable"}
    targets = _plan_pool_indices(total, frame_pool_size) or [0]
    meta["n_planned"] = len(targets)
    frames, kept = _read_indices(v_path, targets)
    if len(frames) < len(targets):
        true_total = _count_decodable(v_path)
        meta.update(recounted=True, true_total=true_total)
        if true_total != total:
            t2 = _plan_pool_indices(true_total, frame_pool_size)
            if t2:
                meta["n_planned"] = len(t2)
                frames, kept = _read_indices(v_path, t2)
    return frames, kept, {**meta, "status": "ok" if frames else "undecodable"}


EXTRACT_OUTCOME = {}      # video path -> {"status": reused|new|undecodable|insufficient_frames, "n_frames": int, ...}


def _manifest_matches(dir_path):
    mf = Path(dir_path) / "_cache_manifest.json"
    if not mf.exists():
        return True, {}
    got = json.load(open(mf))
    want = {"face_margin": CONFIG["face_margin"], "face_crop_size": CONFIG["face_crop_size"], "frame_pool_size": CONFIG["frame_pool_size"]}
    bad = {k: (got.get(k), v) for k, v in want.items() if got.get(k) != v}
    return (not bad), bad


def extract_selected_to_disk(videos, output_dir, frame_pool_size):
    """Extract face crops only for the videos chosen by the identity-disjoint selection. Existing crops (this run's cache or
    CONFIG['face_cache_input_dir']) are re-used when (i) the cache manifest matches margin / crop size / pool size and (ii) enough crops exist.
    v10: dataset-qualified folder names (a v9 name `parent__stem` is still accepted for re-use); per-video outcome registry (EXTRACT_OUTCOME);
    videos with fewer than `min_usable_frames` usable frames are dropped (short_video_policy='drop') instead of being padded later;
    full crop / sampling metadata is written to _face_boxes.json (saved incrementally)."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    boxes_path = out_path / "_face_boxes.json"
    boxes = json.load(open(boxes_path)) if boxes_path.exists() else {}
    local_ok, bad = _manifest_matches(out_path)
    if not local_ok:
        print(f"!! Local face cache manifest differs from CONFIG {bad}: existing local crops will NOT be re-used (they are overwritten).")
    min_needed = CONFIG["min_usable_frames"] if CONFIG["short_video_policy"] == "drop" else min(frame_pool_size, CONFIG["seq_len"])
    folder_of = {}
    n_fallback = n_new = n_reused = 0
    print(f"Preparing face crops for {len(videos)} selected videos ({frame_pool_size} frames/video; only videos without valid cached crops are decoded)...")
    for k, v in enumerate(tqdm(videos, desc="Extracting Faces")):
        v_path = Path(v["path"])
        sub = "real" if v["label"] == 0 else "fake"
        name_new = f"{v['dataset']}__{v_path.parent.name}__{v_path.stem}"
        name_old = f"{v_path.parent.name}__{v_path.stem}"                      # v9 naming
        bases = ([Path(CONFIG["face_cache_input_dir"])] if CONFIG["face_cache_input_dir"] else []) + ([out_path] if local_ok else [])
        reused = None
        for base in bases:
            for nm in (name_new, name_old):
                c = base / sub / nm
                if c.exists() and len(list(c.glob("*.jpg"))) >= min_needed:
                    reused = (c, nm)
                    break
            if reused:
                break
        if reused is not None:
            folder_of[v["path"]] = str(reused[0])
            n_reused += 1
            EXTRACT_OUTCOME[v["path"]] = {"status": "reused", "n_frames": len(list(reused[0].glob("*.jpg"))), "v9_name": reused[1] == name_old}
            continue
        frames, kept, meta = _read_pool_frames(v_path, frame_pool_size)
        if not frames:
            EXTRACT_OUTCOME[v["path"]] = {**meta, "status": "undecodable", "n_frames": 0}
            continue
        if CONFIG["short_video_policy"] == "drop" and len(frames) < CONFIG["min_usable_frames"]:
            EXTRACT_OUTCOME[v["path"]] = {**meta, "status": "insufficient_frames", "n_frames": len(frames)}
            continue
        n_new += 1
        vid_out_dir = out_path / sub / name_new
        vid_out_dir.mkdir(parents=True, exist_ok=True)
        for old in vid_out_dir.glob("*.jpg"):
            old.unlink()
        x0, y0, x1, y1, ok, rate, stats = _video_face_square(frames, CONFIG["face_margin"])
        n_fallback += (not ok)
        cs = CONFIG["face_crop_size"]
        for j, fr in enumerate(frames):
            crop = cv2.resize(fr[y0:y1, x0:x1], (cs, cs), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(vid_out_dir / f"frame_{j:03d}.jpg"), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        boxes[name_new] = {"box": [x0, y0, x1, y1], "hw": [int(frames[0].shape[0]), int(frames[0].shape[1])], "detect_rate": float(rate),
                           "ok": bool(ok), **stats, "frame_indices": [int(i) for i in kept], "reported_total": meta["reported_total"],
                           "recounted": meta["recounted"], "true_total": meta["true_total"], "video_path": str(v_path),
                           "dataset": v["dataset"], "label": int(v["label"]), "method": v["method"], "crop_size": cs,
                           "margin": CONFIG["face_margin"], "pool": frame_pool_size, "jpeg_quality": 95, "extractor": "v10"}
        folder_of[v["path"]] = str(vid_out_dir)
        EXTRACT_OUTCOME[v["path"]] = {**meta, "status": "new", "n_frames": len(frames)}
        if (n_new % 200) == 0:
            atomic_json_save(boxes, str(boxes_path))
    atomic_json_save(boxes, str(boxes_path))
    n_bad = {s: sum(1 for o in EXTRACT_OUTCOME.values() if o["status"] == s) for s in ("undecodable", "insufficient_frames")}
    print(f"Newly extracted: {n_new}; re-used from cache: {n_reused}; undecodable: {n_bad['undecodable']}; fewer than "
          f"{CONFIG['min_usable_frames']} usable frames: {n_bad['insufficient_frames']}; centre-crop fallback for {n_fallback} of the new ones "
          f"(per-video detection rates and crop boxes are in {boxes_path}).")
    return folder_of, boxes


def check_cache_provenance(cache_dir):
    """Provenance check before re-using a face-crop cache from another session. Returns the dir if compatible, else None.
    Compatible = its manifest (if present) matches margin / crop size / pool size, and sampled crops have the expected size."""
    if not cache_dir:
        return None
    p = Path(cache_dir)
    if not p.exists():
        print(f"Face cache {cache_dir} not found -> ignored.")
        return None
    want = {"face_margin": CONFIG["face_margin"], "face_crop_size": CONFIG["face_crop_size"],
            "frame_pool_size": CONFIG["frame_pool_size"]}
    mf = p / "_cache_manifest.json"
    if mf.exists():
        got = json.load(open(mf))
        bad = {k: (got.get(k), v) for k, v in want.items() if got.get(k) != v}
        if bad:
            print(f"Face cache manifest mismatch {bad} -> cache IGNORED (crops would be re-extracted).")
            return None
    else:
        print("Face cache has no manifest; falling back to a size check on sampled crops.")
    sample = list(p.glob("*/*/*.jpg"))[:5]
    for jp in sample:
        im = cv2.imread(str(jp))
        if im is None or im.shape[0] != CONFIG["face_crop_size"] or im.shape[1] != CONFIG["face_crop_size"]:
            print(f"Sampled crop {jp.name} has wrong size/unreadable -> cache IGNORED.")
            return None
    print(f"Face cache {cache_dir} passed provenance checks ({len(sample)} sampled crops).")
    return str(p)


def discover_face_cache():
    """Kaggle: look (max 4 levels deep) under /kaggle/input for a face-crop cache written by an earlier session (_face_boxes.json)."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for dirpath, dirnames, filenames in os.walk(root):
        if "_face_boxes.json" in filenames:
            return dirpath
        if len(Path(dirpath).relative_to(root).parts) >= 4:
            dirnames[:] = []
    return None


if CONFIG["face_cache_input_dir"] is None and CONFIG.get("auto_discover_inputs"):
    _found = discover_face_cache()
    if _found:
        print(f"Auto-discovered a face-crop cache: {_found} (will be used only if it passes the provenance checks)")
        CONFIG["face_cache_input_dir"] = _found
    else:
        print("No face-crop cache found under /kaggle/input: crops will be extracted (~1.7 h on Kaggle). TIP: after this run, save "
              "/kaggle/working/face_frames as a Kaggle dataset and add it to the next session; it is picked up automatically.")
CONFIG["face_cache_input_dir"] = check_cache_provenance(CONFIG["face_cache_input_dir"])

_all_selected = [v for s in SPLITS for v in selected[s]]
folder_of, FACE_BOXES = extract_selected_to_disk(_all_selected, CONFIG["face_frames_dir"], CONFIG["frame_pool_size"])
atomic_json_save({"face_margin": CONFIG["face_margin"], "face_crop_size": CONFIG["face_crop_size"], "frame_pool_size": CONFIG["frame_pool_size"],
                  "extractor": "v10", "min_usable_frames": CONFIG["min_usable_frames"], "short_video_policy": CONFIG["short_video_policy"]},
                 str(Path(CONFIG["face_frames_dir"]) / "_cache_manifest.json"))

for _s in SPLITS:                                  # fates after extraction (exclusive: a video has exactly one)
    for _v in selected[_s]:
        _st = EXTRACT_OUTCOME.get(_v["path"], {}).get("status")
        if _st in ("undecodable", "insufficient_frames"):
            _set_fate(_v, _st, _s)


def balance_by_min(records, seed):
    real = [r for r in records if r[1] == 0]
    fake = [r for r in records if r[1] == 1]
    n = min(len(real), len(fake))
    rng = random.Random(seed)
    rng.shuffle(real)
    rng.shuffle(fake)
    out = real[:n] + fake[:n]
    rng.shuffle(out)
    return out


def balance_by_min_per_dataset(records, seed, info):
    """v10: trims (only if some videos failed to decode) WITHIN each dataset so every dataset stays class-balanced. Returns (kept, trimmed)."""
    rng = random.Random(seed)
    by_ds = defaultdict(lambda: {0: [], 1: []})
    for r in records:
        by_ds[info[r[0]]["dataset"]][r[1]].append(r)
    out, trimmed = [], []
    for d in sorted(by_ds):
        real, fake = by_ds[d][0], by_ds[d][1]
        rng.shuffle(real)
        rng.shuffle(fake)
        n = min(len(real), len(fake))
        out += real[:n] + fake[:n]
        trimmed += real[n:] + fake[n:]
    rng.shuffle(out)
    return out, trimmed


record_info = {}          # folder -> {"method", "ids", "split", "video_path", "dataset"}
split_records = {}
_path_of_folder = {}
for s in SPLITS:
    recs = []
    for v in selected[s]:
        folder = folder_of.get(v["path"])
        if folder is None:                         # undecodable / too few frames: dropped and counted in VIDEO_FATE
            continue
        recs.append((folder, v["label"]))
        record_info[folder] = {"method": v["method"], "ids": v["ids"], "split": s, "video_path": v["path"], "dataset": v["dataset"]}
        _path_of_folder[folder] = v
    if s != "train" or CONFIG["train_fake_to_real_ratio"] == 1.0:
        if MODE == "legacy_v9":
            recs = balance_by_min(recs, CONFIG["split_seed"])
        else:
            recs, _trimmed = balance_by_min_per_dataset(recs, CONFIG["split_seed"], record_info)
            for f, _l in _trimmed:
                _set_fate(_path_of_folder[f], "trimmed_after_failures", s)
                record_info.pop(f, None)
    split_records[s] = recs
for s in SPLITS:
    for f, _l in split_records[s]:
        _set_fate(_path_of_folder[f], "included", s)

train_records, val_records, test_records = split_records["train"], split_records["val"], split_records["test"]
extracted_records = train_records + val_records + test_records


def split_report(name, records):
    n_real = sum(1 for _, l in records if l == 0)
    n_fake = sum(1 for _, l in records if l == 1)
    n_ids = len({i for f, _ in records for i in record_info[f]["ids"]})
    print(f"{name:<6}: {len(records):5d} videos  |  real={n_real:5d}  fake={n_fake:5d}  |  identities={n_ids:4d}")


print("=" * 60)
print("IDENTITY-DISJOINT SPLIT REPORT")
print("=" * 60)
split_report("Train", train_records)
split_report("Val", val_records)
split_report("Test", test_records)
print("=" * 60)


# ---------------- Dataset ----------------
_VALID_MIN = torch.tensor([(0.0 - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)]).view(3, 1, 1)
_VALID_MAX = torch.tensor([(1.0 - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)]).view(3, 1, 1)


def apply_social_media_compression(img_rgb, scale_factor, should_blur, quality):
    h, w, _ = img_rgb.shape
    new_h, new_w = max(1, int(h * scale_factor)), max(1, int(w * scale_factor))
    img_resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    img_resized = cv2.resize(img_resized, (w, h), interpolation=cv2.INTER_LINEAR)

    if should_blur:
        k = CONFIG["compression_blur_kernel"]
        k = k if k % 2 == 1 else k + 1
        img_resized = cv2.GaussianBlur(img_resized, (k, k), 0)

    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_resized, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg


class PreextractedVideoDataset(Dataset):
    def __init__(self, sample_records, seq_len=16, img_size=224, is_train=True):
        self.sample_records = sample_records
        self.seq_len = seq_len
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return len(self.sample_records)

    def _sample_frame_indices(self, n_available, rng):
        if n_available <= self.seq_len:
            return list(range(n_available))

        bounds = np.linspace(0, n_available, self.seq_len + 1).astype(int)
        indices = []
        for i in range(self.seq_len):
            lo, hi = int(bounds[i]), int(bounds[i + 1])
            hi = max(hi, lo + 1)
            if self.is_train:
                indices.append(rng.randint(lo, hi - 1))
            else:
                indices.append((lo + hi - 1) // 2)
        return indices

    def planned_frame_files(self, idx, seed=0):
        """The crop files that __getitem__ would read (eval: deterministic; train: seeded draw). Used by the tests and by Grad-CAM alignment."""
        folder = Path(self.sample_records[idx][0])
        files = sorted(folder.glob("*.jpg"))
        return [files[i] for i in self._sample_frame_indices(len(files), random.Random(seed))]

    def __getitem__(self, idx):
        folder_path, label = self.sample_records[idx]
        folder = Path(folder_path)
        frame_files = sorted(list(folder.glob("*.jpg")))
        n_available = len(frame_files)
        if n_available == 0:
            raise RuntimeError(f"no face crops in {folder}")
        if n_available < self.seq_len and CONFIG["short_video_policy"] == "drop":
            raise RuntimeError(f"{folder}: {n_available} crops < seq_len={self.seq_len} (should have been filtered by the extraction step)")

        rng = random.Random() if self.is_train else random.Random(0)

        frame_indices = self._sample_frame_indices(n_available, rng)
        selected_files = [frame_files[i] for i in frame_indices]

        should_compress = self.is_train and rng.random() < CONFIG["compression_prob"]
        compress_quality = (rng.randint(*CONFIG["compression_jpeg_quality_range"])
                             if should_compress else None)
        should_blur = should_compress and rng.random() < CONFIG["compression_blur_prob"]
        scale_factor = rng.uniform(*CONFIG["compression_downscale_range"])

        if self.is_train:
            do_flip = rng.random() < CONFIG["aug_flip_prob"]
            rotation_angle = rng.uniform(*CONFIG["aug_rotation_range"])
            brightness_factor = rng.uniform(*CONFIG["aug_brightness_range"])
            contrast_factor = rng.uniform(*CONFIG["aug_contrast_range"])
            saturation_factor = rng.uniform(*CONFIG["aug_saturation_range"])
            hue_factor = rng.uniform(*CONFIG["aug_hue_range"])
            crop_padding = CONFIG["aug_crop_padding"]
            crop_top = rng.randint(0, 2 * crop_padding)
            crop_left = rng.randint(0, 2 * crop_padding)
            add_noise = rng.random() < CONFIG["aug_noise_prob"]

        processed_frames = []
        for f_path in selected_files:
            img = cv2.imread(str(f_path))
            if img is None:
                raise RuntimeError(f"unreadable face crop: {f_path}")
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            if should_compress:
                img_rgb = apply_social_media_compression(
                    img_rgb, scale_factor, should_blur, compress_quality
                )

            pil_img = transforms.ToPILImage()(img_rgb)
            pil_img = TF.resize(pil_img, (self.img_size, self.img_size))

            if self.is_train:
                pil_img = TF.rotate(pil_img, rotation_angle)
                pil_img = TF.pad(pil_img, padding=crop_padding, padding_mode='reflect')
                pil_img = TF.crop(pil_img, top=crop_top, left=crop_left,
                                   height=self.img_size, width=self.img_size)
                if do_flip:
                    pil_img = TF.hflip(pil_img)
                pil_img = TF.adjust_brightness(pil_img, brightness_factor)
                pil_img = TF.adjust_contrast(pil_img, contrast_factor)
                if hue_factor != 0.0:      # v10: with a zero hue range v9 still did an HSV round trip (uint8 rounding noise of up to ~3 grey levels, TRAIN ONLY)
                    pil_img = TF.adjust_hue(pil_img, hue_factor)
                pil_img = TF.adjust_saturation(pil_img, saturation_factor)

            tensor_frame = TF.to_tensor(pil_img)
            tensor_frame = TF.normalize(tensor_frame, mean=IMAGENET_MEAN, std=IMAGENET_STD)

            if self.is_train and add_noise:
                tensor_frame = tensor_frame + torch.randn_like(tensor_frame) * CONFIG["aug_noise_std"]
            # Review item 6: every input is a VALID image (noise is added after normalisation, so clamp)
            tensor_frame = torch.max(torch.min(tensor_frame, _VALID_MAX), _VALID_MIN)

            processed_frames.append(tensor_frame)

        while len(processed_frames) < self.seq_len:
            if len(processed_frames) > 0:
                processed_frames.append(processed_frames[-1].clone())
            else:
                processed_frames.append(torch.zeros(3, self.img_size, self.img_size))

        video_tensor = torch.stack(processed_frames[:self.seq_len])
        label_tensor = torch.tensor(label, dtype=torch.float32)
        return video_tensor, label_tensor


# ---- Datasets ----
train_dataset = PreextractedVideoDataset(train_records, seq_len=CONFIG["seq_len"],
                                          img_size=CONFIG["img_size"], is_train=True)
val_dataset = PreextractedVideoDataset(val_records, seq_len=CONFIG["seq_len"],
                                        img_size=CONFIG["img_size"], is_train=False)
test_dataset = PreextractedVideoDataset(test_records, seq_len=CONFIG["seq_len"],
                                         img_size=CONFIG["img_size"], is_train=False)


# --- DATALOADER WORKER INITIALIZATION FIX (preserved) ---
def worker_init_fn(worker_id):
    """Ensures torch._utils is loaded in worker processes before torchvision attempts to use it."""
    import torch
    import torch._utils


train_labels_arr = np.array([r[1] for r in train_records])
class_counts = np.bincount(train_labels_arr, minlength=2)
print(f"Train class counts -> real={class_counts[0]}, fake={class_counts[1]} "
      f"(ratio 1:{class_counts[1] / max(class_counts[0], 1):.2f})")

# The sampler is only needed if the train split is deliberately imbalanced (train_fake_to_real_ratio != 1).
USE_SAMPLER = bool(abs(class_counts[1] / max(class_counts[0], 1) - 1.0) > 0.05)
_sample_weights = (1.0 / np.maximum(class_counts, 1))[train_labels_arr]


def make_train_loader(batch_size):
    kw = dict(batch_size=batch_size, num_workers=CONFIG["num_workers"], worker_init_fn=worker_init_fn,
              drop_last=True, pin_memory=True)
    if USE_SAMPLER:
        sampler = WeightedRandomSampler(weights=torch.as_tensor(_sample_weights, dtype=torch.double),
                                        num_samples=len(_sample_weights), replacement=True)
        return DataLoader(train_dataset, sampler=sampler, **kw)
    return DataLoader(train_dataset, shuffle=True, **kw)


def make_eval_loader(dataset, batch_size):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=CONFIG["num_workers"],
                      worker_init_fn=worker_init_fn, pin_memory=True)


train_loader = make_train_loader(CONFIG["batch_size"])
train_loader_fgsm = make_train_loader(CONFIG["fgsm_batch_size"])
val_loader = make_eval_loader(val_dataset, CONFIG["eval_batch_size"])

# Split fingerprint: stored in every checkpoint; re-used checkpoints must match it (provenance check)
SPLIT_FINGERPRINT = hashlib.md5("|".join(
    [s + ":" + ",".join(sorted(Path(f).name + str(l) for f, l in split_records[s])) for s in SPLITS]).encode()).hexdigest()[:12]
print(f"Split fingerprint (train+val+test records): {SPLIT_FINGERPRINT}")


class GuardedLoader:
    """Wraps the TEST loader. It refuses to yield a batch until TEST_UNLOCKED['open'] is set (Cell 10), which only happens
    after acceptance_criteria.json has been written. Guarantees that no test video is read during pilot / training / selection."""
    def __init__(self, loader):
        self._loader = loader
        self.dataset = loader.dataset
        self.batch_size = loader.batch_size

    def __len__(self):
        return len(self._loader)

    def __iter__(self):
        if not TEST_UNLOCKED["open"]:
            raise RuntimeError("TEST SET ACCESS DENIED: acceptance criteria have not been frozen yet (Cell 10).")
        return iter(self._loader)


TEST_UNLOCKED = {"open": False}
test_loader = GuardedLoader(make_eval_loader(test_dataset, CONFIG["eval_batch_size"]))
test_attack_loader_bs = CONFIG["eval_attack_batch_size"]


def make_test_attack_loader():
    return GuardedLoader(make_eval_loader(test_dataset, CONFIG["eval_attack_batch_size"]))


# Fixed, balanced VALIDATION subset used for the per-epoch FGSM metrics (selection) and the pilot. Never test data.
_half = CONFIG["val_adv_n"] // 2
val_adv_records = [r for r in val_records if r[1] == 0][:_half] + [r for r in val_records if r[1] == 1][:_half]
val_attack_loader = make_eval_loader(val_dataset, CONFIG["eval_attack_batch_size"])       # FULL validation set, attack batch size
val_adv_loader = make_eval_loader(
    PreextractedVideoDataset(val_adv_records, seq_len=CONFIG["seq_len"], img_size=CONFIG["img_size"], is_train=False),
    CONFIG["eval_attack_batch_size"])

print(f"\nSetup complete! Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | "
      f"Test batches: {len(test_loader)} (LOCKED until Cell 10)")
print(f"Sampler: {'WeightedRandomSampler ENABLED (train is intentionally imbalanced)' if USE_SAMPLER else 'not needed -- train split is balanced'}")
print(f"FGSM validation subset: {len(val_adv_records)} videos")

In [ ]:
# ============================================================
# CELL 2b (v10): Dataset audit table, discard accounting, manipulation-method distribution, face-crop and frame-sampling diagnostics
# ============================================================
# One row per (dataset, class, method). Every video has exactly ONE terminal fate (VIDEO_FATE), so the discard columns never double count:
#   discovered = duplicate + label_conflict_duplicate + identity_straddle + not_selected_budget_balance + undecodable
#                + insufficient_frames + trimmed_after_failures + included          (asserted below)
# Face-crop flags (no_face_found / fallback_crop / low_detect_rate) are properties of INCLUDED videos, not discards.
from scipy.stats import fisher_exact

_lab = {0: "real", 1: "fake"}
_rows = []
for p, r in VIDEO_FATE.items():
    _rows.append({"dataset": r["dataset"], "class": _lab[r["label"]], "method": r["method"], "fate": r["fate"],
                  "split_assigned": r.get("split_assigned"), "split_final": r.get("split_selected")})
_fd = pd.DataFrame(_rows)
_fd["fate"] = _fd["fate"].replace({"assigned": "not_selected_budget_balance", "selected": "not_selected_budget_balance"})  # defensive: should not occur
_key = ["dataset", "class", "method"]
AUDIT = _fd.groupby(_key).size().rename("discovered").to_frame()
for f in FATE_ORDER:
    AUDIT[f] = _fd[_fd.fate == f].groupby(_key).size()
AUDIT = AUDIT.fillna(0).astype(int)
AUDIT["eligible_after_filtering"] = AUDIT["discovered"] - AUDIT["duplicate"] - AUDIT["label_conflict_duplicate"] - AUDIT["identity_straddle"]
_elig = _fd[~_fd.fate.isin(["duplicate", "label_conflict_duplicate", "identity_straddle"])]
for s in SPLITS:
    AUDIT[f"assigned_{s}"] = _elig[_elig.split_assigned == s].groupby(_key).size()
_sel = _fd[~_fd.fate.isin(["duplicate", "label_conflict_duplicate", "identity_straddle", "not_selected_budget_balance"])]
for s in SPLITS:
    AUDIT[f"selected_{s}"] = _sel[_sel.split_final == s].groupby(_key).size()
    AUDIT[f"included_{s}"] = _fd[(_fd.fate == "included") & (_fd.split_final == s)].groupby(_key).size()
AUDIT = AUDIT.fillna(0).astype(int).rename(columns={"included": "included_total"})
assert (AUDIT[[c for c in FATE_ORDER if c != "included"] + ["included_total"]].sum(axis=1) == AUDIT["discovered"]).all(), \
    "audit conservation failed: some video is counted twice or not at all"
assert AUDIT["included_total"].sum() == len(extracted_records), "audit 'included' != number of records used for training/evaluation"

# face-crop flags on the INCLUDED videos
_fl = []
for f, l in extracted_records:
    info = record_info[f]
    b = FACE_BOXES.get(Path(f).name) or {}
    _fl.append({"dataset": info["dataset"], "class": _lab[l], "method": info["method"], "has_meta": "detect_rate" in b,
                "no_face_found": ("any_detection" in b) and (not b["any_detection"]) or (("any_detection" not in b) and b.get("detect_rate", 1) == 0),
                "fallback_crop": ("ok" in b) and (not b["ok"]),
                "low_detect_rate": ("detect_rate" in b) and b["detect_rate"] < 0.5 and b.get("ok", True),
                "detect_rate": b.get("detect_rate", np.nan), "drift": b.get("centre_drift_over_side", np.nan),
                "outside": b.get("faces_outside_static_crop_rate", np.nan), "recounted": b.get("recounted", False),
                "n_frames": len(list(Path(f).glob('*.jpg')))})
FACEDF = pd.DataFrame(_fl)
for c in ["no_face_found", "fallback_crop", "low_detect_rate"]:
    AUDIT[c + "_included"] = FACEDF.groupby(_key)[c].sum().reindex(AUDIT.index).fillna(0).astype(int)
AUDIT.to_csv(os.path.join(OUT_DIR, "dataset_audit_v10.csv"))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
print("=" * 100); print("DATASET AUDIT (rows: dataset / class / method; every video has exactly one terminal fate)"); print("=" * 100)
print(AUDIT.reset_index().to_string(index=False))
print("\nTOTALS per dataset x class:")
_tot = AUDIT.groupby(level=["dataset", "class"]).sum()
print(_tot.to_string())
print("\nSplit sizes actually used (included):", {s: int(AUDIT[f"included_{s}"].sum()) for s in SPLITS})
_miss = [(d, s) for d in CONFIG["datasets"] for s in ("val", "test")
         if min(AUDIT.xs(d, level="dataset")[f"included_{s}"].groupby(level="class").sum().reindex(["real", "fake"]).fillna(0)) < CONFIG["min_dataset_videos_per_class_eval"]] \
    if len(AUDIT) else []
for d, s in _miss:
    print(f"WARNING: dataset '{d}' has fewer than {CONFIG['min_dataset_videos_per_class_eval']} videos per class in {s}: per-dataset {s} metrics are NOT reliable (reported but flagged).")
_methods_expected = {d: m for d, m in EXPECTED_METHODS.items() if d in CONFIG["datasets"]}
for d, ms in _methods_expected.items():
    present = set(AUDIT.xs(d, level="dataset").query("included_total > 0").index.get_level_values("method")) if d in AUDIT.index.get_level_values("dataset") else set()
    for m in ms:
        if m not in present:
            print(f"WARNING: manipulation method '{m}' of {d} has NO included video (not found on disk, dropped by the split, or not selected).")

# ---- per-dataset face diagnostics: real vs fake difference in centre-crop fallback (Fisher exact) ----
_fr = []
for (d, c), g in FACEDF.groupby(["dataset", "class"]):
    _fr.append({"dataset": d, "class": c, "n": len(g), "mean_detect_rate": g.detect_rate.mean(), "fallback_rate": g.fallback_crop.mean(),
                "no_face_rate": g.no_face_found.mean(), "low_rate_share": g.low_detect_rate.mean(), "centre_drift_over_side": g.drift.mean(),
                "faces_outside_static_crop": g.outside.mean(), "no_metadata": int((~g.has_meta).sum())})
FACE_DIAG = pd.DataFrame(_fr)
FACE_DIAG.to_csv(os.path.join(OUT_DIR, "face_crop_diagnostics_v10.csv"), index=False)
print("\nFACE-CROP DIAGNOSTICS (included videos):"); print(FACE_DIAG.round(3).to_string(index=False))
for d in FACEDF.dataset.unique():
    g = FACEDF[FACEDF.dataset == d]
    t = pd.crosstab(g["class"], g.fallback_crop).reindex(index=["real", "fake"], columns=[False, True]).fillna(0).values
    if t.sum(axis=1).min() > 0:
        print(f"  [{d}] fallback real {int(t[0,1])}/{int(t[0].sum())} vs fake {int(t[1,1])}/{int(t[1].sum())}  Fisher p={fisher_exact(t)[1]:.3f}")
for _, r in FACE_DIAG.iterrows():
    if r.fallback_rate > CONFIG["face_fallback_warn_rate"]:
        print(f"WARNING: {r.dataset}/{r['class']}: centre-crop fallback rate {r.fallback_rate:.1%} > {CONFIG['face_fallback_warn_rate']:.0%}; "
              f"evaluate fallback samples separately (Cell 12b reports metrics on non-fallback videos).")
_by_m = FACEDF.groupby(["dataset", "method"]).agg(n=("fallback_crop", "size"), fallback=("fallback_crop", "mean"), detect=("detect_rate", "mean")).round(3)
_by_m.to_csv(os.path.join(OUT_DIR, "face_crop_by_method_v10.csv"))
print("\nBy manipulation method:"); print(_by_m.to_string())

# ---- frame-sampling diagnostics ----
_fs = FACEDF.groupby(["dataset", "class"]).agg(n=("n_frames", "size"), min_frames=("n_frames", "min"), mean_frames=("n_frames", "mean"),
                                               share_below_pool=("n_frames", lambda x: float((x < CONFIG["frame_pool_size"]).mean())),
                                               share_recounted=("recounted", "mean"))
_fs["model_frames"] = CONFIG["seq_len"]
_fs["policy"] = f"pool={CONFIG['frame_pool_size']} linspace over the video; model uses {CONFIG['seq_len']}: 1 of 2 frames per bin (train random, eval first of bin); short videos: {CONFIG['short_video_policy']}"
_fs.to_csv(os.path.join(OUT_DIR, "frame_sampling_diagnostics_v10.csv"))
print("\nFRAME-SAMPLING DIAGNOSTICS:"); print(_fs.drop(columns="policy").round(3).to_string())
print("Policy:", _fs.policy.iloc[0] if len(_fs) else "n/a")

# ---- figures ----
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
_inc = _fd[_fd.fate == "included"]
_pv = _inc.groupby(["split_final", "dataset", "class"]).size().unstack(["dataset", "class"]).reindex(list(SPLITS)).fillna(0)
_pv.plot(kind="bar", stacked=True, ax=ax[0]); ax[0].set_title("Included videos: split x dataset x class"); ax[0].tick_params(axis="x", rotation=0); ax[0].legend(fontsize=7)
_pm = _inc[_inc["class"] == "fake"].groupby(["method", "split_final"]).size().unstack().reindex(columns=list(SPLITS)).fillna(0)
_pm.plot(kind="bar", ax=ax[1]); ax[1].set_title("Included FAKE videos per manipulation method"); ax[1].tick_params(axis="x", rotation=30)
_dc = AUDIT.groupby(level="dataset")[[c if c != "included" else "included_total" for c in FATE_ORDER]].sum()
_dc.plot(kind="bar", stacked=True, ax=ax[2], colormap="tab20"); ax[2].set_title("Video fates per dataset (exclusive)"); ax[2].tick_params(axis="x", rotation=0); ax[2].legend(fontsize=7)
plt.tight_layout(); save_fig(fig, "fig_v10_dataset_distribution.png"); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if len(FACE_DIAG):
    FACE_DIAG.pivot(index="dataset", columns="class", values="fallback_rate").plot(kind="bar", ax=ax[0]); ax[0].set_title("Centre-crop fallback rate"); ax[0].tick_params(axis="x", rotation=0)
    for (d, c), g in FACEDF.groupby(["dataset", "class"]):
        ax[1].hist(g.detect_rate.dropna(), bins=20, alpha=.45, label=f"{d}/{c}")
    ax[1].set_title("Per-video face-detection rate"); ax[1].legend(fontsize=7)
plt.tight_layout(); save_fig(fig, "fig_v10_face_diagnostics.png"); plt.show()
print("\nSaved: dataset_audit_v10.csv, face_crop_diagnostics_v10.csv, face_crop_by_method_v10.csv, frame_sampling_diagnostics_v10.csv")


In [ ]:
# ============================================================
# CELL 3: Data distribution analysis and face-crop QA (review item 8)
# ============================================================
# (The v4 sampler demo was removed: on balanced data it compared 0 all-fake batches with 0.)

split_counts = pd.DataFrame([
    {"Split": name,
     "Real": sum(1 for _, l in recs if l == 0),
     "Fake": sum(1 for _, l in recs if l == 1)}
    for name, recs in [("Train", train_records), ("Val", val_records), ("Test", test_records)]
])
print("Class distribution per split:")
print(split_counts.to_string(index=False))

method_rows = []
for name, recs in [("Train", train_records), ("Val", val_records), ("Test", test_records)]:
    cnt = pd.Series([record_info[f]["method"] for f, _ in recs]).value_counts()
    for m, c in cnt.items():
        method_rows.append({"Split": name, "Method": m, "Videos": int(c)})
method_df = pd.DataFrame(method_rows)
print("\nVideos per split and manipulation method:")
print(method_df.pivot(index="Method", columns="Split", values="Videos").fillna(0).astype(int).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
split_counts.set_index("Split")[["Real", "Fake"]].plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title("Class distribution per split")
axes[0].set_ylabel("Number of videos")
axes[0].tick_params(axis="x", rotation=0)
method_df.pivot(index="Split", columns="Method", values="Videos").fillna(0).loc[["Train", "Val", "Test"]].plot(
    kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Videos per manipulation method")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + "/class_distribution_by_split_and_method.png", dpi=150)
plt.show()

# ---- Face-crop quality: detection rates and a visual grid (inspect this before trusting any result) ----
def _box_info(folder):
    return FACE_BOXES.get(Path(folder).name)


infos = [(f, l, _box_info(f)) for f, l in extracted_records]
rates = [i["detect_rate"] for _, _, i in infos if i is not None]
n_fb = sum(1 for _, _, i in infos if i is not None and not i["ok"])
n_unknown = sum(1 for _, _, i in infos if i is None)
if rates:
    print(f"\nFace-detector statistics over {len(rates)} videos with stored boxes: "
          f"mean detection rate={np.mean(rates):.2f}, median={np.median(rates):.2f}, "
          f"videos below 50% detection={sum(r < 0.5 for r in rates)}, centre-crop fallbacks={n_fb}")
if n_unknown:
    print(f"({n_unknown} videos came from a cache without box metadata; they cannot be scored.)")


def _mid_frame(folder):
    files = sorted(Path(folder).glob("*.jpg"))
    return cv2.cvtColor(cv2.imread(str(files[len(files) // 2])), cv2.COLOR_BGR2RGB)


def show_crop_grid(items, title, path, ncols=6):
    nrows = int(math.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (folder, label, info) in zip(axes, items):
        ax.imshow(_mid_frame(folder))
        rate = f"{info['detect_rate']:.2f}" if info else "n/a"
        ax.set_title(f"{'FAKE' if label else 'REAL'} | det={rate}" + ("" if (info is None or info['ok']) else " FALLBACK"),
                     fontsize=8)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.savefig(path, dpi=130)
    plt.show()


_qa_rng = random.Random(0)
real_items = [x for x in infos if x[1] == 0]
fake_items = [x for x in infos if x[1] == 1]
qa_items = _qa_rng.sample(real_items, min(12, len(real_items))) + _qa_rng.sample(fake_items, min(12, len(fake_items)))
show_crop_grid(qa_items, "Random face crops (top: real, bottom: fake) -- check framing, no background/false positives",
               OUT_DIR + "/face_crop_qa_random.png")

scored = sorted([x for x in infos if x[2] is not None], key=lambda x: x[2]["detect_rate"])[:12]
if scored:
    show_crop_grid(scored, "Lowest detection-rate clips (most likely wrong crops)", OUT_DIR + "/face_crop_qa_worst.png")

# ---- Sample training frames (post-augmentation) ----
images, labels = next(iter(train_loader))
images_to_show = torch.cat([images[0, :4], images[1, :4]], dim=0)
grid = make_grid(images_to_show, nrow=4)
mean_t = torch.tensor(IMAGENET_MEAN, device=grid.device).view(3, 1, 1)
std_t = torch.tensor(IMAGENET_STD, device=grid.device).view(3, 1, 1)
grid_unnorm = torch.clamp(grid * std_t + mean_t, 0.0, 1.0)
label_mapping = {0: "Real", 1: "Fake"}
titles = f"Vid 1 ({label_mapping[int(labels[0].item())]}) | Vid 2 ({label_mapping[int(labels[1].item())]})"
plt.figure(figsize=(12, 6))
plt.imshow(grid_unnorm.permute(1, 2, 0).cpu().numpy())
plt.title(f"Sample training frames (post-augmentation): {titles}")
plt.axis("off")
plt.show()
print("\nTIP (review item 8): after this run, 'Save Version' and add /kaggle/working/face_frames as a Kaggle "
      "dataset; set CONFIG['face_cache_input_dir'] to its path to skip extraction next time.")

In [ ]:
# ============================================================
# CELL 4: Model architecture (unchanged from v4-v7)
# ============================================================
# MAIN model: EfficientNet-B1 (pretrained, frame-level feature extractor) -> 1280 -> Linear
# projection -> 512 -> 2-layer UNIDIRECTIONAL LSTM (hidden=512) -> MEAN pooling -> classifier ->
# 1 logit. The attention-pooling path exists only as an optional switch and is NOT enabled.

class TemporalAttentionPooling(nn.Module):
    """Optional (ablation-only) alternative to mean pooling."""
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.Tanh(),
            nn.Linear(input_dim // 2, 1),
        )

    def forward(self, x):
        scores = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(weights * x, dim=1)
        return pooled, weights.squeeze(-1)


class FFPPDeepfakeDetectorV2(nn.Module):
    def __init__(self, lstm_hidden_size=None, num_layers=None, bidirectional=None,
                 dropout=None, use_attention=None):
        super().__init__()
        lstm_hidden_size = lstm_hidden_size or CONFIG["lstm_hidden_size"]
        num_layers = num_layers or CONFIG["lstm_num_layers"]
        bidirectional = CONFIG["lstm_bidirectional"] if bidirectional is None else bidirectional
        dropout = CONFIG["dropout"] if dropout is None else dropout
        use_attention = CONFIG["use_temporal_attention"] if use_attention is None else use_attention

        # ---- EfficientNet-B1 spatial feature extractor (pretrained, unchanged) ----
        weights = EfficientNet_B1_Weights.DEFAULT
        base_model = efficientnet_b1(weights=weights)
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])  # [stages, avgpool]
        self.feature_dim = 1280
        self.dropout_cnn = nn.Dropout(p=dropout)
        self.flatten = nn.Flatten()

        # ---- Projection: 1280 -> 512 ----
        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )

        # ---- Temporal model: 2-layer UNIDIRECTIONAL LSTM by default ----
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=lstm_hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        lstm_out_dim = lstm_hidden_size * (2 if bidirectional else 1)

        self.temporal_pool = TemporalAttentionPooling(lstm_out_dim) if use_attention else None

        # ---- Classification head ----
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, 1),
        )

        self.last_attention_weights = None

    def forward(self, x, verbose_shape_check=False):
        batch_size, seq_len, c, h, w = x.shape
        assert (c, h, w) == (3, CONFIG["img_size"], CONFIG["img_size"]), (
            f"Expected input channels/H/W = (3,{CONFIG['img_size']},{CONFIG['img_size']}), got {(c, h, w)}"
        )
        if verbose_shape_check:
            print(f"  Input                    : {tuple(x.shape)}")

        x = x.view(batch_size * seq_len, c, h, w)
        if verbose_shape_check:
            print(f"  Reshaped for CNN         : {tuple(x.shape)}")

        features = self.feature_extractor(x)
        features = self.flatten(features)
        if verbose_shape_check:
            print(f"  EfficientNet-B1 output   : {tuple(features.shape)}")
        assert features.shape[1] == self.feature_dim, (
            f"Expected {self.feature_dim}-D EfficientNet-B1 features, got {features.shape[1]}"
        )

        features = self.dropout_cnn(features)
        features = self.projection(features)
        if verbose_shape_check:
            print(f"  After projection (->512) : {tuple(features.shape)}")
        assert features.shape[1] == 512, f"Expected 512-D projected features, got {features.shape[1]}"

        features = features.view(batch_size, seq_len, -1)
        if verbose_shape_check:
            print(f"  Reshaped for LSTM        : {tuple(features.shape)}")

        lstm_out, _ = self.lstm(features)
        if verbose_shape_check:
            print(f"  LSTM output              : {tuple(lstm_out.shape)}")

        if self.use_attention:
            pooled, attn_weights = self.temporal_pool(lstm_out)
            self.last_attention_weights = attn_weights.detach()
        else:
            pooled = torch.mean(lstm_out, dim=1)
            self.last_attention_weights = None
        if verbose_shape_check:
            print(f"  Pooled ({'attention' if self.use_attention else 'mean':<9}) : {tuple(pooled.shape)}")

        out = self.classifier(pooled)
        if verbose_shape_check:
            print(f"  Classifier output        : {tuple(out.shape)}")

        out = out.squeeze(1)
        if verbose_shape_check:
            print(f"  Final logits (per video) : {tuple(out.shape)}")
        return out

    def train(self, mode=True):
        """FIX: model.train() used to put the FROZEN backbone's BatchNorm layers in train
        mode, so their running stats drifted every batch while their weights stayed frozen.
        BN layers with no trainable params are now kept in eval mode."""
        super().train(mode)
        if mode:
            bn_eval = getattr(self, "bn_eval", False)      # set True during FGSM-AT -> ALL BN layers use running stats
            for m in self.feature_extractor.modules():
                if isinstance(m, nn.modules.batchnorm._BatchNorm) and \
                        (bn_eval or not any(p.requires_grad for p in m.parameters())):
                    m.eval()
        return self

    # ---- Staged transfer-learning helpers ----
    def freeze_backbone(self):
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n):
        backbone_blocks = list(self.feature_extractor[0].children())
        for block in backbone_blocks[-n:]:
            for p in block.parameters():
                p.requires_grad = True

    def backbone_param_groups(self, backbone_lr, head_lr):
        backbone_params = [p for p in self.feature_extractor.parameters() if p.requires_grad]
        head_params = (
            list(self.projection.parameters())
            + list(self.lstm.parameters())
            + (list(self.temporal_pool.parameters()) if self.use_attention else [])
            + list(self.classifier.parameters())
        )
        groups = [{"params": head_params, "lr": head_lr}]
        if backbone_params:
            groups.append({"params": backbone_params, "lr": backbone_lr})
        return groups


def build_model(**overrides):
    m = FFPPDeepfakeDetectorV2(**overrides).to(CONFIG["device"])
    m.freeze_backbone()  # backbone starts frozen; run_staged_training unfreezes the last N blocks
    return m


model = build_model()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model initialized:")
print(f"  EfficientNet-B1 -> projection(1280->512) -> "
      f"{'Bi' if CONFIG['lstm_bidirectional'] else ''}LSTM(hidden={CONFIG['lstm_hidden_size']}, "
      f"layers={CONFIG['lstm_num_layers']}, bidirectional={CONFIG['lstm_bidirectional']}) -> "
      f"{'attention' if CONFIG['use_temporal_attention'] else 'mean'} pooling -> classifier(256) -> 1 logit")
print(f"  Total params: {total_params:,} | Trainable (Phase 1, backbone frozen): {trainable_params:,}")

print("\nShape verification (dummy forward pass, batch_size=2):")
with torch.no_grad():
    dummy_input = torch.randn(2, CONFIG["seq_len"], 3, CONFIG["img_size"], CONFIG["img_size"],
                               device=CONFIG["device"])
    _ = model(dummy_input, verbose_shape_check=True)
print("Shape verification passed.")

In [ ]:
# ============================================================
# CELL 5: FGSM attack library, prediction helpers and 12 automated unit tests
# ============================================================
# FGSM is the ONLY adversarial attack in this notebook (single gradient step, L-inf):
#       x_adv = clip_[0,1]( x + eps * sign( grad_x  BCEWithLogits(model(x), y) ) )
# Conventions (verified by the unit tests below):
#   * label 1 = FAKE = positive class; the model outputs ONE logit z, P(fake) = sigmoid(z).
#   * The loss is the plain BCE of the logit against the label that is being ATTACKED (true label for evaluation), so
#     ascending it pushes a fake towards z<0 and a real towards z>0.
#   * Tensors are [B,T,3,H,W] in ImageNet-NORMALISED space. eps is a PIXEL-scale (0..1) budget; it is converted per channel
#     with eps/std, and the result is clamped to the normalised image of [0,1]. Every frame of the video is perturbed.
#   * Attack mode: BatchNorm/Dropout in eval (frozen running statistics, deterministic). cuDNN's LSTM backward only exists in
#     train mode, so LSTM modules are switched to train() with dropout 0.0 for the gradient pass and restored afterwards.
#   * The gradient is taken with torch.autograd.grad w.r.t. the INPUT only: no parameter .grad is written, no graph is retained,
#     no optimizer/EMA is touched. The returned tensor is detached.
#   * The gradient does not depend on eps, so an epsilon sweep computes sign(grad) once per batch (`fgsm_sweep_logits`) --
#     mathematically identical to calling fgsm_attack once per epsilon (checked in the unit tests).

_MEAN = torch.tensor(IMAGENET_MEAN).view(1, 1, 3, 1, 1)
_STD = torch.tensor(IMAGENET_STD).view(1, 1, 3, 1, 1)
ATTACK_DIAG = {"samples": 0, "zero_grad_samples": 0, "nonfinite_grad_samples": 0}


def reset_attack_diag():
    for k in ATTACK_DIAG:
        ATTACK_DIAG[k] = 0


def _prepare_model_for_attack(model):
    prev = {m: m.training for m in model.modules()}
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)):
            m.train()
            m._stored_dropout = m.dropout
            m.dropout = 0.0
    return prev


def _restore_model_modes(model, prev):
    for m in model.modules():
        if m in prev:
            m.train(prev[m])
        if isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)) and hasattr(m, "_stored_dropout"):
            m.dropout = m._stored_dropout
            del m._stored_dropout


def _norm_bounds(device):
    mean, std = _MEAN.to(device), _STD.to(device)
    return (0.0 - mean) / std, (1.0 - mean) / std


def _eps_normalized(eps, device):
    if not torch.is_tensor(eps):
        eps = torch.tensor(float(eps))
    return eps.to(device).float() / _STD.to(device)


def fgsm_grad(model, videos, labels, use_amp=False, label_source="true"):
    """Input gradient of the summed BCE (the sum keeps fp16 gradients away from underflow; sign() is scale-free).
    label_source='pred' attacks the model's OWN hard prediction (no label information used); 'flipped' (audit only) uses 1-y.
    Returns (grad, clean_logits_float). Model modes are restored; parameters/BN/EMA are untouched."""
    prev = _prepare_model_for_attack(model)
    try:
        x = videos.detach().clone().requires_grad_(True)
        ctx = torch.amp.autocast("cuda") if (use_amp and videos.is_cuda) else nullcontext()
        with ctx:
            logits = model(x)
        logits = logits.float()
        if label_source == "pred":
            y = (logits.detach() > 0).float()
        elif label_source == "flipped":                      # audit only: ascend the loss of the WRONG label (a 'helpful' perturbation)
            y = 1.0 - labels.float()
        else:
            y = labels.float()
        loss = Fn.binary_cross_entropy_with_logits(logits, y, reduction="sum")
        if use_amp and videos.is_cuda:
            scale = 1024.0
            grad = torch.autograd.grad(loss * scale, x)[0] / scale
        else:
            grad = torch.autograd.grad(loss, x)[0]
    finally:
        _restore_model_modes(model, prev)
    finite = torch.isfinite(grad.flatten(1)).all(dim=1)
    ATTACK_DIAG["nonfinite_grad_samples"] += int((~finite).sum().item())
    grad = torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
    zero = (grad.flatten(1).abs().amax(dim=1) == 0)
    ATTACK_DIAG["samples"] += int(grad.size(0))
    ATTACK_DIAG["zero_grad_samples"] += int(zero.sum().item())
    return grad.detach(), logits.detach()


def fgsm_apply(x0, grad, eps):
    """x_adv = clamp(x0 + eps*sign(grad)); eps: float or tensor broadcastable to [B,1,1,1,1] (pixel scale)."""
    vmin, vmax = _norm_bounds(x0.device)
    eps_n = _eps_normalized(eps, x0.device)
    with torch.no_grad():
        x_adv = x0.detach() + eps_n * torch.sign(grad)
        x_adv = torch.max(torch.min(x_adv, vmax), vmin)
    return x_adv.detach()


def fgsm_attack(model, videos, labels, epsilon, use_amp=False, label_source="true", return_info=False):
    grad, logits = fgsm_grad(model, videos, labels, use_amp=use_amp, label_source=label_source)
    x_adv = fgsm_apply(videos, grad, epsilon)
    if return_info:
        return x_adv, {"grad": grad, "logits": logits}
    return x_adv


def gradient_alignment(model, videos, labels, eps, use_amp=False):
    """Catastrophic-overfitting heuristic: mean cosine similarity between the input gradients at two INDEPENDENT random
    points of the eps-ball. Close to 1 = locally linear loss (FGSM is a good attack); close to 0 = FGSM is unreliable."""
    vmin, vmax = _norm_bounds(videos.device)
    eps_n = _eps_normalized(eps, videos.device)
    gs = []
    for _ in range(2):
        noise = (torch.rand_like(videos) * 2 - 1) * eps_n
        xr = torch.max(torch.min(videos + noise, vmax), vmin)
        g, _ = fgsm_grad(model, xr, labels, use_amp=use_amp)
        gs.append(g.flatten(1))
    return float(Fn.cosine_similarity(gs[0], gs[1], dim=1, eps=1e-12).mean().item())


# ---- Prediction helpers (shared by every later cell) ----
def get_logits(model, loader, attack_fn=None, use_amp=None, desc="Evaluating"):
    """(labels, raw logits) in loader order. attack_fn(model, videos, labels)->adv_videos or None. fp32 by default."""
    use_amp = CONFIG["eval_use_amp"] if use_amp is None else use_amp
    model.eval()
    all_logits, all_labels = [], []
    for videos, labels in tqdm(loader, desc=desc, leave=False):
        videos = videos.to(CONFIG["device"], non_blocking=True)
        labels_dev = labels.to(CONFIG["device"], non_blocking=True)
        if attack_fn is not None:
            videos = attack_fn(model, videos, labels_dev)
        with torch.no_grad():
            ctx = torch.amp.autocast("cuda") if (use_amp and CONFIG["device"].type == "cuda") else nullcontext()
            with ctx:
                logits = model(videos)
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels).astype(int), np.concatenate(all_logits)


def fgsm_sweep_logits(model, loader, eps_list, label_source="true", desc="FGSM sweep"):
    """Clean logits and FGSM logits for EVERY epsilon in `eps_list` (pixel scale) in one pass over `loader`.
    Returns labels, {eps: logits}; eps=0 is the clean forward pass. Same videos / same order for any model."""
    model.eval()
    labels_all, out = [], {float(e): [] for e in eps_list}
    for videos, labels in tqdm(loader, desc=desc, leave=False):
        videos = videos.to(CONFIG["device"], non_blocking=True)
        labels_dev = labels.to(CONFIG["device"], non_blocking=True)
        grad = None
        for e in eps_list:
            if e == 0:
                x = videos
            else:
                if grad is None:
                    grad, _ = fgsm_grad(model, videos, labels_dev, use_amp=False, label_source=label_source)
                x = fgsm_apply(videos, grad, e)
            with torch.no_grad():
                out[float(e)].append(model(x).float().cpu().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(labels_all).astype(int), {e: np.concatenate(v) for e, v in out.items()}


def quantize_to_8bit(x_norm):
    """Round a normalised video to the 8-bit pixel grid (what a saved/re-decoded image could actually contain)."""
    mean, std = _MEAN.to(x_norm.device), _STD.to(x_norm.device)
    pix = torch.round((x_norm * std + mean).clamp(0, 1) * 255.0) / 255.0
    return (pix - mean) / std


# ============================================================
# UNIT TESTS (all must pass before any training)
# ============================================================
def _valid_batch(B=3, T=4, seed=0, lo=0.05, hi=0.95):
    g = torch.Generator().manual_seed(seed)
    pix = lo + (hi - lo) * torch.rand(B, T, 3, CONFIG["img_size"], CONFIG["img_size"], generator=g)
    return ((pix - _MEAN) / _STD).to(CONFIG["device"]), torch.tensor([0, 1, 0][:B] + [1] * max(0, B - 3)).to(CONFIG["device"])


def _pix(x):               # normalised -> pixel scale
    return x * _STD.to(x.device) + _MEAN.to(x.device)


def run_fgsm_unit_tests(model):
    results = []
    dev = CONFIG["device"]

    def record(name, ok, detail=""):
        results.append((name, bool(ok), detail))
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}  {detail}")

    was_training = model.training
    x, y = _valid_batch()
    model.train()                                  # tests also prove the attack handles / restores a model left in train mode
    n_params0 = {k: v.detach().clone() for k, v in model.state_dict().items()}

    # 1. epsilon = 0 -> identity
    xa = fgsm_attack(model, x, y, 0.0)
    record("1. eps=0 returns the input unchanged", torch.equal(xa, x), f"max|diff|={float((xa - x).abs().max()):.2e}")

    # 2. L-inf bound (pixel space)
    worst = 0.0
    for e in [0.25, 1, 4, 16]:
        xa = fgsm_attack(model, x, y, e / 255)
        worst = max(worst, float((_pix(xa) - _pix(x)).abs().max()) * 255 - e)
    record("2. L-inf perturbation <= eps (pixel space)", worst <= 1e-3, f"max excess over eps = {worst:.2e}/255")

    # 3. valid pixel range, including inputs sitting exactly on 0 and 1
    xe, _ = _valid_batch(lo=0.0, hi=1.0)
    xe[:, :, :, :8, :8] = ((0.0 - _MEAN) / _STD).to(dev)
    xe[:, :, :, -8:, -8:] = ((1.0 - _MEAN) / _STD).to(dev)
    xa = fgsm_attack(model, xe, y, 16 / 255)
    pa = _pix(xa)
    record("3. adversarial pixels stay in [0,1]", float(pa.min()) >= -1e-5 and float(pa.max()) <= 1 + 1e-5,
           f"range=[{float(pa.min()):.5f}, {float(pa.max()):.5f}]")

    # 4. perturbation is non-zero (and exactly eps) wherever the gradient is non-zero and not clipped
    eps4 = 2 / 255
    xa, info = fgsm_attack(model, x, y, eps4, return_info=True)
    d = (_pix(xa) - _pix(x)).abs()
    interior = (_pix(x) > eps4 + 1e-4) & (_pix(x) < 1 - eps4 - 1e-4)
    mask = (info["grad"].abs() > 0) & interior
    frac_nonzero_grad = float((info["grad"].abs() > 0).float().mean())
    ok4 = mask.any() and bool((torch.abs(d[mask] - eps4) < 1e-5).all())
    record("4. non-zero perturbation (= eps) wherever grad != 0", ok4, f"fraction of pixels with non-zero grad = {frac_nonzero_grad:.3f}")

    # 5. finite gradients, zero-gradient samples counted
    reset_attack_diag()
    _, info = fgsm_attack(model, x, y, 1 / 255, return_info=True)
    ok5 = bool(torch.isfinite(info["grad"]).all()) and ATTACK_DIAG["zero_grad_samples"] == 0 and ATTACK_DIAG["nonfinite_grad_samples"] == 0
    record("5. gradients finite; zero-gradient samples = 0", ok5,
           f"zero-grad {ATTACK_DIAG['zero_grad_samples']}/{ATTACK_DIAG['samples']}, non-finite {ATTACK_DIAG['nonfinite_grad_samples']}")

    # 6. deterministic evaluation (clean forward exactly; attack sign pattern up to cuDNN LSTM-backward non-determinism)
    model.eval()
    with torch.no_grad():
        z1, z2 = model(x).float(), model(x).float()
    a1, a2 = fgsm_attack(model, x, y, 2 / 255), fgsm_attack(model, x, y, 2 / 255)
    agree = float((a1 == a2).float().mean())
    record("6. deterministic evaluation", torch.equal(z1, z2) and agree >= 0.99, f"clean logits identical={torch.equal(z1, z2)}, adversarial pixel agreement={agree:.5f}")

    # 7/8. parameters and BatchNorm statistics unchanged; module modes restored; no parameter gradients written
    model.train()
    sd_before = {k: v.detach().clone() for k, v in model.state_dict().items()}
    modes_before = {m: m.training for m in model.modules()}
    model.zero_grad(set_to_none=True)
    _ = fgsm_attack(model, x, y, 4 / 255)
    sd_after = model.state_dict()
    params_same = all(torch.equal(sd_before[k], sd_after[k]) for k, v in sd_before.items()
                      if k in dict(model.named_parameters()))
    buffers_same = all(torch.equal(sd_before[k], sd_after[k]) for k in sd_before if k not in dict(model.named_parameters()))
    grads_clean = all(p.grad is None for p in model.parameters())
    modes_same = all(m.training == modes_before[m] for m in model.modules())
    record("7. model parameters unchanged (and no .grad written)", params_same and grads_clean, f"params identical={params_same}, grads None={grads_clean}")
    record("8. BatchNorm running stats / counters unchanged; modes restored", buffers_same and modes_same,
           f"buffers identical={buffers_same}, training flags restored={modes_same}")

    # 9. temporal dimension: any number of frames, every frame perturbed, shape preserved
    ok9, det9 = True, []
    for T in (2, 6):
        xt, yt = _valid_batch(B=2, T=T, seed=T)
        xa = fgsm_attack(model, xt, yt[:2], 2 / 255)
        per_frame = (xa - xt).abs().flatten(2).amax(dim=2)          # [B,T]
        ok9 &= (xa.shape == xt.shape) and bool((per_frame > 0).all())
        det9.append(f"T={T}: shape ok, min per-frame |delta|={float(per_frame.min()):.3e}")
    record("9. temporal dimension preserved, every frame perturbed", ok9, "; ".join(det9))

    # 10. last incomplete batch (5 videos, batch size 4 -> batches of 4 and 1)
    model.eval()
    xs, ys = _valid_batch(B=5, T=2, seed=7)
    ys = torch.tensor([0, 1, 0, 1, 1])
    loader = DataLoader(torch.utils.data.TensorDataset(xs.cpu(), ys), batch_size=4, shuffle=False)
    sizes, outs = [], []
    for xb, yb in loader:
        sizes.append(len(xb))
        outs.append(fgsm_attack(model, xb.to(dev), yb.to(dev), 2 / 255).cpu())
    whole = fgsm_attack(model, xs, ys.to(dev), 2 / 255).cpu()
    cat = torch.cat(outs)
    record("10. last incomplete batch handled (4+1), same result as one batch of 5", sizes == [4, 1] and cat.shape == whole.shape and float((cat == whole).float().mean()) >= 0.99,
           f"batch sizes={sizes}, agreement={float((cat == whole).float().mean()):.5f}")

    # 11. direction + label convention: margin (1-2y)*logit must increase, the BCE must increase.
    #     Run on a tiny deterministic LINEAR probe (logit = scaled mean of w*x) so the test is guaranteed to be sensitive to the input: a freshly
    #     initialised head (logits ~ 0) or a random-init backbone cannot hide a sign error in float noise. The backbone model is tested
    #     for the same property below as an INFORMATION line only (it can legitimately be insensitive before training).
    class _LinearProbe(nn.Module):
        def __init__(self):
            super().__init__()
            g = torch.Generator().manual_seed(11)
            self.w = nn.Parameter(torch.randn(1, 1, 3, CONFIG["img_size"], CONFIG["img_size"], generator=g))
        def forward(self, x):
            return 400.0 * (x * self.w).mean(dim=(1, 2, 3, 4))
    probe = _LinearProbe().to(dev).eval()
    xb, yb = _valid_batch(B=4, T=4, seed=3)
    yb = torch.tensor([0, 1, 0, 1]).to(dev)
    with torch.no_grad():
        z0 = probe(xb).float()
    xa = fgsm_attack(probe, xb, yb, 2 / 255)
    with torch.no_grad():
        z1 = probe(xa).float()
    yf = yb.float()
    l0 = Fn.binary_cross_entropy_with_logits(z0, yf).item()
    l1 = Fn.binary_cross_entropy_with_logits(z1, yf).item()
    dm = ((1 - 2 * yf) * (z1 - z0))
    record("11. FGSM ascends the loss (fake -> lower logit, real -> higher logit)", l1 > l0 and bool((dm > 0).all()),
           f"BCE {l0:.4f}->{l1:.4f}; margin change per sample={[round(float(v), 4) for v in dm]}")
    del probe
    xb2, yb2 = _valid_batch(B=3, T=4, seed=3)
    xb2a = fgsm_attack(model, xb2, yb2, 8 / 255)          # (needs autograd: attack OUTSIDE no_grad)
    with torch.no_grad():
        zb0 = model(xb2).float()
        zb1 = model(xb2a).float()
    print(f"  [INFO] backbone model, same direction check at 8/255: margin change {[round(float(v), 4) for v in (1 - 2 * yb2.float()) * (zb1 - zb0)]} "
          f"(~0 is normal for an untrained head; the linear probe above is the binding test)")

    # 12. no retained graph, and sweep == per-epsilon attack
    xa = fgsm_attack(model, x, y, 2 / 255)
    g, _ = fgsm_grad(model, x, y)
    same = torch.equal(fgsm_apply(x, g, 2 / 255), xa) or float((fgsm_apply(x, g, 2 / 255) == xa).float().mean()) >= 0.99
    record("12. output detached (no graph); sweep path == single-eps attack", (not xa.requires_grad) and xa.grad_fn is None and same,
           f"requires_grad={xa.requires_grad}, grad_fn={xa.grad_fn}")

    model.train(was_training)
    failed = [n for n, ok, _ in results if not ok]
    print(f"\nFGSM unit tests: {len(results) - len(failed)}/{len(results)} passed.")
    assert not failed, f"FGSM unit tests FAILED: {failed}"
    return results


print("FGSM library ready (fgsm_grad / fgsm_apply / fgsm_attack / gradient_alignment / fgsm_sweep_logits).")
print("Running the automated FGSM unit tests on the freshly built model (before any training):")
UNIT_TEST_RESULTS = run_fgsm_unit_tests(model)
atomic_json_save([{"test": n, "passed": ok, "detail": d} for n, ok, d in UNIT_TEST_RESULTS], os.path.join(OUT_DIR, "fgsm_unit_tests.json"))

In [ ]:
# ============================================================
# CELL 5b (v10): Automated checks (synthetic-data checks run anywhere; data checks run on the real records). Critical failures STOP the notebook.
# ============================================================
import tempfile, shutil
from scipy.stats import rankdata

CHECKS = []          # (name, status, detail, critical)


def independent_auc(labels, scores):
    """ROC-AUC from average ranks (Mann-Whitney U), NO sklearn: the independent re-computation used to verify roc_auc_score on saved raw scores."""
    labels = np.asarray(labels).astype(int)
    s = np.asarray(scores, dtype=np.float64)
    n1, n0 = int((labels == 1).sum()), int((labels == 0).sum())
    if n1 == 0 or n0 == 0 or not np.all(np.isfinite(s)):
        return float("nan")
    r = rankdata(s)
    return float((r[labels == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def check_sequence_tensor(x, seq_len=None, img=None):
    """x: [T,3,H,W] in ImageNet-normalised space. Returns (ok, detail)."""
    seq_len = seq_len or CONFIG["seq_len"]
    img = img or CONFIG["img_size"]
    if tuple(x.shape) != (seq_len, 3, img, img):
        return False, f"shape {tuple(x.shape)} != {(seq_len, 3, img, img)}"
    if not torch.isfinite(x).all():
        return False, "non-finite values"
    lo, hi = _VALID_MIN.to(x.device) - 1e-4, _VALID_MAX.to(x.device) + 1e-4
    if bool((x < lo).any() or (x > hi).any()):
        return False, "values outside the valid normalised pixel range"
    return True, f"shape {tuple(x.shape)}, range [{float(x.min()):.3f}, {float(x.max()):.3f}]"


def run_check(name, fn, critical=True):
    try:
        res = fn()
        ok, detail = (res if isinstance(res, tuple) else (bool(res), ""))
    except Exception as e:
        ok, detail = False, f"EXCEPTION {type(e).__name__}: {e}"
    CHECKS.append({"check": name, "status": "PASS" if ok else ("FAIL" if critical else "WARN"), "detail": str(detail)[:300], "critical": critical})
    print(f"  [{'PASS' if ok else ('FAIL' if critical else 'WARN')}] {name}  {detail}")


def _touch(p):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(b"x")


# ---------------- synthetic-data checks ----------------
def chk_discovery_and_mapping():
    d = Path(tempfile.mkdtemp())
    try:
        for n in ["000", "001"]:
            _touch(d / "ff/original" / f"{n}.mp4")
        for n in ["000_001", "001_000"]:
            _touch(d / "ff/Deepfakes" / f"{n}.mp4")
        _touch(d / "cd/Celeb-real/id0_0000.mp4"); _touch(d / "cd/YouTube-real/00007.mp4"); _touch(d / "cd/Celeb-synthesis/id0_id16_0003.mp4")
        _touch(d / "ua/real/0047.mp4"); _touch(d / "ua/fake/0047_fake.mp4")
        ff, cd, ua = parse_ff_c23_v10(d / "ff"), parse_celeb_df_v10(d / "cd"), parse_uadfv_v10(d / "ua")
        ok = all(v["label"] == (0 if v["method"] == "real" else 1) for v in ff + cd + ua)
        ok &= {v["method"] for v in ff if v["label"] == 1} == {"Deepfakes"}
        by = {Path(v["path"]).stem: v for v in cd + ua + ff}
        regress = by["id0_0000"]["ids"] == ("celebdf_id0",)                 # v9 produced ('celebdf_i','celebdf_d','celebdf_0')
        syn = by["id0_id16_0003"]["ids"] == ("celebdf_id0", "celebdf_id16")
        yt = by["00007"]["ids"] == ("celebdf_yt_00007",)
        ua_ok = by["0047"]["ids"] == by["0047_fake"]["ids"] == ("uadfv_0047",)
        return bool(ok and regress and syn and yt and ua_ok), f"labels/methods ok={ok}, celeb id not char-level={regress}, fake ids={syn}, YouTube-real own group={yt}, UADFV pair grouped={ua_ok}"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def _ff_like(n=120, seed=0):
    rng = random.Random(seed)
    ids = [f"{i:03d}" for i in range(n)]
    vs = [{"path": f"/x/original/{i}.mp4", "label": 0, "method": "real", "ids": (f"ffpp_{i}",), "dataset": "ffpp"} for i in ids]
    for m in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]:
        perm = ids[:]
        rng.shuffle(perm)
        vs += [{"path": f"/x/{m}/{a}_{b}.mp4", "label": 1, "method": m, "ids": (f"ffpp_{a}", f"ffpp_{b}"), "dataset": "ffpp"} for a, b in zip(ids, perm) if a != b]
    return vs


def chk_split_leakage():
    vs = _ff_like()
    split_of, assigned, straddle, info = identity_disjoint_split_v10(vs, 42, 0.15, 0.2, 20)
    seen = {s: {i for v in assigned[s] for i in v["ids"]} for s in SPLITS}
    clean = not (seen["train"] & seen["val"] or seen["train"] & seen["test"] or seen["val"] & seen["test"])
    n_in = sum(len(assigned[s]) for s in SPLITS)
    better = info["straddle_dropped"] <= info["straddle_dropped_random_assignment_trial0"]
    return bool(clean and n_in + len(straddle) == len(vs) and better), \
        f"no id in two splits={clean}; kept {n_in}/{len(vs)}; dropped {len(straddle)} (random assignment: {info['straddle_dropped_random_assignment_trial0']})"


def chk_duplicates():
    d = Path(tempfile.mkdtemp())
    try:
        (d / "a").mkdir(); (d / "b").mkdir()
        (d / "a/1.mp4").write_bytes(b"AAAA" * 5000); (d / "b/1.mp4").write_bytes(b"AAAA" * 5000)       # same bytes, same label
        (d / "a/2.mp4").write_bytes(b"BBBB" * 5000); (d / "b/2.mp4").write_bytes(b"BBBB" * 5000)       # same bytes, different labels
        (d / "a/3.mp4").write_bytes(b"CCCC" * 5000)
        mk = lambda p, ds, lab: {"path": str(p), "label": lab, "method": "x", "ids": (str(p),), "dataset": ds}
        vs = [mk(d / "a/1.mp4", "A", 0), mk(d / "b/1.mp4", "B", 0), mk(d / "a/2.mp4", "A", 0), mk(d / "b/2.mp4", "B", 1), mk(d / "a/3.mp4", "A", 1)]
        kept, dropped = dedupe_videos(vs, {"A": 0, "B": 1})
        reasons = sorted(w for _, w in dropped)
        return bool(len(kept) == 2 and reasons == ["duplicate", "label_conflict_duplicate", "label_conflict_duplicate"]), f"kept {len(kept)}, dropped reasons {reasons}"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def chk_reproducible_selection():
    vs = _ff_like(150, 3)
    a = {s: [] for s in SPLITS}
    for v in vs:
        a[SPLITS[hash(v["ids"][0]) % 3 if False else (int(v["ids"][0][-3:]) % 3)]].append(v)
    cfg = dict(CONFIG)
    cfg.update({"max_videos_to_use": 120, "split_budget_fractions": {"train": 0.7, "val": 0.15, "test": 0.15}})
    s1, s2, s3 = (stratified_select_v10(a, cfg, sd) for sd in (7, 7, 8))
    same = all([v["path"] for v in s1[s]] == [v["path"] for v in s2[s]] for s in SPLITS)
    diff = any({v["path"] for v in s1[s]} != {v["path"] for v in s3[s]} for s in SPLITS)
    tr = Counter(v["method"] for v in s1["train"] if v["label"] == 1)
    spread = max(tr.values()) - min(tr.values()) <= 1 and len(tr) == 4           # round-robin over the manipulation methods
    bal = all(sum(v["label"] == 0 for v in s1[s]) == sum(v["label"] == 1 for v in s1[s]) for s in SPLITS)
    cap = sum(len(s1[s]) for s in SPLITS) <= 120
    return bool(same and diff and spread and bal and cap), f"same seed identical={same}, other seed differs={diff}, methods balanced={dict(tr)}, class-balanced={bal}, within budget={cap}"


def chk_alphabetical_truncation_gone():
    vs = _ff_like(150, 5)
    a = {"train": vs, "val": [], "test": []}
    cfg = dict(CONFIG); cfg.update({"max_videos_to_use": 400, "split_budget_fractions": {"train": 1.0, "val": 0.0, "test": 0.0}})
    sel = stratified_select_v10(a, cfg, 1)["train"]
    got = {v["method"] for v in sel if v["label"] == 1}
    return got == {"Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"}, f"fake methods in a budget-limited selection: {sorted(got)} (v9 kept only the alphabetically first methods)"


def _write_video(p, n, size=(96, 96)):
    w = cv2.VideoWriter(str(p), cv2.VideoWriter_fourcc(*"mp4v"), 10, size)
    for i in range(n):
        w.write(np.full((size[1], size[0], 3), (i * 5) % 250, np.uint8))
    w.release()


def chk_frame_decoding():
    d = Path(tempfile.mkdtemp())
    try:
        _write_video(d / "long.mp4", 80); _write_video(d / "short.mp4", 10); (d / "bad.mp4").write_bytes(b"nope" * 100)
        fr, idx, meta = _read_pool_frames(d / "long.mp4", 32)
        want = [int(i) for i in np.linspace(0, 79, 32, dtype=int)]
        ok1 = len(fr) == 32 and idx == want and idx == sorted(idx)
        fr2, idx2, m2 = _read_pool_frames(d / "short.mp4", 32)
        ok2 = len(fr2) == 10 and idx2 == list(range(10)) and len(fr2) < CONFIG["min_usable_frames"]      # flagged as insufficient by the extractor
        fr3, idx3, m3 = _read_pool_frames(d / "bad.mp4", 32)
        ok3 = fr3 == [] and m3["status"] == "undecodable"
        return bool(ok1 and ok2 and ok3), f"80-frame video -> 32 linspace frames in order={ok1}; 10-frame video -> 10 frames (dropped by policy)={ok2}; unreadable -> undecodable={ok3}"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def chk_face_crop_geometry():
    h, w = 120, 200
    frames = [np.full((h, w, 3), 100, np.uint8) for _ in range(8)]       # no face -> must fall back to the centre square and say so
    x0, y0, x1, y1, ok, rate, st = _video_face_square(frames, CONFIG["face_margin"])
    side_ok = (x1 - x0) == (y1 - y0) and 0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h
    return bool((not ok) and rate == 0 and (not st["any_detection"]) and side_ok), f"fallback flagged={not ok}, any_detection={st['any_detection']}, square inside frame={side_ok} ({x0},{y0},{x1},{y1})"


def chk_rgb_and_normalisation():
    d = Path(tempfile.mkdtemp())
    try:
        (d / "v").mkdir()
        for i in range(CONFIG["seq_len"]):
            cv2.imwrite(str(d / "v" / f"frame_{i:03d}.jpg"), np.dstack([np.zeros((64, 64), np.uint8), np.zeros((64, 64), np.uint8), np.full((64, 64), 255, np.uint8)]))   # BGR red
        x, y = PreextractedVideoDataset([(str(d / "v"), 1)], CONFIG["seq_len"], CONFIG["img_size"], is_train=False)[0]
        m = x.mean(dim=(0, 2, 3)).numpy()
        want = np.array([(1 - IMAGENET_MEAN[0]) / IMAGENET_STD[0], (0 - IMAGENET_MEAN[1]) / IMAGENET_STD[1], (0 - IMAGENET_MEAN[2]) / IMAGENET_STD[2]])
        ok = np.allclose(m, want, atol=0.1)
        s_ok, s_d = check_sequence_tensor(x)
        return bool(ok and s_ok and float(y) == 1.0), f"channel means {np.round(m, 3)} vs expected RGB-red {np.round(want, 3)}; {s_d}"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def chk_train_eval_preprocessing_identical():
    d = Path(tempfile.mkdtemp())
    keys = ["compression_prob", "aug_flip_prob", "aug_rotation_range", "aug_brightness_range", "aug_contrast_range", "aug_saturation_range",
            "aug_hue_range", "aug_crop_padding", "aug_noise_prob"]
    old = {k: CONFIG[k] for k in keys}
    try:
        (d / "v").mkdir()
        rng = np.random.RandomState(0)
        for i in range(CONFIG["seq_len"]):                      # exactly seq_len crops -> train and eval pick the same frame indices
            cv2.imwrite(str(d / "v" / f"frame_{i:03d}.jpg"), rng.randint(0, 255, (80, 80, 3), np.uint8))
        CONFIG.update({"compression_prob": 0.0, "aug_flip_prob": 0.0, "aug_rotation_range": (0.0, 0.0), "aug_brightness_range": (1.0, 1.0),
                       "aug_contrast_range": (1.0, 1.0), "aug_saturation_range": (1.0, 1.0), "aug_hue_range": (0.0, 0.0), "aug_crop_padding": 0, "aug_noise_prob": 0.0})
        tr = PreextractedVideoDataset([(str(d / "v"), 0)], CONFIG["seq_len"], CONFIG["img_size"], is_train=True)[0][0]
        ev = PreextractedVideoDataset([(str(d / "v"), 0)], CONFIG["seq_len"], CONFIG["img_size"], is_train=False)[0][0]
        diff = float((tr - ev).abs().max())
        return diff < 1e-5, f"max |train(no aug) - eval| = {diff:.2e}"
    finally:
        CONFIG.update(old)
        shutil.rmtree(d, ignore_errors=True)


def chk_eval_frame_indices_deterministic():
    d = Path(tempfile.mkdtemp())
    try:
        (d / "v").mkdir()
        for i in range(32):
            cv2.imwrite(str(d / "v" / f"frame_{i:03d}.jpg"), np.full((16, 16, 3), i * 7, np.uint8))
        ds = PreextractedVideoDataset([(str(d / "v"), 0)], CONFIG["seq_len"], CONFIG["img_size"], is_train=False)
        a, b = ds.planned_frame_files(0), ds.planned_frame_files(0)
        names = [p.name for p in a]
        ok = a == b and names == sorted(names) and len(set(names)) == CONFIG["seq_len"]
        return ok, f"eval picks {CONFIG['seq_len']} distinct, time-ordered, identical crops on every call (first: {names[:3]})"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def chk_fake_prob_and_auc():
    rng = np.random.RandomState(1)
    y = np.r_[np.zeros(200), np.ones(200)].astype(int)
    z = np.r_[rng.randn(200) - 1.5, rng.randn(200) + 1.5]
    p = sigmoid_np(z)
    a_sk, a_ind, a_flip = safe_roc_auc(y, p), independent_auc(y, z), safe_roc_auc(y, 1 - p)
    m = compute_metrics(y, p, 0.5)
    ok = abs(a_sk - a_ind) < 1e-9 and a_sk > 0.9 and abs(a_sk + a_flip - 1) < 1e-9 and m["tp"] + m["fn"] == 200 and m["recall_fake"] == m["sensitivity"]
    return bool(ok), f"sklearn AUC {a_sk:.6f} == independent rank AUC {a_ind:.6f}; AUC(1-p)={a_flip:.4f}; positive class = fake (tp+fn=200)"


class _TinyVid(nn.Module):
    """Small differentiable video model used only to test the attack / Grad-CAM plumbing quickly on CPU."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 4, 3, padding=1)
        self.fc = nn.Linear(4, 1)

    def forward(self, x):
        b, t = x.shape[:2]
        f = torch.relu(self.conv(x.flatten(0, 1))).mean(dim=(2, 3))
        return self.fc(f).view(b, t).mean(dim=1)


def fgsm_perturbation_stats(x0, x_adv, eps_pixel):
    """Pixel-space statistics of a perturbation: actual L-inf (/255), mean |delta| (/255), share of elements moved by exactly eps, share clipped."""
    dpix = ((x_adv - x0) * _STD.to(x0.device)).abs()
    e = float(eps_pixel)
    moved = (dpix > 1e-7)
    full = (dpix >= e - 1e-6)
    return {"linf_255": float(dpix.max()) * 255, "mean_abs_255": float(dpix.mean()) * 255, "eps_255": e * 255,
            "frac_at_full_eps": float(full.float().mean()), "frac_clipped": float((moved & ~full).float().mean()),
            "frac_unmoved": float((~moved).float().mean())}


def chk_fgsm_bounds_and_finiteness():
    torch.manual_seed(0)
    m = _TinyVid().eval()
    pix = 0.02 + 0.96 * torch.rand(2, 3, 3, 24, 24)
    x = (pix - _MEAN) / _STD
    y = torch.tensor([0, 1])
    worst, finite = 0.0, True
    for e in (0.25, 1, 4, 16):
        xa, info = fgsm_attack(m, x, y, e / 255, return_info=True)
        st = fgsm_perturbation_stats(x, xa, e / 255)
        worst = max(worst, st["linf_255"] - e)
        pa = xa * _STD + _MEAN
        finite &= bool(torch.isfinite(xa).all() and torch.isfinite(info["grad"]).all() and float(pa.min()) >= -1e-6 and float(pa.max()) <= 1 + 1e-6)
    logits = m(x)
    loss = Fn.binary_cross_entropy_with_logits(logits, y.float())
    return bool(worst <= 1e-4 and finite and torch.isfinite(loss) and logits.shape == (2,)), f"max L-inf excess {worst:.2e}/255; pixels in [0,1]; loss/logits/grad finite={finite}; output shape {tuple(logits.shape)}"


def chk_model_output_shape():
    mdl = globals().get("model")
    if mdl is None:
        return True, "main model not built yet (checked by Cell 4's shape verification)"
    mdl.eval()
    with torch.no_grad():
        z = mdl(torch.randn(2, CONFIG["seq_len"], 3, CONFIG["img_size"], CONFIG["img_size"], device=CONFIG["device"]))
    return bool(z.shape == (2,) and torch.isfinite(z).all()), f"logits shape {tuple(z.shape)}, finite"


# ---------------- real-data checks ----------------
def preflight_sequences(records, n_decode=48, seed=0, name=""):
    """Every record: >= seq_len crops. A random sample of n_decode records is fully loaded through the eval dataset and verified (shape / dtype
    range / finiteness / deterministic frame plan / identical reload)."""
    short = [f for f, _ in records if len(list(Path(f).glob("*.jpg"))) < CONFIG["seq_len"]]
    if short:
        return False, f"{name}: {len(short)} records have fewer than seq_len crops, e.g. {short[:2]}"
    ds = PreextractedVideoDataset(records, CONFIG["seq_len"], CONFIG["img_size"], is_train=False)
    idxs = random.Random(seed).sample(range(len(records)), min(n_decode, len(records)))
    for i in idxs:
        x, y = ds[i]
        ok, d = check_sequence_tensor(x)
        if not ok:
            return False, f"{name}: record {i} {d}"
        if int(y) != records[i][1]:
            return False, f"{name}: label mismatch at {i}"
    x1, _ = ds[idxs[0]]
    x2, _ = ds[idxs[0]]
    return bool(torch.equal(x1, x2)), f"{name}: all {len(records)} records have >= {CONFIG['seq_len']} crops; {len(idxs)} sampled sequences valid and reload-identical"


def chk_real_data():
    out = []
    labs_ok = all(("/real/" in f.replace("\\", "/")) == (l == 0) and ("/fake/" in f.replace("\\", "/")) == (l == 1) for f, l in extracted_records)
    out.append(("class mapping: crop folder (real/fake) == label", labs_ok))
    vp = [record_info[f]["video_path"] for f, _ in extracted_records]
    out.append(("no video used twice in train/val/test", len(vp) == len(set(vp))))
    fps = defaultdict(set)
    if CONFIG.get("dedupe_videos", True):
        for f, _ in extracted_records:
            fps[video_fingerprint(record_info[f]["video_path"])].add(record_info[f]["split"])
    out.append(("no byte-identical video in two splits", all(len(s) == 1 for s in fps.values()) if fps else True))
    ids = {s: {i for f, _ in split_records[s] for i in record_info[f]["ids"]} for s in SPLITS}
    out.append(("no identity / source id shared between splits", not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"])))
    return bool(all(o[1] for o in out)), "; ".join(f"{n}={'ok' if v else 'FAIL'}" for n, v in out)


print("AUTOMATED CHECKS (v10)")
print(" -- synthetic-data checks")
run_check("dataset discovery, class mapping, identity parsing (v9 char-level id regression)", chk_discovery_and_mapping)
run_check("identity-disjoint split: no leakage, refinement not worse than random", chk_split_leakage)
run_check("duplicate / label-conflict detection", chk_duplicates)
run_check("reproducible stratified selection (seeded, class-balanced, round-robin methods)", chk_reproducible_selection)
run_check("budget-limited selection keeps ALL manipulation methods (v9 truncation regression)", chk_alphabetical_truncation_gone)
run_check("frame decoding: pool length, linspace positions, short / unreadable videos", chk_frame_decoding)
run_check("face crop: fallback is flagged, square, inside the frame", chk_face_crop_geometry)
run_check("RGB conversion and ImageNet normalisation", chk_rgb_and_normalisation)
run_check("train preprocessing (augmentation off) == eval preprocessing", chk_train_eval_preprocessing_identical)
run_check("eval frame plan deterministic, ordered, distinct", chk_eval_frame_indices_deterministic)
run_check("P(fake) / ROC-AUC: positive class = fake, equals independent rank AUC", chk_fake_prob_and_auc)
run_check("FGSM L-inf bound in pixel space, [0,1] clipping, finite loss / grads / outputs", chk_fgsm_bounds_and_finiteness)
run_check("main model output shape [B] and finite", chk_model_output_shape)
print(" -- real-data checks")
run_check("records: class mapping, no repeated video, no duplicate / identity leakage across splits", chk_real_data)
for _n, _r in (("train", train_records), ("val", val_records), ("test", test_records)):
    run_check(f"sequences ({_n}): count, shape, channel order, value range, frame plan", lambda r=_r, n=_n: preflight_sequences(r, name=n))
_fb = FACE_DIAG[FACE_DIAG.fallback_rate > CONFIG["face_fallback_warn_rate"]] if "FACE_DIAG" in globals() and len(FACE_DIAG) else []
run_check("face-crop fallback rate below the warning level in every dataset x class", lambda: (len(_fb) == 0, f"{len(_fb)} groups above {CONFIG['face_fallback_warn_rate']:.0%}"), critical=False)
atomic_json_save(CHECKS, os.path.join(OUT_DIR, "automated_checks_v10.json"))
_crit = [c for c in CHECKS if c["status"] == "FAIL"]
print(f"\n{sum(c['status'] == 'PASS' for c in CHECKS)} passed, {len(_crit)} critical failures, {sum(c['status'] == 'WARN' for c in CHECKS)} warnings.")
assert not _crit, f"CRITICAL automated checks failed -> stop: {[c['check'] for c in _crit]}"


In [ ]:
# ============================================================
# CELL 6: Training machinery (clean control + FGSM adversarial training), resumable, with stability guards (v9)
# ============================================================
# One shared train_epoch / run_fit is used for BOTH models so the control and the FGSM-trained model differ ONLY in the
# adversarial component. Design decisions (all documented here, all logged per epoch):
#   * Batch composition: `adv_ratio` of every batch is replaced by its FGSM version, the rest stays clean (configurable;
#     0.5 = half clean / half adversarial). The perturbed samples keep their TRUE labels as targets.
#   * Label leakage: (a) the perturbation is crafted from the TRUE label only when adv_label_source == "true" (standard
#     Goodfellow-style FGSM-AT); "pred" crafts it from the model's own prediction so no label information reaches the
#     perturbation. (b) Cell 15 measures the leaking signature directly (FGSM accuracy above clean accuracy; true-label vs
#     pred-label FGSM; transfer attack). (c) Test labels are never used for any decision (GuardedLoader).
#   * BatchNorm: all BN layers use their (frozen) running statistics during FGSM-AT (model.bn_eval=True). The attack, the
#     clean half and the adversarial half therefore see exactly the same network and the BN statistics cannot drift towards
#     adversarial inputs. The clean-trained control keeps the v6/v7 behaviour (only unfrozen BN layers update).
#   * EMA: the averaged weights are what is validated / saved / evaluated; the attack always runs on the LIVE model during
#     training and on the evaluated model at test time. verify_ema() checks finiteness and that EMA != live after training.
#   * Numerical safety (v9): a non-finite fp16 loss is first RE-COMPUTED in fp32 on the same batch (v8 simply skipped 3 steps and could not say
#     why); only if fp32 is non-finite too is the step skipped. Every event is logged (max |logit|, which part was non-finite, input finite?).
#     10 consecutive skips abort. GradScaler overflow skips are detected and the LR scheduler is NOT advanced for a skipped optimizer step.
#   * Gradient clipping (v9): AdamW is invariant to a UNIFORM rescaling of the gradient, so v8's fixed clip at 1.0 (66-76% of steps clipped, mean
#     norm 5-7) did almost nothing useful. "adaptive" clipping clips only spikes (> mult x running mean norm); rate and threshold are logged.
#   * Anti-hedging (v9): adversarial logit pairing Huber(z_adv, stopgrad(z_clean)) + eps ramp + adversarial-ratio ramp + lower LR + EMA 0.995.
#   * Early stopping (v9): never before `early_stop_min_epochs`, and only if the gates failed several epochs in a row AND the raw score is not rising.

criterion = nn.BCEWithLogitsLoss()          # plain BCE: validation, attacks and the FGSM-AT losses


class SmoothBCEWithLogits(nn.Module):
    def __init__(self, smoothing=0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        t = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return Fn.binary_cross_entropy_with_logits(logits, t)


train_criterion = SmoothBCEWithLogits(CONFIG["label_smoothing"])
train_labels_arr = np.array([r[1] for r in train_records])
print(f"Train label distribution -> real={int((train_labels_arr == 0).sum())}, fake={int((train_labels_arr == 1).sum())}")


class ModelEMA:
    """Exponential moving average of weights and BN statistics; `module` is what gets validated / checkpointed / evaluated."""
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        self.module.lstm.flatten_parameters()
        self.decay = decay
        self.n = 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1.0 + self.n) / (10.0 + self.n))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])


@torch.no_grad()
def verify_ema(ema, model):
    """EMA sanity: all finite; after >1 update it differs from the live weights (otherwise it is not averaging)."""
    finite = all(torch.isfinite(v).all().item() for v in ema.module.state_dict().values() if v.dtype.is_floating_point)
    live = model.state_dict()
    dist = max(float((v - live[k]).abs().max()) for k, v in ema.module.state_dict().items()
               if v.dtype.is_floating_point and v.numel() > 1)
    return {"ema_finite": bool(finite), "ema_max_abs_diff_to_live": dist, "ema_updates": ema.n,
            "ema_differs_from_live": bool(dist > 0 or ema.n <= 1)}


def build_optimizer(model, lr_scale=1.0):
    groups = model.backbone_param_groups(backbone_lr=CONFIG["backbone_lr"] * lr_scale, head_lr=CONFIG["head_lr"] * lr_scale)
    return torch.optim.AdamW(groups, lr=CONFIG["head_lr"] * lr_scale, weight_decay=CONFIG["weight_decay"])


def build_scheduler(optimizer, total_steps, warmup_steps=0, min_ratio=0.02):
    """PER-STEP linear warm-up then cosine decay to min_ratio x peak. Stepped AFTER optimizer.step() and only when the
    optimizer actually stepped."""
    total_steps = max(int(total_steps), 1)
    warmup_steps = int(warmup_steps)

    def lr_lambda(step):
        if warmup_steps > 0 and step < warmup_steps:
            return max(1e-3, (step + 1) / warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * min(p, 1.0)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def config_snapshot():
    return {k: (str(v) if isinstance(v, (torch.device, Path)) else v) for k, v in CONFIG.items()}


def make_meta(kind, extra=None):
    m = {"kind": kind, "split_fingerprint": SPLIT_FINGERPRINT, "arch_fingerprint": ARCH_FINGERPRINT,
         "torch": torch.__version__, "seed": CONFIG["seed"], "config": config_snapshot()}
    m.update(extra or {})
    return m


def check_provenance(ckpt, kind):
    """Reuse of a checkpoint is allowed ONLY if it was trained on this exact split and architecture."""
    meta = ckpt.get("meta")
    if meta is None:
        return False, "checkpoint has no provenance metadata (legacy file)"
    if meta.get("kind") != kind:
        return False, f"checkpoint kind is {meta.get('kind')!r}, expected {kind!r}"
    if meta.get("split_fingerprint") != SPLIT_FINGERPRINT:
        return False, f"split fingerprint {meta.get('split_fingerprint')} != current {SPLIT_FINGERPRINT}"
    if meta.get("arch_fingerprint") != ARCH_FINGERPRINT:
        return False, f"architecture fingerprint {meta.get('arch_fingerprint')} != current {ARCH_FINGERPRINT}"
    return True, "ok"


def _clip_gradients(params, cfg, state):
    """Global-norm clipping. fixed: clip at cfg['norm'].  adaptive: clip at max(norm_floor, mult x running mean of the clipped norm), i.e. only
    genuine spikes are clipped. Returns (total norm before clipping, threshold used).
    Why: AdamW's update size is invariant to a UNIFORM rescaling of the gradient, so clipping every step (v8: 66-76% of steps clipped at 1.0
    while the mean norm was 5-7) changes nothing except the relative weight of steps; its only useful job is to stop single spikes from
    poisoning Adam's second-moment estimate. Adaptive clipping does exactly that and its rate is logged."""
    if cfg is None:
        return torch.nn.utils.clip_grad_norm_(params, 1e9), float("inf")
    if cfg["mode"] == "fixed":
        thr = cfg["norm"]
    else:
        ema = state.get("clip_ema")
        thr = max(cfg["norm"], cfg["mult"] * ema) if ema is not None else 1e9
    gn = torch.nn.utils.clip_grad_norm_(params, thr)
    if cfg["mode"] != "fixed" and torch.isfinite(gn):
        g = float(min(gn, thr))
        state["clip_ema"] = g if state.get("clip_ema") is None else 0.98 * state["clip_ema"] + 0.02 * g
    return gn, thr


def train_epoch(model, loader, optimizer, scaler, use_amp, scheduler, ema, adv_cfg=None, grad_clip_cfg=None,
                max_steps=None, desc="Training", step_state=None):
    """One epoch (optionally capped at `max_steps` batches). adv_cfg=None -> clean training (label-smoothed BCE).
    adv_cfg -> FGSM adversarial training: the first n_adv videos of every batch are replaced by their FGSM version (eps ramped, optional
    per-sample random eps), the rest stays clean; every video is used exactly once per step.
    loss = clean_w*BCE(clean part) + adv_w*BCE(adversarial part) + alp_w*Huber(z_adv, stopgrad(z_clean of the same video))."""
    model.train()
    step_state = step_state if step_state is not None else {"step": 0}
    S = dict(loss=0.0, clean_loss=0.0, adv_loss=0.0, alp=0.0, n=0, n_adv=0, n_clean=0, correct_clean=0, correct_adv=0,
             grad_norm=0.0, clipped=0, stepped=0, nan_skips=0, fp32_retries=0, clip_thr=0.0, thr_n=0, eps_t=0.0, ratio_t=0.0)
    nan_events, ga_vals, consecutive_bad = [], [], 0
    reset_attack_diag()
    n_batches = min(len(loader), max_steps) if max_steps else len(loader)
    t0 = time.time()
    attack_amp = bool(adv_cfg.get("attack_amp", use_amp)) and use_amp if adv_cfg else False

    for bi, (videos, labels) in enumerate(tqdm(loader, desc=desc, leave=False, total=n_batches)):
        if max_steps and bi >= max_steps:
            break
        videos = videos.to(CONFIG["device"], non_blocking=True)
        labels = labels.to(CONFIG["device"], non_blocking=True)
        bs = videos.size(0)
        yf = labels.float()
        n_adv, z_ref = 0, None
        if adv_cfg is not None:
            st = step_state["step"]
            ratio_t = adv_cfg["ratio_start"] + (adv_cfg["ratio_end"] - adv_cfg["ratio_start"]) * min(1.0, st / max(1, adv_cfg["ratio_ramp_steps"]))
            n_adv = int(round(ratio_t * bs))
            if ratio_t > 0:
                n_adv = max(1, n_adv)
            eps_t = adv_cfg["eps"] * min(1.0, (st + 1) / max(1, adv_cfg["ramp_steps"]))
            S["eps_t"], S["ratio_t"] = eps_t, ratio_t
            eps_arg = eps_t
            if adv_cfg.get("random_eps") and n_adv > 0:
                eps_arg = (eps_t * torch.rand(n_adv, device=videos.device)).view(-1, 1, 1, 1, 1)
        step_state["step"] += 1

        if n_adv > 0:
            adv, info = fgsm_attack(model, videos[:n_adv], labels[:n_adv], eps_arg, use_amp=attack_amp,
                                    label_source=adv_cfg["label_source"], return_info=True)
            z_ref = info["logits"].float()                     # clean logits of the attacked videos (attack mode, no grad)
            if adv_cfg.get("ga_every") and step_state["step"] % adv_cfg["ga_every"] == 0:
                ga_vals.append(gradient_alignment(model, videos[:n_adv], labels[:n_adv], eps_t, use_amp=attack_amp))
            model.train()
            inp = torch.cat([videos[n_adv:], adv], dim=0)          # [clean part ; adversarial part]
        else:
            inp = videos
        n_clean = bs - n_adv

        def compute_losses(lg):
            if adv_cfg is None:
                c = train_criterion(lg, yf)
                zero = torch.zeros((), device=lg.device)
                return c, c, zero, zero
            cl = criterion(lg[:n_clean], yf[n_adv:]) if n_clean > 0 else torch.zeros((), device=lg.device)
            al = criterion(lg[n_clean:], yf[:n_adv]) if n_adv > 0 else torch.zeros((), device=lg.device)
            alp = (Fn.huber_loss(lg[n_clean:], z_ref, delta=adv_cfg["alp_delta"]) if (n_adv > 0 and adv_cfg["alp_w"] > 0)
                   else torch.zeros((), device=lg.device))
            if n_clean > 0 and n_adv > 0:
                tot = adv_cfg["clean_w"] * cl + adv_cfg["adv_w"] * al + adv_cfg["alp_w"] * alp
            else:
                tot = (cl if n_adv == 0 else al + adv_cfg["alp_w"] * alp)
            return tot, cl, al, alp

        optimizer.zero_grad(set_to_none=True)
        ctx = torch.amp.autocast("cuda") if use_amp else nullcontext()
        with ctx:
            logits = model(inp)
        logits = logits.float()
        loss, clean_loss, adv_loss, alp_loss = compute_losses(logits)
        amp_step = use_amp

        if not torch.isfinite(loss):
            ev = {"step": step_state["step"], "n_adv": n_adv, "max_abs_logit": float(torch.nan_to_num(logits.detach(), nan=1e9).abs().max()),
                  "nonfinite_logits": int((~torch.isfinite(logits)).sum()), "input_nonfinite": bool(not torch.isfinite(inp).all()),
                  "clean_part_finite": bool(torch.isfinite(clean_loss)), "adv_part_finite": bool(torch.isfinite(adv_loss)),
                  "amp": bool(use_amp), "fp32_retry_finite": None}
            if CONFIG["nan_fp32_retry"] and use_amp:        # was it an fp16 overflow? re-compute this very batch in fp32
                logits32 = model(inp).float()
                loss32, c32, a32, p32 = compute_losses(logits32)
                ev["fp32_retry_finite"] = bool(torch.isfinite(loss32))
                if torch.isfinite(loss32):
                    logits, loss, clean_loss, adv_loss, alp_loss, amp_step = logits32, loss32, c32, a32, p32, False
                    S["fp32_retries"] += 1
            if len(nan_events) < 20:
                nan_events.append(ev)
            if not torch.isfinite(loss):
                S["nan_skips"] += 1
                consecutive_bad += 1
                optimizer.zero_grad(set_to_none=True)
                if consecutive_bad >= 10:
                    raise FloatingPointError("10 consecutive non-finite losses -- training aborted (see nan_events in the log).")
                continue
        consecutive_bad = 0

        params = [p for p in model.parameters() if p.requires_grad]
        stepped = True
        if amp_step:
            scale_before = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            gn, thr = _clip_gradients(params, grad_clip_cfg, step_state)
            scaler.step(optimizer)
            scaler.update()
            stepped = scaler.get_scale() >= scale_before            # a lowered scale means the step was skipped (inf/nan grads)
        else:
            loss.backward()
            gn, thr = _clip_gradients(params, grad_clip_cfg, step_state)
            if torch.isfinite(gn):
                optimizer.step()
            else:
                stepped = False
        if stepped:
            S["stepped"] += 1
            if scheduler is not None:
                scheduler.step()                                    # AFTER optimizer.step()
            if ema is not None:
                ema.update(model)
        gnf = float(gn) if torch.isfinite(gn) else float("nan")
        if gnf == gnf:
            S["grad_norm"] += gnf
            if thr < 1e6:                                   # the first steps have no running mean yet (no clipping): not averaged
                S["clip_thr"] += thr; S["thr_n"] += 1
            S["clipped"] += int(gnf > thr)

        with torch.no_grad():
            pred = (logits > 0).long()
            S["n"] += bs; S["n_adv"] += n_adv; S["n_clean"] += n_clean
            S["loss"] += float(loss) * bs
            S["clean_loss"] += float(clean_loss) * max(n_clean, 0)
            S["adv_loss"] += float(adv_loss) * n_adv
            S["alp"] += float(alp_loss) * n_adv
            if adv_cfg is None:
                S["correct_clean"] += int((pred == labels).sum())
            else:
                S["correct_clean"] += int((pred[:n_clean] == labels[n_adv:]).sum())
                S["correct_adv"] += int((pred[n_clean:] == labels[:n_adv]).sum())
        del logits, loss, inp
        if (bi + 1) % 25 == 0:
            free_cuda_memory()

    n_steps = max(S["stepped"], 1)
    m = {"loss": S["loss"] / max(S["n"], 1), "clean_loss": S["clean_loss"] / max(S["n_clean"], 1),
         "acc": S["correct_clean"] / max(S["n_clean"], 1), "grad_norm": S["grad_norm"] / n_steps,
         "clip_rate": S["clipped"] / n_steps, "clip_thr": S["clip_thr"] / max(S["thr_n"], 1), "nan_skips": S["nan_skips"],
         "fp32_retries": S["fp32_retries"], "optimizer_steps": S["stepped"], "seconds": time.time() - t0,
         "zero_grad_samples": ATTACK_DIAG["zero_grad_samples"], "attacked_samples": ATTACK_DIAG["samples"],
         "nan_events": nan_events}
    if adv_cfg is not None:
        m.update({"adv_loss": S["adv_loss"] / max(S["n_adv"], 1), "alp_loss": S["alp"] / max(S["n_adv"], 1),
                  "adv_acc": S["correct_adv"] / max(S["n_adv"], 1), "eps_t_255": S["eps_t"] * 255, "ratio_t": S["ratio_t"],
                  "grad_align": float(np.mean(ga_vals)) if ga_vals else float("nan")})
    return m


def validate_epoch(model, loader, loss_fn, threshold=0.5, use_amp=False):
    model.eval()
    probs, labs, run, n = [], [], 0.0, 0
    with torch.no_grad():
        for videos, labels in tqdm(loader, desc="Validating", leave=False):
            videos = videos.to(CONFIG["device"], non_blocking=True)
            labels = labels.to(CONFIG["device"], non_blocking=True)
            ctx = torch.amp.autocast("cuda") if (use_amp and CONFIG["device"].type == "cuda") else nullcontext()
            with ctx:
                z = model(videos)
            run += float(loss_fn(z.float(), labels.float())) * videos.size(0)
            n += videos.size(0)
            probs.extend(torch.sigmoid(z.float()).cpu().numpy())
            labs.extend(labels.cpu().numpy())
    m = compute_metrics(labs, probs, threshold=threshold)
    m["loss"] = run / n
    return m


# ---------------- generic resumable fit loop ----------------
def run_fit(model, tag, kind, epochs, loader, steps_per_epoch, lr_scale, warmup_steps, adv_cfg, eval_fn, select_fn,
            ema_decay, grad_clip_cfg, best_path=None, soft_path=None, resume_path=None, patience_no_improve=None,
            early_stop=None, extra_meta=None, unfreeze=True):
    """Trains `model`; validates the EMA weights every epoch with eval_fn(ema_module)->dict; select_fn(history)->
    {'score','raw','gate_ok'} decides the best epoch (validation only). Saves (atomically) the best GATED checkpoint, the best SOFT
    (raw score, gate ignored) checkpoint if `soft_path` is given, and every epoch a resume file; resumes from it when it exists AND
    passes the provenance check. early_stop = {min_epochs, bad_epochs, no_gain}: stop only after min_epochs, with `bad_epochs`
    consecutive failed gates AND no raw-score gain over the earlier best (v8 stopped at epoch 3 while still improving)."""
    if unfreeze:
        model.unfreeze_last_n_blocks(CONFIG["unfreeze_last_n_blocks"])
    steps_per_epoch = steps_per_epoch or len(loader)
    total_steps = epochs * steps_per_epoch
    optimizer = build_optimizer(model, lr_scale)
    scheduler = build_scheduler(optimizer, total_steps, warmup_steps, CONFIG["min_lr_ratio"])
    scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
    ema = ModelEMA(model, ema_decay)
    history, best, step_state, start_epoch = [], {"score": -1e9, "epoch": None, "soft_raw": -1e9, "soft_epoch": None}, {"step": 0}, 0
    no_improve = bad_run = 0

    if resume_path and Path(resume_path).exists():
        ck = torch.load(resume_path, map_location=CONFIG["device"], weights_only=False)
        ok, why = check_provenance(ck, kind + "_resume")
        if ok and ck["meta"].get("tag") == tag and ck["meta"].get("epochs") == epochs:
            model.load_state_dict(ck["model"]); ema.module.load_state_dict(ck["ema"]); ema.n = ck["ema_n"]
            optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"])
            if scaler is not None and ck.get("scaler"):
                scaler.load_state_dict(ck["scaler"])
            history, best, step_state = ck["history"], ck["best"], ck["step_state"]
            start_epoch, no_improve, bad_run = ck["epoch"], ck["no_improve"], ck["bad_run"]
            print(f"[{tag}] RESUMED from {resume_path} after epoch {start_epoch} (provenance ok).")
        else:
            print(f"[{tag}] resume file ignored: {why if not ok else 'tag/epochs differ'} -> starting from scratch.")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'=' * 64}\n[{tag}] {epochs} epochs x {steps_per_epoch} steps | trainable params {trainable:,} | "
          f"warm-up {warmup_steps} steps | LR scale {lr_scale}\n{'=' * 64}")
    stopped_early = False
    for epoch in range(start_epoch + 1, epochs + 1):
        t_ep = time.time()
        tm = train_epoch(model, loader, optimizer, scaler, USE_AMP, scheduler, ema, adv_cfg, grad_clip_cfg,
                         max_steps=steps_per_epoch, desc=f"{tag} ep{epoch}", step_state=step_state)
        evm = eval_fn(ema.module)
        em = verify_ema(ema, model)
        row = {"epoch": epoch, "lr": optimizer.param_groups[0]["lr"], **{f"train_{k}": v for k, v in tm.items()}, **evm, **em}
        history.append(row)
        sel = select_fn(history)
        row.update({"select_raw": sel["raw"], "select_score": sel["score"], "gate_ok": sel["gate_ok"]})
        extra = (f", adv {tm['adv_loss']:.4f}, ALP {tm['alp_loss']:.4f}, adv-acc {tm['adv_acc']:.3f}, GA cos {tm['grad_align']:.3f}, "
                 f"eps {tm['eps_t_255']:.2f}/255, adv-ratio {tm['ratio_t']:.2f}") if adv_cfg else f", acc {tm['acc']:.3f}"
        print(f"[{tag}] epoch {epoch}/{epochs} [{time.time() - t_ep:.0f}s]  train loss {tm['loss']:.4f} (clean {tm['clean_loss']:.4f}{extra}) | "
              f"gradnorm {tm['grad_norm']:.2f} clip {tm['clip_rate']:.0%} (thr~{tm['clip_thr']:.1f}) nan-skips {tm['nan_skips']} "
              f"fp32-retries {tm['fp32_retries']} zero-grad {tm['zero_grad_samples']}")
        if tm["nan_events"]:
            print(f"      NON-FINITE EVENTS: {tm['nan_events'][:3]}")
        print("      " + " | ".join(f"{k}={evm[k]:.4f}" for k in evm if isinstance(evm[k], float) and k.startswith(
            ("val_clean_auc", "val_fgsm_mean", "val_fgsm_min", "val_fgsm_target_auc", "val_fgsm_target_sens", "val_fgsm_target_spec",
             "val_clean_logit_std", "val_auc", "val_acc")))
              + f" | select raw={sel['raw']:.4f} score={sel['score']:.4f} gate={'OK' if sel['gate_ok'] else 'FAIL'}")
        if not em["ema_finite"]:
            raise FloatingPointError("EMA weights contain NaN/Inf")

        meta_row = {k: v for k, v in row.items() if not isinstance(v, (dict, list))}
        if sel["gate_ok"] and sel["score"] > best["score"]:
            best.update({"score": float(sel["score"]), "epoch": epoch})
            no_improve = 0
            if best_path:
                atomic_torch_save({"model_state_dict": ema.module.state_dict(), "val_score": float(sel["score"]), "epoch": epoch,
                                   "weights": "ema", "val_row": meta_row, "meta": make_meta(kind, extra_meta)}, best_path)
                print(f"      --> best {tag} checkpoint saved (epoch {epoch}, selection score {sel['score']:.4f})")
        else:
            no_improve += 1
        if soft_path and sel["raw"] > best["soft_raw"]:
            best.update({"soft_raw": float(sel["raw"]), "soft_epoch": epoch})
            atomic_torch_save({"model_state_dict": ema.module.state_dict(), "val_score": float("nan"), "epoch": epoch,
                               "weights": "ema (soft best: gate ignored)", "val_row": meta_row,
                               "meta": make_meta(kind, {**(extra_meta or {}), "gate_never_passed": True})}, soft_path)
        bad_run = 0 if sel["gate_ok"] else bad_run + 1

        if resume_path:
            atomic_torch_save({"model": model.state_dict(), "ema": ema.module.state_dict(), "ema_n": ema.n,
                               "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                               "scaler": scaler.state_dict() if scaler is not None else None, "history": history, "best": best,
                               "step_state": step_state, "epoch": epoch, "no_improve": no_improve, "bad_run": bad_run,
                               "meta": make_meta(kind + "_resume", {"tag": tag, "epochs": epochs})}, resume_path)
        if patience_no_improve and no_improve >= patience_no_improve:
            print(f"[{tag}] early stopping: no improvement for {no_improve} epochs.")
            break
        if early_stop and epoch >= early_stop["min_epochs"] and bad_run >= early_stop["bad_epochs"]:
            raws = [h["select_raw"] for h in history]
            recent, earlier = max(raws[-3:]), (max(raws[:-3]) if len(raws) > 3 else -1.0)
            if recent < earlier + early_stop["no_gain"]:
                print(f"[{tag}] EARLY STOP at epoch {epoch}: gate failed {bad_run} epochs in a row and the raw score gained "
                      f"{recent - earlier:+.4f} (< {early_stop['no_gain']}) over the earlier best -> validation FGSM metrics unusable.")
                stopped_early = True
                break
    del optimizer, scheduler, scaler
    free_cuda_memory()
    best["stopped_early_bad_gate"] = stopped_early
    best["ema_module"] = ema.module
    return history, best


# ---------------- clean control ----------------
def clean_eval_fn(ema_module):
    m = validate_epoch(ema_module, val_loader, criterion, threshold=0.5, use_amp=USE_AMP)
    return {"val_loss": m["loss"], "val_accuracy": m["accuracy"], "val_f1": m["f1"], "val_roc_auc": m["roc_auc"]}


def clean_select_fn(history):
    a = history[-1]["val_roc_auc"]
    ok = a == a
    return {"raw": a if ok else -1.0, "score": a if ok else -1.0, "gate_ok": ok}


def train_clean_model(seed, best_path, resume_path):
    set_global_seed(seed)
    m = build_model()
    hist, best = run_fit(m, f"clean-seed{seed}", "clean", CONFIG["phase2_epochs"], train_loader, None, 1.0,
                         int(CONFIG["warmup_epochs"] * len(train_loader)), None, clean_eval_fn, clean_select_fn,
                         CONFIG["ema_decay"], {"mode": "fixed", "norm": CONFIG["grad_clip_norm"]}, best_path=best_path, resume_path=resume_path,
                         patience_no_improve=CONFIG["early_stopping_patience"], extra_meta={"seed": seed})
    return m, hist, best


print("Training machinery ready: ModelEMA/verify_ema, train_epoch (clean or FGSM-AT), resumable run_fit, provenance checks.")

In [ ]:
# ============================================================
# CELL 7: Clean-trained CONTROL model, validation curves, calibration and validation-only threshold
# ============================================================
# The control is trained exactly like v6/v7. A checkpoint from an earlier session may be re-used ONLY if its provenance
# (split fingerprint + architecture fingerprint + kind) matches this run; otherwise it is ignored and the model is retrained.
# Nothing here reads the test set.

def _history_path(ckpt_path):
    return str(Path(ckpt_path).with_suffix("")) + "_history.json"


def load_clean_if_valid(path):
    if not path or not Path(path).exists():
        return None
    ck = torch.load(path, map_location=CONFIG["device"], weights_only=False)
    ok, why = check_provenance(ck, "clean")
    if not ok:
        print(f"Clean checkpoint {path} REJECTED ({why}) -> will retrain.")
        return None
    print(f"Clean checkpoint {path} accepted (epoch {ck['epoch']}, val ROC-AUC {ck['val_score']:.4f}, provenance ok).")
    return ck


def discover_clean_checkpoint():
    """v9: look for best_clean_model.pth under /kaggle/input (<=4 levels) so an uploaded checkpoint is picked up without editing CONFIG."""
    if not CONFIG.get("auto_discover_inputs") or not os.path.isdir("/kaggle/input"):
        return None
    base = "/kaggle/input"
    for root, dirs, files in os.walk(base):
        if root[len(base):].count(os.sep) >= 4:
            dirs[:] = []
        if "best_clean_model.pth" in files:
            return os.path.join(root, "best_clean_model.pth")
    return None


_disc = None if CONFIG["clean_checkpoint_input"] else discover_clean_checkpoint()
if _disc:
    print(f"Auto-discovered clean checkpoint: {_disc}")
    CONFIG["clean_checkpoint_input"] = _disc
T_CLEAN0 = time.time()
_ck = load_clean_if_valid(CONFIG["clean_checkpoint_input"]) or (
    load_clean_if_valid(CONFIG["save_clean_checkpoint"]) if Path(_history_path(CONFIG["save_clean_checkpoint"])).exists() else None)
clean_model = build_model()
if _ck is not None:
    import shutil
    clean_model.load_state_dict(_ck["model_state_dict"])
    clean_best_val_auc = float(_ck["val_score"])
    if CONFIG["clean_checkpoint_input"] and str(Path(CONFIG["clean_checkpoint_input"]).resolve()) != str(Path(CONFIG["save_clean_checkpoint"]).resolve()):
        shutil.copy(CONFIG["clean_checkpoint_input"], CONFIG["save_clean_checkpoint"])
    _hp = _history_path(CONFIG["clean_checkpoint_input"] or CONFIG["save_clean_checkpoint"])
    clean_history = json.load(open(_hp)) if Path(_hp).exists() else [
        {"epoch": int(_ck["epoch"]), "lr": float("nan"), "train_loss": float("nan"), "train_acc": float("nan"),
         "val_loss": float("nan"), "val_accuracy": float("nan"), "val_f1": float("nan"), "val_roc_auc": clean_best_val_auc}]
    print("Clean training SKIPPED (verified checkpoint re-used).")
else:
    with timed("clean_training"):
        clean_model, clean_history, _best = train_clean_model(CONFIG["seed"], CONFIG["save_clean_checkpoint"], CONFIG["resume_clean_path"])
    clean_best_val_auc = _best["score"]
    atomic_json_save(clean_history, _history_path(CONFIG["save_clean_checkpoint"]))
    del _best
print(f"Clean control ready. Best validation ROC-AUC (EMA weights): {clean_best_val_auc:.4f}   ({(time.time() - T_CLEAN0) / 60:.1f} min)")

# ---- load the BEST-validation weights (what is evaluated is exactly what was selected) ----
free_cuda_memory()
clean_checkpoint = torch.load(CONFIG["save_clean_checkpoint"], map_location=CONFIG["device"], weights_only=False)
clean_model.load_state_dict(clean_checkpoint["model_state_dict"])
clean_model.eval()
assert abs(clean_checkpoint["val_score"] - clean_best_val_auc) < 1e-9, "checkpoint val AUC != tracked best val AUC"
clean_df = pd.DataFrame(clean_history)
print(f"Loaded best clean checkpoint: epoch {clean_checkpoint['epoch']} of {int(clean_df['epoch'].max())} run, val ROC-AUC {clean_checkpoint['val_score']:.4f}")


# ---- calibration + threshold (VALIDATION ONLY) ----
def select_threshold(labels, probs):
    """EER threshold (FPR = FNR), macro-F1-optimal threshold on a grid, and the one selected by CONFIG['threshold_method']."""
    fpr, tpr, thr = roc_curve(labels, probs)
    fnr = 1.0 - tpr
    i = int(np.nanargmin(np.abs(fpr - fnr)))
    eer_thr = float(np.clip(thr[i], 0.0, 1.0))
    grid = np.arange(CONFIG["threshold_search_min"], CONFIG["threshold_search_max"] + 1e-9, CONFIG["threshold_search_step"])
    f1s = [compute_metrics(labels, probs, float(t))["f1"] for t in grid]
    f1_thr = float(grid[int(np.argmax(f1s))])
    selected = eer_thr if CONFIG["threshold_method"] == "eer" else f1_thr
    return {"eer": eer_thr, "f1": f1_thr, "selected": selected, "eer_value": float((fpr[i] + fnr[i]) / 2)}


def calibrate_and_threshold(model, tag=""):
    labels, logits = get_logits(model, val_loader, None, desc=f"val {tag}")
    T = fit_temperature(logits, labels)
    probs = sigmoid_np(logits, T)
    thr = select_threshold(labels, probs)
    print(f"[{tag}] temperature T={T:.3f}  ECE {expected_calibration_error(labels, sigmoid_np(logits, 1.0)):.4f}->"
          f"{expected_calibration_error(labels, probs):.4f}  | threshold EER={thr['eer']:.3f} (EER={thr['eer_value']:.3f}) "
          f"F1-opt={thr['f1']:.3f} -> using {CONFIG['threshold_method']}={thr['selected']:.3f}")
    return T, thr["selected"], thr


clean_T, clean_selected_threshold, clean_thr_info = calibrate_and_threshold(clean_model, "Clean-trained")

# ---- convention audit (positive class = FAKE; tells 'AUC ~ 0' = inverted ranking apart from a label/sign bug) ----
_lab, _z = get_logits(clean_model, val_loader, None, desc="audit")
_auc_plain = safe_roc_auc(_lab, _z)
_auc_inv = safe_roc_auc(_lab, -_z)
_zf, _zr = float(np.mean(_z[_lab == 1])), float(np.mean(_z[_lab == 0]))
print(f"Convention audit (validation): mean logit of FAKE={_zf:+.3f}, REAL={_zr:+.3f}; AUC(score=+logit)={_auc_plain:.4f}, "
      f"AUC(score=-logit)={_auc_inv:.4f}")
assert _zf > _zr and _auc_plain > 0.5 and abs(_auc_plain + _auc_inv - 1.0) < 1e-6, "convention bug: label 1 (fake) must receive the HIGHER logit and sklearn must be fed P(fake)"
print("Verified: positive class = FAKE (label 1); roc_auc_score(labels, P(fake)); AUC(-logit) = 1 - AUC(+logit).")

# ---- clean-trained model on the VALIDATION adversarial subset: reference for pilot gates and acceptance criteria ----
_eps_ref = sorted({0.0, CONFIG["target_eval_eps"] * EPS_UNIT, *[e * EPS_UNIT for e in CONFIG["val_eps_grid"]]})
_l, _zz = fgsm_sweep_logits(clean_model, val_adv_loader, _eps_ref, desc="clean model val FGSM")
_tgt = CONFIG["target_eval_eps"] * EPS_UNIT
_aucs = [safe_roc_auc(_l, _zz[e * EPS_UNIT]) for e in CONFIG["val_eps_grid"]]
BASELINE_VAL = {"clean_val_auc_full": float(clean_best_val_auc),
                "clean_val_auc_subset": safe_roc_auc(_l, _zz[0.0]),
                "clean_val_logit_std": float(np.std(_zz[0.0])),
                "fgsm_target_auc": safe_roc_auc(_l, _zz[_tgt]),
                "fgsm_target_acc": float(np.mean((_zz[_tgt] > 0).astype(int) == _l)),
                "fgsm_target_sens_at0": float(np.mean(_zz[_tgt][_l == 1] > 0)),
                "fgsm_target_spec_at0": float(np.mean(_zz[_tgt][_l == 0] <= 0)),
                "fgsm_mean_auc": float(np.nanmean(_aucs)), "fgsm_min_auc": float(np.nanmin(_aucs))}
print(f"Clean-trained reference on the validation FGSM subset: clean AUC {BASELINE_VAL['clean_val_auc_subset']:.4f}, "
      f"FGSM {CONFIG['target_eval_eps']:g}/255 AUC {BASELINE_VAL['fgsm_target_auc']:.4f} acc {BASELINE_VAL['fgsm_target_acc']:.4f} "
      f"(fake recall {BASELINE_VAL['fgsm_target_sens_at0']:.3f}, real specificity {BASELINE_VAL['fgsm_target_spec_at0']:.3f} at logit 0); "
      f"mean/min AUC over val eps {BASELINE_VAL['fgsm_mean_auc']:.3f}/{BASELINE_VAL['fgsm_min_auc']:.3f}")

In [ ]:
# ============================================================
# CELL 7b: END-TO-END ATTACK AUDIT of the clean-trained model (validation data only)
# ============================================================
# Question: is the clean model's collapse under tiny FGSM (v8: 0% accuracy / AUC 0.000 at <1/255) a REAL fragility or a pipeline bug?
# Each check isolates one possible bug. Nothing here touches the test set or changes any model.
#   A1 class-wise logits / probabilities / sensitivity / specificity under attack (what does the ranking actually look like?)
#   A2 epsilon units: the realised max |x_adv - x| in 0-255 pixel units must equal eps*255 (unless clipped at 0/1)
#   A3 positive controls: FLIPPED-label FGSM must RAISE AUC (gradient direction is meaningful), a RANDOM-sign perturbation of the
#      same size must barely move it (fragility is adversarial, not noise), true-label FGSM must lower it
#   A4 loss linearity: actual loss increase vs first-order prediction <g, delta> (FGSM is only a good attack if ~linear)
#   A5 AMP vs fp32 attack: sign agreement of the input gradient and resulting AUC -> decides whether training attacks may use fp16
#   A6 8-bit quantisation: does rounding the adversarial image to integer pixels change the result? (eps < 0.5/255 is sub-quantum)

def _audit_batch(n_total):
    per = max(1, n_total // 2)
    xs, ys, cnt = [], [], {0: 0, 1: 0}
    for v, l in val_adv_loader:
        keep = [i for i in range(len(l)) if cnt[int(l[i])] < per]
        for i in keep:
            cnt[int(l[i])] += 1
        if keep:
            xs.append(v[keep]); ys.append(l[keep])
        if min(cnt.values()) >= per:
            break
    return torch.cat(xs), torch.cat(ys)


def _pix_np(x_norm):
    return (x_norm.detach().cpu() * _STD + _MEAN).numpy()


def run_attack_audit(model, tag, n_videos, eps_list_255=(0.25, 0.5, 1.0, 2.0), decide_amp=False):
    dev = CONFIG["device"]
    model.eval()
    X, Y = _audit_batch(n_videos)
    bs = max(1, CONFIG["eval_attack_batch_size"])
    eps_list = [e * EPS_UNIT for e in eps_list_255]
    keys = ["clean", "true", "flipped", "random", "amp", "true_q"]
    Z = {k: {e: [] for e in eps_list} for k in keys}
    unit_err, lin_pred, lin_act, sign_agree = [], {e: [] for e in eps_list}, {e: [] for e in eps_list}, []
    amp_ok = (dev.type == "cuda")
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs].to(dev), Y[i:i + bs].to(dev)
        g_true, z0 = fgsm_grad(model, xb, yb, use_amp=False, label_source="true")
        g_flip, _ = fgsm_grad(model, xb, yb, use_amp=False, label_source="flipped")
        g_amp = fgsm_grad(model, xb, yb, use_amp=True, label_source="true")[0] if amp_ok else g_true
        sign_agree.append(float((torch.sign(g_amp) == torch.sign(g_true)).float().mean()))
        with torch.no_grad():
            base_loss = Fn.binary_cross_entropy_with_logits(z0, yb.float(), reduction="none")
        for e in eps_list:
            with torch.no_grad():
                Z["clean"][e].append(z0.float().cpu().numpy())
            xa = fgsm_apply(xb, g_true, e)
            pa, p0 = _pix_np(xa), _pix_np(xb)
            unit_err.append(float(np.abs(pa - p0).max() * 255.0 / (e * 255.0)))
            with torch.no_grad():
                za = model(xa).float()
                Z["true"][e].append(za.cpu().numpy())
                Z["flipped"][e].append(model(fgsm_apply(xb, g_flip, e)).float().cpu().numpy())
                Z["random"][e].append(model(fgsm_apply(xb, torch.randn_like(xb), e)).float().cpu().numpy())
                Z["amp"][e].append(model(fgsm_apply(xb, g_amp, e)).float().cpu().numpy())
                Z["true_q"][e].append(model(quantize_to_8bit(xa)).float().cpu().numpy())
                adv_loss = Fn.binary_cross_entropy_with_logits(za, yb.float(), reduction="none")
            # first-order prediction: <grad of summed BCE, delta> per video (delta = realised perturbation)
            lin_pred[e].append(((g_true * (xa - xb)).flatten(1).sum(1)).cpu().numpy())
            lin_act[e].append((adv_loss - base_loss).cpu().numpy())
    Yn = Y.numpy().astype(int)
    cat = lambda k, e: np.concatenate(Z[k][e])
    res = {"tag": tag, "n_videos": int(len(Yn)), "eps_255": list(eps_list_255)}
    base_auc = safe_roc_auc(Yn, cat("clean", eps_list[0]))
    res["clean_auc"] = base_auc
    res["clean_logit_mean_fake"] = float(cat("clean", eps_list[0])[Yn == 1].mean())
    res["clean_logit_mean_real"] = float(cat("clean", eps_list[0])[Yn == 0].mean())
    rows = []
    for e255, e in zip(eps_list_255, eps_list):
        zt = cat("true", e)
        rows.append({
            "eps_255": e255,
            "A1_logit_fake": float(zt[Yn == 1].mean()), "A1_logit_real": float(zt[Yn == 0].mean()),
            "A1_prob_fake_given_fake": float(sigmoid_np(zt[Yn == 1]).mean()), "A1_prob_fake_given_real": float(sigmoid_np(zt[Yn == 0]).mean()),
            "A1_fake_recall@0": float((zt[Yn == 1] > 0).mean()), "A1_real_spec@0": float((zt[Yn == 0] <= 0).mean()),
            "A2_max_perturbation_ratio": float(np.max(unit_err)),     # <= 1.0 (clipping can only shrink it)
            "A3_auc_true": safe_roc_auc(Yn, zt), "A3_auc_flipped": safe_roc_auc(Yn, cat("flipped", e)),
            "A3_auc_random": safe_roc_auc(Yn, cat("random", e)),
            "A4_linearity_ratio": float(np.mean(np.concatenate(lin_act[e])) / (np.mean(np.concatenate(lin_pred[e])) + 1e-12)),
            "A5_auc_amp": safe_roc_auc(Yn, cat("amp", e)),
            "A6_auc_quantised": safe_roc_auc(Yn, cat("true_q", e)),
            "A6_sub_quantum": bool(e255 < 0.5),
        })
    res["rows"] = rows
    res["A5_sign_agreement_amp_vs_fp32"] = float(np.mean(sign_agree))
    df = pd.DataFrame(rows)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    print(f"\n=== ATTACK AUDIT [{tag}]  ({res['n_videos']} validation videos, clean AUC {base_auc:.3f}, mean logit fake {res['clean_logit_mean_fake']:+.2f} / real {res['clean_logit_mean_real']:+.2f}) ===")
    print(df.round(3).to_string(index=False))
    print(f"A5 input-gradient sign agreement AMP vs fp32: {res['A5_sign_agreement_amp_vs_fp32']:.3f}" + ("" if amp_ok else "  (CPU: AMP not available, identical by construction)"))
    # ---- verdicts ----
    v = {}
    v["A1_convention_ok"] = bool(res["clean_logit_mean_fake"] > res["clean_logit_mean_real"])
    v["A2_units_ok"] = bool(max(r["A2_max_perturbation_ratio"] for r in rows) <= 1.0 + 1e-3)
    v["A3_flipped_raises_auc"] = bool(all(r["A3_auc_flipped"] >= base_auc - 0.02 for r in rows))
    v["A3_true_lowers_auc"] = bool(rows[-1]["A3_auc_true"] < base_auc - 0.05)
    v["A3_random_noise_harmless"] = bool(all(r["A3_auc_random"] >= base_auc - 0.1 for r in rows))
    v["A4_linearity_ok"] = bool(all(0.3 <= r["A4_linearity_ratio"] <= 3.0 for r in rows))
    v["A5_amp_fp32_agree"] = bool(res["A5_sign_agreement_amp_vs_fp32"] >= 0.95
                                  and max(abs(r["A5_auc_amp"] - r["A3_auc_true"]) for r in rows) <= 0.03)
    v["A6_quantisation_changes_little"] = bool(all(abs(r["A6_auc_quantised"] - r["A3_auc_true"]) <= 0.05
                                                   for r in rows if not r["A6_sub_quantum"]))
    res["verdicts"] = v
    for k, b in v.items():
        print(f"  {'PASS' if b else 'CHECK'}  {k}")
    if v["A1_convention_ok"] and v["A2_units_ok"] and v["A3_flipped_raises_auc"] and v["A3_true_lowers_auc"] and v["A3_random_noise_harmless"]:
        print("  => The collapse is a REAL adversarial fragility: right sign convention, right epsilon units, a meaningful gradient direction "
              "(flipped labels help, true labels hurt) and random noise of the same size is harmless. It is not a pipeline bug.")
    else:
        print("  => At least one end-to-end check did not behave as expected: read the CHECK lines before trusting any attacked number.")
    if any(r["A6_sub_quantum"] for r in rows):
        print("  NOTE: eps < 0.5/255 cannot survive saving as an 8-bit image; those rows measure the float-input threat model only.")
    return res


AUDIT = run_attack_audit(clean_model, "clean-trained", CONFIG["audit_n_videos"], decide_amp=True)
atomic_json_save(AUDIT, os.path.join(OUT_DIR, "fgsm_attack_audit.json"))
if CONFIG["train_attack_amp"] is None:
    CONFIG["train_attack_amp"] = bool(USE_AMP and AUDIT["verdicts"]["A5_amp_fp32_agree"])
    print(f"\ntrain_attack_amp decided by audit A5 -> {CONFIG['train_attack_amp']} "
          f"({'fp16 attack is faithful' if CONFIG['train_attack_amp'] else 'training attacks use fp32 so the perturbation matches the fp32 evaluation attack'})")

In [ ]:
# ============================================================
# CELL 8: FGSM validation metrics, selection rule (with hedge gate) and the recipe PILOT (validation data only)
# ============================================================
# v8 failure being fixed: the FGSM-trained model "hedged" (logits ~ 0, loss = ln 2), which makes a label-aware FGSM invert the ranking
# (AUC < 0.5) and puts fakes below the clean-EER threshold. v9 therefore (1) detects hedging directly (clean logit std gate),
# (2) pilots four recipes (fixed / random epsilon, with / without adversarial logit pairing) and (3) selects with mean/min FGSM AUC over
# several validation epsilons instead of a single one. Training epsilon != evaluation epsilon; test data is never touched here.

W = CONFIG["select_weights"]
VAL_EPS = [e * EPS_UNIT for e in CONFIG["val_eps_grid"]]
TARGET = CONFIG["target_eval_eps"] * EPS_UNIT


def make_fgsm_eval_fn(full_val=True):
    """eval_fn(ema_module) -> validation metrics: clean (full val or subset), FGSM AUC at every val epsilon (mean/min), target-epsilon
    sensitivity/specificity at logit 0, hedge indicator (clean logit std), label-leak margin. FGSM attacks run in fp32."""
    eps_list = sorted({0.0, float(TARGET), *[float(e) for e in VAL_EPS]})

    def eval_fn(module):
        out = {}
        if full_val:
            m = validate_epoch(module, val_loader, criterion, threshold=0.5, use_amp=False)
            out.update({"val_loss": m["loss"], "val_accuracy": m["accuracy"], "val_f1": m["f1"], "val_clean_auc": m["roc_auc"]})
        lab, zz = fgsm_sweep_logits(module, val_adv_loader, eps_list, desc="val FGSM")
        acc = lambda z: float(np.mean((z > 0).astype(int) == lab))
        out["val_subset_clean_auc"] = safe_roc_auc(lab, zz[0.0])
        out["val_subset_clean_acc"] = acc(zz[0.0])
        if not full_val:
            out["val_clean_auc"] = out["val_subset_clean_auc"]
        aucs = [safe_roc_auc(lab, zz[float(e)]) for e in VAL_EPS]
        out["val_fgsm_mean_auc"] = float(np.mean(aucs)) if all(a == a for a in aucs) else float("nan")
        out["val_fgsm_min_auc"] = float(np.min(aucs)) if all(a == a for a in aucs) else float("nan")
        zt = zz[float(TARGET)]
        out["val_fgsm_target_auc"] = safe_roc_auc(lab, zt)
        out["val_fgsm_target_acc"] = acc(zt)
        out["val_fgsm_target_sens"] = float(np.mean(zt[lab == 1] > 0))      # fake recall at logit 0
        out["val_fgsm_target_spec"] = float(np.mean(zt[lab == 0] <= 0))     # real specificity at logit 0
        out["val_clean_logit_std"] = float(np.std(zz[0.0]))
        out["val_leak_margin"] = float(max(acc(zz[float(e)]) for e in VAL_EPS) - out["val_subset_clean_acc"])
        return out
    return eval_fn


def make_fgsm_select_fn(full_val=True):
    """VALIDATION-ONLY. raw = 0.4*clean AUC + 0.4*mean FGSM AUC + 0.2*min FGSM AUC. GATES (all must hold): finite; clean AUC >= clean-trained
    baseline - select_max_clean_auc_drop; mean AND target FGSM AUC >= select_min_fgsm_auc; clean logit std >= select_min_logit_std (no hedge);
    FGSM accuracy <= clean accuracy + label-leak margin. score = mean(raw of last K epochs) - penalty*std (rewards stable epochs)."""
    base = BASELINE_VAL["clean_val_auc_full" if full_val else "clean_val_auc_subset"]

    def select_fn(history):
        r = history[-1]
        vals = [r["val_clean_auc"], r["val_fgsm_mean_auc"], r["val_fgsm_min_auc"], r["val_fgsm_target_auc"]]
        finite = all(v == v for v in vals)
        raw = (W["clean_auc"] * r["val_clean_auc"] + W["fgsm_mean_auc"] * r["val_fgsm_mean_auc"]
               + W["fgsm_min_auc"] * r["val_fgsm_min_auc"]) if finite else -1.0
        r["select_raw"] = raw
        gate = bool(finite and r["val_clean_auc"] >= base - CONFIG["select_max_clean_auc_drop"]
                    and r["val_fgsm_mean_auc"] >= CONFIG["select_min_fgsm_auc"]
                    and r["val_fgsm_target_auc"] >= CONFIG["select_min_fgsm_auc"]
                    and r["val_clean_logit_std"] >= CONFIG["select_min_logit_std"]
                    and r["val_leak_margin"] <= CONFIG["accept_max_label_leak_margin"])
        raws = [h.get("select_raw", raw) for h in history[-CONFIG["select_smooth_epochs"]:]]
        score = float(np.mean(raws) - CONFIG["select_stability_penalty"] * (np.std(raws) if len(raws) > 1 else 0.0))
        return {"raw": float(raw), "score": score, "gate_ok": gate}
    return select_fn


def fgsm_adv_cfg(eps, ramp_epochs, steps_per_epoch, random_eps=False, alp_w=None, label_source=None):
    return {"eps": eps, "ramp_steps": int(ramp_epochs * steps_per_epoch), "random_eps": random_eps,
            "ratio_start": CONFIG["adv_ratio_start"], "ratio_end": CONFIG["adv_ratio"],
            "ratio_ramp_steps": int(CONFIG["adv_ratio_ramp_epochs"] * steps_per_epoch),
            "label_source": label_source or CONFIG["adv_label_source"],
            "clean_w": CONFIG["fgsm_clean_loss_weight"], "adv_w": CONFIG["fgsm_adv_loss_weight"],
            "alp_w": CONFIG["alp_weight"] if alp_w is None else alp_w, "alp_delta": CONFIG["alp_huber_delta"],
            "ga_every": CONFIG["ga_check_every"], "attack_amp": bool(CONFIG["train_attack_amp"])}


def fgsm_clip_cfg():
    return {"mode": CONFIG["fgsm_grad_clip_mode"], "norm": CONFIG["fgsm_grad_clip_norm"], "mult": CONFIG["fgsm_grad_clip_mult"]}


def new_fgsm_model_from_clean(path=None):
    m = build_model()
    m.load_state_dict(torch.load(path or CONFIG["save_clean_checkpoint"], map_location=CONFIG["device"], weights_only=False)["model_state_dict"])
    m.bn_eval = CONFIG["fgsm_bn_eval"]
    return m


PILOT_ROWS = []
PILOT_ALL_FAILED = False
if CONFIG["fgsm_train_eps"] is not None:
    print(f"fgsm_train_eps fixed in CONFIG ({fmt_eps(CONFIG['fgsm_train_eps'])}/255): pilot skipped.")
else:
    base_c = BASELINE_VAL["clean_val_auc_subset"]
    for (vname, v_eps, v_rand, v_alp) in CONFIG["pilot_variants"]:
        set_global_seed(CONFIG["seed"])
        pm = new_fgsm_model_from_clean()
        spe = CONFIG["pilot_steps"]
        with timed("pilot"):
            hist, _ = run_fit(pm, f"pilot {vname}", "fgsm_pilot", CONFIG["pilot_epochs"], train_loader_fgsm, spe,
                              CONFIG["fgsm_lr_scale"], 30, fgsm_adv_cfg(v_eps * EPS_UNIT, CONFIG["pilot_ramp_epochs"], spe, v_rand, v_alp),
                              make_fgsm_eval_fn(full_val=False), make_fgsm_select_fn(full_val=False), CONFIG["ema_decay_fgsm"],
                              fgsm_clip_cfg(), best_path=None, resume_path=None)
        r = hist[-1]
        healthy = bool(r["val_clean_auc"] >= base_c - CONFIG["pilot_min_clean_auc_drop"]
                       and r["val_clean_logit_std"] >= CONFIG["pilot_min_clean_logit_std"]
                       and r["val_fgsm_target_auc"] >= CONFIG["select_min_fgsm_auc"]
                       and r["val_leak_margin"] <= CONFIG["accept_max_label_leak_margin"]
                       and math.isfinite(r["train_adv_loss"]) and r["train_nan_skips"] == 0)
        PILOT_ROWS.append({"variant": vname, "train_eps_255": v_eps, "random_eps": v_rand, "alp_weight": v_alp,
                           "val_clean_auc": r["val_clean_auc"], "val_fgsm_mean_auc": r["val_fgsm_mean_auc"],
                           "val_fgsm_target_auc": r["val_fgsm_target_auc"], "val_fgsm_target_sens": r["val_fgsm_target_sens"],
                           "val_fgsm_target_spec": r["val_fgsm_target_spec"], "val_clean_logit_std": r["val_clean_logit_std"],
                           "val_clean_acc": r["val_subset_clean_acc"], "adv_loss_first": hist[0]["train_adv_loss"],
                           "adv_loss_last": r["train_adv_loss"], "grad_align": r["train_grad_align"],
                           "nan_skips": r["train_nan_skips"], "fp32_retries": r["train_fp32_retries"], "healthy": healthy,
                           "compare_score": 0.4 * r["val_clean_auc"] + 0.4 * r["val_fgsm_mean_auc"] + 0.2 * r["val_fgsm_min_auc"],
                           "sec_per_step": r["train_seconds"] / max(r["train_optimizer_steps"], 1)})
        del pm, hist
        free_cuda_memory()
    pilot_df = pd.DataFrame(PILOT_ROWS)
    pilot_df.to_csv(os.path.join(OUT_DIR, "fgsm_pilot.csv"), index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    print(f"\nPILOT RESULTS (validation subset). Clean-trained reference: clean AUC {base_c:.4f}, clean logit std "
          f"{BASELINE_VAL['clean_val_logit_std']:.2f}, FGSM@{CONFIG['target_eval_eps']:g}/255 AUC {BASELINE_VAL['fgsm_target_auc']:.4f}")
    print(pilot_df.round(4).to_string(index=False))
    ok = pilot_df[pilot_df["healthy"]]
    PILOT_ALL_FAILED = ok.empty
    if PILOT_ALL_FAILED:
        pick = pilot_df.sort_values("compare_score", ascending=False).iloc[0]
        print("\n!! NO variant passed the pilot health gates (no hedge, clean AUC kept, FGSM AUC above chance, finite). Falling back to the variant "
              "with the best validation compare score; the main run is still gated. Treat all later results as a FAILURE analysis.")
    else:
        pick = ok.sort_values("compare_score", ascending=False).iloc[0]
    CONFIG["fgsm_train_eps"] = float(pick["train_eps_255"]) * EPS_UNIT
    CONFIG["adv_random_eps"] = bool(pick["random_eps"])
    CONFIG["alp_weight"] = float(pick["alp_weight"])
    print(f"\nCHOSEN recipe: '{pick['variant']}' -> train eps {pick['train_eps_255']:g}/255, random-per-sample {bool(pick['random_eps'])}, "
          f"ALP {pick['alp_weight']:g} (compare score {pick['compare_score']:.4f})")

    sps = float(pilot_df["sec_per_step"].median())
    ep_sec = sps * CONFIG["fgsm_steps_per_epoch"] + 120
    elapsed_h = (time.time() - NOTEBOOK_START) / 3600
    n_extra = len(CONFIG["extra_seeds"])
    # reserve: evaluation (~0.7 h) + (optionally) extra seeds are time-guarded later, so they do not shrink the main run
    avail_h = CONFIG["time_budget_hours"] - elapsed_h - 0.8
    fit = int(max(0, avail_h) * 3600 // ep_sec)
    CONFIG["fgsm_epochs_effective"] = int(max(CONFIG["early_stop_min_epochs"], min(CONFIG["fgsm_epochs"], fit)))
    print(f"\nRuntime: {sps:.2f} s/step -> ~{ep_sec / 60:.1f} min per FGSM epoch; elapsed {elapsed_h:.2f} h; {avail_h:.2f} h left before the "
          f"evaluation reserve -> main run = {CONFIG['fgsm_epochs_effective']} epochs (~{CONFIG['fgsm_epochs_effective'] * ep_sec / 3600:.2f} h). "
          f"Extra seeds ({n_extra}) start only if the time guard in Cell 18 says they fit.")

In [ ]:
# ============================================================
# CELL 9: FGSM adversarial training (main run) with validation-only checkpoint selection
# ============================================================
# Starts from the clean checkpoint (never from scratch). Resumable: re-running the cell continues after the last finished
# epoch if resume_fgsm.pth exists and matches this split/architecture/config.

FGSM_EPS = CONFIG["fgsm_train_eps"]
assert FGSM_EPS is not None
set_global_seed(CONFIG["seed"])
fgsm_model = new_fgsm_model_from_clean()
E = CONFIG["fgsm_epochs_effective"]
SPE = CONFIG["fgsm_steps_per_epoch"]
print(f"FGSM-AT (v9): training eps {fmt_eps(FGSM_EPS)}/255 (random-per-sample={CONFIG['adv_random_eps']}), target eval eps {CONFIG['target_eval_eps']:g}/255, "
      f"adv ratio {CONFIG['adv_ratio_start']}->{CONFIG['adv_ratio']}, ALP {CONFIG['alp_weight']}, label source '{CONFIG['adv_label_source']}', "
      f"clip {CONFIG['fgsm_grad_clip_mode']}, attack AMP {CONFIG['train_attack_amp']}, epochs {E} x {SPE} steps")
SOFT_PATH = os.path.join(OUT_DIR, "soft_best_fgsm_model.pth")

_V2_REUSE = v2_reuse_fgsm_main()          # v2: verified v1 checkpoint is re-used (never silently: provenance + recipe are checked)
with timed("fgsm_training"):
    if _V2_REUSE is not None:
        fgsm_history, fgsm_best = _V2_REUSE
    else:
        fgsm_history, fgsm_best = run_fit(
            fgsm_model, "FGSM-AT", "fgsm", E, train_loader_fgsm, SPE, CONFIG["fgsm_lr_scale"], CONFIG["fgsm_warmup_steps"],
            fgsm_adv_cfg(FGSM_EPS, CONFIG["fgsm_eps_ramp_epochs"], SPE, CONFIG["adv_random_eps"], CONFIG["alp_weight"]),
            make_fgsm_eval_fn(True), make_fgsm_select_fn(True), CONFIG["ema_decay_fgsm"], fgsm_clip_cfg(),
            best_path=CONFIG["save_fgsm_checkpoint"], soft_path=SOFT_PATH, resume_path=CONFIG["resume_fgsm_path"],
            early_stop={"min_epochs": CONFIG["early_stop_min_epochs"], "bad_epochs": CONFIG["early_stop_bad_epochs"],
                        "no_gain": CONFIG["early_stop_no_gain"]},
            extra_meta={"train_eps": FGSM_EPS, "adv_ratio": CONFIG["adv_ratio"], "label_source": CONFIG["adv_label_source"],
                        "alp_weight": CONFIG["alp_weight"], "random_eps": CONFIG["adv_random_eps"]})
if _V2_REUSE is None:
    atomic_json_save(fgsm_history, os.path.join(OUT_DIR, "fgsm_training_history.json"))

FGSM_GATE_NEVER_PASSED = fgsm_best["epoch"] is None
if FGSM_GATE_NEVER_PASSED:
    print("\n!! No epoch passed ALL validation gates (clean AUC kept, FGSM AUC above chance, no hedge, no label leak). Using the SOFT-best "
          "checkpoint (best raw validation score, gates ignored) so the failure can be analysed; the acceptance criteria will report FAILURE.")
    import shutil
    if Path(SOFT_PATH).exists():
        shutil.copy(SOFT_PATH, CONFIG["save_fgsm_checkpoint"])
    else:
        atomic_torch_save({"model_state_dict": fgsm_best["ema_module"].state_dict(), "val_score": float("nan"),
                           "epoch": int(fgsm_history[-1]["epoch"]), "weights": "ema (gate never passed)",
                           "meta": make_meta("fgsm", {"train_eps": FGSM_EPS, "gate_never_passed": True})}, CONFIG["save_fgsm_checkpoint"])
del fgsm_model
free_cuda_memory()

# ---- load the SELECTED checkpoint and verify it ----
fgsm_ckpt = torch.load(CONFIG["save_fgsm_checkpoint"], map_location=CONFIG["device"], weights_only=False)
ok, why = check_provenance(fgsm_ckpt, "fgsm")
assert ok, f"FGSM checkpoint provenance failed: {why}"
fgsm_eval_model = build_model()
fgsm_eval_model.load_state_dict(fgsm_ckpt["model_state_dict"])
fgsm_eval_model.eval()
assert all(torch.isfinite(v).all() for v in fgsm_eval_model.state_dict().values() if v.dtype.is_floating_point)
_sel_row = next(r for r in fgsm_history if r["epoch"] == fgsm_ckpt["epoch"])
print(f"\nSelected FGSM checkpoint: epoch {fgsm_ckpt['epoch']} of {len(fgsm_history)} run (selection score {fgsm_ckpt['val_score']:.4f}); "
      f"val clean AUC {_sel_row['val_clean_auc']:.4f}, FGSM@{CONFIG['target_eval_eps']:g}/255 AUC {_sel_row['val_fgsm_target_auc']:.4f}, "
      f"mean/min FGSM AUC {_sel_row['val_fgsm_mean_auc']:.4f}/{_sel_row['val_fgsm_min_auc']:.4f}, clean logit std {_sel_row['val_clean_logit_std']:.2f}")
fgsm_T, fgsm_selected_threshold, fgsm_thr_info = calibrate_and_threshold(fgsm_eval_model, "FGSM-trained")
print(f"Thresholds (validation-selected) -- clean-trained: {clean_selected_threshold:.3f} | FGSM-trained: {fgsm_selected_threshold:.3f}")
fgsm_df = pd.DataFrame(fgsm_history)

# ---- ATTACK-AWARE threshold (validation only) ----
# v8 applied the CLEAN-EER threshold (0.666) to attacked scores; attacked logits are compressed towards 0, so most fakes fell below it
# (16/200 detected). The attack-aware threshold is the EER point of the POOLED validation scores {clean} U {FGSM at the target eps}.
# It is reported NEXT TO the clean threshold, for both models, and fixed here, before the test set is unlocked.
def attack_aware_threshold(model, T, tag):
    lab, zz = fgsm_sweep_logits(model, val_attack_loader, [0.0, float(TARGET)], desc=f"val pooled {tag}")
    pooled_lab = np.concatenate([lab, lab])
    pooled_p = np.concatenate([sigmoid_np(zz[0.0], T), sigmoid_np(zz[float(TARGET)], T)])
    thr = select_threshold(pooled_lab, pooled_p)
    pa = sigmoid_np(zz[float(TARGET)], T)
    print(f"[{tag}] attack-aware EER threshold {thr['eer']:.3f} (pooled EER {thr['eer_value']:.3f}); on attacked validation: fake recall "
          f"{np.mean(pa[lab == 1] >= thr['eer']):.3f}, real specificity {np.mean(pa[lab == 0] < thr['eer']):.3f}  "
          f"(clean-EER thr {'n/a' if tag.startswith('x') else ''}-> recall {np.mean(pa[lab == 1] >= (clean_selected_threshold if tag.startswith('Clean') else fgsm_selected_threshold)):.3f})")
    return float(thr["eer"])


ATTACK_AWARE_THR = {"clean": attack_aware_threshold(clean_model, clean_T, "Clean-trained"),
                    "fgsm": attack_aware_threshold(fgsm_eval_model, fgsm_T, "FGSM-trained")}

In [ ]:
# ============================================================
# CELL 9b (v10): FGSM correctness audit on VALIDATION data (never test): raw scores, independent AUC, perturbation magnitudes,
# controls, per-epsilon diagnostics and an evidence-based diagnosis of the clean model's collapse.
# ============================================================
# Question: is the clean-trained model's collapse at small epsilon (and the partial recovery at large epsilon) a genuine vulnerability, a score
# reversal, a class-mapping / threshold problem, a preprocessing inconsistency or an implementation error? Every candidate has a measurable
# signature; the verdict below is derived from the numbers, not assumed. Epsilon is in PIXEL units (0..1 scale, quoted /255); the normalised-space
# budget is eps / std_c per channel (fgsm_apply), and the result is clipped to the normalised image of [0,1].

def eps_diag_table(tag, labels, zdict, T_, thr, thr_name, eps_list):
    """One row per epsilon with everything the review asks for (accuracy, ROC-AUC (sklearn AND independent), macro F1, balanced accuracy, per-class
    precision / recall, confusion matrix, logit mean/std per class, mean P(fake) per class, predicted-class shares, flips in both directions)."""
    labels = np.asarray(labels).astype(int)
    p0 = sigmoid_np(zdict[0.0], T_)
    c0 = (p0 >= thr).astype(int) == labels
    rows = []
    for e in eps_list:
        z = np.asarray(zdict[e], dtype=np.float64)
        p = sigmoid_np(z, T_)
        m = compute_metrics(labels, p, thr)
        pred = (p >= thr).astype(int)
        c = pred == labels
        real, fake = labels == 0, labels == 1
        rows.append({"model": tag, "threshold_type": thr_name, "threshold": float(thr), "eps_255": e * 255, "n": len(labels),
                     "accuracy": m["accuracy"], "roc_auc_sklearn": m["roc_auc"], "roc_auc_independent": independent_auc(labels, z),
                     "macro_f1": m["f1"], "balanced_accuracy": m["balanced_accuracy"],
                     "precision_real": m["precision_real"], "recall_real": m["recall_real"], "precision_fake": m["precision_fake"], "recall_fake": m["recall_fake"],
                     "tn": m["tn"], "fp": m["fp"], "fn": m["fn"], "tp": m["tp"],
                     "logit_mean_real": float(z[real].mean()), "logit_std_real": float(z[real].std()),
                     "logit_mean_fake": float(z[fake].mean()), "logit_std_fake": float(z[fake].std()),
                     "p_fake_mean_real": float(p[real].mean()), "p_fake_mean_fake": float(p[fake].mean()),
                     "share_pred_fake": float(pred.mean()), "share_pred_real": float(1 - pred.mean()),
                     "correct_to_incorrect": int((c0 & ~c).sum()), "incorrect_to_correct": int((~c0 & c).sum())})
    df = pd.DataFrame(rows)
    df["auc_abs_diff_sklearn_vs_independent"] = (df.roc_auc_sklearn - df.roc_auc_independent).abs()
    return df


def perturbation_audit(model, loader, eps_list_255):
    """Run the attack exactly as in evaluation and measure what it does to the pixels (computed once per batch: sign(grad) is eps-independent)."""
    agg = {e: {"linf": 0.0, "mean_abs": [], "full": [], "clipped": [], "unmoved": []} for e in eps_list_255}
    reset_attack_diag()
    for videos, labels in loader:
        videos, labels = videos.to(CONFIG["device"]), labels.to(CONFIG["device"])
        g, _ = fgsm_grad(model, videos, labels)
        for e in eps_list_255:
            xa = fgsm_apply(videos, g, e / 255)
            st = fgsm_perturbation_stats(videos, xa, e / 255)
            a = agg[e]
            a["linf"] = max(a["linf"], st["linf_255"]); a["mean_abs"].append(st["mean_abs_255"]); a["full"].append(st["frac_at_full_eps"])
            a["clipped"].append(st["frac_clipped"]); a["unmoved"].append(st["frac_unmoved"])
    return pd.DataFrame([{"eps_255": e, "linf_255_max": a["linf"], "within_bound": a["linf"] <= e + 1e-3, "mean_abs_delta_255": float(np.mean(a["mean_abs"])),
                          "frac_elements_at_full_eps": float(np.mean(a["full"])), "frac_elements_clipped_by_[0,1]": float(np.mean(a["clipped"])),
                          "frac_elements_unmoved": float(np.mean(a["unmoved"])),
                          "zero_grad_samples": ATTACK_DIAG["zero_grad_samples"], "nonfinite_grad_samples": ATTACK_DIAG["nonfinite_grad_samples"]} for e, a in agg.items()])


def controls_scores(model, loader, eps_list):
    """Raw logits (same videos / order) for: clean, FGSM true label, FGSM FLIPPED label (a helpful perturbation), random-sign noise of the same L-inf size."""
    model.eval()
    labs, out = [], {k: {float(e): [] for e in eps_list} for k in ("true", "flipped", "random")}
    out["clean"] = []
    vmin, vmax = None, None
    for videos, labels in loader:
        videos, labels = videos.to(CONFIG["device"]), labels.to(CONFIG["device"])
        vmin, vmax = _norm_bounds(videos.device)
        gt, _ = fgsm_grad(model, videos, labels, label_source="true")
        gf, _ = fgsm_grad(model, videos, labels, label_source="flipped")
        with torch.no_grad():
            out["clean"].append(model(videos).float().cpu().numpy())
            for e in eps_list:
                rs = torch.sign(torch.randn_like(videos))
                xr = torch.max(torch.min(videos + _eps_normalized(e, videos.device) * rs, vmax), vmin)
                for k, x in (("true", fgsm_apply(videos, gt, e)), ("flipped", fgsm_apply(videos, gf, e)), ("random", xr)):
                    out[k][float(e)].append(model(x).float().cpu().numpy())
        labs.append(labels.cpu().numpy())
    res = {k: ({e: np.concatenate(v) for e, v in d.items()} if isinstance(d, dict) else np.concatenate(d)) for k, d in out.items()}
    return np.concatenate(labs).astype(int), res


AUDIT_EPS = [e * EPS_UNIT for e in (0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0)]
FGSM_AUDIT = {}
_val_threshold = {"Clean-trained": clean_selected_threshold, "FGSM-trained": fgsm_selected_threshold}
_val_T = {"Clean-trained": clean_T, "FGSM-trained": fgsm_T}
_audit_models = {"Clean-trained": clean_model, "FGSM-trained": fgsm_eval_model}
_raw_dump, _tables, _ptabs = {}, [], []
for tag, mdl in _audit_models.items():
    lab, R = controls_scores(mdl, val_adv_loader, AUDIT_EPS)
    zd = {0.0: R["clean"], **R["true"]}
    _tables.append(eps_diag_table(tag, lab, zd, _val_T[tag], _val_threshold[tag], "val_selected", [0.0] + AUDIT_EPS))
    _raw_dump[f"{tag}|labels"] = lab
    _raw_dump[f"{tag}|clean"] = R["clean"]
    for k in ("true", "flipped", "random"):
        for e in AUDIT_EPS:
            _raw_dump[f"{tag}|{k}|{e * 255:g}"] = R[k][e]
    pa = perturbation_audit(mdl, val_adv_loader, [e * 255 for e in AUDIT_EPS])
    pa.insert(0, "model", tag)
    _ptabs.append(pa)
    FGSM_AUDIT[tag] = {"labels": lab, "R": R}
np.savez_compressed(os.path.join(OUT_DIR, "val_raw_scores_v10.npz"), **_raw_dump)
FGSM_EPS_VAL_TABLE = pd.concat(_tables, ignore_index=True)
FGSM_PERT_TABLE = pd.concat(_ptabs, ignore_index=True)
FGSM_EPS_VAL_TABLE.to_csv(os.path.join(OUT_DIR, "fgsm_eps_diagnostics_val_v10.csv"), index=False)
FGSM_PERT_TABLE.to_csv(os.path.join(OUT_DIR, "fgsm_perturbation_audit_val_v10.csv"), index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("PERTURBATION AUDIT (validation subset; eps and measured magnitudes in units of 1/255 pixel):")
print(FGSM_PERT_TABLE.round(4).to_string(index=False))
print("\nPER-EPSILON DIAGNOSTICS (validation subset, validation-selected threshold):")
print(FGSM_EPS_VAL_TABLE[["model", "eps_255", "accuracy", "roc_auc_sklearn", "roc_auc_independent", "macro_f1", "balanced_accuracy", "recall_real", "recall_fake",
                          "logit_mean_real", "logit_mean_fake", "p_fake_mean_real", "p_fake_mean_fake", "share_pred_fake", "correct_to_incorrect", "incorrect_to_correct"]].round(3).to_string(index=False))

# ---- verdict derived from evidence ----
def _auc(m, k, e):
    d = FGSM_AUDIT[m]
    return independent_auc(d["labels"], d["R"][k][e] if k != "clean" else d["R"]["clean"])


EV = []
for tag in _audit_models:
    d = FGSM_AUDIT[tag]
    lab = d["labels"]
    e_small = AUDIT_EPS[2]       # 1/255
    clean_auc = independent_auc(lab, d["R"]["clean"])
    mean_f, mean_r = d["R"]["clean"][lab == 1].mean(), d["R"]["clean"][lab == 0].mean()
    auc_true, auc_flip, auc_rand = _auc(tag, "true", e_small), _auc(tag, "flipped", e_small), _auc(tag, "random", e_small)
    sk_gap = float(FGSM_EPS_VAL_TABLE[FGSM_EPS_VAL_TABLE.model == tag].auc_abs_diff_sklearn_vs_independent.max())
    pa = FGSM_PERT_TABLE[FGSM_PERT_TABLE.model == tag]
    EV.append({"model": tag, "clean_auc": clean_auc, "mean_logit_fake_minus_real": float(mean_f - mean_r), "auc_fgsm_true@1": auc_true, "auc_fgsm_flipped@1": auc_flip,
               "auc_random_sign@1": auc_rand, "max_auc_gap_sklearn_vs_independent": sk_gap, "all_perturbations_within_bound": bool(pa.within_bound.all()),
               "zero_or_nonfinite_grads": int(pa.zero_grad_samples.max() + pa.nonfinite_grad_samples.max())})
EVDF = pd.DataFrame(EV)
print("\nEVIDENCE:"); print(EVDF.round(4).to_string(index=False))
VERDICT_LINES = []
for r in EV:
    t = r["model"]
    mapping_ok = r["clean_auc"] > 0.5 and r["mean_logit_fake_minus_real"] > 0
    metric_ok = r["max_auc_gap_sklearn_vs_independent"] < 1e-9
    impl_ok = r["all_perturbations_within_bound"] and r["zero_or_nonfinite_grads"] == 0
    attack_works = r["auc_fgsm_true@1"] < r["clean_auc"] - 0.3
    direction_ok = r["auc_fgsm_flipped@1"] > r["clean_auc"] - 0.02 and (r["auc_fgsm_flipped@1"] >= r["auc_fgsm_true@1"] + 0.2 or not attack_works)
    noise_harmless = r["auc_random_sign@1"] > r["clean_auc"] - 0.1
    if not (mapping_ok and metric_ok and impl_ok):
        v = "IMPLEMENTATION / MAPPING PROBLEM: fix before interpreting any robustness number"
    elif attack_works and direction_ok and noise_harmless:
        v = ("GENUINE VULNERABILITY: class mapping and AUC are correct, the perturbation obeys the bound, the true-label gradient lowers AUC, the flipped-label "
             "gradient raises it, and random noise of the same size is harmless. AUC < 0.5 is a ranking REVERSAL caused by the attack, not a label/sign bug")
    elif not attack_works:
        v = "NO COLLAPSE at 1/255 on this model (nothing to explain)"
    else:
        v = "INCONCLUSIVE: some control behaved unexpectedly (see EVIDENCE); do not claim a cause"
    VERDICT_LINES.append(f"{t}: {v}")
print("\nVERDICT (derived from the numbers above):")
for l in VERDICT_LINES:
    print(" -", l)
print(" - Large-epsilon recovery is assessed with the linearity ratio (actual / first-order loss increase) in Cell 14: a ratio far below 1 means the single "
      "linear FGSM step overshoots on a non-linear loss surface, i.e. a WEAKER attack at large eps, not a more robust model.")
atomic_json_save({"evidence": EV, "verdict": VERDICT_LINES, "eps_space": "pixel scale [0,1]; normalised budget = eps/std_c; clipped to normalised [0,1]"},
                 os.path.join(OUT_DIR, "fgsm_correctness_audit_v10.json"))


In [ ]:
# ============================================================
# CELL 10: Freeze the ACCEPTANCE CRITERIA, re-run the unit tests on the trained models, then unlock the test set
# ============================================================
# Everything that decides "did the experiment succeed" is written to acceptance_criteria.json BEFORE the first test video is
# read (the test loader raises until TEST_UNLOCKED is set below). The file stores the thresholds, the selected checkpoint,
# the training / target epsilons and the validation-side evidence that already exists.

print("Unit tests re-run on the SELECTED (EMA) weights of both models:")
for _name, _m in [("clean-trained", clean_model), ("FGSM-trained", fgsm_eval_model)]:
    print(f"-- {_name}")
    run_fgsm_unit_tests(_m)
clean_model.eval(); fgsm_eval_model.eval()

val_row = _sel_row
val_gain = val_row["val_fgsm_target_auc"] - BASELINE_VAL["fgsm_target_auc"]
CRITERIA = {
    "frozen_before_test_access": True,
    "frozen_at_unix": time.time(),
    "split_fingerprint": SPLIT_FINGERPRINT,
    "scope": "FGSM only. Success here says NOTHING about iterative-attack (PGD-type) robustness; PGD was deliberately excluded.",
    "train_eps_255": FGSM_EPS * 255, "target_eps_255": CONFIG["target_eval_eps"],
    "selected_epoch": int(fgsm_ckpt["epoch"]), "gate_never_passed": bool(FGSM_GATE_NEVER_PASSED), "pilot_all_failed": bool(PILOT_ALL_FAILED),
    "thresholds": {"clean_trained": clean_selected_threshold, "fgsm_trained": fgsm_selected_threshold, "rule": CONFIG["threshold_method"] + " on validation",
                   "attack_aware": ATTACK_AWARE_THR, "attack_aware_rule": "EER of pooled validation scores {clean} U {FGSM at target eps}"},
    "recipe": {"random_eps": CONFIG["adv_random_eps"], "alp_weight": CONFIG["alp_weight"], "train_attack_amp": CONFIG["train_attack_amp"],
               "clip_mode": CONFIG["fgsm_grad_clip_mode"], "epochs_run": len(fgsm_history)},
    "criteria": {
        "C1_clean_auc_retained": f"test clean ROC-AUC of FGSM-trained >= {CONFIG['accept_min_test_clean_auc_ratio']} x clean-trained",
        "C2_val_fgsm_auc_target": f"validation FGSM AUC at target eps >= {CONFIG['accept_min_val_fgsm_auc_target']}",
        "C3_val_fgsm_auc_gain": f"validation FGSM AUC at target eps exceeds the clean-trained model's by >= {CONFIG['accept_min_val_fgsm_auc_gain']}",
        "C4_no_label_leak": f"test FGSM accuracy at any eps <= clean accuracy + {CONFIG['accept_max_label_leak_margin']}",
        "C5_paired_gain": "paired-bootstrap 95% CI of [AUC(FGSM-trained) - AUC(clean-trained)] has lower bound > 0 at the target AND the training epsilon (test)",
        "C6_numerics": "no NaN/Inf, zero-gradient samples = 0 during evaluation, and every non-finite TRAINING step rescued by the fp32 retry or skipped (counted)",
        "C7_class_balance_under_attack": f"test, FGSM at target eps, attack-aware threshold: BOTH fake recall and real specificity >= {CONFIG['accept_min_class_recall']} (v8: fake recall 8% at 1/255)",
        "C8_multi_seed": f"the FGSM validation gate passes in >= {CONFIG['accept_min_seeds_passing']} training seeds (evaluated after Cell 18)",
    },
    "validation_evidence_already_known": {
        "C2_value": val_row["val_fgsm_target_auc"], "C2_met": bool(val_row["val_fgsm_target_auc"] >= CONFIG["accept_min_val_fgsm_auc_target"]),
        "C3_value": val_gain, "C3_met": bool(val_gain >= CONFIG["accept_min_val_fgsm_auc_gain"]),
        "val_clean_auc_fgsm_model": val_row["val_clean_auc"], "val_clean_logit_std": val_row["val_clean_logit_std"],
        "val_fgsm_target_sens": val_row["val_fgsm_target_sens"], "val_fgsm_target_spec": val_row["val_fgsm_target_spec"], "val_clean_auc_clean_model": BASELINE_VAL["clean_val_auc_full"],
    },
}
atomic_json_save(CRITERIA, CONFIG["acceptance_json"])
atomic_json_save(config_snapshot(), CONFIG["run_config_json"])
CRITERIA_HASH = hashlib.md5(open(CONFIG["acceptance_json"], "rb").read()).hexdigest()[:12]
print(f"\nAcceptance criteria written to {CONFIG['acceptance_json']} (md5 {CRITERIA_HASH}) BEFORE test access:")
print(json.dumps(CRITERIA["criteria"], indent=2))
print("Validation-side evidence:", json.dumps(CRITERIA["validation_evidence_already_known"], indent=2))
TEST_UNLOCKED["open"] = True
print("\nTEST SET UNLOCKED (first test access happens after this line).")

In [ ]:
# ============================================================
# CELL 11: Full test evaluation -- clean-trained vs FGSM-trained, every evaluation epsilon, same videos, fp32
# ============================================================
# Evaluation epsilons (/255): 0, 0.25, 0.5, 1, 1.5, 2, 4, 8, 12, 16 (NOT the training epsilon). All videos of the identity-disjoint
# test set are used for BOTH models, in the same order (paired tests). Probabilities are temperature-calibrated on validation;
# metrics are reported at the fixed 0.5 AND at the validation-selected threshold. The threshold is never re-tuned under attack.
#
# ASR (attack success rate), for a given threshold:  ASR = #{videos classified correctly when clean AND wrongly under FGSM}
#                                                         / #{videos classified correctly when clean}
#   ASR_real / ASR_fake restrict numerator and denominator to true-real / true-fake videos. "flips" are counts of videos whose
#   decision changes: correct->wrong (the ASR numerator), wrong->correct (recovered), and any prediction change.

EVAL_EPS = [e / 255 for e in CONFIG["eval_eps_grid"]]
MODELS = {"Clean-trained": (clean_model, clean_T, clean_selected_threshold),
          "FGSM-trained": (fgsm_eval_model, fgsm_T, fgsm_selected_threshold)}
THR_AA = {"Clean-trained": ATTACK_AWARE_THR["clean"], "FGSM-trained": ATTACK_AWARE_THR["fgsm"]}   # fixed on validation in Cell 9
# Three threshold types are reported: 0.5, the clean-EER threshold (v8's choice) and the ATTACK-AWARE threshold (EER of pooled clean +
# FGSM@target validation scores). The attack-aware one is the fix for v8's 16/200 fake recall; it is chosen on validation only.
test_attack_loader = make_test_attack_loader()
RAW = {}        # model tag -> (labels, {eps: logits})
reset_attack_diag()
for tag, (m, T_, thr_) in MODELS.items():
    with timed(f"test_eval_{tag}"):
        RAW[tag] = fgsm_sweep_logits(m, test_attack_loader, EVAL_EPS, desc=f"{tag} FGSM sweep")
EVAL_DIAG = dict(ATTACK_DIAG)
assert np.array_equal(RAW["Clean-trained"][0], RAW["FGSM-trained"][0]), "the two models saw different test samples/orders"
test_labels = RAW["Clean-trained"][0]
n_test = len(test_labels)
print(f"Test set: {n_test} videos (real={int((test_labels == 0).sum())}, fake={int((test_labels == 1).sum())}); "
      f"zero-gradient samples {EVAL_DIAG['zero_grad_samples']}/{EVAL_DIAG['samples']}, non-finite {EVAL_DIAG['nonfinite_grad_samples']}")
for e in EVAL_EPS[:1]:
    assert np.allclose(RAW["Clean-trained"][1][0.0], get_logits(clean_model, test_loader, None, desc="clean check")[1], atol=1e-4), \
        "eps=0 sweep logits differ from the plain clean forward pass"


def eval_rows(tag, labels, zdict, T_, thr_sel, thr_aa):
    """One row per (epsilon, threshold type)."""
    rows = []
    _auc_ci_cache = {}
    for thr_name, thr in (("0.5", 0.5), ("val_selected", thr_sel), ("attack_aware", thr_aa)):
        probs0 = sigmoid_np(zdict[0.0], T_)
        pred0 = (probs0 >= thr).astype(int)
        corr0 = pred0 == labels
        for e in EVAL_EPS:
            z = zdict[e]
            probs = sigmoid_np(z, T_)
            pred = (probs >= thr).astype(int)
            corr = pred == labels
            m = compute_metrics(labels, probs, threshold=thr)
            lo, hi = wilson_ci(int(corr.sum()), len(labels))
            if e not in _auc_ci_cache:      # AUC is threshold-free: one bootstrap per (model, eps) serves both threshold rows
                _auc_ci_cache[e] = bootstrap_ci(labels, probs, 0.5, seed=CONFIG["seed"])["auc"]
            auc_lo, auc_hi = _auc_ci_cache[e]
            def asr(mask):
                den = int((corr0 & mask).sum())
                return (int((corr0 & ~corr & mask).sum()) / den) if den else float("nan")
            real, fake = labels == 0, labels == 1
            rows.append({
                "model": tag, "eps_255": e * 255, "threshold_type": thr_name, **m,
                "acc_ci_lo": lo, "acc_ci_hi": hi, "auc_ci_lo": auc_lo, "auc_ci_hi": auc_hi,
                "asr": asr(np.ones_like(real)), "asr_real": asr(real), "asr_fake": asr(fake),
                "n_fake": int(fake.sum()), "n_fake_detected": int((pred[fake] == 1).sum()), "n_real": int(real.sum()), "n_real_kept": int((pred[real] == 0).sum()),
                "n_initially_correct": int(corr0.sum()), "flips_correct_to_wrong": int((corr0 & ~corr).sum()),
                "flips_wrong_to_correct": int((~corr0 & corr).sum()), "pred_changed": int((pred != pred0).sum()),
                "acc_drop_vs_own_clean": float(corr0.mean() - corr.mean()),
                "auc_drop_vs_own_clean": safe_roc_auc(labels, probs0) - m["roc_auc"],
                "mean_p_fake_real": float(probs[real].mean()), "mean_p_fake_fake": float(probs[fake].mean()),
                "mean_logit_real": float(z[real].mean()), "mean_logit_fake": float(z[fake].mean()),
                "logit_std": float(z.std()), "n_videos": len(labels)})
    return rows


rows = []
for tag, (m, T_, thr_) in MODELS.items():
    rows += eval_rows(tag, RAW[tag][0], RAW[tag][1], T_, thr_, THR_AA[tag])
metrics_df = pd.DataFrame(rows)

# FGSM-trained minus clean-trained, per epsilon and threshold type
key = ["eps_255", "threshold_type"]
a = metrics_df[metrics_df.model == "Clean-trained"].set_index(key)
b = metrics_df[metrics_df.model == "FGSM-trained"].set_index(key)
for col in ["accuracy", "roc_auc", "f1", "balanced_accuracy", "asr"]:
    metrics_df[f"delta_{col}_fgsm_minus_clean"] = [
        (b.loc[(r.eps_255, r.threshold_type), col] - a.loc[(r.eps_255, r.threshold_type), col]) for r in metrics_df.itertuples()]
metrics_df.to_csv(CONFIG["metrics_csv"], index=False)
print(f"\nMetrics CSV written: {CONFIG['metrics_csv']}  ({len(metrics_df)} rows, {metrics_df.shape[1]} columns)")

pd.set_option("display.width", 260); pd.set_option("display.max_columns", 40)
show = ["model", "eps_255", "accuracy", "acc_ci_lo", "acc_ci_hi", "roc_auc", "auc_ci_lo", "auc_ci_hi", "f1", "balanced_accuracy",
        "sensitivity", "specificity", "asr", "asr_real", "asr_fake", "flips_correct_to_wrong"]
print("\n=== TEST, threshold 0.5 ===")
print(metrics_df[metrics_df.threshold_type == "0.5"][show].round(4).to_string(index=False))
print("\n=== TEST, validation-selected (clean-EER) threshold ===")
print(metrics_df[metrics_df.threshold_type == "val_selected"][show].round(4).to_string(index=False))
print("\n=== TEST, ATTACK-AWARE threshold (fixed on validation) ===")
print(metrics_df[metrics_df.threshold_type == "attack_aware"][show + ["n_fake_detected", "n_fake"]].round(4).to_string(index=False))
_t = metrics_df[(np.isclose(metrics_df.eps_255, CONFIG["target_eval_eps"])) & (metrics_df.model == "FGSM-trained")]
for _, r in _t.iterrows():
    print(f"FGSM-trained @ {CONFIG['target_eval_eps']:g}/255, threshold {r['threshold_type']}: fakes detected {int(r['n_fake_detected'])}/{int(r['n_fake'])}, "
          f"real kept {int(r['n_real_kept'])}/{int(r['n_real'])}   (v8: 16/200 at the clean-EER threshold)")

# ---- detailed clean report (eps = 0) ----
for tag, (m, T_, thr_) in MODELS.items():
    p0 = sigmoid_np(RAW[tag][1][0.0], T_)
    print(f"\n[{tag}] clean test, classification report @ validation-selected threshold {thr_:.3f}:")
    print(classification_report(test_labels, (p0 >= thr_).astype(int), target_names=["Real", "Fake"], zero_division=0))
    r05 = metrics_df[(metrics_df.model == tag) & (metrics_df.eps_255 == 0) & (metrics_df.threshold_type == "0.5")].iloc[0]
    print(f"   @0.5: acc {r05['accuracy']:.4f} | ROC-AUC {r05['roc_auc']:.4f} [{r05['auc_ci_lo']:.4f}, {r05['auc_ci_hi']:.4f}] | "
          f"PR-AUC {r05['pr_auc']:.4f} | FPR {r05['fpr']:.4f} FNR {r05['fnr']:.4f}")

In [ ]:
# ============================================================
# CELL 11b (v10): Experiment A (mixed-dataset) results SEPARATELY per dataset, per-epsilon diagnostics on test, raw scores, fallback-crop subset
# ============================================================
# Test order == test_records order (the loader does not shuffle), so every logit can be attributed to its dataset / method / crop type.
# Thresholds and temperatures are the ones fixed on the (mixed) validation set before the test was unlocked; nothing is re-tuned per dataset.
_t_ds = np.array([record_info[f]["dataset"] for f, _ in test_records])
_t_fb = np.array([bool((FACE_BOXES.get(Path(f).name) or {}).get("ok", True) is False) for f, _ in test_records])
assert len(_t_ds) == len(test_labels) and np.array_equal(np.array([l for _, l in test_records]), test_labels), "test record order != evaluated order"

_rows, _diag_tabs = [], []
for tag, (m_, T_, thr_) in MODELS.items():
    zd = RAW[tag][1]
    for name, mask in [("ALL", np.ones(len(test_labels), bool))] + [(d, _t_ds == d) for d in CONFIG["datasets"]] + [("non-fallback crops", ~_t_fb), ("fallback crops", _t_fb)]:
        if mask.sum() == 0:
            continue
        lab = test_labels[mask]
        n_r, n_f = int((lab == 0).sum()), int((lab == 1).sum())
        reliable = min(n_r, n_f) >= CONFIG["min_dataset_videos_per_class_eval"]
        sub = {e: zd[e][mask] for e in EVAL_EPS}
        for thr_name, thr in (("val_selected", thr_), ("attack_aware", THR_AA[tag])):
            t = eps_diag_table(tag, lab, sub, T_, thr, thr_name, EVAL_EPS)
            t.insert(1, "subset", name); t["n_real"], t["n_fake"], t["reliable"] = n_r, n_f, reliable
            _diag_tabs.append(t)
PER_DATASET_DIAG = pd.concat(_diag_tabs, ignore_index=True)
PER_DATASET_DIAG.to_csv(os.path.join(OUT_DIR, "mixed_per_dataset_eps_diagnostics_test_v10.csv"), index=False)
np.savez_compressed(os.path.join(OUT_DIR, "test_raw_scores_v10.npz"), labels=test_labels, dataset=_t_ds, fallback=_t_fb,
                    **{f"{tag}|{e * 255:g}": RAW[tag][1][e] for tag in MODELS for e in EVAL_EPS})
# independent AUC check on the full test scores (saved raw logits -> rank AUC) against the AUC in the main metrics table
_gap = PER_DATASET_DIAG[PER_DATASET_DIAG.subset == "ALL"].auc_abs_diff_sklearn_vs_independent.max()
assert _gap < 1e-9, f"ROC-AUC from sklearn and from the independent rank formula differ by {_gap}"
print(f"Independent ROC-AUC recomputation from saved raw test scores agrees with sklearn (max |diff| = {_gap:.1e}).")

_show = PER_DATASET_DIAG[(PER_DATASET_DIAG.threshold_type == "val_selected") & (PER_DATASET_DIAG.eps_255.isin([0, 1, 2, 4, 8]))]
print("\nEXPERIMENT A -- mixed-dataset test results per dataset (validation-selected threshold; 'reliable' = >= "
      f"{CONFIG['min_dataset_videos_per_class_eval']} videos per class):")
print(_show[["model", "subset", "n_real", "n_fake", "reliable", "eps_255", "accuracy", "roc_auc_independent", "macro_f1", "balanced_accuracy",
             "recall_real", "recall_fake", "share_pred_fake"]].round(3).to_string(index=False))
for d in CONFIG["datasets"]:
    r = PER_DATASET_DIAG[(PER_DATASET_DIAG.subset == d)]
    if r.empty:
        print(f"NOTE: dataset '{d}' has NO test videos in this split -> no mixed-dataset test result for it (see the audit table).")
    elif not bool(r.reliable.iloc[0]):
        print(f"NOTE: dataset '{d}' has only {int(r.n_real.iloc[0])} real / {int(r.n_fake.iloc[0])} fake test videos -> its numbers are indicative only.")
print("\nREMINDER: this is the MIXED-dataset experiment (train, validation and test videos of the same datasets, identity-disjoint). It does NOT measure "
      "cross-dataset generalisation; see the leave-one-dataset-out experiment in Cell 19b.")


In [ ]:
# ============================================================
# CELL 12: Statistics -- test size/balance, paired comparisons FGSM-trained vs clean-trained
# ============================================================
# * Split is identity-disjoint (asserted in Cell 2). CIs: Wilson for accuracy (in the CSV), percentile bootstrap over videos for AUC.
# * Paired tests use the SAME videos for both models: exact McNemar on per-video correctness, and a paired bootstrap of
#   AUC(FGSM-trained) - AUC(clean-trained). 'significant' = p<0.05 / CI excluding 0; with ~500 videos, accuracy differences
#   below ~4 points are within sampling noise.

n_real, n_fake = int((test_labels == 0).sum()), int((test_labels == 1).sum())
print(f"Test size: {n_test} videos | real {n_real} ({n_real / n_test:.1%}) / fake {n_fake} ({n_fake / n_test:.1%}) | "
      f"identities in test: {len({i for f, _ in test_records for i in record_info[f]['ids']})}")
print(f"Minimum detectable accuracy change (approx., 95%): +/-{1.96 * math.sqrt(0.25 / n_test) * 100:.1f} points for a single model.")

lab = test_labels
rows = []
for e in EVAL_EPS:
    zc, zf = RAW["Clean-trained"][1][e], RAW["FGSM-trained"][1][e]
    pc, pf = sigmoid_np(zc, clean_T), sigmoid_np(zf, fgsm_T)
    cor_c = (pc >= clean_selected_threshold).astype(int) == lab
    cor_f = (pf >= fgsm_selected_threshold).astype(int) == lab
    cor_c_aa = (pc >= THR_AA["Clean-trained"]).astype(int) == lab
    cor_f_aa = (pf >= THR_AA["FGSM-trained"]).astype(int) == lab
    b, c, p = mcnemar_exact(cor_c, cor_f)          # b = clean-only correct, c = FGSM-only correct
    d, lo, hi = paired_bootstrap_auc_delta(lab, pc, pf, seed=CONFIG["seed"])
    rows.append({"eps_255": e * 255, "acc_clean_trained": cor_c.mean(), "acc_fgsm_trained": cor_f.mean(),
                 "acc_delta": cor_f.mean() - cor_c.mean(),
                 "acc_clean_trained_attack_aware_thr": cor_c_aa.mean(), "acc_fgsm_trained_attack_aware_thr": cor_f_aa.mean(), "clean_only_correct": b, "fgsm_only_correct": c,
                 "mcnemar_p": p, "auc_clean_trained": safe_roc_auc(lab, pc), "auc_fgsm_trained": safe_roc_auc(lab, pf),
                 "auc_delta": d, "auc_delta_ci_lo": lo, "auc_delta_ci_hi": hi,
                 "auc_gain_significant": bool(lo == lo and lo > 0), "auc_loss_significant": bool(hi == hi and hi < 0)})
paired_df = pd.DataFrame(rows)
paired_df.to_csv(os.path.join(OUT_DIR, "fgsm_v9_paired_comparison.csv"), index=False)
print("\nPaired comparison, FGSM-trained vs clean-trained (validation-selected thresholds):")
print(paired_df.round(4).to_string(index=False))

In [ ]:
# ============================================================
# CELL 13: Per-manipulation results with counts and uncertainty
# ============================================================
# Real videos: 'rate' = specificity (called real). Fake videos: 'rate' = detection rate (called fake). Wilson 95% intervals; the
# per-method sample sizes are small, so read the intervals, not the point estimates.

test_methods = np.array([record_info[f]["method"] for f, _ in test_records])
assert len(test_methods) == n_test, "test_records order/length does not match the evaluated videos"
PM_EPS = sorted({0.0, CONFIG["target_eval_eps"] * EPS_UNIT, FGSM_EPS, 8 * EPS_UNIT})
rows = []
for tag, (m, T_, thr_) in MODELS.items():
    for e in PM_EPS:
        probs = sigmoid_np(RAW[tag][1][e], T_)
        pred = (probs >= thr_).astype(int)
        for meth in sorted(set(test_methods)):
            idx = test_methods == meth
            ok = (pred[idx] == 0) if meth == "real" else (pred[idx] == 1)
            lo, hi = wilson_ci(int(ok.sum()), int(idx.sum()))
            rows.append({"model": tag, "eps_255": e * 255, "method": meth, "n_videos": int(idx.sum()), "n_correct": int(ok.sum()),
                         "rate": float(ok.mean()), "rate_type": "specificity" if meth == "real" else "detection", "ci_lo": lo, "ci_hi": hi,
                         "mean_p_fake": float(probs[idx].mean())})
per_method_df = pd.DataFrame(rows)
per_method_df.to_csv(os.path.join(OUT_DIR, "fgsm_v9_per_method.csv"), index=False)
for e in PM_EPS:
    sub = per_method_df[np.isclose(per_method_df.eps_255, e * 255)]
    wide = sub.pivot(index="method", columns="model", values="rate").round(3)
    cnt = sub[sub.model == "Clean-trained"].set_index("method")["n_videos"]
    wide.insert(0, "n", cnt)
    print(f"\nPer-method rate at eps={e * 255:g}/255:")
    print(wide.to_string())

In [ ]:
# ============================================================
# CELL 14: Failure-mode investigation (FGSM-only evidence; no iterative attack is used)
# ============================================================
# Every diagnostic below is measured on a balanced test subset (or reuses the full-test logits). They answer:
#  D1 AUC ~ 0 / inverted ranking?   -> AUC vs eps for both models + mean logit(fake) - mean logit(real); AUC<0.5 means the
#                                      attack REVERSES the ranking (a convention bug is excluded by the Cell 7 audit and test 11).
#  D2 non-monotonic accuracy / AUC?  -> flagged when a LARGER eps gives HIGHER accuracy/AUC by more than 2 binomial SE; explained with
#                                      D3 (linearity of the loss) and D4 (random-sign noise of the same L-inf size).
#  D3 FGSM linearity                 -> actual BCE increase / first-order prediction eps*||grad||_1. Ratio << 1 at large eps = the
#                                      single linear step overshoots (non-linear loss) -> weaker, non-monotonic FGSM.
#  D4 random-sign noise control      -> same L-inf budget, random signs: if the model collapses under noise too, the drop is not an
#                                      adversarial effect.
#  D5 label leaking                  -> FGSM with the true label vs with the model's own prediction; FGSM accuracy above clean accuracy.
#  D6 gradient masking               -> transfer attack (crafted on the OTHER model) must NOT be stronger than the white-box FGSM.
#  D7 gradient alignment             -> cosine of input gradients at two random points of the eps-ball (catastrophic-overfitting sign).
#  D8 hedging                        -> logit spread and mean confidence under attack.

n_d = CONFIG["diag_n_per_class"]
_diag_real = [r for r in test_records if r[1] == 0][:n_d]
_diag_fake = [r for r in test_records if r[1] == 1][:n_d]
diag_loader = GuardedLoader(make_eval_loader(
    PreextractedVideoDataset(_diag_real + _diag_fake, seq_len=CONFIG["seq_len"], img_size=CONFIG["img_size"], is_train=False),
    CONFIG["eval_attack_batch_size"]))
DIAG_EPS = [0.5 / 255, 1 / 255, 2 / 255, 4 / 255, 8 / 255, 16 / 255]
_m_c, _m_f = clean_model, fgsm_eval_model


def _bce_ps(z, y):
    return Fn.binary_cross_entropy_with_logits(z, y.float(), reduction="none")


def run_diag_pass(model, other):
    keys = ["z0", "z_true", "z_pred", "z_noise", "z_transfer", "loss0", "loss_true", "loss_pred_lin", "ga"]
    acc = {k: {e: [] for e in DIAG_EPS} for k in keys}
    labs = []
    model.eval(); other.eval()
    for videos, labels in tqdm(diag_loader, desc="diagnostics", leave=False):
        videos = videos.to(CONFIG["device"]); y = labels.to(CONFIG["device"])
        g_t, z0 = fgsm_grad(model, videos, y, label_source="true")
        g_p, _ = fgsm_grad(model, videos, y, label_source="pred")
        g_o, _ = fgsm_grad(other, videos, y, label_source="true")
        l1_pix = (g_t.abs() / _STD.to(videos.device)).flatten(1).sum(dim=1)      # sum |dL/dx_pixel| per sample
        vmin, vmax = _norm_bounds(videos.device)
        with torch.no_grad():
            for e in DIAG_EPS:
                xt = fgsm_apply(videos, g_t, e)
                xp = fgsm_apply(videos, g_p, e)
                xo = fgsm_apply(videos, g_o, e)
                rs = torch.sign(torch.randn_like(videos))
                xn = torch.max(torch.min(videos + _eps_normalized(e, videos.device) * rs, vmax), vmin)
                zt, zp, zn, zo = (model(x).float() for x in (xt, xp, xn, xo))
                acc["z0"][e].append(z0.cpu().numpy()); acc["z_true"][e].append(zt.cpu().numpy())
                acc["z_pred"][e].append(zp.cpu().numpy()); acc["z_noise"][e].append(zn.cpu().numpy())
                acc["z_transfer"][e].append(zo.cpu().numpy())
                acc["loss0"][e].append(_bce_ps(z0, y).cpu().numpy())
                acc["loss_true"][e].append(_bce_ps(zt, y).cpu().numpy())
                acc["loss_pred_lin"][e].append((_bce_ps(z0, y) + e * l1_pix).cpu().numpy())
        labs.append(labels.numpy())
    out = {k: {e: np.concatenate(v) for e, v in d.items() if len(v)} for k, d in acc.items() if k != "ga"}
    # D7 gradient alignment on the first diag batches
    ga = {e: [] for e in DIAG_EPS}
    for bi, (videos, labels) in enumerate(diag_loader):
        if bi >= 6:
            break
        for e in DIAG_EPS:
            ga[e].append(gradient_alignment(model, videos.to(CONFIG["device"]), labels.to(CONFIG["device"]), e))
    out["ga"] = {e: float(np.mean(v)) for e, v in ga.items()}
    return np.concatenate(labs).astype(int), out


with timed("diagnostics"):
    d_lab, D_clean = run_diag_pass(_m_c, _m_f)       # attacks crafted on clean-trained; transfer comes from the FGSM-trained model
    _, D_fgsm = run_diag_pass(_m_f, _m_c)            # attacks on FGSM-trained; transfer comes from the clean-trained model
DIAG = {"Clean-trained": D_clean, "FGSM-trained": D_fgsm}
acc_of = lambda z, y: float(np.mean((z > 0).astype(int) == y))
rows = []
for tag, D in DIAG.items():
    for e in DIAG_EPS:
        pred_ratio = (D["loss_true"][e].mean() - D["loss0"][e].mean()) / max(D["loss_pred_lin"][e].mean() - D["loss0"][e].mean(), 1e-12)
        rows.append({"model": tag, "eps_255": e * 255, "acc_clean": acc_of(D["z0"][e], d_lab),
                     "acc_fgsm_true_label": acc_of(D["z_true"][e], d_lab), "acc_fgsm_pred_label": acc_of(D["z_pred"][e], d_lab),
                     "acc_random_sign_noise": acc_of(D["z_noise"][e], d_lab),
                     "acc_transfer_from_other_model": acc_of(D["z_transfer"][e], d_lab),
                     "auc_fgsm_true_label": safe_roc_auc(d_lab, D["z_true"][e]), "auc_transfer": safe_roc_auc(d_lab, D["z_transfer"][e]),
                     "linearity_ratio_actual_over_predicted": pred_ratio, "grad_align_cos": D["ga"][e],
                     "logit_std_clean": float(D["z0"][e].std()), "logit_std_fgsm": float(D["z_true"][e].std()),
                     "mean_conf_clean": float(np.maximum(sigmoid_np(D["z0"][e]), 1 - sigmoid_np(D["z0"][e])).mean()),
                     "mean_conf_fgsm": float(np.maximum(sigmoid_np(D["z_true"][e]), 1 - sigmoid_np(D["z_true"][e])).mean())})
diag_df = pd.DataFrame(rows)
diag_df.to_csv(os.path.join(OUT_DIR, "fgsm_v9_diagnostics.csv"), index=False)
print(f"Diagnostic subset: {len(d_lab)} test videos ({n_d} real + {n_d} fake)")
print(diag_df.round(3).to_string(index=False))

FINDINGS = []        # human-readable, evidence-backed statements consumed by Cell 17
for tag, (m, T_, thr_) in MODELS.items():
    z = RAW[tag][1]
    aucs = np.array([safe_roc_auc(test_labels, sigmoid_np(z[e], T_)) for e in EVAL_EPS])
    accs = np.array([float(np.mean((sigmoid_np(z[e], T_) >= thr_).astype(int) == test_labels)) for e in EVAL_EPS])
    se = 2 * math.sqrt(0.25 / n_test)
    nonmono_acc = [(EVAL_EPS[i] * 255, EVAL_EPS[j] * 255) for i in range(len(EVAL_EPS)) for j in range(i + 1, len(EVAL_EPS)) if accs[j] > accs[i] + se]
    nonmono_auc = [(EVAL_EPS[i] * 255, EVAL_EPS[j] * 255) for i in range(len(EVAL_EPS)) for j in range(i + 1, len(EVAL_EPS)) if aucs[j] > aucs[i] + se]
    inverted = [EVAL_EPS[i] * 255 for i in range(len(EVAL_EPS)) if aucs[i] == aucs[i] and aucs[i] < 0.5]
    FINDINGS.append(f"[D1] {tag}: AUC<0.5 (inverted ranking) at eps(/255) = {inverted if inverted else 'none'}; lowest AUC {np.nanmin(aucs):.3f}.")
    FINDINGS.append(f"[D2] {tag}: non-monotonic accuracy (larger eps, higher acc by >2 SE) in pairs {nonmono_acc if nonmono_acc else 'none'}; "
                    f"AUC pairs {nonmono_auc if nonmono_auc else 'none'}.")
    dd = diag_df[diag_df.model == tag]
    FINDINGS.append(f"[D3] {tag}: loss-linearity ratio (actual/first-order) per eps 0.5,1,2,4,8,16 = {[round(v, 2) for v in dd.linearity_ratio_actual_over_predicted]}.")
    FINDINGS.append(f"[D4] {tag}: random-sign noise accuracy {[round(v, 3) for v in dd.acc_random_sign_noise]} vs FGSM {[round(v, 3) for v in dd.acc_fgsm_true_label]} "
                    f"(clean {dd.acc_clean.iloc[0]:.3f}).")
    leak = [(r.eps_255, round(r.acc_fgsm_true_label - r.acc_clean, 3)) for r in dd.itertuples() if r.acc_fgsm_true_label > r.acc_clean + CONFIG["accept_max_label_leak_margin"]]
    gap = [(r.eps_255, round(r.acc_fgsm_pred_label - r.acc_fgsm_true_label, 3)) for r in dd.itertuples()]
    FINDINGS.append(f"[D5] {tag}: FGSM accuracy exceeds clean accuracy by > margin at {leak if leak else 'no epsilon'}; pred-label minus true-label FGSM accuracy = {gap}.")
    mask = [(r.eps_255, round(r.acc_transfer_from_other_model - r.acc_fgsm_true_label, 3)) for r in dd.itertuples() if r.acc_transfer_from_other_model < r.acc_fgsm_true_label - 0.03]
    FINDINGS.append(f"[D6] {tag}: transfer attack STRONGER than white-box FGSM (masking signature) at {mask if mask else 'no epsilon'}.")
    FINDINGS.append(f"[D7] {tag}: gradient-alignment cosine per eps 0.5,1,2,4,8,16 = {[round(v, 3) for v in dd.grad_align_cos]}.")
    FINDINGS.append(f"[D8] {tag}: logit std clean->FGSM per eps = {[(round(a, 2), round(b, 2)) for a, b in zip(dd.logit_std_clean, dd.logit_std_fgsm)]}; "
                    f"mean confidence clean->FGSM {[(round(a, 3), round(b, 3)) for a, b in zip(dd.mean_conf_clean, dd.mean_conf_fgsm)]}.")
for tag, (m, T_, thr_) in MODELS.items():       # D9 class-wise behaviour at the target eps on the full test set
    zt_ = RAW[tag][1][CONFIG["target_eval_eps"] * EPS_UNIT]
    p_ = sigmoid_np(zt_, T_)
    f_, r_ = test_labels == 1, test_labels == 0
    FINDINGS.append(f"[D9] {tag} @ {CONFIG['target_eval_eps']:g}/255: mean logit fake {zt_[f_].mean():+.2f} / real {zt_[r_].mean():+.2f}; mean P(fake) "
                    f"fake {p_[f_].mean():.3f} / real {p_[r_].mean():.3f}; fake recall clean-EER thr {np.mean(p_[f_] >= thr_):.3f}, attack-aware thr "
                    f"{np.mean(p_[f_] >= THR_AA[tag]):.3f}, @0.5 {np.mean(p_[f_] >= 0.5):.3f}; real specificity attack-aware {np.mean(p_[r_] < THR_AA[tag]):.3f}.")
print("\nEVIDENCE-BASED FINDINGS (all numbers measured in this run):")
for f in FINDINGS:
    print(" -", f)

In [ ]:
# ============================================================
# CELL 15: Figures (all saved to /kaggle/working/figures/)
# ============================================================
C_CLEAN, C_FGSM, C_GREY = "#4C78A8", "#E8832A", "#7f7f7f"
COL = {"Clean-trained": C_CLEAN, "FGSM-trained": C_FGSM}
XT = list(range(len(EVAL_EPS)))
XL = [f"{e * 255:g}" for e in EVAL_EPS]
FIGS = []


def _finish(fig, name):
    fig.tight_layout()
    FIGS.append(save_fig(fig, name))
    plt.show()


def _rows(tag, thr="val_selected"):
    return metrics_df[(metrics_df.model == tag) & (metrics_df.threshold_type == thr)].sort_values("eps_255")


# 1 loss curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(clean_df["epoch"], clean_df["train_loss"], "--", color=C_CLEAN, label="train (smoothed BCE)")
ax[0].plot(clean_df["epoch"], clean_df["val_loss"], color=C_CLEAN, label="val BCE")
ax[0].set_title("Clean-trained: loss")
ax[1].plot(fgsm_df["epoch"], fgsm_df["train_clean_loss"], "--", color=C_CLEAN, label="train clean part")
ax[1].plot(fgsm_df["epoch"], fgsm_df["train_adv_loss"], "--", color=C_FGSM, label="train FGSM part")
ax[1].plot(fgsm_df["epoch"], fgsm_df["train_loss"], color=C_GREY, label="train total")
if "val_loss" in fgsm_df:
    ax[1].plot(fgsm_df["epoch"], fgsm_df["val_loss"], color="black", label="val clean BCE")
ax[1].axhline(math.log(2), color="red", ls=":", lw=1, label="ln 2 (hedging)")
ax[1].set_title("FGSM-trained: loss")
for a in ax:
    a.set_xlabel("Epoch"); a.set_ylabel("BCE"); a.grid(alpha=.3); a.legend(fontsize=8)
_finish(fig, "fig01_loss_curves.png")

# 2 clean accuracy by epoch, 4 clean AUC by epoch
for name, col_c, col_f, ttl in [("fig02_clean_accuracy_by_epoch.png", "val_accuracy", "val_accuracy", "Validation CLEAN accuracy @0.5"),
                                ("fig04_clean_auc_by_epoch.png", "val_roc_auc", "val_clean_auc", "Validation CLEAN ROC-AUC")]:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(clean_df["epoch"], clean_df[col_c], color=C_CLEAN, marker="o"); ax[0].set_title("Clean-trained (control)")
    ax[1].plot(fgsm_df["epoch"], fgsm_df[col_f], color=C_FGSM, marker="o"); ax[1].set_title("FGSM-trained")
    ref = BASELINE_VAL["clean_val_auc_full"] if "auc" in name else None
    if ref:
        ax[1].axhline(ref, color=C_CLEAN, ls="--", label="clean-trained best"); ax[1].legend(fontsize=8)
    for a in ax:
        a.set_xlabel("Epoch"); a.set_ylabel(ttl); a.grid(alpha=.3)
    _finish(fig, name)

# 3 FGSM accuracy by epoch, 5 FGSM AUC by epoch (validation subset)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(fgsm_df["epoch"], fgsm_df["val_subset_clean_acc"], color=C_GREY, marker="o", label="clean (same subset)")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_target_acc"], color=C_FGSM, marker="o", label=f"FGSM {CONFIG['target_eval_eps']:g}/255 accuracy @0.5")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_target_sens"], color="#a02020", marker="^", ls="--", label="fake recall @ logit 0 (target eps)")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_target_spec"], color="#2c7f2c", marker="v", ls="--", label="real specificity @ logit 0 (target eps)")
ax.axvline(fgsm_ckpt["epoch"], color="green", lw=1, alpha=.6, label="selected epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation FGSM accuracy / class rates (subset)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
_finish(fig, "fig03_fgsm_accuracy_by_epoch.png")
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(fgsm_df["epoch"], fgsm_df["val_subset_clean_auc"], color=C_GREY, marker="o", label="clean (same subset)")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_target_auc"], color=C_FGSM, marker="o", label=f"FGSM {CONFIG['target_eval_eps']:g}/255 AUC")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_mean_auc"], color="#B5441B", marker="s", ls="--", label="mean FGSM AUC over val eps")
ax.plot(fgsm_df["epoch"], fgsm_df["val_fgsm_min_auc"], color="#6a3d9a", marker="x", ls="--", label="min FGSM AUC over val eps")
ax.axhline(BASELINE_VAL["fgsm_target_auc"], color=C_CLEAN, ls=":", label="clean-trained @ target eps"); ax.axhline(0.5, color="red", lw=0.8, ls=":")
ax.axvline(fgsm_ckpt["epoch"], color="green", lw=1, alpha=.6, label="selected epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation FGSM ROC-AUC (subset)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
_finish(fig, "fig05_fgsm_auc_by_epoch.png")

# 6-8 accuracy / AUC / F1 vs epsilon (test)
for name, col, ttl, ci in [("fig06_accuracy_vs_eps.png", "accuracy", "Test accuracy", ("acc_ci_lo", "acc_ci_hi")),
                           ("fig07_auc_vs_eps.png", "roc_auc", "Test ROC-AUC", None),
                           ("fig08_f1_vs_eps.png", "f1", "Test macro F1", None)]:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for tag in MODELS:
        r = _rows(tag, "0.5" if col == "roc_auc" else "val_selected")
        ax.plot(XT, r[col].values, marker="o", color=COL[tag], label=tag)
        if ci:
            ax.fill_between(XT, r[ci[0]].values, r[ci[1]].values, color=COL[tag], alpha=.15)
        if col == "roc_auc":
            ax.fill_between(XT, r["auc_ci_lo"].values, r["auc_ci_hi"].values, color=COL[tag], alpha=.15)
    if col == "roc_auc":
        ax.axhline(0.5, color="red", ls=":", lw=1)
    ax.axvline(XL.index(f"{CONFIG['target_eval_eps']:g}"), color=C_GREY, ls=":", lw=1)
    ax.axvline(XL.index(f"{FGSM_EPS * 255:g}") if f"{FGSM_EPS * 255:g}" in XL else 0, color=C_FGSM, ls=":", lw=1)
    ax.set_xticks(XT); ax.set_xticklabels(XL); ax.set_xlabel("FGSM evaluation epsilon (/255)"); ax.set_ylabel(ttl)
    ax.set_title(f"{ttl} vs FGSM epsilon (shaded: 95% CI; dotted verticals: target and training eps)")
    ax.grid(alpha=.3); ax.legend()
    _finish(fig, name)

# 9 ASR vs eps
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, tag in zip(ax, MODELS):
    r = _rows(tag)
    a.plot(XT, r["asr"], marker="o", color=COL[tag], label="overall")
    a.plot(XT, r["asr_real"], marker="s", ls="--", color="#2c7f2c", label="real videos")
    a.plot(XT, r["asr_fake"], marker="^", ls="--", color="#a02020", label="fake videos")
    a.set_xticks(XT); a.set_xticklabels(XL); a.set_xlabel("FGSM epsilon (/255)"); a.set_title(f"{tag}: attack success rate")
    a.grid(alpha=.3); a.legend(fontsize=8)
ax[0].set_ylabel("ASR (of initially-correct videos)")
_finish(fig, "fig09_asr_vs_eps.png")

# 10 clean-trained vs FGSM-trained comparison
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
w = 0.38
for a, col, ttl in zip(ax[:2], ["accuracy", "roc_auc"], ["Accuracy", "ROC-AUC"]):
    rc, rf = _rows("Clean-trained", "0.5"), _rows("FGSM-trained", "0.5")
    a.bar(np.array(XT) - w / 2, rc[col], w, color=C_CLEAN, label="Clean-trained")
    a.bar(np.array(XT) + w / 2, rf[col], w, color=C_FGSM, label="FGSM-trained")
    a.set_xticks(XT); a.set_xticklabels(XL); a.set_title(f"{ttl} @0.5 by FGSM eps"); a.set_xlabel("eps (/255)"); a.legend(fontsize=8); a.grid(alpha=.3, axis="y")
d = paired_df.sort_values("eps_255")
ax[2].bar(XT, d["auc_delta"], color=[C_FGSM if v > 0 else C_GREY for v in d["auc_delta"]])
ax[2].errorbar(XT, d["auc_delta"], yerr=[np.clip(d["auc_delta"] - d["auc_delta_ci_lo"], 0, None), np.clip(d["auc_delta_ci_hi"] - d["auc_delta"], 0, None)], fmt="none", ecolor="black", capsize=3)
ax[2].axhline(0, color="black", lw=.8); ax[2].set_xticks(XT); ax[2].set_xticklabels(XL)
ax[2].set_title("AUC(FGSM-trained) - AUC(clean-trained), paired 95% CI"); ax[2].set_xlabel("eps (/255)"); ax[2].grid(alpha=.3, axis="y")
_finish(fig, "fig10_clean_vs_fgsm_trained.png")

# 11 confusion matrices
CM_EPS = sorted({0.0, CONFIG["target_eval_eps"] * EPS_UNIT, FGSM_EPS, 8 * EPS_UNIT})
fig, ax = plt.subplots(2, len(CM_EPS), figsize=(3.6 * len(CM_EPS), 7))
for i, (tag, (m, T_, thr_)) in enumerate(MODELS.items()):
    for j, e in enumerate(CM_EPS):
        pred = (sigmoid_np(RAW[tag][1][e], T_) >= thr_).astype(int)
        sns.heatmap(confusion_matrix(test_labels, pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[i, j],
                    xticklabels=["Pred real", "Pred fake"], yticklabels=["True real", "True fake"])
        ax[i, j].set_title(f"{tag}\nFGSM {e * 255:g}/255 (thr {thr_:.2f})", fontsize=9)
_finish(fig, "fig11_confusion_matrices.png")

# 12 per-manipulation
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
methods = sorted(set(test_methods))
for a, tag in zip(ax, MODELS):
    for k, e in enumerate(PM_EPS):
        s = per_method_df[(per_method_df.model == tag) & np.isclose(per_method_df.eps_255, e * 255)].set_index("method").loc[methods]
        xs = np.arange(len(methods)) + (k - (len(PM_EPS) - 1) / 2) * 0.18
        a.bar(xs, s["rate"], 0.18, yerr=[np.clip(s["rate"] - s["ci_lo"], 0, None), np.clip(s["ci_hi"] - s["rate"], 0, None)], capsize=2, label=f"eps {e * 255:g}/255")
    a.set_xticks(np.arange(len(methods))); a.set_xticklabels([f"{m}\n(n={int((test_methods == m).sum())})" for m in methods], fontsize=8)
    a.set_title(f"{tag}: correct-decision rate per manipulation (Wilson 95% CI)"); a.grid(alpha=.3, axis="y"); a.legend(fontsize=7)
ax[0].set_ylabel("rate (specificity for real, detection for fakes)")
_finish(fig, "fig12_per_manipulation.png")

# 13 probability / logit distributions
fig, ax = plt.subplots(2, len(CM_EPS), figsize=(3.6 * len(CM_EPS), 6.5), sharex=True)
for i, (tag, (m, T_, thr_)) in enumerate(MODELS.items()):
    for j, e in enumerate(CM_EPS):
        z = RAW[tag][1][e]
        bins = np.linspace(min(-12, np.percentile(z, 0.5)), max(12, np.percentile(z, 99.5)), 41)
        ax[i, j].hist(z[test_labels == 0], bins=bins, alpha=.6, color="#2c7f2c", label="real")
        ax[i, j].hist(z[test_labels == 1], bins=bins, alpha=.6, color="#a02020", label="fake")
        ax[i, j].axvline(0, color="black", lw=.8)
        ax[i, j].set_title(f"{tag}\nlogit @ FGSM {e * 255:g}/255", fontsize=9)
        if i == 1:
            ax[i, j].set_xlabel("logit  (P(fake) = sigmoid(logit/T))")
ax[0, 0].legend(fontsize=8)
_finish(fig, "fig13_logit_distributions.png")

# 14 stability / learning rate / numerics
fig, ax = plt.subplots(2, 3, figsize=(16, 7.5))
ax[0, 0].plot(fgsm_df["epoch"], fgsm_df["lr"], color=C_GREY, marker="o"); ax[0, 0].set_title("Head LR at end of epoch (per-step warm-up + cosine)")
ax[0, 1].plot(fgsm_df["epoch"], fgsm_df["train_grad_norm"], color=C_FGSM, marker="o", label="mean grad norm (pre-clip)")
ax[0, 1].plot(fgsm_df["epoch"], fgsm_df["train_clip_thr"], color="red", ls=":", label="mean clip threshold"); ax[0, 1].legend(fontsize=8); ax[0, 1].set_title("Gradient norm")
ax[0, 2].plot(fgsm_df["epoch"], fgsm_df["train_clip_rate"], color=C_FGSM, marker="o"); ax[0, 2].set_ylim(0, 1.02); ax[0, 2].set_title("Fraction of steps clipped")
ax[1, 0].plot(fgsm_df["epoch"], fgsm_df["train_grad_align"], color="#6a3d9a", marker="o"); ax[1, 0].axhline(0, color="black", lw=.6)
ax[1, 0].set_title("Gradient-alignment cosine (CO heuristic; low = FGSM unreliable)")
ax[1, 1].plot(fgsm_df["epoch"], fgsm_df["select_raw"], color=C_FGSM, marker="o", label="raw score")
ax[1, 1].plot(fgsm_df["epoch"], fgsm_df["select_score"], color="black", marker="s", label="smoothed - stability penalty")
ax[1, 1].scatter(fgsm_df["epoch"][~fgsm_df["gate_ok"].astype(bool)], fgsm_df["select_raw"][~fgsm_df["gate_ok"].astype(bool)], color="red", zorder=5, label="gate FAILED")
ax[1, 1].legend(fontsize=8); ax[1, 1].set_title("Validation selection score (stability across epochs)")
ax[1, 2].bar(fgsm_df["epoch"] - 0.2, fgsm_df["train_nan_skips"], 0.4, color="red", label="non-finite-loss steps skipped")
ax[1, 2].bar(fgsm_df["epoch"] + 0.2, fgsm_df["train_zero_grad_samples"], 0.4, color=C_GREY, label="zero-gradient attacked samples"); ax[1, 2].bar(fgsm_df["epoch"], fgsm_df["train_fp32_retries"], 0.15, color="#2c7fb8", label="fp16 overflow rescued by fp32 retry"); ax[1, 2].legend(fontsize=7)
ax[1, 2].set_title("Numerical-health counters")
for a in ax.ravel():
    a.set_xlabel("FGSM-AT epoch"); a.grid(alpha=.3)
_finish(fig, "fig14_stability_and_lr.png")
# 16 class-wise rates vs eps for the three thresholds (the fake-recall problem)
fig, ax = plt.subplots(2, 3, figsize=(16, 7.5), sharey=True)
for i, tag in enumerate(MODELS):
    for j, thr in enumerate(["0.5", "val_selected", "attack_aware"]):
        r = _rows(tag, thr)
        ax[i, j].plot(XT, r["n_fake_detected"] / r["n_fake"], marker="^", color="#a02020", label="fake recall")
        ax[i, j].plot(XT, r["n_real_kept"] / r["n_real"], marker="v", color="#2c7f2c", label="real specificity")
        ax[i, j].plot(XT, r["accuracy"], marker="o", color="black", ls="--", label="accuracy")
        ax[i, j].set_xticks(XT); ax[i, j].set_xticklabels(XL); ax[i, j].set_title(f"{tag}, threshold: {thr}", fontsize=9); ax[i, j].grid(alpha=.3)
ax[0, 0].legend(fontsize=8); ax[1, 1].set_xlabel("FGSM eps (/255)")
_finish(fig, "fig16_classwise_rates_vs_eps.png")

# 17 attack audit of the clean-trained model (A3 controls) + 18 pilot comparison
_ad = pd.DataFrame(AUDIT["rows"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(_ad.eps_255, _ad.A3_auc_true, marker="o", color="#a02020", label="true-label FGSM")
ax[0].plot(_ad.eps_255, _ad.A3_auc_flipped, marker="s", color="#2c7f2c", label="flipped-label FGSM (control)")
ax[0].plot(_ad.eps_255, _ad.A3_auc_random, marker="^", color=C_GREY, label="random-sign noise (control)")
ax[0].plot(_ad.eps_255, _ad.A6_auc_quantised, marker="x", ls="--", color="black", label="true-label, 8-bit quantised")
ax[0].axhline(0.5, color="red", ls=":", lw=1); ax[0].set_xlabel("eps (/255)"); ax[0].set_ylabel("validation AUC (audit subset)")
ax[0].set_title("Clean-trained model: attack audit"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
if len(PILOT_ROWS):
    _pd = pd.DataFrame(PILOT_ROWS)
    xx = np.arange(len(_pd))
    ax[1].bar(xx - 0.2, _pd["val_clean_auc"], 0.4, color=C_CLEAN, label="val clean AUC")
    ax[1].bar(xx + 0.2, _pd["val_fgsm_target_auc"], 0.4, color=C_FGSM, label="val FGSM@target AUC")
    ax[1].set_xticks(xx); ax[1].set_xticklabels([v.replace(" + ", "\n+ ").replace(", ", ",\n") for v in _pd["variant"]], fontsize=7)
    ax[1].axhline(0.5, color="red", ls=":", lw=1); ax[1].set_title("Pilot recipes (validation)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y")
_finish(fig, "fig17_audit_and_pilot.png")
print(f"\n{len(FIGS)} figures saved in {FIG_DIR}")


# ============================================================
# 15b: Grad-CAM -- clean-trained vs FGSM-trained, on clean AND FGSM input (white-box, crafted against each model)
# ============================================================
# Target layer: the last EfficientNet-B1 block. fp32, BN/dropout frozen via _prepare_model_for_attack (LSTM keeps its training-mode
# cuDNN backward). Hooks are removed after every call. CAVEAT: Grad-CAM shows where evidence came from; it does not prove the artifact.

class VideoGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self.handles = [target_layer.register_forward_hook(self._save_act),
                        target_layer.register_full_backward_hook(self._save_grad)]

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def remove(self):
        for h in self.handles:
            h.remove()

    def heatmaps(self, video_tensor):
        """video_tensor: [1, T, 3, H, W] -> (P(fake) uncalibrated, per-frame heatmaps [T, h, w])."""
        prev = _prepare_model_for_attack(self.model)
        try:
            x = video_tensor.clone().detach().requires_grad_(True)
            logits = self.model(x)
            prob = torch.sigmoid(logits.float()).item()
            self.model.zero_grad()
            logits.sum().backward()
            weights = self.gradients.mean(dim=(2, 3), keepdim=True)
            cams = Fn.relu((weights * self.activations).sum(dim=1))
            maps = []
            for cam in cams:
                cam = cam - cam.min()
                maps.append((cam / (cam.max() + 1e-7)).cpu().numpy())
        finally:
            _restore_model_modes(self.model, prev)
            self.model.zero_grad(set_to_none=True)
        return prob, np.array(maps)


def _cam_maps(model, video):
    cam = VideoGradCAM(model, model.feature_extractor[0][-1])
    try:
        return cam.heatmaps(video)
    finally:
        cam.remove()


def _overlay(frame_norm, heat):
    img = np.clip(np.array(IMAGENET_STD) * frame_norm + np.array(IMAGENET_MEAN), 0, 1)
    img8 = (img * 255).astype(np.uint8)
    h = cv2.resize(heat, (img8.shape[1], img8.shape[0]))
    colour = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * h), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return img, cv2.addWeighted(img8, 0.6, colour, 0.4, 0)


def gradcam_comparison(sample_idx, n_show=6):
    video, label = test_dataset[sample_idx]
    video = video.unsqueeze(0).to(CONFIG["device"])
    y = torch.as_tensor(label).view(1).to(CONFIG["device"])
    attack_eps = sorted({CONFIG["target_eval_eps"] * EPS_UNIT, FGSM_EPS})
    kinds = ["clean"] + [f"FGSM {e * 255:g}/255" for e in attack_eps]
    results = {}
    for name, (mdl, T_, thr_) in MODELS.items():
        inputs = [video] + [fgsm_attack(mdl, video, y, e) for e in attack_eps]     # crafted against THIS model
        for kind, inp in zip(kinds, inputs):
            with torch.no_grad():
                p = float(torch.sigmoid(mdl(inp).float() / T_).item())
            _, maps = _cam_maps(mdl, inp)
            results[(name, kind)] = (inp, p, maps, thr_)
    fidx = np.linspace(0, CONFIG["seq_len"] - 1, n_show, dtype=int)
    nrows = 1 + len(MODELS) * len(kinds)
    fig, axes = plt.subplots(nrows, n_show, figsize=(2.3 * n_show + 2.5, 2.5 * nrows))
    base = video[0].cpu().numpy()
    for j, f in enumerate(fidx):
        axes[0, j].imshow(_overlay(base[f].transpose(1, 2, 0), np.zeros((7, 7)))[0])
        axes[0, j].set_title(f"frame {f}", fontsize=10)
    axes[0, 0].set_ylabel("input", fontsize=10, rotation=0, ha="right", va="center", labelpad=8)
    row = 1
    true_txt = "fake" if int(label) else "real"
    for name in MODELS:
        for kind in kinds:
            inp, p, maps, thr_ = results[(name, kind)]
            arr = inp[0].detach().cpu().numpy()
            for j, f in enumerate(fidx):
                axes[row, j].imshow(_overlay(arr[f].transpose(1, 2, 0), maps[min(f, len(maps) - 1)])[1])
            pred = "fake" if p >= thr_ else "real"
            axes[row, 0].set_ylabel(f"{name}\n{kind}\nP(fake)={p:.2f} -> {pred}", fontsize=9, rotation=0, ha="right", va="center", labelpad=8)
            print(f"  sample {sample_idx} (true={true_txt}) | {name:<14} | {kind:<14} | P(fake)={p:.3f} (thr {thr_:.3f}) -> predicted {pred}")
            row += 1
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"LEGACY v9 Grad-CAM (superseded by Cell 16b; fake-logit target, per-frame normalisation) -- test sample {sample_idx} (true={true_txt}). Heatmaps localise evidence; they do not prove it.", fontsize=11)
    FIGS.append(save_fig(fig, f"fig15_gradcam_sample{sample_idx}.png"))
    plt.show()


print("INTERPRETATION CAVEAT: Grad-CAM shows which spatial regions contributed to the prediction. It does NOT by itself prove the "
      "manipulation artifact -- a localisation aid, not ground-truth evidence.\n")
_fake_idx = next(i for i, (_, l) in enumerate(test_records) if l == 1)
_real_idx = next(i for i, (_, l) in enumerate(test_records) if l == 0)
for _i in (_fake_idx, _real_idx):
    gradcam_comparison(_i)
free_cuda_memory()

In [ ]:
# ============================================================
# CELL 15c (v10): Corrected Grad-CAM for the CNN-LSTM video model + explanation tests (faithfulness proxies; no fabricated masks)
# ============================================================
# What the v9 implementation (Cell 15b above, kept as 'legacy') got wrong, verified by reading the code:
#   G1  target = logits.sum(): with ONE logit that is the FAKE logit, so for a video predicted REAL the map explained evidence for the class that was
#       NOT predicted. v10 explains an explicit class: score = +logit for 'fake', -logit for 'real' (the single logit's negation).
#   G2  every frame's map was independently min-max normalised (cam - min) / max: intensity could not be compared between frames, models or epsilons,
#       and a map of pure noise looked as 'strong' as a real one. v10 saves RAW maps and displays them with ONE shared scale per figure.
#   G3  full_backward_hook on the last block + stored tensors that could be stale; v10 uses a forward hook + a tensor hook with fresh storage per call, shape /
#       finiteness / near-zero checks and flags in the result.
#   G4  the displayed 'input' row was the model input (face crop) and there was no mapping to the original frame; v10 keeps the crop box and the sampled
#       frame indices (written by the extractor) and can project a heat map back onto the ORIGINAL frame.
# Frame association: the model reshapes [B,T,...] -> [B*T,...]; the gradient of the VIDEO-level score w.r.t. frame t's activation tensor is the gradient
# through the LSTM and mean pooling of that frame's features, so map t = ReLU(sum_k mean_hw(dS/dA_t,k) * A_t,k) is the contribution of frame t to the video score.
# CAVEAT: Grad-CAM shows where evidence for a prediction came from; it does NOT prove the manipulation region. Faithfulness (does removing the highlighted
# region change the score?) is a different property from localisation accuracy (does it match a ground-truth manipulation mask?).

def _default_target_layer(model):
    return model.feature_extractor[0][-1]          # last EfficientNet-B1 block: 7x7x1280 per frame at 224 px


class VideoGradCAMv2:
    def __init__(self, model, target_layer=None):
        self.model = model
        self.layer = target_layer if target_layer is not None else _default_target_layer(model)
        self.calls = 0

    def explain(self, video, cls="pred", logit_thr=0.0):
        """video [1,T,3,H,W] (normalised). cls: 'fake' | 'real' | 'pred' (class the model assigns at logit_thr). Returns dict with RAW maps [T,h,w]."""
        assert video.dim() == 5 and video.size(0) == 1, "explain() takes exactly one video"
        T = video.size(1)
        store = {"act": None, "grad": None, "call": self.calls + 1}
        self.calls += 1

        def fwd_hook(mod, inp, out):
            store["act"] = out
            if out.requires_grad:
                out.register_hook(lambda g: store.__setitem__("grad", g))

        h = self.layer.register_forward_hook(fwd_hook)
        prev = _prepare_model_for_attack(self.model)
        try:
            x = video.detach().clone().requires_grad_(True)
            logit = self.model(x).float().reshape(())
            target = int(float(logit) > logit_thr) if cls == "pred" else (1 if cls == "fake" else 0)
            score = logit if target == 1 else -logit
            self.model.zero_grad(set_to_none=True)
            score.backward()
            act, grad = store["act"].detach(), store["grad"]
            flags = []
            if grad is None:
                raise RuntimeError("no gradient reached the target layer")
            if act.shape[0] != T or grad.shape != act.shape:
                raise RuntimeError(f"activation/gradient shapes {tuple(act.shape)}/{tuple(grad.shape)} do not match T={T}")
            if not (torch.isfinite(act).all() and torch.isfinite(grad).all()):
                flags.append("non_finite")
            w = grad.mean(dim=(2, 3), keepdim=True)                       # [T,C,1,1]: one weight vector PER FRAME
            cams = Fn.relu((w * act).sum(dim=1)).float().cpu().numpy()       # [T,h,w], RAW (not normalised)
            if float(np.abs(grad.cpu().numpy()).max()) < 1e-12:
                flags.append("near_zero_gradient")
            if float(cams.max()) <= 0:
                flags.append("all_zero_map")
        finally:
            h.remove()
            _restore_model_modes(self.model, prev)
            self.model.zero_grad(set_to_none=True)
        return {"raw": cams, "logit": float(logit), "prob_uncal": float(torch.sigmoid(logit)), "target_class": target, "flags": flags, "call": store["call"]}


def shared_display_scale(raw_list, pct=None):
    pct = pct or CONFIG["gradcam_ref_percentile"]
    v = np.concatenate([np.asarray(r).ravel() for r in raw_list])
    v = v[v > 0]
    return float(np.percentile(v, pct)) if v.size else 1.0


def to_display(raw, ref):
    return np.clip(np.asarray(raw) / max(ref, 1e-12), 0.0, 1.0)


def heat_to_crop(heat_hw, size):
    return cv2.resize(np.asarray(heat_hw, np.float32), (size, size), interpolation=cv2.INTER_LINEAR)


def heat_to_original(heat_hw, box, frame_hw):
    """Project a heat map defined on the model's face crop back onto the ORIGINAL frame: the crop is the square `box`=(x0,y0,x1,y1) of the frame, resized."""
    x0, y0, x1, y1 = box
    H, W = frame_hw
    canvas = np.zeros((H, W), np.float32)
    canvas[y0:y1, x0:x1] = cv2.resize(np.asarray(heat_hw, np.float32), (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    return canvas


def original_frames_for(folder, n_show):
    """Re-read the sampled ORIGINAL frames behind a face-crop folder (needs the source video + recorded frame indices). Returns (frames_rgb, indices) or (None, None)."""
    b = FACE_BOXES.get(Path(folder).name) or {}
    vp, idx = b.get("video_path") or record_info.get(folder, {}).get("video_path"), b.get("frame_indices")
    if not vp or not os.path.exists(vp):
        return None, None
    if not idx:                                   # v9 cache without metadata: rebuild the deterministic plan
        cap = cv2.VideoCapture(vp); tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        idx = _plan_pool_indices(tot, CONFIG["frame_pool_size"])
    pick = [idx[i] for i in n_show]
    frames, ok = [], []
    for fi in pick:
        cap = cv2.VideoCapture(vp); cap.set(cv2.CAP_PROP_POS_FRAMES, fi); r, fr = cap.read(); cap.release()
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if r else None)
    return frames, pick


# ---------------- faithfulness proxies (clearly labelled: NOT localisation accuracy) ----------------
def _cell_mask_to_input(cell_mask, size):
    """[T,h,w] boolean cell mask -> [T,1,size,size] float mask (nearest upsampling: a cell covers exactly its receptive grid block)."""
    m = torch.from_numpy(cell_mask.astype(np.float32)).unsqueeze(1)
    return Fn.interpolate(m, size=(size, size), mode="nearest")


def _class_score(model, x, target):
    with torch.no_grad():
        z = float(model(x).float().reshape(()))
    return z if target == 1 else -z


def deletion_faithfulness(model, video, raw_maps, target, fracs=None, repeats=None, seed=0):
    """Replace the top-k% / random k% / bottom-k% attributed cells (ranked jointly over all frames of the video) by the dataset mean colour (0 in normalised
    space) and measure the drop of the explained class score. Faithful explanation: drop(top) > drop(random) > drop(bottom)."""
    fracs = fracs or CONFIG["gradcam_occlusion_fracs"]
    repeats = repeats or CONFIG["gradcam_random_repeats"]
    T, h, w = raw_maps.shape
    flat = raw_maps.reshape(-1)
    order = np.argsort(-flat, kind="stable")
    base = _class_score(model, video, target)
    size = video.size(-1)
    rng = np.random.RandomState(seed)
    out = []
    for f in fracs:
        k = max(1, int(round(f * flat.size)))
        def drop(idx):
            m = np.zeros(flat.size, bool); m[idx] = True
            mask = _cell_mask_to_input(m.reshape(T, h, w), size).to(video.device).unsqueeze(0)       # [1,T,1,H,W]
            return base - _class_score(model, video * (1 - mask), target)
        d_top, d_bot = drop(order[:k]), drop(order[-k:])
        d_rand = float(np.mean([drop(rng.choice(flat.size, k, replace=False)) for _ in range(repeats)]))
        out.append({"frac": f, "drop_top": d_top, "drop_random": d_rand, "drop_bottom": d_bot, "base_score": base})
    return out


def explanation_stability(a_raw, b_raw, top_frac=0.2):
    """Per-video mean over frames of: Spearman correlation between two raw maps of the same frame, and IoU of their top-`top_frac` cells."""
    from scipy.stats import spearmanr
    rs, ious = [], []
    for A, B in zip(a_raw, b_raw):
        a, b = A.ravel(), B.ravel()
        if a.std() > 0 and b.std() > 0:
            rs.append(spearmanr(a, b)[0])
        k = max(1, int(round(top_frac * a.size)))
        ta, tb = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k])
        ious.append(len(ta & tb) / len(ta | tb))
    return float(np.mean(rs)) if rs else float("nan"), float(np.mean(ious))


# ---------------- ground-truth masks: discovery + alignment helpers (used only when real masks exist) ----------------
def discover_masks():
    found = {}
    for d, info in CONFIG["datasets"].items():
        root = Path(info["root"])
        hits = []
        if root.is_dir():
            for p in root.rglob("*"):
                if p.is_dir() and p.name.lower() in ("masks", "mask", "ground_truth_masks", "manipulation_masks") and len(p.relative_to(root).parts) <= 4:
                    hits.append(str(p))
                if len(hits) >= 3:
                    break
        found[d] = hits
    return found


def mask_to_crop(mask_full_frame, box, size):
    """Align a full-frame binary mask to the model's face crop (same box, same resize as the crops)."""
    x0, y0, x1, y1 = box
    return (cv2.resize(mask_full_frame[y0:y1, x0:x1].astype(np.float32), (size, size), interpolation=cv2.INTER_NEAREST) > 0.5)


def mask_localisation(raw_map_hw, mask_crop, top_frac=0.2):
    """Share of the top-attributed cells that fall inside the mask (pointing precision) and IoU of the top region with the mask."""
    heat = heat_to_crop(raw_map_hw, mask_crop.shape[0])
    k = int(round(top_frac * heat.size))
    thr = np.sort(heat.ravel())[::-1][k - 1]
    top = heat >= thr
    return float((top & mask_crop).sum() / max(top.sum(), 1)), float((top & mask_crop).sum() / max((top | mask_crop).sum(), 1))


# ---------------- figure + quantitative driver (needs trained models and data) ----------------
def _cond_inputs(video, y, eps):
    adv_clean = fgsm_attack(clean_model, video, y, eps)           # crafted against the clean-trained model
    adv_fgsm = fgsm_attack(fgsm_eval_model, video, y, eps)        # white-box against the FGSM-trained model
    return [("Clean-trained | clean input", clean_model, clean_T, clean_selected_threshold, video),
            ("Clean-trained | FGSM (own)", clean_model, clean_T, clean_selected_threshold, adv_clean),
            ("FGSM-trained | clean input", fgsm_eval_model, fgsm_T, fgsm_selected_threshold, video),
            ("FGSM-trained | FGSM (own, white-box)", fgsm_eval_model, fgsm_T, fgsm_selected_threshold, adv_fgsm),
            ("FGSM-trained | SAME FGSM input as row 2", fgsm_eval_model, fgsm_T, fgsm_selected_threshold, adv_clean)]


def gradcam_v2_figure(sample_idx, eps=None, n_show=6, show_legacy=True):
    eps = eps or CONFIG["target_eval_eps"] * EPS_UNIT
    folder, label = test_records[sample_idx]
    video, _ = test_dataset[sample_idx]
    video = video.unsqueeze(0).to(CONFIG["device"])
    y = torch.tensor([label], device=CONFIG["device"])
    fidx = np.linspace(0, CONFIG["seq_len"] - 1, n_show, dtype=int)
    conds = _cond_inputs(video, y, eps)
    res = []
    for name, mdl, T_, thr_, x in conds:
        with torch.no_grad():
            p = float(torch.sigmoid(mdl(x).float() / T_))
        pred = int(p >= thr_)
        ex = VideoGradCAMv2(mdl).explain(x, cls="fake" if pred else "real")      # explain the class the model ASSIGNED (at its calibrated threshold)
        res.append((name, x, p, thr_, pred, ex))
    ref = shared_display_scale([r[5]["raw"] for r in res])
    orig, orig_idx = original_frames_for(folder, [int(i * 2) for i in fidx])          # eval plan = first frame of each 2-frame bin of the 32-frame pool
    legacy = []
    if show_legacy:
        for nm, mdl, x in (("Clean-trained | clean", clean_model, conds[0][4]), ("FGSM-trained | clean", fgsm_eval_model, conds[2][4])):
            _, mp = _cam_maps(mdl, x)
            legacy.append((nm, x, mp))
    nrows = 2 + len(res) + len(legacy)
    fig, axes = plt.subplots(nrows, n_show, figsize=(2.2 * n_show + 3, 2.3 * nrows))
    box = (FACE_BOXES.get(Path(folder).name) or {}).get("box")
    for j, f in enumerate(fidx):
        a0 = axes[0, j]
        a0.imshow(orig[j] if (orig and orig[j] is not None) else np.zeros((8, 8, 3)))
        if orig and orig[j] is not None and box:
            a0.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec="lime", lw=1.2))
        a0.set_title(f"frame {f}", fontsize=9)
        axes[1, j].imshow(np.clip(np.array(IMAGENET_STD) * video[0, f].cpu().numpy().transpose(1, 2, 0) + np.array(IMAGENET_MEAN), 0, 1))
    axes[0, 0].set_ylabel("original sampled frame\n(green = crop box)" if orig else "original frame\n(source video not available)", fontsize=8, rotation=0, ha="right", va="center")
    axes[1, 0].set_ylabel("face crop given\nto the model", fontsize=8, rotation=0, ha="right", va="center")
    r0 = 2
    true_txt = "fake" if label else "real"
    for k, (name, x, p, thr_, pred, ex) in enumerate(res):
        for j, f in enumerate(fidx):
            img = np.clip(np.array(IMAGENET_STD) * x[0, f].detach().cpu().numpy().transpose(1, 2, 0) + np.array(IMAGENET_MEAN), 0, 1)
            heat = to_display(heat_to_crop(ex["raw"][f], img.shape[0]), ref)
            axes[r0 + k, j].imshow(img); axes[r0 + k, j].imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        ok = "correct" if pred == label else "WRONG"
        axes[r0 + k, 0].set_ylabel(f"{name}\ntrue={true_txt} pred={'fake' if pred else 'real'} ({ok})\nP(fake)={p:.2f} thr={thr_:.2f} eps={eps * 255:g}/255\nexplains class '{'fake' if ex['target_class'] else 'real'}'"
                                   + (f" {ex['flags']}" if ex["flags"] else ""), fontsize=7, rotation=0, ha="right", va="center")
    for k, (nm, x, mp) in enumerate(legacy):
        for j, f in enumerate(fidx):
            img = np.clip(np.array(IMAGENET_STD) * x[0, f].detach().cpu().numpy().transpose(1, 2, 0) + np.array(IMAGENET_MEAN), 0, 1)
            axes[r0 + len(res) + k, j].imshow(img); axes[r0 + len(res) + k, j].imshow(heat_to_crop(mp[f], img.shape[0]), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[r0 + len(res) + k, 0].set_ylabel(f"LEGACY v9: {nm}\n(per-frame min-max, fake-logit target)", fontsize=7, rotation=0, ha="right", va="center", color="firebrick")
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Grad-CAM v2, test sample {sample_idx} ({Path(folder).name}); heat = RAW class-score contribution on ONE shared scale (p{CONFIG['gradcam_ref_percentile']:g} = {ref:.3g}). "
                 "Indicates contribution to a prediction; it does not prove manipulation localisation.", fontsize=9)
    path = save_fig(fig, f"fig15c_gradcam_v2_sample{sample_idx}.png")
    plt.show()
    return {"sample": sample_idx, "folder": folder, "ref": ref, "raw": {r[0]: r[5]["raw"] for r in res}, "probs": {r[0]: r[2] for r in res}, "path": path}


def gradcam_quantitative(n_videos=None, eps=None):
    """Faithfulness (deletion vs random vs bottom) and stability under FGSM, on a class-balanced set of TEST videos (diagnostic use only; nothing is tuned on it)."""
    n_videos = n_videos or CONFIG["gradcam_n_videos"]
    eps = eps or CONFIG["target_eval_eps"] * EPS_UNIT
    sel = [i for i, (_, l) in enumerate(test_records) if l == 0][: n_videos // 2] + [i for i, (_, l) in enumerate(test_records) if l == 1][: n_videos // 2]
    rows = []
    for i in tqdm(sel, desc="Grad-CAM quantitative", leave=False):
        folder, label = test_records[i]
        v, _ = test_dataset[i]
        v = v.unsqueeze(0).to(CONFIG["device"]); y = torch.tensor([label], device=CONFIG["device"])
        for tag, mdl, T_, thr_ in (("Clean-trained", clean_model, clean_T, clean_selected_threshold), ("FGSM-trained", fgsm_eval_model, fgsm_T, fgsm_selected_threshold)):
            with torch.no_grad():
                pred = int(float(torch.sigmoid(mdl(v).float() / T_)) >= thr_)
            cam = VideoGradCAMv2(mdl)
            ex = cam.explain(v, cls="fake" if pred else "real")
            xa = fgsm_attack(mdl, v, y, eps)
            ex_a = cam.explain(xa, cls="fake" if pred else "real")
            xr = v + _eps_normalized(eps, v.device) * torch.sign(torch.randn_like(v))
            ex_r = cam.explain(xr, cls="fake" if pred else "real")
            st_adv, iou_adv = explanation_stability(ex["raw"], ex_a["raw"])
            st_rnd, iou_rnd = explanation_stability(ex["raw"], ex_r["raw"])
            for d in deletion_faithfulness(mdl, v, ex["raw"], ex["target_class"]):
                rows.append({"model": tag, "video": i, "label": label, "pred": pred, "correct": pred == label, **d,
                             "stability_spearman_fgsm": st_adv, "stability_iou_fgsm": iou_adv, "stability_spearman_randsign": st_rnd, "stability_iou_randsign": iou_rnd,
                             "flags": ",".join(ex["flags"])})
    return pd.DataFrame(rows)


def summarize_quantitative(df):
    from scipy.stats import wilcoxon
    out = []
    for (tag, f), g in df.groupby(["model", "frac"]):
        def p(a, b):
            try:
                return float(wilcoxon(g[a] - g[b], alternative="greater").pvalue)
            except Exception:
                return float("nan")
        out.append({"model": tag, "frac_deleted": f, "n_videos": len(g), "mean_drop_top": g.drop_top.mean(), "mean_drop_random": g.drop_random.mean(),
                    "mean_drop_bottom": g.drop_bottom.mean(), "p_top_gt_random": p("drop_top", "drop_random"), "p_random_gt_bottom": p("drop_random", "drop_bottom"),
                    "stability_spearman_fgsm": g.stability_spearman_fgsm.mean(), "stability_spearman_randsign": g.stability_spearman_randsign.mean(),
                    "stability_iou_fgsm": g.stability_iou_fgsm.mean(), "stability_iou_randsign": g.stability_iou_randsign.mean()})
    return pd.DataFrame(out)


# ---------------- unit tests (synthetic, run immediately): alignment, class-specificity, frame association, flags ----------------
def _gradcam_unit_tests():
    res = []
    torch.manual_seed(0)
    m = _TinyVid().eval()
    with torch.no_grad():
        m.conv.weight.zero_(); m.conv.bias.zero_(); m.conv.weight[0, :, 1, 1] = 1.0           # channel 0 = local brightness
        m.fc.weight.zero_(); m.fc.weight[0, 0] = 5.0; m.fc.bias.zero_()
    x = torch.zeros(1, 3, 3, 24, 24)
    x[0, 1, :, 4:12, 8:16] = 1.0                                                           # bright square (rows 4..11, cols 8..15) only in frame 1
    ex = VideoGradCAMv2(m, m.conv).explain(x, cls="fake")
    raw = ex["raw"]
    r, c = np.unravel_index(np.argmax(raw[1]), raw[1].shape)
    res.append(("peak of frame 1 lies on the bright square", 4 <= r < 12 and 8 <= c < 16, f"peak at row {r}, col {c}"))
    res.append(("frame association: only the frame with the square has a non-zero map", raw[1].max() > 0 and raw[0].max() == 0 and raw[2].max() == 0, f"max per frame {np.round(raw.max(axis=(1, 2)), 4)}"))
    ex_real = VideoGradCAMv2(m, m.conv).explain(x, cls="real")
    res.append(("class specificity: explaining 'real' (-logit) gives a different (here empty) map than 'fake'", ex_real["raw"].max() == 0 and ex["raw"].max() > 0, f"fake max {ex['raw'].max():.3f}, real max {ex_real['raw'].max():.3f}"))
    ex2 = VideoGradCAMv2(m, m.conv).explain(torch.zeros(1, 3, 3, 24, 24), cls="fake")
    res.append(("degenerate input is flagged, not silently normalised", "all_zero_map" in ex2["flags"], str(ex2["flags"])))
    heat = np.zeros((7, 7), np.float32); heat[2, 5] = 1.0
    box, fhw = (60, 20, 130, 90), (120, 200)                                              # 70x70 crop inside a 120x200 frame
    full = heat_to_original(heat, box, fhw)
    yy, xx = np.unravel_index(np.argmax(full), full.shape)
    exp_x, exp_y = 60 + (5 + 0.5) * 10, 20 + (2 + 0.5) * 10
    res.append(("heat map projects to the correct place in the original frame", abs(xx - exp_x) <= 8 and abs(yy - exp_y) <= 8 and full[:20].max() == 0, f"peak ({xx},{yy}) vs expected ({exp_x:.0f},{exp_y:.0f})"))
    fm = np.zeros((120, 200), np.uint8); fm[40:60, 90:110] = 1
    mc = mask_to_crop(fm, box, 70)
    res.append(("full-frame mask aligns with the crop (same box)", mc.sum() > 0 and mc[:20].sum() == 0 and abs(np.argwhere(mc).mean(axis=0)[1] - 40) < 4, f"mask centre col {np.argwhere(mc).mean(axis=0)[1]:.1f} (expect ~40)"))
    imgs = [np.abs(np.random.RandomState(1).randn(7, 7))] * 3
    s, iou = explanation_stability(imgs, imgs)
    res.append(("stability of identical maps = 1", abs(s - 1) < 1e-9 and iou == 1.0, f"spearman {s:.3f}, IoU {iou:.3f}"))
    out = deletion_faithfulness(m, x, raw, 1, fracs=[0.05], repeats=3)
    res.append(("deletion: removing the attributed region lowers the explained score more than random / least-attributed cells", out[0]["drop_top"] > out[0]["drop_random"] >= out[0]["drop_bottom"] - 1e-9,
                f"drops top/random/bottom = {out[0]['drop_top']:.3f}/{out[0]['drop_random']:.3f}/{out[0]['drop_bottom']:.3f}"))
    print("Grad-CAM v2 unit tests:")
    for n, ok, d in res:
        print(f"  [{'PASS' if ok else 'FAIL'}] {n}  {d}")
    assert all(ok for _, ok, _ in res), "Grad-CAM v2 unit tests failed"
    return res


GRADCAM_TESTS = _gradcam_unit_tests()
MASKS_FOUND = discover_masks()
print("\nGround-truth manipulation masks found under the dataset roots:", MASKS_FOUND)
if not any(MASKS_FOUND.values()):
    print("-> NO masks available: localisation accuracy is NOT evaluated (no masks were fabricated). Only the labelled proxy tests below are run:\n"
          "   deletion faithfulness vs random / least-attributed occlusion, and explanation stability under small FGSM / random-sign perturbations.")
else:
    print("-> mask directories exist, but this notebook does not assume their format: align them with mask_to_crop() using the stored crop boxes and the sampled "
          "frame indices before computing mask_localisation(); this evaluation is PENDING.")
_gc_samples = [next(i for i, (_, l) in enumerate(test_records) if l == 1), next(i for i, (_, l) in enumerate(test_records) if l == 0)]
GRADCAM_FIGS = [gradcam_v2_figure(i) for i in _gc_samples]
np.savez_compressed(os.path.join(OUT_DIR, "gradcam_raw_maps_v10.npz"),
                    **{f"s{g['sample']}|{k}": v for g in GRADCAM_FIGS for k, v in g["raw"].items()}, **{f"s{g['sample']}|ref_scale": np.array(g["ref"]) for g in GRADCAM_FIGS})
GC_Q = gradcam_quantitative()
GC_Q.to_csv(os.path.join(OUT_DIR, "gradcam_quantitative_raw_v10.csv"), index=False)
GC_SUM = summarize_quantitative(GC_Q)
GC_SUM.to_csv(os.path.join(OUT_DIR, "gradcam_quantitative_summary_v10.csv"), index=False)
print("\nEXPLANATION FAITHFULNESS / STABILITY (proxy tests, test videos, diagnostic only):")
print(GC_SUM.round(4).to_string(index=False))
print("Reading: mean_drop_top > mean_drop_random > mean_drop_bottom (paired Wilcoxon p) supports that the map marks regions the model USES; it says nothing about whether "
      "they coincide with the manipulation. Stability under FGSM vs random-sign noise shows how much the explanation itself is attacked.")


In [ ]:
# ============================================================
# CELL 16: Version comparison v4 -> v9 (values copied from the executed v4-v7 notebooks / v4 review; nothing re-estimated)
# ============================================================
# "Not reported" = the earlier version never printed that number. "Not directly comparable" = different test set / subset size / threshold
# rule. v5-v7 use the SAME identity-disjoint split and caps as v8 ('Clean (full test)' = 400 videos, validation-selected threshold); their
# FGSM numbers were measured on a balanced 200- or 300-video SUBSET, v8/v9 on the whole test set. Iterative-attack results of earlier versions
# are intentionally not tabulated here: v8/v9 are FGSM-only, and FGSM numbers of an adversarially trained model must not be read as robustness.
NR = "Not reported"
HIST = [
    # version, model, n_fgsm_videos, clean acc (full test), clean AUC, FGSM acc 1,2,4,8 ; FGSM AUC 1,2,4,8 ; comparability
    ("v4", "Clean-trained", 300, 0.870, 0.935, NR, 0.003, 0.010, 0.370, NR, NR, NR, NR, "Not directly comparable (non-identity-disjoint split, 300-video test set)"),
    ("v4", "ART (FGSM+label-leaking)", 300, NR, NR, NR, 0.930, 0.920, 0.620, NR, NR, NR, NR, "Not directly comparable; accuracy ABOVE clean = label-leaking signature"),
    ("v5", "Clean-trained", 300, 0.868, 0.9354, NR, 0.000, 0.0067, 0.100, NR, 0.0000, 0.0003, 0.0097, "Clean numbers comparable; FGSM subset 300"),
    ("v5", "ART fine-tuned", 300, 0.865, 0.9272, NR, 0.500, 0.500, 0.500, NR, 0.4368, 0.4864, 0.7533, "Clean numbers comparable; FGSM subset 300"),
    ("v6", "Clean-trained", 200, 0.875, 0.9456, NR, 0.000, 0.010, 0.035, NR, 0.0000, 0.0008, 0.0037, "Clean numbers comparable; FGSM subset 200"),
    ("v6", "ART fine-tuned", 200, 0.858, 0.9373, NR, 0.495, 0.500, 0.480, NR, 0.8503, 0.8119, 0.6530, "Clean numbers comparable; FGSM subset 200"),
    ("v7", "Clean-trained", 200, 0.870, 0.9457, 0.000, 0.000, 0.005, 0.055, 0.0000, 0.0000, 0.0001, 0.0061, "Clean numbers comparable; FGSM subset 200"),
    ("v7", "ART fine-tuned", 200, 0.848, 0.9158, 0.215, 0.220, 0.230, 0.255, 0.0738, 0.0678, 0.0762, 0.1516, "Clean numbers comparable; FGSM subset 200"),
    # v8 values copied from the user's executed v8 notebook (full test set, 400 videos, clean-EER validation-selected threshold)
    ("v8", "Clean-trained (FGSM-only)", 400, 0.9025, 0.9569, 0.0000, 0.0000, 0.0125, 0.0400, 0.0000, 0.0000, 0.0000, 0.0025, "Same split, full test set"),
    ("v8", "FGSM-trained (FGSM-only)", 400, 0.8250, 0.8731, 0.5375, 0.5150, 0.4975, 0.4950, 0.3982, 0.3688, 0.3301, 0.3853, "Same split, full test set; AUC < 0.5 = hedged model attacked with true labels"),
]
cols = ["version", "model", "n_videos_fgsm", "clean_acc_full_test", "clean_auc_full_test", "fgsm1_acc", "fgsm2_acc", "fgsm4_acc", "fgsm8_acc",
        "fgsm1_auc", "fgsm2_auc", "fgsm4_auc", "fgsm8_auc", "comparability"]
cmp_rows = [dict(zip(cols, r)) for r in HIST]


def _v9_row(tag):
    g = lambda e, c, thr="val_selected": float(metrics_df[(metrics_df.model == tag) & np.isclose(metrics_df.eps_255, e) & (metrics_df.threshold_type == thr)][c].iloc[0])
    return {"version": "v9", "model": tag + " (FGSM-only)", "n_videos_fgsm": n_test, "clean_acc_full_test": g(0, "accuracy"),
            "clean_auc_full_test": g(0, "roc_auc"), **{f"fgsm{e}_acc": g(e, "accuracy") for e in (1, 2, 4, 8)},
            **{f"fgsm{e}_auc": g(e, "roc_auc") for e in (1, 2, 4, 8)},
            "comparability": "Clean numbers comparable (same split); FGSM measured on the FULL test set"}


for t in MODELS:
    cmp_rows.append(_v9_row(t))
compare_df = pd.DataFrame(cmp_rows)
compare_df.to_csv(os.path.join(OUT_DIR, "fgsm_v9_version_comparison.csv"), index=False)
print(compare_df.drop(columns=["comparability"]).to_string(index=False))
print("\nComparability notes:")
for r in cmp_rows:
    print(f"  {r['version']} {r['model']}: {r['comparability']}")
print("\nReading guide: 'FGSM acc' of an adversarially trained model that is HIGHER than its clean accuracy (v4) or pinned at 0.50 with "
      "AUC near chance (v5/v6) is hedging / label-leaking, not robustness. v9's Cell 14 measures these signatures directly.")

In [ ]:
# ============================================================
# CELL 17: Verdict against the criteria that were frozen BEFORE the test set was opened
# ============================================================
crit = json.load(open(CONFIG["acceptance_json"]))
assert hashlib.md5(open(CONFIG["acceptance_json"], "rb").read()).hexdigest()[:12] == CRITERIA_HASH, "acceptance_criteria.json was modified after freezing"
g = lambda tag, e, c, thr="val_selected": float(metrics_df[(metrics_df.model == tag) & np.isclose(metrics_df.eps_255, e) & (metrics_df.threshold_type == thr)][c].iloc[0])
tgt, trn = CONFIG["target_eval_eps"], FGSM_EPS * 255
res = {}
auc_c, auc_f = g("Clean-trained", 0, "roc_auc", "0.5"), g("FGSM-trained", 0, "roc_auc", "0.5")
res["C1"] = (auc_f >= CONFIG["accept_min_test_clean_auc_ratio"] * auc_c, f"clean test AUC FGSM-trained {auc_f:.4f} vs clean-trained {auc_c:.4f} (ratio {auc_f / auc_c:.3f}, need >= {CONFIG['accept_min_test_clean_auc_ratio']})")
res["C2"] = (crit["validation_evidence_already_known"]["C2_met"], f"validation FGSM AUC @target = {crit['validation_evidence_already_known']['C2_value']:.4f}")
res["C3"] = (crit["validation_evidence_already_known"]["C3_met"], f"validation FGSM AUC gain over clean-trained = {crit['validation_evidence_already_known']['C3_value']:+.4f}")
own_clean = g("FGSM-trained", 0, "accuracy")
leaks = [(e, g("FGSM-trained", e, "accuracy")) for e in CONFIG["eval_eps_grid"] if g("FGSM-trained", e, "accuracy") > own_clean + CONFIG["accept_max_label_leak_margin"]]
res["C4"] = (not leaks, f"FGSM accuracy above clean accuracy + margin at: {leaks if leaks else 'no epsilon'}")


def _delta_lo(e):
    r = paired_df[np.isclose(paired_df.eps_255, e)].iloc[0]
    return r["auc_delta"], r["auc_delta_ci_lo"], r["auc_delta_ci_hi"]


dt, dr = _delta_lo(tgt), _delta_lo(trn)
res["C5"] = (dt[1] > 0 and dr[1] > 0, f"AUC delta @{tgt:g}/255 = {dt[0]:+.4f} [{dt[1]:+.4f}, {dt[2]:+.4f}]; @{trn:g}/255 = {dr[0]:+.4f} [{dr[1]:+.4f}, {dr[2]:+.4f}]")
nan_total = int(fgsm_df["train_nan_skips"].sum())     # steps that stayed non-finite even in fp32 and were skipped
res["C6"] = (EVAL_DIAG["zero_grad_samples"] == 0 and EVAL_DIAG["nonfinite_grad_samples"] == 0 and nan_total == 0,
             f"eval zero-grad {EVAL_DIAG['zero_grad_samples']}, eval non-finite grads {EVAL_DIAG['nonfinite_grad_samples']}, training NaN-skipped steps {nan_total}")
_tt = metrics_df[(metrics_df.model == "FGSM-trained") & np.isclose(metrics_df.eps_255, tgt) & (metrics_df.threshold_type == "attack_aware")].iloc[0]
_rec, _spc = _tt["n_fake_detected"] / _tt["n_fake"], _tt["n_real_kept"] / _tt["n_real"]
res["C7"] = (_rec >= CONFIG["accept_min_class_recall"] and _spc >= CONFIG["accept_min_class_recall"],
             f"@{tgt:g}/255, attack-aware threshold {THR_AA['FGSM-trained']:.3f}: fake recall {int(_tt['n_fake_detected'])}/{int(_tt['n_fake'])} = {_rec:.3f}, "
             f"real specificity {int(_tt['n_real_kept'])}/{int(_tt['n_real'])} = {_spc:.3f} (need both >= {CONFIG['accept_min_class_recall']})")
_c_ov = metrics_df[(metrics_df.model == "FGSM-trained") & np.isclose(metrics_df.eps_255, 0) & (metrics_df.threshold_type == "val_selected")].iloc[0]["accuracy"]
_c_cl = metrics_df[(metrics_df.model == "Clean-trained") & np.isclose(metrics_df.eps_255, 0) & (metrics_df.threshold_type == "val_selected")].iloc[0]["accuracy"]
print(f"Clean accuracy gap (test, clean-EER thr): clean-trained {_c_cl:.4f} vs FGSM-trained {_c_ov:.4f} -> gap {(_c_cl - _c_ov) * 100:.2f} pts (v8: 7.75 pts). Reported, not a pass/fail criterion.")
print("ACCEPTANCE CRITERIA (frozen before test access; C8 follows Cell 18)")
for k, (ok, msg) in res.items():
    print(f"  {k}: {'MET ' if ok else 'NOT MET'} | {crit['criteria'][{'C1':'C1_clean_auc_retained','C2':'C2_val_fgsm_auc_target','C3':'C3_val_fgsm_auc_gain','C4':'C4_no_label_leak','C5':'C5_paired_gain','C6':'C6_numerics','C7':'C7_class_balance_under_attack'}[k]]}\n          -> {msg}")
n_met = sum(ok for ok, _ in res.values())
if crit["gate_never_passed"]:
    VERDICT = "FAILED (no epoch ever passed the validation gates)"
elif n_met == len(res):
    VERDICT = "SUCCEEDED against FGSM on the main seed (C1-C7 met; C8 multi-seed pending Cell 18)"
elif res["C5"][0] and res["C4"][0] and res["C6"][0]:
    VERDICT = f"PARTIAL against FGSM ({n_met}/{len(res)} criteria met; a measurable FGSM gain exists but a stricter criterion failed)"
else:
    VERDICT = f"FAILED against FGSM ({n_met}/{len(res)} criteria met)"
print(f"\nVERDICT: {VERDICT}")
print("SCOPE: this is an FGSM-ONLY experiment. PGD / iterative attacks were deliberately excluded, so NO claim of general adversarial "
      "robustness is made; a model can pass every FGSM criterion and still be fully breakable by multi-step attacks.")

summary = {"verdict": VERDICT, "criteria": {k: {"met": bool(ok), "evidence": msg} for k, (ok, msg) in res.items()},
           "train_eps_255": trn, "target_eps_255": tgt, "selected_epoch": int(fgsm_ckpt["epoch"]), "n_test_videos": n_test,
           "criteria_hash": CRITERIA_HASH, "findings": FINDINGS, "audit_verdicts": AUDIT["verdicts"], "attack_aware_thresholds": ATTACK_AWARE_THR,
           "nan_events_first": [e for h in fgsm_history for e in h.get("train_nan_events", [])][:5],
           "key_numbers": {"clean_test_auc_clean_trained": auc_c, "clean_test_auc_fgsm_trained": auc_f,
                           **{f"{tag}_acc_eps{e:g}": g(tag, e, "accuracy") for tag in MODELS for e in (0, tgt, trn, 8)},
                           **{f"{tag}_auc_eps{e:g}": g(tag, e, "roc_auc", "0.5") for tag in MODELS for e in (0, tgt, trn, 8)}}}
atomic_json_save(summary, os.path.join(OUT_DIR, "fgsm_v9_results_summary.json"))

with open(os.path.join(OUT_DIR, "fgsm_v9_final_summary.md"), "w") as f:
    f.write(f"# v9 FGSM-only adversarial training -- final summary (auto-generated from this run)\n\n**Verdict:** {VERDICT}\n\n")
    f.write(f"Training epsilon {trn:g}/255, target epsilon {tgt:g}/255, selected epoch {fgsm_ckpt['epoch']}, test videos {n_test}.\n\n")
    f.write("| epsilon (/255) | clean-trained acc | FGSM-trained acc | clean-trained AUC | FGSM-trained AUC |\n|---|---|---|---|---|\n")
    for e in CONFIG["eval_eps_grid"]:
        f.write(f"| {e:g} | {g('Clean-trained', e, 'accuracy'):.3f} | {g('FGSM-trained', e, 'accuracy'):.3f} | "
                f"{g('Clean-trained', e, 'roc_auc', '0.5'):.3f} | {g('FGSM-trained', e, 'roc_auc', '0.5'):.3f} |\n")
    f.write("\nCriteria:\n" + "\n".join(f"- {k}: {'met' if ok else 'NOT met'} -- {msg}" for k, (ok, msg) in res.items()))
    f.write("\n\nScope: FGSM only. PGD was deliberately excluded; nothing here supports a claim of general adversarial robustness.\n")
print("\nWritten:", os.path.join(OUT_DIR, "fgsm_v9_final_summary.md"))

In [ ]:
# ============================================================
# CELL 18: Extra training seeds (time-guarded) and runtime / memory report
# ============================================================
# Extra seeds retrain BOTH models (clean control and FGSM-AT, same split, same epsilon, same epochs) and are evaluated on the same
# test set at CONFIG['multiseed_eps_grid']. A seed is started only if its predicted duration fits in the remaining budget; otherwise it
# is skipped and the report says so. Seeds are never selected or filtered by their test results.

def _est_seed_hours(mode):
    fg = RUNTIME_LOG.get("fgsm_training", {"seconds": 0})["seconds"]
    cl = RUNTIME_LOG.get("clean_training", {"seconds": 0})["seconds"]
    if cl == 0:                       # clean control was re-used from a checkpoint -> assume a fresh clean run costs ~ the FGSM run
        cl = 0.9 * fg
    ev = sum(v["seconds"] for k, v in RUNTIME_LOG.items() if k.startswith("test_eval_")) / 2 * (len(CONFIG["multiseed_eps_grid"]) / len(CONFIG["eval_eps_grid"]))
    n_models = 2 if mode == "full" else 1
    return (fg + (cl if mode == "full" else 0) + ev * n_models + 420) / 3600


SEED_ROWS = []
main_rows = metrics_df[metrics_df.threshold_type == "val_selected"]
for tag in MODELS:
    for e in CONFIG["multiseed_eps_grid"]:
        r = main_rows[(main_rows.model == tag) & np.isclose(main_rows.eps_255, e)].iloc[0]
        ra = metrics_df[(metrics_df.model == tag) & (metrics_df.threshold_type == "attack_aware") & np.isclose(metrics_df.eps_255, e)].iloc[0]
        SEED_ROWS.append({"seed": CONFIG["seed"], "mode": "main", "gate_passed": (not FGSM_GATE_NEVER_PASSED) if tag == "FGSM-trained" else True,
                          "model": tag, "eps_255": e, "accuracy": r["accuracy"],
                          "roc_auc": safe_roc_auc(RAW[tag][0], sigmoid_np(RAW[tag][1][e * EPS_UNIT], MODELS[tag][1])), "f1": r["f1"],
                          "fake_recall_attack_aware": ra["n_fake_detected"] / ra["n_fake"], "real_spec_attack_aware": ra["n_real_kept"] / ra["n_real"]})
SEEDS_RUN, SEEDS_SKIPPED, SEED_GATES = [CONFIG["seed"]], [], {CONFIG["seed"]: (not FGSM_GATE_NEVER_PASSED)}
# Mode per extra seed (time-guarded, decided before the seed starts):
#   "full"       : retrain the clean control AND the FGSM model with that seed (cleanest; ~2x cost)
#   "fgsm_only"  : re-use the main clean checkpoint, retrain only FGSM-AT with that seed (seed variance of the FGSM stage only)
for sd in CONFIG["extra_seeds"]:
    elapsed_h = (time.time() - NOTEBOOK_START) / 3600
    left = CONFIG["time_budget_hours"] - elapsed_h - CONFIG["seed_reserve_hours"]
    _re = v2_reuse_seed(sd)                  # v2: verified v1 artifacts of this seed -> no retraining, no time guard needed
    mode = "full" if _re is not None else next((m for m in ("full", "fgsm_only") if _est_seed_hours(m) <= left), None)
    if mode is None:
        SEEDS_SKIPPED.append((sd, f"'fgsm_only' needs ~{_est_seed_hours('fgsm_only'):.2f} h, only {left:.2f} h left after the reserve"))
        print(f"Seed {sd} SKIPPED: {SEEDS_SKIPPED[-1][1]}")
        continue
    print(f"\n##### Extra seed {sd}: mode '{mode}', estimated {_est_seed_hours(mode):.2f} h; elapsed {elapsed_h:.2f} h")
    paths = {k: os.path.join(OUT_DIR, f"seed{sd}_{k}.pth") for k in ("clean", "fgsm", "soft", "resume_clean", "resume_fgsm")}
    with timed(f"seed{sd}"):
        if _re is None:
            if mode == "full":
                cm_, ch_, cb_ = train_clean_model(sd, paths["clean"], paths["resume_clean"])
                del cm_
                clean_src = paths["clean"]
            else:
                clean_src = CONFIG["save_clean_checkpoint"]
            set_global_seed(sd)
            sm = new_fgsm_model_from_clean(clean_src)
            fh_, fb_ = run_fit(sm, f"FGSM-AT seed{sd}", "fgsm", E, train_loader_fgsm, SPE, CONFIG["fgsm_lr_scale"], CONFIG["fgsm_warmup_steps"],
                               fgsm_adv_cfg(FGSM_EPS, CONFIG["fgsm_eps_ramp_epochs"], SPE, CONFIG["adv_random_eps"], CONFIG["alp_weight"]),
                               make_fgsm_eval_fn(True), make_fgsm_select_fn(True), CONFIG["ema_decay_fgsm"], fgsm_clip_cfg(),
                               best_path=paths["fgsm"], soft_path=paths["soft"], resume_path=paths["resume_fgsm"],
                               early_stop={"min_epochs": CONFIG["early_stop_min_epochs"], "bad_epochs": CONFIG["early_stop_bad_epochs"],
                                           "no_gain": CONFIG["early_stop_no_gain"]},
                               extra_meta={"train_eps": FGSM_EPS, "seed": sd})
            del sm
            free_cuda_memory()
            passed = fb_["epoch"] is not None
        else:
            passed = _re["passed"]
            print(f"Seed {sd}: v1 artifacts re-used (no retraining); selected epoch {_re['best_epoch']}, gate passed at selection: {passed}")
        SEED_GATES[sd] = passed
        if not passed:
            print(f"Seed {sd}: FGSM validation gate never passed -> counted as a FAILED seed (soft-best weights evaluated for transparency).")
            fg_path = paths["soft"]
        else:
            fg_path = paths["fgsm"]
        todo = [("FGSM-trained", fg_path)] + ([("Clean-trained", paths["clean"])] if mode == "full" else [])
        for tag_, path_ in todo:
            if not Path(path_).exists():
                continue
            mdl = build_model()
            mdl.load_state_dict(torch.load(path_, map_location=CONFIG["device"], weights_only=False)["model_state_dict"])
            mdl.eval()
            T_, thr_, _ = calibrate_and_threshold(mdl, f"{tag_} seed{sd}")
            thr_aa = attack_aware_threshold(mdl, T_, f"{tag_} seed{sd}")
            lab_, zz_ = fgsm_sweep_logits(mdl, test_attack_loader, [e / 255 for e in CONFIG["multiseed_eps_grid"]], desc=f"seed{sd} {tag_}")
            for e in CONFIG["multiseed_eps_grid"]:
                p_ = sigmoid_np(zz_[e / 255], T_)
                m_ = compute_metrics(lab_, p_, thr_)
                SEED_ROWS.append({"seed": sd, "mode": mode, "gate_passed": passed if tag_ == "FGSM-trained" else True, "model": tag_, "eps_255": e,
                                  "accuracy": m_["accuracy"], "roc_auc": m_["roc_auc"], "f1": m_["f1"],
                                  "fake_recall_attack_aware": float(np.mean(p_[lab_ == 1] >= thr_aa)),
                                  "real_spec_attack_aware": float(np.mean(p_[lab_ == 0] < thr_aa))})
            del mdl
            free_cuda_memory()
    SEEDS_RUN.append(sd)

seed_df = pd.DataFrame(SEED_ROWS)
seed_df.to_csv(os.path.join(OUT_DIR, "fgsm_v9_multiseed.csv"), index=False)
print(f"\nSEEDS ACTUALLY RUN: {SEEDS_RUN} (n={len(SEEDS_RUN)}); skipped: {SEEDS_SKIPPED if SEEDS_SKIPPED else 'none'}")
if len(SEEDS_RUN) < 3:
    print(f"!! Only {len(SEEDS_RUN)} seed(s) ran. Fewer than 3 seeds: NO seed-to-seed variance can be estimated, every number above is a single-run value, "
          "and differences smaller than the CIs must not be interpreted.")
agg = []
for tag in MODELS:
    for e in CONFIG["multiseed_eps_grid"]:
        s = seed_df[(seed_df.model == tag) & (seed_df.eps_255 == e)]
        agg.append({"model": tag, "eps_255": e, "n_seeds": len(s), "accuracy": mean_std_str(s.accuracy), "roc_auc": mean_std_str(s.roc_auc), "f1": mean_std_str(s.f1)})
print(pd.DataFrame(agg).to_string(index=False))

# ---- C8 and the FINAL verdict (the single-seed verdict of Cell 17 is kept above for transparency) ----
n_pass = int(sum(bool(v) for v in SEED_GATES.values()))
c8_met = n_pass >= CONFIG["accept_min_seeds_passing"]
print(f"\nC8 multi-seed: FGSM validation gate passed in {n_pass}/{len(SEED_GATES)} seeds {SEED_GATES} -> {'MET' if c8_met else 'NOT MET'} (need >= {CONFIG['accept_min_seeds_passing']}). "
      f"Seeds that did not run: {[s_[0] for s_ in SEEDS_SKIPPED] if SEEDS_SKIPPED else 'none'}.")
summary["criteria"]["C8"] = {"met": bool(c8_met), "evidence": f"gate passed in {n_pass}/{len(SEED_GATES)} seeds {SEED_GATES}; skipped: {SEEDS_SKIPPED}"}
if crit["gate_never_passed"]:
    VERDICT_FINAL = VERDICT
elif all(ok for ok, _ in res.values()) and c8_met:
    VERDICT_FINAL = f"SUCCEEDED against FGSM (C1-C8 met, {len(SEED_GATES)} seed(s))"
elif all(ok for ok, _ in res.values()):
    VERDICT_FINAL = "PARTIAL against FGSM (C1-C7 met on the main seed; C8 multi-seed NOT met -> single-model result, do not generalise)"
else:
    VERDICT_FINAL = VERDICT + (" | C8 met" if c8_met else " | C8 NOT met")
summary["verdict_final"] = VERDICT_FINAL
atomic_json_save(summary, os.path.join(OUT_DIR, "fgsm_v9_results_summary.json"))
with open(os.path.join(OUT_DIR, "fgsm_v9_final_summary.md"), "a") as f:
    f.write(f"\n\n**C8 (multi-seed):** {'met' if c8_met else 'NOT met'} -- {summary['criteria']['C8']['evidence']}\n\n**Final verdict:** {VERDICT_FINAL}\n")
print(f"FINAL VERDICT: {VERDICT_FINAL}")
print("SCOPE REMINDER: FGSM only; PGD / iterative attacks were deliberately excluded, so no general adversarial robustness is claimed.")

# ---- runtime / memory ----
total_h = (time.time() - NOTEBOOK_START) / 3600
rt = pd.DataFrame([{"section": k, "minutes": v["seconds"] / 60, "peak_gpu_mb": v["peak_gpu_mb"]} for k, v in RUNTIME_LOG.items()])
print("\nRUNTIME / MEMORY (this session; sections re-used from checkpoints are not timed):")
print(rt.round(1).to_string(index=False) if len(rt) else "(no timed sections)")
print(f"Total wall-clock since notebook start: {total_h:.2f} h (budget {CONFIG['time_budget_hours']} h)")
atomic_json_save({"sections": RUNTIME_LOG, "total_hours": total_h, "seeds_run": SEEDS_RUN, "seeds_skipped": SEEDS_SKIPPED,
                  "fgsm_epochs_effective": CONFIG["fgsm_epochs_effective"]}, os.path.join(OUT_DIR, "runtime_log.json"))
print("\nOUTPUT FILES in", OUT_DIR)
for p in sorted(Path(OUT_DIR).glob("*")):
    if p.is_file() and p.suffix in (".csv", ".json", ".md", ".pth"):
        print(f"  {p.name:<42} {p.stat().st_size / 1024:>10.1f} KB")

In [ ]:
# ============================================================
# CELL 18c (v2): PRIORITY 5 -- checkpoint provenance, clean/FGSM identity check, and reconciliation of the validation-gate status of every seed
# ============================================================
# Question answered here: seed 2 printed "EARLY STOP ... gate failed 4 epochs in a row ... validation FGSM metrics unusable" (Cell 18) while the summary says
# the gate PASSED in 3/3 seeds. Both statements come from the same code path: `passed = best['epoch'] is not None` is True as soon as ONE epoch passed all gates
# (that epoch's EMA weights are the saved checkpoint), while the early-stop message refers to the LAST epochs. This cell recomputes every gate component for every
# epoch from the saved per-epoch validation rows, so the status is established from the logs and checkpoint metadata, not from a printed label.
# No test data is read in this cell (only validation, for the behavioural identity check).
import hashlib as _hl


def sha256_file(path, chunk=1 << 22):
    h = _hl.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def _cfg_hash(cfg):
    try:
        return _hl.md5(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:10]
    except Exception:
        return "n/a"


def _ckpt_candidates():
    c = [("clean", 42, CONFIG["save_clean_checkpoint"]), ("fgsm", 42, CONFIG["save_fgsm_checkpoint"]), ("fgsm_soft", 42, os.path.join(OUT_DIR, "soft_best_fgsm_model.pth"))]
    for sd in CONFIG["extra_seeds"]:
        for k in ("clean", "fgsm", "soft"):
            c.append((k if k != "soft" else "fgsm_soft", sd, os.path.join(OUT_DIR, f"seed{sd}_{k}.pth")))
    return [(r, s, p) for r, s, p in c if os.path.exists(p)]


def gate_components(row, base_clean_auc):
    """The EXACT gate of make_fgsm_select_fn (Cell 8), decomposed. Returns {component: bool}."""
    fin = all(row.get(k, float("nan")) == row.get(k, float("nan")) for k in ("val_clean_auc", "val_fgsm_mean_auc", "val_fgsm_min_auc", "val_fgsm_target_auc"))
    return {"finite": fin,
            "clean_auc_kept": bool(row["val_clean_auc"] >= base_clean_auc - CONFIG["select_max_clean_auc_drop"]),
            "fgsm_mean_auc_ok": bool(row["val_fgsm_mean_auc"] >= CONFIG["select_min_fgsm_auc"]),
            "fgsm_target_auc_ok": bool(row["val_fgsm_target_auc"] >= CONFIG["select_min_fgsm_auc"]),
            "no_hedge": bool(row["val_clean_logit_std"] >= CONFIG["select_min_logit_std"]),
            "no_label_leak": bool(row["val_leak_margin"] <= CONFIG["accept_max_label_leak_margin"])}


def _val_profile_for(path):
    m = build_model()
    m.load_state_dict(torch.load(path, map_location=CONFIG["device"], weights_only=False)["model_state_dict"])
    m.eval()
    lab, zz = fgsm_sweep_logits(m, val_adv_loader, [0.0, float(TARGET)], desc="identity check")
    out = {"val_subset_auc_clean": safe_roc_auc(lab, zz[0.0]), "val_subset_auc_fgsm_target": safe_roc_auc(lab, zz[float(TARGET)])}
    del m
    free_cuda_memory()
    return out


# ---------------- 1. provenance table ----------------
PROV = []
for role, sd, path in _ckpt_candidates():
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck.get("meta", {}) or {}
    kind_expected = "clean" if role == "clean" else "fgsm"
    ok, why = check_provenance(ck, kind_expected)
    vrow = ck.get("val_row", {}) or {}
    row = {"file": os.path.basename(path), "role": role, "seed": meta.get("seed", sd), "kind_in_meta": meta.get("kind"), "architecture": "EfficientNet-B1 + 2-layer LSTM (+temporal attention)"
           if CONFIG.get("use_temporal_attention") else "EfficientNet-B1 + LSTM", "arch_fingerprint": meta.get("arch_fingerprint"),
           "split_id": meta.get("split_fingerprint"), "split_matches_current": meta.get("split_fingerprint") == SPLIT_FINGERPRINT,
           "provenance_ok": ok, "provenance_note": why, "selected_epoch": ck.get("epoch"), "weights": ck.get("weights"),
           "val_selection_score": ck.get("val_score"), "val_clean_auc_at_epoch": vrow.get("val_clean_auc", vrow.get("val_roc_auc")),
           "val_fgsm_target_auc_at_epoch": vrow.get("val_fgsm_target_auc"), "train_eps_255": (meta.get("train_eps") or float("nan")) * 255 if meta.get("train_eps") else float("nan"),
           "gate_never_passed": bool(meta.get("gate_never_passed", False)), "config_hash": _cfg_hash(meta.get("config")), "size_mb": round(os.path.getsize(path) / 1e6, 1),
           "sha256": sha256_file(path), "threshold_selection": "see frozen_thresholds_v2.json (Cell 18d): clean-val EER, attack-aware (validation)"}
    PROV.append(row)
    del ck
PROV_DF = pd.DataFrame(PROV)

# ---------------- 2. behavioural identity: is the 'clean' checkpoint really non-robust and the 'FGSM' checkpoint robust? (validation subset, no test data) ----------------
for i, r in PROV_DF.iterrows():
    if r["role"] in ("clean", "fgsm"):
        prof = _val_profile_for(os.path.join(OUT_DIR, r["file"]))
        PROV_DF.loc[i, "val_subset_auc_clean"] = prof["val_subset_auc_clean"]
        PROV_DF.loc[i, "val_subset_auc_fgsm_1_255"] = prof["val_subset_auc_fgsm_target"]
        behaves_robust = prof["val_subset_auc_fgsm_target"] >= 0.5
        PROV_DF.loc[i, "behaviour_matches_role"] = bool((r["role"] == "fgsm") == behaves_robust)
PROV_DF.to_csv(os.path.join(OUT_DIR, "checkpoint_provenance_v2.csv"), index=False)
print("CHECKPOINT PROVENANCE (sha256 shortened):")
_show = PROV_DF.copy(); _show["sha256"] = _show.sha256.str[:12]
print(_show[["file", "role", "seed", "kind_in_meta", "selected_epoch", "val_selection_score", "split_matches_current", "provenance_ok", "gate_never_passed",
             "val_subset_auc_clean", "val_subset_auc_fgsm_1_255", "behaviour_matches_role", "sha256"]].round(4).to_string(index=False))
assert PROV_DF.provenance_ok.all(), "a checkpoint on disk fails the provenance check (wrong split / architecture / kind) -- do not evaluate it"
_bad = PROV_DF[PROV_DF.behaviour_matches_role == False]
# The authoritative identity check is the stored provenance 'kind' (asserted above via check_provenance). The behavioural probe is a weak sanity signal on a small validation
# subset (a clean model can by chance keep AUC >= 0.5 under 1/255 FGSM), so it WARNS instead of failing the run.
if not _bad.empty:
    print(f"WARNING (behavioural probe only; provenance 'kind' is consistent): {_bad.file.tolist()} did not behave as the weak role heuristic expects -- inspect val_subset_auc_* above.")
for sd in CONFIG["v2_seeds"]:       # clean and FGSM checkpoint of one seed must carry the same seed tag
    s = PROV_DF[(PROV_DF.seed == sd) & PROV_DF.role.isin(["clean", "fgsm"])]
    assert s.seed.nunique() <= 1, f"seed {sd}: clean / FGSM checkpoints carry different seeds"
print("Verified: every checkpoint was trained on THIS split and architecture; stored kind = role for clean/FGSM files; behavioural probe reported above; seeds consistent.")

# ---------------- 3. gate reconciliation per seed ----------------
def _history_of(sd):
    if sd == CONFIG["seed"]:
        p = os.path.join(OUT_DIR, "fgsm_training_history.json")
        return json.load(open(p)) if os.path.exists(p) else None
    p = os.path.join(OUT_DIR, f"seed{sd}_resume_fgsm.pth")
    if os.path.exists(p):
        return torch.load(p, map_location="cpu", weights_only=False).get("history")
    return None


def _clean_val_auc_of(sd):
    p = CONFIG["save_clean_checkpoint"] if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_clean.pth")
    return float(torch.load(p, map_location="cpu", weights_only=False)["val_score"]) if os.path.exists(p) else float("nan")


GATE_ROWS, SEED_SUMMARY = [], []
for sd in CONFIG["v2_seeds"]:
    hist = _history_of(sd)
    fp = CONFIG["save_fgsm_checkpoint"] if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_fgsm.pth")
    sp = os.path.join(OUT_DIR, "soft_best_fgsm_model.pth") if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_soft.pth")
    if not os.path.exists(fp) and os.path.exists(sp):
        fp = sp                                                 # the gate never passed: only the soft-best (failure-analysis) file exists for this seed
    if hist is None or not os.path.exists(fp):
        SEED_SUMMARY.append({"seed": sd, "status": "UNKNOWN (no per-epoch history or checkpoint on disk)", "valid_seed": False})
        continue
    base_main = BASELINE_VAL["clean_val_auc_full"]            # the baseline Cell 18 actually used for EVERY seed's gate
    base_own = _clean_val_auc_of(sd)                            # the seed's OWN clean-trained validation AUC
    ck = torch.load(fp, map_location="cpu", weights_only=False)
    sel_epoch = ck.get("epoch")
    gate_never = bool(ck.get("meta", {}).get("gate_never_passed", False))
    ok_epochs, rows = [], []
    for r in hist:
        gc_main = gate_components(r, base_main)
        gc_own = gate_components(r, base_own)
        gok_main, gok_own = all(gc_main.values()), all(gc_own.values())
        stored = r.get("gate_ok")
        rows.append({"seed": sd, "epoch": r["epoch"], "val_clean_auc": r["val_clean_auc"], "val_fgsm_target_auc": r["val_fgsm_target_auc"], "val_fgsm_mean_auc": r["val_fgsm_mean_auc"],
                     "val_leak_margin": r["val_leak_margin"], "val_clean_logit_std": r["val_clean_logit_std"], "gate_ok_recomputed_main_baseline": gok_main,
                     "gate_ok_recomputed_own_baseline": gok_own, "gate_ok_stored": stored,
                     "failing_components_main_baseline": ",".join(k for k, v in gc_main.items() if not v) or "-",
                     "failing_components_own_baseline": ",".join(k for k, v in gc_own.items() if not v) or "-", "is_selected_epoch": (r["epoch"] == sel_epoch and not gate_never)})
        if gok_main:
            ok_epochs.append(r["epoch"])
    GATE_ROWS += rows
    last = rows[-1]
    sel_row = next((x for x in rows if x["epoch"] == sel_epoch), None)
    n_stored_mismatch = sum(1 for x in rows if x["gate_ok_stored"] is not None and bool(x["gate_ok_stored"]) != x["gate_ok_recomputed_main_baseline"])
    sel_ok_main = bool(sel_row and sel_row["gate_ok_recomputed_main_baseline"])
    sel_ok_own = bool(sel_row and sel_row["gate_ok_recomputed_own_baseline"])
    late_fail = [x["epoch"] for x in rows if x["epoch"] > (sel_epoch or 0) and not x["gate_ok_recomputed_main_baseline"]]
    late_components = sorted({c for x in rows if x["epoch"] > (sel_epoch or 0) for c in x["failing_components_main_baseline"].split(",") if c != "-"})
    if gate_never:
        status, valid = "GATE NEVER PASSED: the soft-best checkpoint was evaluated (failure analysis only)", False
    elif not sel_ok_main:
        status, valid = "INCONSISTENT: selected epoch does not pass the recomputed gate -- inspect", False
    elif not sel_ok_own:
        status, valid = "VALID WITH CAVEAT: selected epoch passes the gate with the main-seed clean baseline (what Cell 18 used) but NOT with this seed's own clean baseline", True
    else:
        status, valid = "VALID: selected epoch passes all gates (main-seed and own clean baseline)", True
    SEED_SUMMARY.append({"seed": sd, "epochs_run": len(hist), "epochs_gate_ok": ",".join(map(str, ok_epochs)) or "none", "selected_epoch": sel_epoch,
                         "selected_epoch_gate_ok_main_baseline": sel_ok_main, "selected_epoch_gate_ok_own_baseline": sel_ok_own,
                         "baseline_clean_auc_used_by_cell18": base_main, "seed_own_clean_val_auc": base_own, "last_epoch": last["epoch"], "last_epoch_gate_ok": last["gate_ok_recomputed_main_baseline"],
                         "epochs_after_selection_failing_gate": ",".join(map(str, late_fail)) or "-", "components_failing_after_selection": ",".join(late_components) or "-",
                         "stored_vs_recomputed_gate_mismatches": n_stored_mismatch, "final_evaluated_checkpoint": os.path.basename(fp),
                         "final_evaluated_sha256": sha256_file(fp)[:16], "gate_never_passed_flag": gate_never, "status": status, "valid_seed": valid})
GATE_DF, SEEDSUM_DF = pd.DataFrame(GATE_ROWS), pd.DataFrame(SEED_SUMMARY)
GATE_DF.to_csv(os.path.join(OUT_DIR, "seed_gate_by_epoch_v2.csv"), index=False)
SEEDSUM_DF.to_csv(os.path.join(OUT_DIR, "seed_gate_reconciliation_v2.csv"), index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)
print("\nSEED VALIDATION-GATE RECONCILIATION (recomputed from the saved per-epoch validation rows):")
print(SEEDSUM_DF.drop(columns=["final_evaluated_sha256"], errors="ignore").T.to_string())
print("\nPer-epoch gate decomposition for epochs where the recomputed gate failed (main-seed baseline = what Cell 18 used):")
_f = GATE_DF[~GATE_DF.gate_ok_recomputed_main_baseline][["seed", "epoch", "val_clean_auc", "val_fgsm_target_auc", "val_leak_margin", "failing_components_main_baseline", "failing_components_own_baseline"]]
print(_f.round(4).to_string(index=False) if len(_f) else "none")

RECON_LINES = []
for r in SEED_SUMMARY:
    if "epochs_run" not in r:
        RECON_LINES.append(f"seed {r['seed']}: {r['status']}")
        continue
    txt = (f"seed {r['seed']}: {r['status']}. Selected epoch {r['selected_epoch']} of {r['epochs_run']} run (gate-OK epochs: {r['epochs_gate_ok']}). "
           f"Epochs after the selected one that fail the gate: {r['epochs_after_selection_failing_gate']} (failing components: {r['components_failing_after_selection']}). "
           f"The early-stop message therefore refers to later epochs, while the evaluated checkpoint is the earlier gate-passing one ({r['final_evaluated_checkpoint']}, sha256 {r['final_evaluated_sha256']}).")
    RECON_LINES.append(txt)
    print(txt)
if (SEEDSUM_DF.get("components_failing_after_selection", pd.Series([], dtype=str)) == "no_label_leak").any():
    print("\nNOTE: the late-epoch gate failures are the label-leak gate (FGSM accuracy on validation exceeds clean accuracy by more than "
          f"{CONFIG['accept_max_label_leak_margin']}). An adversarially trained model that scores BETTER on label-aware FGSM inputs than on clean inputs is the classic signature of "
          "label leaking / a weak single-step attack (the perturbation direction itself carries the label) -- see the label-free FGSM evaluation in Cell 18d.")

VALID_SEEDS = [int(r["seed"]) for r in SEED_SUMMARY if r.get("valid_seed")]
INVALID_SEEDS = [int(r["seed"]) for r in SEED_SUMMARY if not r.get("valid_seed")]
SEED_CKPTS = {}
for sd in CONFIG["v2_seeds"]:
    c = CONFIG["save_clean_checkpoint"] if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_clean.pth")
    f_ = CONFIG["save_fgsm_checkpoint"] if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_fgsm.pth")
    s_ = os.path.join(OUT_DIR, "soft_best_fgsm_model.pth") if sd == CONFIG["seed"] else os.path.join(OUT_DIR, f"seed{sd}_soft.pth")
    ssum = next((r for r in SEED_SUMMARY if r["seed"] == sd), None)
    use_f = s_ if (ssum and ssum.get("gate_never_passed_flag")) else f_
    if os.path.exists(c) and os.path.exists(use_f):
        SEED_CKPTS[sd] = {"clean": c, "fgsm": use_f, "valid": sd in VALID_SEEDS}
print(f"\nValid independent seeds: {VALID_SEEDS} (n={len(VALID_SEEDS)}); invalid / unknown: {INVALID_SEEDS}. Repeated evaluations of one checkpoint are NEVER counted as seeds.")
if INVALID_SEEDS:
    print("!! Invalid seeds are EXCLUDED from the completed-seed summary (not relabelled as passed); rerun them to restore n=3.")
atomic_json_save({"valid_seeds": VALID_SEEDS, "invalid_seeds": INVALID_SEEDS, "lines": RECON_LINES, "checkpoints": {str(k): v for k, v in SEED_CKPTS.items()}},
                 os.path.join(OUT_DIR, "seed_status_v2.json"))
print("Remaining non-determinism (reported, not removed): cuDNN's fused LSTM backward is not bit-exact deterministic and AMP training is used, so rerunning a seed reproduces "
      "the protocol and the statistics, not the exact numbers.")


In [ ]:
# ============================================================
# CELL 18d (v2): PRIORITY 1 -- FGSM threshold collapse: audit, two threshold-selection strategies, frozen thresholds, new evaluation, 3-seed consistency
# ============================================================
# Protocol (nothing here is tuned on test data):
#   STEP 1  per seed and model: calibrate on the clean VALIDATION set (temperature), compute validation logits at eps in {0} U v2_thr_val_eps (+ label-free FGSM)
#   STEP 2  Method A = clean-validation EER threshold (same function as v1).  Method B = attack-aware threshold: the threshold maximising, over the validation FGSM
#           conditions 0.5, 1 and 2 /255, the mean of min(fake recall, real specificity) (a max-min objective: it cannot be satisfied by predicting one class).
#           Ties: every threshold within v2_thr_tol of the best objective is 'near optimal'; the MEDIAN near-optimal threshold is used (stability preference).
#           Legacy method C = v1's pooled {clean} U {FGSM@1/255} EER threshold, reported for continuity only.
#   STEP 3  ALL thresholds of ALL seeds are written to frozen_thresholds_v2.json (+ md5) BEFORE any test logit of this cell is computed; the test evaluation reads them back.
#   STEP 4  test evaluation at eps 0, 0.5, 1, 2, 4, 8 /255, each threshold applied unchanged to every eps.
# Disclosure: the test set had already been unlocked by v1's Cell 10 (the v1 analysis used it); thresholds here nevertheless come from validation only, in code order.
from scipy import stats as _st

THR_VAL_EPS = [float(e) for e in CONFIG["v2_thr_val_eps"]]
EPS_V2 = [e / 255 for e in CONFIG["v2_eval_eps_grid"]]
VAL_EPS_V2 = [0.0] + [e / 255 for e in THR_VAL_EPS]
COLLAPSE_RESOLVED_MIN = 0.70          # predefined: 'resolved' = fake recall AND real specificity >= this at 0.5, 1 and 2/255 (frozen test) for every valid seed
CLEAN_BALACC_MAX_DROP = 0.03          # ... and clean balanced accuracy at the attack-aware threshold within this of the clean-EER threshold


def _load_model(path):
    m = build_model()
    m.load_state_dict(torch.load(path, map_location=CONFIG["device"], weights_only=False)["model_state_dict"])
    m.eval()
    return m


def attack_aware_threshold_v2(lab, probs_by_eps, tol):
    """max over t of mean_e min(recall_e(t), specificity_e(t)); returns the median near-optimal threshold and the whole objective curve."""
    allp = np.unique(np.concatenate([np.asarray(p, float) for p in probs_by_eps.values()]))
    cand = np.concatenate([[allp[0] - 1e-9], (allp[1:] + allp[:-1]) / 2, [allp[-1] + 1e-9]])
    obj = np.zeros(len(cand))
    for p in probs_by_eps.values():
        pf, pr = np.sort(p[lab == 1]), np.sort(p[lab == 0])
        rec = 1.0 - np.searchsorted(pf, cand, side="left") / len(pf)      # fraction of fakes with p >= t
        spec = np.searchsorted(pr, cand, side="left") / len(pr)           # fraction of reals with p < t
        obj += np.minimum(rec, spec)
    obj /= len(probs_by_eps)
    best = float(obj.max())
    near = np.where(obj >= best - tol)[0]
    med = float(np.median(cand[near]))
    k = near[np.argmin(np.abs(cand[near] - med))]
    return {"thr": float(cand[k]), "objective_at_thr": float(obj[k]), "objective_max": best, "n_near_optimal": int(len(near)),
            "near_optimal_lo": float(cand[near].min()), "near_optimal_hi": float(cand[near].max()), "cand": cand, "obj": obj}


def _oracle_bal_acc(lab, p):
    allp = np.unique(p)
    cand = np.concatenate([[allp[0] - 1e-9], (allp[1:] + allp[:-1]) / 2, [allp[-1] + 1e-9]])
    pf, pr = np.sort(p[lab == 1]), np.sort(p[lab == 0])
    rec = 1.0 - np.searchsorted(pf, cand, side="left") / len(pf)
    spec = np.searchsorted(pr, cand, side="left") / len(pr)
    return float(np.max(0.5 * (rec + spec)))


def _rs(lab, p, thr):
    pred = p >= thr
    rec, spec = float(np.mean(pred[lab == 1])), float(np.mean(~pred[lab == 0]))
    return rec, spec, 0.5 * (rec + spec)


def _boot_acc_ci(correct, n, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(correct), (n, len(correct)))
    m = correct[idx].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


# ---------------- 0. audit of the existing implementation (facts established by reading Cells 7, 8, 11) ----------------
print("THRESHOLD IMPLEMENTATION AUDIT (v1 code, verified by inspection and by the asserts below):")
for l in ["  - scores: P(fake) = sigmoid(logit / T); T is fitted on the CLEAN validation logits only (Cell 7); label 1 = FAKE, label 0 = REAL; ROC-AUC uses P(fake).",
          "  - clean threshold: EER point of the calibrated clean validation scores (select_threshold); the SAME constant is applied at every epsilon (no per-epsilon tuning).",
          "  - v1 attack-aware threshold: EER of the POOLED {clean} U {FGSM@1/255} validation scores (Cell 8). For the clean-trained model this degenerates to the clean threshold "
          "(pooled EER 0.56: attacked scores are inverted), which is expected, not a bug.",
          "  - seeds 1/2 (Cell 18) calibrate and select thresholds on THEIR OWN validation scores; only the printed 'clean-EER thr -> recall' diagnostic inside attack_aware_threshold() "
          "referenced the main seed's threshold (cosmetic, no effect on any metric).",
          "  - confusion matrices in v1 Cell 15 use the validation-selected threshold of the same model that produced the metrics (same `thr`).",
          "  - Cell 18 gated every extra seed against the MAIN seed's clean validation AUC (BASELINE_VAL), not the seed's own: reconciled in Cell 18c."]:
    print(l)

# ---------------- STEP 1-2: validation logits, calibration, thresholds (per seed, per model) ----------------
FROZEN, VAL_STORE, VAL_DIAG_ROWS = {}, {}, []
for sd, pth in SEED_CKPTS.items():
    FROZEN[sd] = {}
    for tag, key in (("Clean-trained", "clean"), ("FGSM-trained", "fgsm")):
        mdl = _load_model(pth[key])
        lab_v, zv = fgsm_sweep_logits(mdl, val_attack_loader, VAL_EPS_V2, desc=f"val seed{sd} {tag}")
        T = float(fit_temperature(zv[0.0], lab_v))
        p0 = sigmoid_np(zv[0.0], T)
        _map_ok = bool(np.mean(zv[0.0][lab_v == 1]) > np.mean(zv[0.0][lab_v == 0]) and safe_roc_auc(lab_v, zv[0.0]) > 0.5)
        if CONFIG.get("v2_strict_class_mapping_check", True):
            assert _map_ok, f"class mapping check failed (seed {sd}, {tag}): FAKE logits are not higher than REAL logits on clean validation"
        elif not _map_ok:
            print(f"  (class-mapping check disabled by config; seed {sd} {tag} would have FAILED it)")
        thrA = float(select_threshold(lab_v, p0)["selected"])
        pbe = {e: sigmoid_np(zv[e], T) for e in VAL_EPS_V2[1:]}
        B = attack_aware_threshold_v2(lab_v, pbe, CONFIG["v2_thr_tol"])
        pooled_l = np.concatenate([lab_v, lab_v]); pooled_p = np.concatenate([p0, sigmoid_np(zv[float(TARGET)], T)])
        thrC = float(select_threshold(pooled_l, pooled_p)["eer"])
        row = {"T": T, "thr_clean_val_EER": thrA, "thr_attack_aware_v2": B["thr"], "thr_legacy_pooled_EER": thrC, "attack_aware_objective": B["objective_at_thr"],
               "attack_aware_objective_max": B["objective_max"], "near_optimal_range": [B["near_optimal_lo"], B["near_optimal_hi"]], "n_near_optimal_candidates": B["n_near_optimal"],
               "checkpoint": os.path.basename(pth[key]), "checkpoint_sha256": sha256_file(pth[key]), "val_n": int(len(lab_v))}
        FROZEN[sd][tag] = row
        VAL_STORE[(sd, tag)] = {"lab": lab_v, "z": zv, "B": B}
        if label_free := CONFIG["v2_label_free_eval"]:
            _, zvp = fgsm_sweep_logits(mdl, val_attack_loader, VAL_EPS_V2, label_source="pred", desc=f"val label-free seed{sd} {tag}")
            VAL_STORE[(sd, tag)]["z_pred"] = zvp
        # ---- validation diagnostics (Priority 1.5): score shift, calibration, rank-vs-threshold gap ----
        for e in VAL_EPS_V2:
            z, p = zv[e], sigmoid_np(zv[e], T)
            rA, sA, bA = _rs(lab_v, p, thrA); rB, sB, bB = _rs(lab_v, p, B["thr"])
            d = {"seed": sd, "model": tag, "eps_255": e * 255, "auc": safe_roc_auc(lab_v, z), "auc_minus_logit": safe_roc_auc(lab_v, -z),
                 "mean_logit_real": float(z[lab_v == 0].mean()), "mean_logit_fake": float(z[lab_v == 1].mean()), "logit_std": float(z.std()),
                 "logit_threshold_A": float(T * np.log(thrA / (1 - thrA))), "logit_threshold_B": float(T * np.log(B["thr"] / (1 - B["thr"]))),
                 "recall_A": rA, "spec_A": sA, "balacc_A": bA, "recall_B": rB, "spec_B": sB, "balacc_B": bB, "balacc_oracle": _oracle_bal_acc(lab_v, p),
                 "share_pred_fake_A": float(np.mean(p >= thrA)), "share_pred_fake_B": float(np.mean(p >= B["thr"])),
                 "ece": expected_calibration_error(lab_v, p), "brier": float(np.mean((p - lab_v) ** 2))}
            if label_free:
                zp = zvp[e]
                d["auc_label_free"] = safe_roc_auc(lab_v, zp)
                d["acc_A_true_label"] = float(np.mean((p >= thrA) == lab_v)); d["acc_A_label_free"] = float(np.mean((sigmoid_np(zp, T) >= thrA) == lab_v))
            VAL_DIAG_ROWS.append(d)
        del mdl
        free_cuda_memory()
    if sd == CONFIG["seed"]:         # the main seed must reproduce v1's calibration and thresholds exactly (same checkpoints, same validation data)
        assert abs(FROZEN[sd]["FGSM-trained"]["T"] - fgsm_T) < 5e-3 and abs(FROZEN[sd]["Clean-trained"]["T"] - clean_T) < 5e-3, "temperature differs from v1 -> checkpoint mismatch"
        assert abs(FROZEN[sd]["FGSM-trained"]["thr_clean_val_EER"] - fgsm_selected_threshold) < 5e-3 and abs(FROZEN[sd]["Clean-trained"]["thr_clean_val_EER"] - clean_selected_threshold) < 5e-3, \
            "clean-EER threshold differs from v1 -> checkpoint mismatch"
        print(f"Main seed reproduces v1: T clean/FGSM {FROZEN[sd]['Clean-trained']['T']:.3f}/{FROZEN[sd]['FGSM-trained']['T']:.3f}, clean-EER thresholds "
              f"{FROZEN[sd]['Clean-trained']['thr_clean_val_EER']:.3f}/{FROZEN[sd]['FGSM-trained']['thr_clean_val_EER']:.3f}  (v1: {clean_selected_threshold:.3f}/{fgsm_selected_threshold:.3f})")
VAL_DIAG = pd.DataFrame(VAL_DIAG_ROWS)
VAL_DIAG["eps_255"] = VAL_DIAG.eps_255.round(6)
VAL_DIAG.to_csv(os.path.join(OUT_DIR, "threshold_validation_diagnostics_v2.csv"), index=False)

# ---------------- STEP 3: FREEZE ----------------
_frozen_path = os.path.join(OUT_DIR, "frozen_thresholds_v2.json")
atomic_json_save({"protocol": "thresholds selected on VALIDATION data only, written before the v2 test evaluation",
                  "method_A": "clean-validation EER threshold", "method_B": f"attack-aware: max over t of mean_e min(recall, specificity), e in {THR_VAL_EPS}/255, tol {CONFIG['v2_thr_tol']}, median near-optimal",
                  "method_C": "legacy v1 pooled {clean}U{FGSM@1/255} EER", "frozen_at_unix": time.time(), "split_fingerprint": SPLIT_FINGERPRINT,
                  "seeds": {str(s): v for s, v in FROZEN.items()}}, _frozen_path)
FROZEN_MD5 = hashlib.md5(open(_frozen_path, "rb").read()).hexdigest()[:12]
print(f"\nThresholds FROZEN in {_frozen_path} (md5 {FROZEN_MD5}) -- every test number below uses these values unchanged.")
_ft = pd.DataFrame([{"seed": s, "model": m, "T": r["T"], "thr_A_clean_EER": r["thr_clean_val_EER"], "thr_B_attack_aware": r["thr_attack_aware_v2"], "thr_C_legacy_pooled_EER": r["thr_legacy_pooled_EER"],
                     "objective_B": r["attack_aware_objective"], "near_optimal_range_B": f"[{r['near_optimal_range'][0]:.3f}, {r['near_optimal_range'][1]:.3f}]"}
                    for s, d in FROZEN.items() for m, r in d.items()])
print(_ft.round(4).to_string(index=False))
FROZEN = json.load(open(_frozen_path))["seeds"]            # the evaluation below reads the FROZEN file, not the in-memory values
FROZEN = {int(k): v for k, v in FROZEN.items()}

# ---------------- STEP 4: frozen-threshold test evaluation ----------------
THR_ROWS, TEST_STORE = [], {}
for sd, pth in SEED_CKPTS.items():
    for tag, key in (("Clean-trained", "clean"), ("FGSM-trained", "fgsm")):
        fr = FROZEN[sd][tag]
        assert sha256_file(pth[key]) == fr["checkpoint_sha256"], f"checkpoint changed between freezing and evaluation (seed {sd}, {tag})"
        sources = [("true", None)] + ([("pred", "pred")] if CONFIG["v2_label_free_eval"] else [])
        for lname, lsrc in sources:
            if sd == CONFIG["seed"] and lsrc is None and tag in RAW and all(e in RAW[tag][1] for e in EPS_V2):
                labs, zd = RAW[tag][0], {e: RAW[tag][1][e] for e in EPS_V2}         # re-use v1's test logits of the main seed (same checkpoint: hash/T/threshold verified above)
            else:
                mdl = _load_model(pth[key])
                labs, zd = fgsm_sweep_logits(mdl, make_test_attack_loader(), EPS_V2, label_source=(lsrc or "true"), desc=f"test seed{sd} {tag} {lname}")
                del mdl
                free_cuda_memory()
            TEST_STORE[(sd, tag, lname)] = (labs, zd)
            for tname, thr in (("0.5 (reference)", 0.5), ("A: clean-val EER", fr["thr_clean_val_EER"]), ("B: attack-aware (val)", fr["thr_attack_aware_v2"]),
                               ("C: legacy pooled EER", fr["thr_legacy_pooled_EER"])):
                t = eps_diag_table(tag, labs, zd, fr["T"], thr, tname, EPS_V2)
                lo, hi = [], []
                for e in EPS_V2:
                    c = ((sigmoid_np(zd[e], fr["T"]) >= thr).astype(int) == labs).astype(float)
                    a, b = _boot_acc_ci(c, CONFIG["v2_bootstrap_n"], seed=sd)
                    lo.append(a); hi.append(b)
                t["acc_ci_lo"], t["acc_ci_hi"] = lo, hi
                t.insert(0, "seed", sd); t.insert(2, "label_source", lname)
                t["seed_valid"] = bool(SEED_CKPTS[sd]["valid"])
                THR_ROWS.append(t)
THR_DF = pd.concat(THR_ROWS, ignore_index=True)
THR_DF = THR_DF.rename(columns={"recall_fake": "fake_recall", "recall_real": "real_specificity"})
THR_DF["eps_255"] = THR_DF.eps_255.round(6)
THR_DF.to_csv(os.path.join(OUT_DIR, "threshold_comparison_v2.csv"), index=False)
assert THR_DF.auc_abs_diff_sklearn_vs_independent.max() < 1e-9

COLS = ["seed", "model", "eps_255", "threshold_type", "threshold", "accuracy", "roc_auc_independent", "macro_f1", "balanced_accuracy", "fake_recall", "real_specificity", "precision_fake", "precision_real"]
_main = THR_DF[(THR_DF.seed == CONFIG["seed"]) & (THR_DF.label_source == "true") & (THR_DF.eps_255.isin([0, 0.5, 1, 2])) & (THR_DF.threshold_type != "0.5 (reference)")]
print("\nCOMPARISON TABLE (seed 42, frozen thresholds, true-label FGSM):")
print(_main[COLS[1:]].rename(columns={"eps_255": "Epsilon(/255)", "roc_auc_independent": "ROC-AUC"}).round(3).to_string(index=False))

# ---- mean / std / 95% CI over VALID seeds only ----
_val = THR_DF[(THR_DF.seed_valid) & (THR_DF.label_source == "true")]
_plot_src = _val if len(_val) else THR_DF[THR_DF.label_source == "true"]      # figures only: when NO seed is valid they show the (flagged) unvalidated seeds
_PLOT_NOTE = "valid seeds" if len(_val) else "UNVALIDATED seeds (no valid seed)"
agg_cols = ["accuracy", "roc_auc_independent", "macro_f1", "balanced_accuracy", "fake_recall", "real_specificity", "precision_fake"]
rows = []
for (m, e, tn), g in _val.groupby(["model", "eps_255", "threshold_type"]):
    n = len(g)
    d = {"model": m, "eps_255": e, "threshold_type": tn, "n_valid_seeds": n, "threshold_mean": g.threshold.mean(), "threshold_std": g.threshold.std(ddof=1) if n > 1 else float("nan")}
    for c in agg_cols:
        mu, sdv = g[c].mean(), (g[c].std(ddof=1) if n > 1 else float("nan"))
        d[f"{c}_mean"], d[f"{c}_std"] = mu, sdv
        d[f"{c}_ci95_lo"], d[f"{c}_ci95_hi"] = (mu - _st.t.ppf(0.975, n - 1) * sdv / np.sqrt(n), mu + _st.t.ppf(0.975, n - 1) * sdv / np.sqrt(n)) if n > 1 else (float("nan"),) * 2
    rows.append(d)
THR_AGG = pd.DataFrame(rows, columns=None if rows else ["model", "eps_255", "threshold_type", "n_valid_seeds", "threshold_mean", "threshold_std"] + [f"{c}_{k}" for c in agg_cols for k in ("mean", "std", "ci95_lo", "ci95_hi")])
if not rows:
    print("NO VALID SEED: mean +- std is not computed (per-seed rows are still reported and flagged seed_valid=False). Nothing is averaged over unvalidated seeds.")
THR_AGG.to_csv(os.path.join(OUT_DIR, "threshold_summary_mean_std_v2.csv"), index=False)
if len(_val):
  print(f"\nPER-SEED / MEAN +- STD over {sorted(_val.seed.unique().tolist())} (n={_val.seed.nunique()} valid seeds; 95% CI = t-interval with n-1 d.o.f. -- very wide for n=3):")
  _p = THR_AGG[(THR_AGG.eps_255.isin([0, 1, 2])) & (THR_AGG.threshold_type.isin(["A: clean-val EER", "B: attack-aware (val)"]))]
  print(_p[["model", "eps_255", "threshold_type", "n_valid_seeds", "threshold_mean", "threshold_std", "accuracy_mean", "accuracy_std", "roc_auc_independent_mean", "roc_auc_independent_std",
            "fake_recall_mean", "fake_recall_std", "real_specificity_mean", "real_specificity_std"]].round(3).to_string(index=False))
print("\nPer-seed (FGSM-trained, eps 1/255, true-label FGSM):")
print(THR_DF[(THR_DF.model == "FGSM-trained") & (THR_DF.eps_255 == 1) & (THR_DF.label_source == "true") & (THR_DF.threshold_type.str[0].isin(["A", "B", "C"]))]
      [["seed", "seed_valid", "threshold_type", "threshold", "accuracy", "acc_ci_lo", "acc_ci_hi", "roc_auc_independent", "fake_recall", "real_specificity"]].round(3).to_string(index=False))

# ---------------- diagnosis (Priority 1.5): which explanation fits the evidence? ----------------
print("\nDIAGNOSIS OF THE THRESHOLD COLLAPSE (validation evidence, every number measured above):")
DIAG_LINES = []
vd = VAL_DIAG[(VAL_DIAG.model == "FGSM-trained")]
for sd in SEED_CKPTS:
    g0 = vd[(vd.seed == sd) & (vd.eps_255 == 0)].iloc[0]; g1 = vd[(vd.seed == sd) & (np.isclose(vd.eps_255, 1.0))].iloc[0]
    shift_real, shift_fake = g1.mean_logit_real - g0.mean_logit_real, g1.mean_logit_fake - g0.mean_logit_fake
    line = (f"seed {sd}: clean-EER threshold sits at logit {g0.logit_threshold_A:+.2f}; under FGSM 1/255 the logit std goes {g0.logit_std:.2f} -> {g1.logit_std:.2f}, real logits move {shift_real:+.2f}, "
            f"fake {shift_fake:+.2f} => at A: recall {g1.recall_A:.2f}, specificity {g1.spec_A:.2f} (balanced acc {g1.balacc_A:.3f}); at B (logit {g1.logit_threshold_B:+.2f}): recall {g1.recall_B:.2f}, "
            f"specificity {g1.spec_B:.2f} (balanced acc {g1.balacc_B:.3f}); best possible threshold on this validation data: {g1.balacc_oracle:.3f}; AUC {g1.auc:.3f}; ECE clean->FGSM {g0.ece:.3f}->{g1.ece:.3f}.")
    DIAG_LINES.append(line); print("  " + line)
bad_map = VAL_DIAG[(VAL_DIAG.eps_255 == 0) & ((VAL_DIAG.auc + VAL_DIAG.auc_minus_logit - 1).abs() > 1e-6)]
shift_flag = bool((vd[np.isclose(vd.eps_255, 1.0)].set_index("seed").logit_std.values < 0.6 * vd[vd.eps_255 == 0].set_index("seed").logit_std.values).all())
A_collapse = bool((vd[np.isclose(vd.eps_255, 1.0)].spec_A < 0.2).all() or (vd[np.isclose(vd.eps_255, 1.0)].recall_A < 0.2).all())
B_gain = float((vd[np.isclose(vd.eps_255, 1.0)].balacc_B - vd[np.isclose(vd.eps_255, 1.0)].balacc_A).mean())
gap_B = float((vd[np.isclose(vd.eps_255, 1.0)].balacc_oracle - vd[np.isclose(vd.eps_255, 1.0)].balacc_B).mean())
auc_1 = float(vd[np.isclose(vd.eps_255, 1.0)].auc.mean())
verdicts = [
    ("class-mapping error", "RULED OUT" if bad_map.empty else "NOT ruled out", "AUC(+logit) + AUC(-logit) = 1 in every condition; mean logit of FAKE > REAL on clean validation (asserted for every seed/model)."),
    ("checkpoint mismatch", "RULED OUT" if bool(PROV_DF.behaviour_matches_role.dropna().all()) else "NOT ruled out", "Cell 18c: provenance + behavioural identity check; main-seed T and thresholds reproduce v1 (asserted)."),
    ("systematic score-distribution shift", "SUPPORTED" if shift_flag else "not supported", "attacked logits are compressed (std falls to <60% of clean) and shifted, so a threshold placed on clean scores sits outside the attacked score range."),
    ("threshold-selection issue", "SUPPORTED" if A_collapse else "not supported", f"at the clean-EER threshold one class is (almost) never predicted under FGSM; the attack-aware threshold raises balanced accuracy at 1/255 by {B_gain:+.3f} (validation, mean over seeds)."),
    ("calibration difference", "PARTLY", "temperature is fitted on clean validation scores; ECE changes under attack (see ECE column) -- calibration is wrong under attack, but ranking (AUC) is unaffected by it."),
    ("genuine classification limitation", "PRESENT" if gap_B > 0.03 or auc_1 < 0.9 else "small", f"even the best possible threshold on validation leaves balanced accuracy {gap_B:+.3f} above the frozen attack-aware one at 1/255; AUC at 1/255 = {auc_1:.3f}."),
]
for k, v, why in verdicts:
    DIAG_LINES.append(f"{k}: {v} -- {why}"); print(f"  [{v:>12}] {k}: {why}")
if CONFIG["v2_label_free_eval"]:
    lf = VAL_DIAG[(VAL_DIAG.model == "FGSM-trained") & np.isclose(VAL_DIAG.eps_255, 1.0)]
    dl = float((lf.auc - lf.auc_label_free).mean())
    msg = (f"label-leak / weak-attack check (FGSM-trained, validation, 1/255): AUC with the TRUE label {lf.auc.mean():.3f} vs label-free FGSM (predicted label) {lf.auc_label_free.mean():.3f} (difference {dl:+.3f}); "
           f"accuracy at threshold A: true-label {lf.acc_A_true_label.mean():.3f} vs label-free {lf.acc_A_label_free.mean():.3f}. "
           + ("A true-label attack that is WEAKER than the label-free one means the perturbation carries label information (label leaking) or the loss surface is masked: the FGSM-trained "
              "model's robustness numbers are then an upper bound for FGSM." if dl > 0.03 else "No large difference: no evidence of label leaking from this check."))
    DIAG_LINES.append(msg); print("  " + msg)
print("  NOTE: threshold adjustment does not make the model robust; it only removes the artefact of applying a clean-scale threshold to attacked scores. Under FGSM-only evaluation "
      "(no multi-step attack, by design) a high AUC cannot certify robustness.")

# ---- predefined resolution criterion (reported, never used for tuning) ----
res = []
for sd in SEED_CKPTS:
    g = THR_DF[(THR_DF.seed == sd) & (THR_DF.model == "FGSM-trained") & (THR_DF.label_source == "true") & (THR_DF.threshold_type == "B: attack-aware (val)")]
    low = g[g.eps_255.isin([0.5, 1, 2])]
    ga = THR_DF[(THR_DF.seed == sd) & (THR_DF.model == "FGSM-trained") & (THR_DF.label_source == "true") & (THR_DF.threshold_type == "A: clean-val EER") & (THR_DF.eps_255 == 0)].balanced_accuracy.iloc[0]
    gb = g[g.eps_255 == 0].balanced_accuracy.iloc[0]
    ok = bool((low.fake_recall >= COLLAPSE_RESOLVED_MIN).all() and (low.real_specificity >= COLLAPSE_RESOLVED_MIN).all() and gb >= ga - CLEAN_BALACC_MAX_DROP)
    res.append({"seed": sd, "valid": SEED_CKPTS[sd]["valid"], "min_recall_0.5_1_2": float(low.fake_recall.min()), "min_specificity_0.5_1_2": float(low.real_specificity.min()),
                "clean_balacc_A": float(ga), "clean_balacc_B": float(gb), "resolved": ok})
RESOLVED_DF = pd.DataFrame(res)
THR_RESOLVED = bool(RESOLVED_DF[RESOLVED_DF.valid].resolved.all()) if RESOLVED_DF.valid.any() else False
RESOLVED_DF.to_csv(os.path.join(OUT_DIR, "threshold_collapse_resolution_v2.csv"), index=False)
print(f"\nTHRESHOLD COLLAPSE RESOLVED? (predefined: fake recall AND real specificity >= {COLLAPSE_RESOLVED_MIN} at 0.5/1/2 /255 with the frozen attack-aware threshold, clean balanced accuracy "
      f"within {CLEAN_BALACC_MAX_DROP} of the clean-EER threshold, every valid seed) -> {'YES' if THR_RESOLVED else 'NO / PARTLY'}")
print(RESOLVED_DF.round(3).to_string(index=False))

# ---------------- figures ----------------
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix as _cm

# F1: validation objective curve and thresholds (seed 42, both models)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, tag in zip(ax, ("Clean-trained", "FGSM-trained")):
    B = VAL_STORE[(CONFIG["seed"], tag)]["B"]; fr = FROZEN[CONFIG["seed"]][tag]
    a.plot(B["cand"], B["obj"], color="#444", lw=1.5, label="attack-aware objective: mean_e min(recall, spec), e in {0.5,1,2}/255")
    a.axvline(fr["thr_clean_val_EER"], color="tab:red", ls="--", label=f"A clean-val EER = {fr['thr_clean_val_EER']:.3f}")
    a.axvline(fr["thr_attack_aware_v2"], color="tab:green", ls="-", label=f"B attack-aware = {fr['thr_attack_aware_v2']:.3f}")
    a.axvline(fr["thr_legacy_pooled_EER"], color="tab:blue", ls=":", label=f"C legacy pooled EER = {fr['thr_legacy_pooled_EER']:.3f}")
    a.set_title(f"{tag} (seed {CONFIG['seed']}) -- VALIDATION"); a.set_xlabel("threshold on calibrated P(fake)"); a.set_ylabel("objective"); a.set_ylim(-0.02, 1.0); a.legend(fontsize=7, loc="upper right")
fig.suptitle("Clean-validation versus attack-aware threshold (validation data only)")
fig.tight_layout(); _finish(fig, "fig_v2_threshold_validation.png")

# F2: frozen-threshold confusion matrices, seed 42
EPS_CM = [0.0, 0.5 / 255, 1 / 255, 2 / 255]
CM_CFG = [("Clean-trained", "A: clean-val EER", "thr_clean_val_EER"), ("FGSM-trained", "A: clean-val EER", "thr_clean_val_EER"), ("FGSM-trained", "B: attack-aware (val)", "thr_attack_aware_v2")]


def _cm_grid(sd, eps_list, fname, title):
    fig, axs = plt.subplots(len(CM_CFG), len(eps_list), figsize=(3.1 * len(eps_list), 3.0 * len(CM_CFG)))
    for i, (tag, tname, key) in enumerate(CM_CFG):
        labs, zd = TEST_STORE[(sd, tag, "true")]; fr = FROZEN[sd][tag]
        for j, e in enumerate(eps_list):
            pred = (sigmoid_np(zd[e], fr["T"]) >= fr[key]).astype(int)
            sns.heatmap(_cm(labs, pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=axs[i, j], xticklabels=["pred real", "pred fake"], yticklabels=["real", "fake"])
            axs[i, j].set_title(f"{tag}\nthr {tname.split(':')[0]}={fr[key]:.3f}, eps {e * 255:g}/255", fontsize=8)
    fig.suptitle(title); fig.tight_layout(); _finish(fig, fname)


_cm_grid(CONFIG["seed"], EPS_CM, "fig_v2_confusion_frozen_seed42.png", f"Frozen-threshold confusion matrices, seed {CONFIG['seed']} (threshold named in every panel)")
for sd in SEED_CKPTS:
    if sd != CONFIG["seed"]:
        _cm_grid(sd, [0.0, 1 / 255], f"fig_v2_confusion_frozen_seed{sd}.png", f"Frozen-threshold confusion matrices, seed {sd}")

# F3: fake recall / real specificity vs epsilon
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
styles = [("Clean-trained", "A: clean-val EER", "tab:gray", "--"), ("FGSM-trained", "A: clean-val EER", "tab:red", "-"), ("FGSM-trained", "B: attack-aware (val)", "tab:green", "-")]
for a, (col, nm) in zip(ax, (("fake_recall", "Fake recall"), ("real_specificity", "Real specificity"))):
    for tag, tn, c, ls in styles:
        g = _plot_src[(_plot_src.model == tag) & (_plot_src.threshold_type == tn)]
        for sd_, gs in g.groupby("seed"):
            a.plot(gs.eps_255, gs[col], color=c, alpha=0.25, lw=1)
        m_ = g.groupby("eps_255")[col].agg(["mean", "std"]).reset_index()
        a.errorbar(m_.eps_255, m_["mean"], yerr=m_["std"].fillna(0), color=c, ls=ls, marker="o", capsize=3, label=f"{tag} @ {tn.split(':')[0]}")
    a.set_xlabel("FGSM epsilon (/255)"); a.set_title(nm); a.set_ylim(-0.02, 1.02); a.grid(alpha=0.3)
ax[0].legend(fontsize=8); ax[0].set_ylabel("rate")
fig.suptitle(f"Frozen thresholds: class-wise rates across epsilon (mean +- std over {_PLOT_NOTE} {sorted(_plot_src.seed.unique().tolist())}; thin lines = seeds)")
fig.tight_layout(); _finish(fig, "fig_v2_recall_specificity_vs_eps.png")

# F4: per-seed thresholds and metrics at 1/255
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
g = THR_DF[(THR_DF.model == "FGSM-trained") & (THR_DF.eps_255 == 1) & (THR_DF.label_source == "true") & THR_DF.threshold_type.str[0].isin(["A", "B"])]
for a, col in zip(ax, ("threshold", "accuracy", "balanced_accuracy")):
    sns.barplot(data=g, x="seed", y=col, hue="threshold_type", ax=a); a.set_title(f"{col} (FGSM-trained, 1/255, test)"); a.legend(fontsize=7)
fig.tight_layout(); _finish(fig, "fig_v2_per_seed_threshold_metrics.png")
atomic_json_save({"diagnosis": DIAG_LINES, "resolved": THR_RESOLVED, "frozen_md5": FROZEN_MD5}, os.path.join(OUT_DIR, "threshold_diagnosis_v2.json"))


In [ ]:
# ============================================================
# CELL 25a (v2): Experiment runner (config + status + results per run), controlled FGSM-AT pilots, leave-one-dataset-out (held-out dataset) evaluation
# ============================================================
# Nothing in this cell trains anything unless the corresponding flag is set (CONFIG['lodo_enabled'], CONFIG['run_pilots_v10']). Experiments that are not run are
# recorded as PENDING with the estimated cost, so the final report cannot mistake 'code exists' for 'experiment done'.
import traceback, datetime, contextlib

EVAL_EPS = [e / 255 for e in CONFIG["eval_eps_grid"]]      # same definition as Cell 11 (harmless to redefine)
EXP_DIR = os.path.join(OUT_DIR, "experiments")
os.makedirs(EXP_DIR, exist_ok=True)


def experiment_config_snapshot(extra=None):
    keep = ["seed", "split_seed", "selection_mode", "selection_seed", "datasets", "max_videos_to_use", "val_id_fraction", "test_id_fraction",
            "seq_len", "frame_pool_size", "img_size", "min_usable_frames", "short_video_policy", "face_margin", "face_crop_size", "face_detect_every",
            "face_min_detect_rate", "lstm_hidden_size", "lstm_num_layers", "lstm_bidirectional", "dropout", "use_temporal_attention", "batch_size", "fgsm_batch_size",
            "backbone_lr", "head_lr", "weight_decay", "phase2_epochs", "fgsm_epochs", "fgsm_steps_per_epoch", "fgsm_lr_scale", "fgsm_eps_ramp_epochs", "adv_ratio_start",
            "adv_ratio", "adv_ratio_ramp_epochs", "adv_label_source", "adv_random_eps", "fgsm_clean_loss_weight", "fgsm_adv_loss_weight", "alp_weight", "fgsm_train_eps",
            "eval_eps_grid", "val_eps_grid", "target_eval_eps", "select_weights", "threshold_method"]
    snap = {k: (list(CONFIG[k]) if isinstance(CONFIG.get(k), tuple) else CONFIG.get(k)) for k in keep}
    snap["split_fingerprint"] = SPLIT_FINGERPRINT
    snap["time_utc"] = datetime.datetime.utcnow().isoformat()
    snap.update(extra or {})
    return snap


class ExperimentRunner:
    """Resumable experiment registry. One folder per run_id with status.json (completed / failed / pending / running), config.json and result files.
    A completed run is skipped on re-execution (resume); a failed run is kept with its traceback and retried only with force=True."""
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def _p(self, run_id):
        d = self.root / run_id
        d.mkdir(parents=True, exist_ok=True)
        return d

    def status(self, run_id):
        f = self._p(run_id) / "status.json"
        return json.load(open(f)) if f.exists() else None

    def _write(self, run_id, **kw):
        d = self._p(run_id)
        cur = self.status(run_id) or {}
        cur.update(kw)
        atomic_json_save(cur, str(d / "status.json"))

    def mark_pending(self, run_id, config, reason, est_hours=None):
        self._write(run_id, status="pending", reason=reason, est_hours=est_hours)
        atomic_json_save(config, str(self._p(run_id) / "config.json"))

    def register_completed(self, run_id, config, artifacts, summary=None):
        self._write(run_id, status="completed", artifacts=artifacts, summary=summary or {}, finished=datetime.datetime.utcnow().isoformat())
        atomic_json_save(config, str(self._p(run_id) / "config.json"))

    def run(self, run_id, config, fn, force=False):
        st = self.status(run_id)
        if st and st.get("status") == "completed" and not force:
            print(f"[runner] {run_id}: already completed -> skipped (resume).")
            return st
        atomic_json_save(config, str(self._p(run_id) / "config.json"))
        t0 = time.time()
        self._write(run_id, status="running", started=datetime.datetime.utcnow().isoformat(), reason=None)
        try:
            res = fn(self._p(run_id))
            self._write(run_id, status="completed", duration_s=time.time() - t0, summary=res if isinstance(res, dict) else {}, finished=datetime.datetime.utcnow().isoformat())
        except Exception as e:
            self._write(run_id, status="failed", duration_s=time.time() - t0, error=f"{type(e).__name__}: {e}", traceback=traceback.format_exc()[-3000:])
            print(f"[runner] {run_id} FAILED: {type(e).__name__}: {e}")
        return self.status(run_id)

    def table(self):
        rows = []
        for d in sorted(self.root.iterdir()):
            f = d / "status.json"
            if f.exists():
                s = json.load(open(f))
                rows.append({"run_id": d.name, "status": s.get("status"), "duration_min": (s.get("duration_s") or 0) / 60, "est_hours": s.get("est_hours"),
                             "reason_or_error": s.get("error") or s.get("reason") or ""})
        return pd.DataFrame(rows)


RUNNER = ExperimentRunner(EXP_DIR)


@contextlib.contextmanager
def config_override(**kw):
    old = {k: CONFIG.get(k) for k in kw}
    CONFIG.update(kw)
    try:
        yield
    finally:
        CONFIG.update(old)


_FOLD_NAMES = ["train_records", "val_records", "test_records", "split_records", "train_dataset", "val_dataset", "test_dataset", "train_loader", "train_loader_fgsm",
               "val_loader", "val_attack_loader", "val_adv_loader", "val_adv_records", "SPLIT_FINGERPRINT", "class_counts", "USE_SAMPLER", "_sample_weights",
               "train_labels_arr", "BASELINE_VAL"]


@contextlib.contextmanager
def rebind_data(new_vars):
    """Temporarily rebind the notebook-level data globals used by run_fit / validate_epoch / make_*_loader, so the SAME training machinery runs on a different split."""
    g = globals()
    old = {k: g.get(k, None) for k in _FOLD_NAMES}
    g.update(new_vars)
    try:
        yield
    finally:
        g.update(old)


def _records_from(videos, folder_of_new, info_out):
    recs = []
    for v in videos:
        f = folder_of_new.get(v["path"])
        if f is None:
            continue
        recs.append((f, v["label"]))
        info_out[f] = {"method": v["method"], "ids": v["ids"], "split": None, "video_path": v["path"], "dataset": v["dataset"]}
    return recs


def build_fold(holdout, seed):
    """Training/validation = identity-disjoint split of the OTHER datasets only; test = videos of the held-out dataset (class- and method-balanced, seeded).
    The held-out dataset contributes NOTHING to training, validation, checkpoint selection, calibration or threshold choice."""
    kept = [v for v in DISCOVERED if VIDEO_FATE[v["path"]]["fate"] not in ("duplicate", "label_conflict_duplicate")]
    others = [v for v in kept if v["dataset"] != holdout]
    held = [v for v in kept if v["dataset"] == holdout]
    split_of, assigned, straddle, info = identity_disjoint_split_v10(others, CONFIG["split_seed"], CONFIG["val_id_fraction"], 0.0, CONFIG["split_search_trials"],
                                                                    CONFIG["split_refine_passes"], CONFIG["split_quota_slack"])
    cfg = dict(CONFIG)
    cfg["datasets"] = {k: v for k, v in CONFIG["datasets"].items() if k != holdout}
    cfg["train_per_class_cap"] = CONFIG["lodo_train_per_class_cap"]
    cfg["split_budget_fractions"] = {"train": 0.85, "val": 0.15, "test": 0.0}
    cfg["max_videos_to_use"] = CONFIG["max_videos_to_use"]
    sel = stratified_select_v10({"train": assigned["train"], "val": assigned["val"], "test": []}, cfg, seed)
    cfg_h = dict(CONFIG)
    cfg_h["datasets"] = {holdout: {"weight": 1.0}}
    cfg_h["split_budget_fractions"] = {"train": 0.0, "val": 0.0, "test": 1.0}
    cfg_h["max_videos_to_use"] = 2 * CONFIG["lodo_heldout_per_class"]
    cfg_h["eval_per_class_cap"] = CONFIG["lodo_heldout_per_class"]
    sel_h = stratified_select_v10({"train": [], "val": [], "test": held}, cfg_h, seed)["test"]
    return {"train": sel["train"], "val": sel["val"], "test": sel_h}, {"split_info": info, "n_straddle_dropped": len(straddle), "n_heldout_eligible": len(held),
                                                                      "ids_train_val_overlap": bool({i for v in sel["train"] for i in v["ids"]} & {i for v in sel["val"] for i in v["ids"]})}


def lodo_eligible(min_per_class):
    out = {}
    for d in CONFIG["datasets"]:
        c = Counter(VIDEO_FATE[v["path"]]["label"] for v in DISCOVERED if v["dataset"] == d and VIDEO_FATE[v["path"]]["fate"] not in ("duplicate", "label_conflict_duplicate"))
        out[d] = (c[0], c[1], min(c[0], c[1]) >= min_per_class)
    return out



def _elapsed_left_h():
    return CONFIG["time_budget_hours"] - (time.time() - NOTEBOOK_START) / 3600 - CONFIG["v2_phase_reserve_hours"]


def run_lodo_fold(holdout, seed, out_dir, exploratory=False):
    """One held-out-dataset experiment: train BOTH models on the other datasets (training-side validation only), freeze thresholds on that validation set,
    then evaluate on the held-out dataset (first and only contact with it)."""
    t0 = time.time()
    sel, finfo = build_fold(holdout, seed)
    assert not finfo["ids_train_val_overlap"], "identity leakage between fold train and validation"
    assert all(v["dataset"] != holdout for s in ("train", "val") for v in sel[s]), "held-out dataset leaked into training/validation"
    # cross-dataset duplicate check (identical bytes) between the held-out videos and the training-side videos
    fps = json.load(open(os.path.join(OUT_DIR, "_video_fingerprints.json"))) if os.path.exists(os.path.join(OUT_DIR, "_video_fingerprints.json")) else {}
    train_side = {v["path"] for s in ("train", "val") for v in sel[s]}
    _fp_of = {p: (fps.get(p) or [None, None, None])[2] for p in list(train_side) + [v["path"] for v in sel["test"]]}      # cache format: path -> [size, mtime, md5(head+tail+size)]
    cross_dups = len({_fp_of[v["path"]] for v in sel["test"] if _fp_of.get(v["path"])} & {_fp_of[p] for p in train_side if _fp_of.get(p)})
    assert cross_dups == 0, f"{cross_dups} held-out videos have byte-identical copies on the training side"
    allv = sel["train"] + sel["val"] + sel["test"]
    fo, boxes = extract_selected_to_disk(allv, CONFIG["face_frames_dir"], CONFIG["frame_pool_size"])
    FACE_BOXES.update(boxes)
    info = {}
    recs = {s: _records_from(sel[s], fo, info) for s in SPLITS}
    for s in ("train", "val"):
        recs[s], _ = balance_by_min_per_dataset(recs[s], CONFIG["split_seed"], info)
    record_info.update(info)
    fp = hashlib.md5("|".join([s + ":" + ",".join(sorted(Path(f).name + str(l) for f, l in recs[s])) for s in SPLITS]).encode()).hexdigest()[:12]
    lab = np.array([r[1] for r in recs["train"]])
    cc = np.bincount(lab, minlength=2)
    tr_ds = PreextractedVideoDataset(recs["train"], CONFIG["seq_len"], CONFIG["img_size"], True)
    va_ds = PreextractedVideoDataset(recs["val"], CONFIG["seq_len"], CONFIG["img_size"], False)
    te_ds = PreextractedVideoDataset(recs["test"], CONFIG["seq_len"], CONFIG["img_size"], False)
    half = CONFIG["val_adv_n"] // 2
    vadv = [r for r in recs["val"] if r[1] == 0][:half] + [r for r in recs["val"] if r[1] == 1][:half]
    nv = dict(train_records=recs["train"], val_records=recs["val"], test_records=recs["test"], split_records=recs, train_dataset=tr_ds, val_dataset=va_ds, test_dataset=te_ds,
              SPLIT_FINGERPRINT=fp, class_counts=cc, USE_SAMPLER=bool(abs(cc[1] / max(cc[0], 1) - 1) > 0.05), _sample_weights=(1.0 / np.maximum(cc, 1))[lab], train_labels_arr=lab,
              val_adv_records=vadv)
    tag = f"lodo_{holdout}_s{seed}"
    P = {k: os.path.join(out_dir, f"{tag}_{k}.pth") for k in ("clean", "fgsm", "soft", "rc", "rf")}
    ec, ef, spe = CONFIG["lodo_clean_epochs"], CONFIG["lodo_fgsm_epochs"], CONFIG["lodo_fgsm_steps"]
    scale = ef / max(CONFIG["fgsm_epochs_effective"], 1)
    ov = dict(phase2_epochs=ec, fgsm_eps_ramp_epochs=CONFIG["fgsm_eps_ramp_epochs"] * scale, adv_ratio_ramp_epochs=CONFIG["adv_ratio_ramp_epochs"] * scale,
              early_stop_min_epochs=min(CONFIG["early_stop_min_epochs"], max(3, ef // 2)))
    with rebind_data(nv), config_override(**ov):
        g = globals()
        g["train_loader"] = make_train_loader(CONFIG["batch_size"])
        g["train_loader_fgsm"] = make_train_loader(CONFIG["fgsm_batch_size"])
        g["val_loader"] = make_eval_loader(va_ds, CONFIG["eval_batch_size"])
        g["val_attack_loader"] = make_eval_loader(va_ds, CONFIG["eval_attack_batch_size"])
        g["val_adv_loader"] = make_eval_loader(PreextractedVideoDataset(vadv, CONFIG["seq_len"], CONFIG["img_size"], False), CONFIG["eval_attack_batch_size"])
        cm, chist, cbest = train_clean_model(seed, P["clean"], P["rc"])
        ck = torch.load(P["clean"], map_location=CONFIG["device"], weights_only=False)
        cm.load_state_dict(ck["model_state_dict"]); cm.eval()
        _l, _zz = fgsm_sweep_logits(cm, val_adv_loader, sorted({0.0, float(TARGET), *[float(e) for e in VAL_EPS]}), desc="fold clean val FGSM")
        _aucs = [safe_roc_auc(_l, _zz[e * EPS_UNIT]) for e in CONFIG["val_eps_grid"]]
        _lv, _zv = get_logits(cm, val_loader, None, desc="fold val")
        g["BASELINE_VAL"] = {"clean_val_auc_full": safe_roc_auc(_lv, _zv), "clean_val_auc_subset": safe_roc_auc(_l, _zz[0.0]), "clean_val_logit_std": float(np.std(_zz[0.0])),
                             "fgsm_target_auc": safe_roc_auc(_l, _zz[float(TARGET)]), "fgsm_mean_auc": float(np.nanmean(_aucs)), "fgsm_min_auc": float(np.nanmin(_aucs))}
        fm = new_fgsm_model_from_clean(P["clean"])
        feps = CONFIG["fgsm_train_eps"]
        fh, fb = run_fit(fm, f"{tag} FGSM-AT", "fgsm", ef, train_loader_fgsm, spe, CONFIG["fgsm_lr_scale"], CONFIG["fgsm_warmup_steps"],
                         fgsm_adv_cfg(feps, CONFIG["fgsm_eps_ramp_epochs"], spe, CONFIG["adv_random_eps"], CONFIG["alp_weight"]),
                         make_fgsm_eval_fn(True), make_fgsm_select_fn(True), CONFIG["ema_decay_fgsm"], fgsm_clip_cfg(), best_path=P["fgsm"], soft_path=P["soft"], resume_path=P["rf"],
                         early_stop={"min_epochs": CONFIG["early_stop_min_epochs"], "bad_epochs": CONFIG["early_stop_bad_epochs"], "no_gain": CONFIG["early_stop_no_gain"]},
                         extra_meta={"train_eps": feps, "random_eps": CONFIG["adv_random_eps"], "lodo_holdout": holdout, "seed": seed})
        gate_failed = fb["epoch"] is None
        if gate_failed:
            shutil.copy(P["soft"], P["fgsm"])
        fk = torch.load(P["fgsm"], map_location=CONFIG["device"], weights_only=False)
        fev = build_model(); fev.load_state_dict(fk["model_state_dict"]); fev.eval()
        # ---- STEP: thresholds from the TRAINING-SIDE validation set only, frozen before the held-out loader is touched ----
        frozen = {}
        for nm, mdl in (("Clean-trained", cm), ("FGSM-trained", fev)):
            lab_v, zv = fgsm_sweep_logits(mdl, val_attack_loader, VAL_EPS_V2, desc=f"{tag} val {nm}")
            T_ = float(fit_temperature(zv[0.0], lab_v))
            thrA = float(select_threshold(lab_v, sigmoid_np(zv[0.0], T_))["selected"])
            B = attack_aware_threshold_v2(lab_v, {e: sigmoid_np(zv[e], T_) for e in VAL_EPS_V2[1:]}, CONFIG["v2_thr_tol"])
            frozen[nm] = {"T": T_, "thr_clean_val_EER": thrA, "thr_attack_aware_v2": B["thr"], "objective_B": B["objective_at_thr"], "val_n": int(len(lab_v)),
                          "val_datasets": sorted({record_info[f]["dataset"] for f, _ in recs["val"]})}
        fz_path = os.path.join(out_dir, f"{tag}_frozen_thresholds.json")
        atomic_json_save({"holdout": holdout, "seed": seed, "frozen_at_unix": time.time(), "rule": "validation of the TRAINING datasets only", "thresholds": frozen}, fz_path)
        fz_md5 = hashlib.md5(open(fz_path, "rb").read()).hexdigest()[:12]
        print(f"[{tag}] thresholds frozen on training-side validation (md5 {fz_md5}); held-out loader is created only now.")
        frozen = json.load(open(fz_path))["thresholds"]
        # ---- held-out evaluation (first and only contact with the held-out dataset) ----
        te_loader = make_eval_loader(te_ds, CONFIG["eval_attack_batch_size"])
        tabs, raw = [], {}
        for nm, mdl in (("Clean-trained", cm), ("FGSM-trained", fev)):
            labs, zd = fgsm_sweep_logits(mdl, te_loader, EPS_V2, desc=f"{tag} {nm} held-out")
            raw[nm] = (labs, zd); fr = frozen[nm]
            for tname, thr in (("0.5 (reference)", 0.5), ("A: clean-val EER", fr["thr_clean_val_EER"]), ("B: attack-aware (val)", fr["thr_attack_aware_v2"])):
                t = eps_diag_table(nm, labs, zd, fr["T"], thr, tname, EPS_V2)
                ci = [_boot_acc_ci(((sigmoid_np(zd[e], fr["T"]) >= thr).astype(int) == labs).astype(float), CONFIG["v2_bootstrap_n"], seed) for e in EPS_V2]
                t["acc_ci_lo"], t["acc_ci_hi"] = [c[0] for c in ci], [c[1] for c in ci]
                tabs.append(t)
        res = pd.concat(tabs, ignore_index=True).rename(columns={"recall_fake": "fake_recall", "recall_real": "real_specificity"})
        res["eps_255"] = res.eps_255.round(6)
        res.insert(0, "holdout", holdout); res.insert(1, "seed", seed); res["exploratory"] = bool(exploratory)
        res["n_real"], res["n_fake"] = int(sum(1 for _, l in recs["test"] if l == 0)), int(sum(1 for _, l in recs["test"] if l == 1))
        res.to_csv(os.path.join(out_dir, f"{tag}_metrics.csv"), index=False)
        np.savez_compressed(os.path.join(out_dir, f"{tag}_raw_scores.npz"), **{f"{m}|{e * 255:g}": raw[m][1][e] for m in raw for e in EPS_V2}, labels=raw["Clean-trained"][0])
        # ---- confusion matrices at the frozen thresholds ----
        from sklearn.metrics import confusion_matrix as _cmx
        cfgs = [("Clean-trained", "A: clean-val EER", "thr_clean_val_EER"), ("FGSM-trained", "A: clean-val EER", "thr_clean_val_EER"), ("FGSM-trained", "B: attack-aware (val)", "thr_attack_aware_v2")]
        fig, axs = plt.subplots(3, 3, figsize=(9.5, 9))
        for i, (nm, tn, key) in enumerate(cfgs):
            for j, e in enumerate([0.0, 1 / 255, 2 / 255]):
                labs, zd = raw[nm]
                pr = (sigmoid_np(zd[e], frozen[nm]["T"]) >= frozen[nm][key]).astype(int)
                sns.heatmap(_cmx(labs, pr, labels=[0, 1]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=axs[i, j], xticklabels=["pred real", "pred fake"], yticklabels=["real", "fake"])
                axs[i, j].set_title(f"{nm}\nthr {tn.split(':')[0]}={frozen[nm][key]:.3f}, eps {e * 255:g}/255", fontsize=8)
        fig.suptitle(f"Held-out {holdout}{' (EXPLORATORY)' if exploratory else ''}: frozen-threshold confusion matrices"); fig.tight_layout()
        fig.savefig(os.path.join(FIG_DIR, f"fig_v2_lodo_{holdout}_confusion.png"), dpi=130); plt.show()
        summary = {"holdout": holdout, "seed": seed, "exploratory": bool(exploratory), "n_train": len(recs["train"]), "n_val": len(recs["val"]), "n_heldout": len(recs["test"]),
                   "heldout_real": int(res.n_real.iloc[0]), "heldout_fake": int(res.n_fake.iloc[0]),
                   "train_datasets": sorted({info[f]["dataset"] for f, _ in recs["train"]}), "validation_datasets": sorted({info[f]["dataset"] for f, _ in recs["val"]}),
                   "fgsm_gate_never_passed": bool(gate_failed), "fgsm_selected_epoch": fk.get("epoch"), "clean_selected_epoch": ck.get("epoch"),
                   "frozen_thresholds_md5": fz_md5, "frozen_thresholds": frozen, "clean_ckpt": os.path.basename(P["clean"]), "fgsm_ckpt": os.path.basename(P["fgsm"]),
                   "clean_ckpt_sha256": sha256_file(P["clean"]), "fgsm_ckpt_sha256": sha256_file(P["fgsm"]), "n_straddle_dropped": finfo["n_straddle_dropped"],
                   "cross_dataset_byte_duplicates": cross_dups, "split_fingerprint": fp, "duration_h": (time.time() - t0) / 3600,
                   "fold_recipe": {"clean_epochs": ec, "fgsm_epochs": ef, "fgsm_steps": spe, "train_eps_255": feps * 255, "note": "compute-limited; identical for every fold"}}
        atomic_json_save(summary, os.path.join(out_dir, f"{tag}_summary.json"))
    return summary


# ---------------- registry + execution ----------------
LODO_ELIG = lodo_eligible(CONFIG["lodo_min_videos_per_class"])
print("Held-out-dataset eligibility (eligible videos after de-duplication; need >= " f"{CONFIG['lodo_min_videos_per_class']} per class):")
for d, (r, f, ok) in LODO_ELIG.items():
    print(f"  {d:<9} real={r:5d} fake={f:5d}  -> {'eligible' if ok else 'NOT eligible as a definitive held-out set (too few videos): exploratory only'}")
_dsn = {"ffpp": "FF++", "celeb_df": "Celeb-DF", "uadfv": "UADFV"}
LODO_PLAN = [("ffpp", False), ("celeb_df", False)] + [("uadfv", True)]
LODO_PLAN = [(h, ex) for h, ex in LODO_PLAN if (CONFIG["lodo_datasets"] is None or h in CONFIG["lodo_datasets"]) and (h in LODO_ELIG)]
for h, exploratory in LODO_PLAN:
    sd = CONFIG["lodo_seeds"][0]
    rid = f"lodo_holdout-{h}_seed{sd}"
    others = [d for d in CONFIG["datasets"] if d != h]
    cfgs = experiment_config_snapshot({"experiment": "leave-one-dataset-out", "holdout": h, "train_datasets": others, "validation_datasets": f"{others} (training-side identity-disjoint validation only)",
                                       "training_seed": sd, "exploratory": exploratory,
                                       "recipe_selection_note": "FGSM-AT recipe (train eps / random eps / no ALP) was chosen by the v1 MIXED pilot whose validation set contained the held-out dataset: disclosed leak"})
    if exploratory and not CONFIG["lodo_run_uadfv_exploratory"]:
        RUNNER.mark_pending(rid, cfgs, f"UADFV has only {LODO_ELIG[h][0]} real / {LODO_ELIG[h][1]} fake videos: exploratory, off by default (lodo_run_uadfv_exploratory=False)", est_hours=CONFIG["lodo_est_hours_per_fold"])
        continue
    if not CONFIG["lodo_enabled"]:
        RUNNER.mark_pending(rid, cfgs, "CONFIG['lodo_enabled'] is False (not executed)", est_hours=CONFIG["lodo_est_hours_per_fold"])
        continue
    st = RUNNER.status(rid)
    if not (st and st.get("status") == "completed") and _elapsed_left_h() < CONFIG["lodo_est_hours_per_fold"]:
        RUNNER.mark_pending(rid, cfgs, f"time guard: {_elapsed_left_h():.2f} h left (after reserve) < estimated {CONFIG['lodo_est_hours_per_fold']} h", est_hours=CONFIG["lodo_est_hours_per_fold"])
        print(f"[runner] {rid}: SKIPPED by the time guard.")
        continue
    RUNNER.run(rid, cfgs, lambda d, h=h, sd=sd, ex=exploratory: run_lodo_fold(h, sd, str(d), ex))

REG_ROWS = []
for h, exploratory in LODO_PLAN:
    sd = CONFIG["lodo_seeds"][0]
    rid = f"lodo_holdout-{h}_seed{sd}"
    st = RUNNER.status(rid) or {}
    sm = st.get("summary") or {}
    others = [_dsn[d] for d in CONFIG["datasets"] if d != h]
    REG_ROWS.append({"Experiment": f"{_dsn[h]} held out{' (exploratory)' if exploratory else ''}", "Training datasets": " + ".join(others), "Validation datasets": "training-side only (" + " + ".join(others) + ")",
                     "Held-out dataset": _dsn[h], "Status": st.get("status", "pending"), "seed": sd, "clean_checkpoint": sm.get("clean_ckpt"), "fgsm_checkpoint": sm.get("fgsm_ckpt"),
                     "frozen_thresholds_md5": sm.get("frozen_thresholds_md5"), "n_heldout (real/fake)": f"{sm.get('heldout_real')}/{sm.get('heldout_fake')}" if sm else None,
                     "reason_or_error": st.get("error") or st.get("reason") or ""})
REGISTRY_DF = pd.DataFrame(REG_ROWS)
REGISTRY_DF.to_csv(os.path.join(OUT_DIR, "cross_dataset_registry_v2.csv"), index=False)
print("\nCROSS-DATASET EXPERIMENT REGISTRY:")
print(REGISTRY_DF.to_string(index=False))

# ---------------- held-out results: four comparisons, per fold; mixed-dataset results kept SEPARATE ----------------
LODO_TABLES = []
for h, exploratory in LODO_PLAN:
    f = os.path.join(EXP_DIR, f"lodo_holdout-{h}_seed{CONFIG['lodo_seeds'][0]}", f"lodo_{h}_s{CONFIG['lodo_seeds'][0]}_metrics.csv")
    if os.path.exists(f):
        LODO_TABLES.append(pd.read_csv(f))
LODO_DF = pd.concat(LODO_TABLES, ignore_index=True) if LODO_TABLES else pd.DataFrame()
if len(LODO_DF):
    LODO_DF.to_csv(os.path.join(OUT_DIR, "lodo_heldout_metrics_v2.csv"), index=False)
    LODO_DF["condition"] = LODO_DF.model + np.where(LODO_DF.eps_255 == 0, " on clean videos", " on FGSM videos")
    for h in LODO_DF.holdout.unique():
        g = LODO_DF[(LODO_DF.holdout == h) & LODO_DF.threshold_type.isin(["A: clean-val EER", "B: attack-aware (val)"])]
        print(f"\nHELD-OUT {_dsn[h]}{' (EXPLORATORY: sample too small for definitive claims)' if g.exploratory.iloc[0] else ''} -- {int(g.n_real.iloc[0])} real / {int(g.n_fake.iloc[0])} fake videos; "
              f"thresholds frozen on training-side validation:")
        print(g[["model", "eps_255", "threshold_type", "threshold", "accuracy", "acc_ci_lo", "acc_ci_hi", "roc_auc_independent", "macro_f1", "balanced_accuracy", "fake_recall", "real_specificity"]].round(3).to_string(index=False))
    # mixed-dataset results for the SAME dataset (Experiment A of v1), shown next to -- never merged with -- the held-out numbers
    mixed = PER_DATASET_DIAG[(PER_DATASET_DIAG.subset.isin(list(LODO_DF.holdout.unique()))) & (PER_DATASET_DIAG.threshold_type == "val_selected") & PER_DATASET_DIAG.eps_255.isin([0, 1, 2, 4, 8])]
    held = LODO_DF[(LODO_DF.threshold_type == "A: clean-val EER") & LODO_DF.eps_255.isin([0, 1, 2, 4, 8])]
    cmp_ = held[["holdout", "model", "eps_255", "accuracy", "roc_auc_independent", "balanced_accuracy"]].merge(
        mixed[["subset", "model", "eps_255", "accuracy", "roc_auc_independent", "balanced_accuracy"]].rename(columns={"subset": "holdout", "accuracy": "mixed_accuracy", "roc_auc_independent": "mixed_auc",
                                                                                                                        "balanced_accuracy": "mixed_balanced_accuracy"}),
        on=["holdout", "model", "eps_255"], how="left").rename(columns={"accuracy": "heldout_accuracy", "roc_auc_independent": "heldout_auc", "balanced_accuracy": "heldout_balanced_accuracy"})
    cmp_.to_csv(os.path.join(OUT_DIR, "mixed_vs_heldout_v2.csv"), index=False)
    print("\nMIXED-DATASET (identity-disjoint, same datasets seen in training) versus HELD-OUT-DATASET (unseen), clean-EER threshold. Reported side by side, never combined into one metric:")
    print(cmp_.round(3).to_string(index=False))
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
    for a, col, nm in zip(axs, ("accuracy", "roc_auc_independent"), ("Accuracy", "ROC-AUC")):
        for (h, m), g in LODO_DF[LODO_DF.threshold_type == "A: clean-val EER"].groupby(["holdout", "model"]):
            a.plot(g.eps_255, g[col], marker="o", ls="-" if m == "FGSM-trained" else "--", label=f"{_dsn[h]} held out | {m}")
        a.set_xlabel("FGSM epsilon (/255)"); a.set_title(nm + " on the unseen dataset (clean-EER threshold frozen on training-side validation)"); a.grid(alpha=0.3); a.set_ylim(-0.02, 1.02)
    axs[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_v2_lodo_heldout.png"), dpi=130); plt.show()
else:
    print("\nNo held-out-dataset result exists yet: every experiment above is PENDING (see the registry). No cross-dataset claim can be made.")


In [ ]:
# ============================================================
# CELL 25b (v2): PRIORITY 3 -- recover clean performance without losing low-epsilon FGSM robustness (controlled pilots + validated candidate)
# ============================================================
# Protocol: one change at a time vs the CURRENT recipe, same split, same seed, same clean starting checkpoint, same budget, VALIDATION data only (test is never read here
# until the single candidate run at the very end, with thresholds frozen on validation first). A pilot is a hypothesis test, not a result to adopt automatically.
#
# Predefined ranking score S (validation, full validation set, every configuration profiled identically):
#   S = 0.30 * clean balanced accuracy @ threshold A  + 0.20 * clean ROC-AUC
#     + 0.30 * mean_{e in 0.5,1,2} balanced accuracy @ threshold B  + 0.20 * mean_{e in 0.5,1,2} ROC-AUC   - 0.5 * std of the per-epoch selection score over the last 3 epochs
#   A = clean-validation EER threshold, B = attack-aware threshold (Cell 18d definition), both selected on the pilot's own validation logits (identical treatment for every configuration).
# ELIGIBLE for 'best': finite, not hedging, clean balanced accuracy >= baseline - 0.02 (a clean regression cannot be hidden by attacked-AUC gains), and
#   min(fake recall, real specificity) @ B at 1/255 >= accept_min_class_recall.
S_WEIGHTS = {"clean_balacc": 0.30, "clean_auc": 0.20, "fgsm_balacc": 0.30, "fgsm_auc": 0.20, "stability_penalty": 0.5}
CLEAN_REGRESSION_TOL = 0.02


def make_fgsm_select_fn_v2(full_val=True):
    """Per-epoch checkpoint criterion (validation only). Rewards clean AND low-eps FGSM behaviour, never attacked AUC alone:
    raw = 0.25 clean AUC + 0.15 clean acc + 0.15 mean FGSM AUC + 0.10 min FGSM AUC + 0.20 balanced acc @target (logit 0) + 0.15 min(fake recall, real spec) @target.
    Gates: finite; clean AUC >= baseline - drop; target FGSM AUC >= floor; BOTH class rates @target >= accept_min_class_recall; no hedge; no label leak.
    score = mean(raw of last K epochs) - penalty * std (stability across epochs)."""
    base = BASELINE_VAL["clean_val_auc_full" if full_val else "clean_val_auc_subset"]

    def select_fn(history):
        r = history[-1]
        clean_acc = r.get("val_accuracy", r.get("val_subset_clean_acc"))
        comp = [r["val_clean_auc"], r["val_fgsm_mean_auc"], r["val_fgsm_min_auc"], r["val_fgsm_target_auc"], clean_acc, r["val_fgsm_target_sens"], r["val_fgsm_target_spec"]]
        finite = all(c == c for c in comp)
        mn = min(r["val_fgsm_target_sens"], r["val_fgsm_target_spec"])
        raw = (0.25 * r["val_clean_auc"] + 0.15 * clean_acc + 0.15 * r["val_fgsm_mean_auc"] + 0.10 * r["val_fgsm_min_auc"]
               + 0.20 * 0.5 * (r["val_fgsm_target_sens"] + r["val_fgsm_target_spec"]) + 0.15 * mn) if finite else -1.0
        r["select_raw"] = raw
        gate = bool(finite and r["val_clean_auc"] >= base - CONFIG["select_max_clean_auc_drop"] and r["val_fgsm_target_auc"] >= CONFIG["select_min_fgsm_auc"]
                    and mn >= CONFIG["accept_min_class_recall"] and r["val_clean_logit_std"] >= CONFIG["select_min_logit_std"]
                    and r["val_leak_margin"] <= CONFIG["accept_max_label_leak_margin"])
        raws = [h.get("select_raw", raw) for h in history[-CONFIG["select_smooth_epochs"]:]]
        return {"raw": float(raw), "score": float(np.mean(raws) - CONFIG["select_stability_penalty"] * (np.std(raws) if len(raws) > 1 else 0.0)), "gate_ok": gate}
    return select_fn


def val_profile(model):
    """Full-validation profile of one model: thresholds A and B from ITS OWN validation logits; metrics at those thresholds for clean and FGSM 0.5/1/2."""
    lab, zv = fgsm_sweep_logits(model, val_attack_loader, VAL_EPS_V2, desc="validation profile")
    T = float(fit_temperature(zv[0.0], lab))
    p0 = sigmoid_np(zv[0.0], T)
    thrA = float(select_threshold(lab, p0)["selected"])
    B = attack_aware_threshold_v2(lab, {e: sigmoid_np(zv[e], T) for e in VAL_EPS_V2[1:]}, CONFIG["v2_thr_tol"])
    out = {"T": T, "thr_A": thrA, "thr_B": B["thr"], "val_logit_std_clean": float(np.std(zv[0.0]))}
    for e in VAL_EPS_V2:
        p = sigmoid_np(zv[e], T)
        k = f"{e * 255:g}"
        out[f"auc_{k}"] = safe_roc_auc(lab, zv[e])
        for nm, thr in (("A", thrA), ("B", B["thr"])):
            r_, s_, b_ = _rs(lab, p, thr)
            out[f"acc_{k}_{nm}"] = float(np.mean((p >= thr) == lab)); out[f"balacc_{k}_{nm}"] = b_; out[f"recall_{k}_{nm}"] = r_; out[f"spec_{k}_{nm}"] = s_
    ks = [f"{e:g}" for e in THR_VAL_EPS]
    out["fgsm_mean_auc"] = float(np.mean([out[f"auc_{k}"] for k in ks])); out["fgsm_mean_balacc_B"] = float(np.mean([out[f"balacc_{k}_B"] for k in ks]))
    out["min_class_rate_B_1"] = float(min(out["recall_1_B"], out["spec_1_B"]))
    return out


def score_S(prof, stability_std):
    return (S_WEIGHTS["clean_balacc"] * prof["balacc_0_A"] + S_WEIGHTS["clean_auc"] * prof["auc_0"] + S_WEIGHTS["fgsm_balacc"] * prof["fgsm_mean_balacc_B"]
            + S_WEIGHTS["fgsm_auc"] * prof["fgsm_mean_auc"] - S_WEIGHTS["stability_penalty"] * stability_std)


# ---------------- 3.1 inspect what exists ----------------
print("INSPECTION OF EXISTING PILOTS")
_v1p = os.path.join(OUT_DIR, "fgsm_pilot.csv")
if os.path.exists(_v1p):
    _d = pd.read_csv(_v1p)
    print("v1 pilot (4 recipes, 3 epochs x 150 steps, validation subset) -- executed in the v1 run; it selected the CURRENT recipe (random U(0,2/255), no ALP). 3 epochs were too short for the "
          "FGSM metrics to converge (the full run needed ~9 epochs), so it ranks early learning speed, not final robustness:")
    print(_d[["variant", "val_clean_auc", "val_fgsm_mean_auc", "val_fgsm_target_auc", "val_clean_acc", "healthy"]].round(4).to_string(index=False))
else:
    print("v1 pilot table not found in the working directory.")
print("v1 notebook's eight controlled pilots (Cell 19b) were all PENDING in the v1 run (flags off): none had been executed.")

# ---------------- 3.2 recipe inspection (no new runs; evidence from v1 logs + config) ----------------
RECIPE_AUDIT = pd.DataFrame([
    ("warm-up", f"{CONFIG['fgsm_warmup_steps']} steps (FGSM) / {CONFIG['warmup_epochs']} epoch (clean)", "linear warm-up then cosine decay", "no change: no instability seen (nan-skips 0, fp32-retries 0 in all v1 runs)"),
    ("FGSM LR scale", f"{CONFIG['fgsm_lr_scale']} x clean LRs (backbone {CONFIG['backbone_lr']}, head {CONFIG['head_lr']})", "pilot 6 tests a lower scale", "changed only if pilot 6 improves S"),
    ("gradient clipping", f"{CONFIG['fgsm_grad_clip_mode']} (clip rate 10-24% of steps in v1 FGSM logs)", "spikes clipped only", "no change: clipping is not dominating the updates"),
    ("BatchNorm", f"frozen (bn_eval={CONFIG['fgsm_bn_eval']}) during FGSM-AT", "avoids train/eval statistic mismatch for the attack", "no change"),
    ("EMA", f"decay {CONFIG['ema_decay_fgsm']} (FGSM) / {CONFIG['ema_decay']} (clean); EMA weights are validated and saved", "smooths epoch noise", "no change"),
    ("regularisation", f"weight decay {CONFIG['weight_decay']}, dropout {CONFIG['dropout']}", "unchanged since v8", "no change"),
    ("loss weights", f"clean/adv/ALP = {CONFIG['fgsm_clean_loss_weight']}/{CONFIG['fgsm_adv_loss_weight']}/{CONFIG['alp_weight']}", "pilots 3a/3b", "tested"),
    ("adversarial ratio", f"{CONFIG['adv_ratio_start']} -> {CONFIG['adv_ratio']} over {CONFIG['adv_ratio_ramp_epochs']} epochs", "pilot 2", "tested"),
    ("epsilon schedule", f"ramp over {CONFIG['fgsm_eps_ramp_epochs']} epochs to {CONFIG['fgsm_train_eps'] * 255:g}/255, random per sample = {CONFIG['adv_random_eps']}", "pilots 1a/1b/4", "tested"),
], columns=["component", "current setting", "note", "decision rule"])
RECIPE_AUDIT.to_csv(os.path.join(OUT_DIR, "training_recipe_audit_v2.csv"), index=False)
print("\nTRAINING-RECIPE INSPECTION:"); print(RECIPE_AUDIT.to_string(index=False))

PILOT_PLAN_V2 = [("0_baseline_current_recipe", {}, "reference: the current v1 recipe at pilot length"),
                 ("1a_eps_ramp_2_epochs", {"fgsm_eps_ramp_epochs": 2.0}, "C. epsilon schedule: faster ramp"),
                 ("1b_eps_ramp_6_epochs", {"fgsm_eps_ramp_epochs": 6.0}, "C. epsilon schedule: slower ramp (small perturbations first)"),
                 ("2_lower_adv_ratio", {"adv_ratio_start": 0.15, "adv_ratio": 0.30}, "B. adversarial-sample ratio 0.15->0.30 instead of 0.25->0.5"),
                 ("3a_loss_weights_0.7_0.3", {"fgsm_clean_loss_weight": 0.7, "fgsm_adv_loss_weight": 0.3}, "A. clean/adv loss weights 0.7/0.3 instead of 0.6/0.4"),
                 ("3b_loss_weights_0.5_0.5", {"fgsm_clean_loss_weight": 0.5, "fgsm_adv_loss_weight": 0.5}, "A. clean/adv loss weights 0.5/0.5"),
                 ("4_fixed_eps_instead_of_random", {"adv_random_eps": not bool(CONFIG["adv_random_eps"])}, "D. fixed per-sample epsilon instead of U(0, eps)"),
                 ("5_lower_lr_scale", {"fgsm_lr_scale": 0.15}, "E. FGSM-AT learning-rate scale 0.15 instead of 0.25")]


def run_pilot_v2(name, overrides, desc, out_dir):
    E, spe = CONFIG["pilot_v10_epochs"], CONFIG["pilot_v10_steps"]
    scale = E / max(CONFIG["fgsm_epochs_effective"], 1)
    ov = dict(overrides)
    ov["fgsm_eps_ramp_epochs"] = ov.get("fgsm_eps_ramp_epochs", CONFIG["fgsm_eps_ramp_epochs"]) * scale
    ov["adv_ratio_ramp_epochs"] = CONFIG["adv_ratio_ramp_epochs"] * scale
    set_global_seed(CONFIG["seed"])
    with config_override(**ov):
        pm = new_fgsm_model_from_clean()
        hist, best = run_fit(pm, f"pilot-v2 {name}", "fgsm_pilot", E, train_loader_fgsm, spe, CONFIG["fgsm_lr_scale"], 30,
                             fgsm_adv_cfg(FGSM_EPS, CONFIG["fgsm_eps_ramp_epochs"], spe, CONFIG["adv_random_eps"], CONFIG["alp_weight"]),
                             make_fgsm_eval_fn(False), make_fgsm_select_fn_v2(False), CONFIG["ema_decay_fgsm"], fgsm_clip_cfg(), best_path=None, resume_path=None)
        mdl = build_model(); mdl.load_state_dict(best["ema_module"].state_dict()); mdl.eval()
        prof = val_profile(mdl)
    raws = [h["select_raw"] for h in hist[-3:]]
    r = hist[-1]
    prof.update({"variant": name, "description": desc, "epochs": E, "stability_std_last3": float(np.std(raws)) if len(raws) > 1 else 0.0, "val_leak_margin": r["val_leak_margin"],
                 "val_clean_logit_std": r["val_clean_logit_std"], "nan_skips": r["train_nan_skips"], "fp32_retries": r["train_fp32_retries"], "epoch_target_auc": [h["val_fgsm_target_auc"] for h in hist]})
    prof["S"] = score_S(prof, prof["stability_std_last3"])
    pd.DataFrame([{k: v for k, v in prof.items() if not isinstance(v, list)}]).to_csv(os.path.join(out_dir, "pilot_row.csv"), index=False)
    del pm, mdl
    free_cuda_memory()
    return prof


_pilot_est_h = CONFIG["pilot_v10_epochs"] * (CONFIG["pilot_v10_steps"] * 0.5 + 75) / 3600 + 0.05
PILOT_ROWS_V2 = []
for name, ov, desc in PILOT_PLAN_V2:
    rid = f"pilot_{name}"
    cfgs = experiment_config_snapshot({"experiment": "controlled FGSM-AT pilot (validation only)", "overrides": ov, "description": desc, "ranking_score": S_WEIGHTS})
    st = RUNNER.status(rid)
    if st and st.get("status") == "completed":
        PILOT_ROWS_V2.append(st["summary"]); print(f"[runner] {rid}: completed earlier -> re-used.")
        continue
    if not CONFIG["run_pilots_v10"]:
        RUNNER.mark_pending(rid, cfgs, "CONFIG['run_pilots_v10'] is False (not executed)", est_hours=_pilot_est_h); continue
    if _elapsed_left_h() < _pilot_est_h + 0.3:
        RUNNER.mark_pending(rid, cfgs, f"time guard: {_elapsed_left_h():.2f} h left after the reserve", est_hours=_pilot_est_h)
        print(f"[runner] {rid}: SKIPPED by the time guard."); continue
    st = RUNNER.run(rid, cfgs, lambda d, name=name, ov=ov, desc=desc: run_pilot_v2(name, ov, desc, str(d)))
    if st and st.get("status") == "completed":
        PILOT_ROWS_V2.append(st["summary"])

# ---------------- 3.5 comparison table (validation) ----------------
PILOT_DF = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, list)} for r in PILOT_ROWS_V2])
BEST_PILOT, CANDIDATE_REASON = None, "no pilot was executed"
if len(PILOT_DF):
    base = PILOT_DF[PILOT_DF.variant == "0_baseline_current_recipe"]
    base_bal = float(base.balacc_0_A.iloc[0]) if len(base) else float("nan")
    PILOT_DF["finite_not_hedging"] = PILOT_DF.val_clean_logit_std >= CONFIG["select_min_logit_std"]
    PILOT_DF["no_clean_regression"] = PILOT_DF.balacc_0_A >= base_bal - CLEAN_REGRESSION_TOL if len(base) else False
    PILOT_DF["class_rates_ok"] = PILOT_DF.min_class_rate_B_1 >= CONFIG["accept_min_class_recall"]
    PILOT_DF["eligible"] = PILOT_DF.finite_not_hedging & PILOT_DF.no_clean_regression & PILOT_DF.class_rates_ok
    PILOT_DF.to_csv(os.path.join(OUT_DIR, "pilot_comparison_v2.csv"), index=False)
    tab = pd.DataFrame({"Configuration": PILOT_DF.variant, "Clean accuracy (thr A)": PILOT_DF.acc_0_A, "Clean AUC": PILOT_DF.auc_0, "FGSM 1/255 accuracy (thr B)": PILOT_DF.acc_1_B,
                        "FGSM 1/255 AUC": PILOT_DF.auc_1, "FGSM 2/255 accuracy (thr B)": PILOT_DF.acc_2_B, "FGSM 2/255 AUC": PILOT_DF.auc_2, "S": PILOT_DF.S, "eligible": PILOT_DF.eligible})
    print("\nCONTROLLED PILOTS -- VALIDATION ONLY (thresholds A / B selected on each model's own validation logits; every configuration profiled identically):")
    print(tab.round(4).to_string(index=False))
    el = PILOT_DF[PILOT_DF.eligible]
    if len(el):
        BEST_PILOT = el.sort_values("S", ascending=False).iloc[0]
        gain = float(BEST_PILOT.S - (base.S.iloc[0] if len(base) else float("nan")))
        print(f"\nBest ELIGIBLE configuration by the predefined score S: '{BEST_PILOT.variant}' (S = {BEST_PILOT.S:.4f}; baseline pilot S = {float(base.S.iloc[0]):.4f}; gain {gain:+.4f}).")
        CANDIDATE_REASON = "best pilot is the baseline" if BEST_PILOT.variant == "0_baseline_current_recipe" else (f"gain {gain:+.4f} < v2_candidate_min_gain {CONFIG['v2_candidate_min_gain']}" if gain < CONFIG["v2_candidate_min_gain"] else "promoted")
    else:
        print("\nNo pilot is eligible (clean regression / class-rate / hedging gates). No configuration is recommended.")
    print("CAUTION: pilots are short (", CONFIG["pilot_v10_epochs"], "epochs x", CONFIG["pilot_v10_steps"], "steps); a differences below ~0.01 in S is within what one run can resolve. The baseline pilot is the fair reference, not the 14-epoch v1 model.")

# ---------------- candidate: full run of the promoted configuration, frozen thresholds, test comparison with the v1 FGSM-trained model ----------------
CANDIDATE = {"status": "pending", "reason": CANDIDATE_REASON}
CAND_DIR = os.path.join(OUT_DIR, "candidate_v2"); os.makedirs(CAND_DIR, exist_ok=True)


def run_candidate(variant, overrides, seed, clean_path, out_dir):
    tag = f"cand_s{seed}"
    P = {k: os.path.join(out_dir, f"{tag}_{k}.pth") for k in ("fgsm", "soft", "rf")}
    set_global_seed(seed)
    with config_override(**overrides):
        m = new_fgsm_model_from_clean(clean_path)
        fh, fb = run_fit(m, f"FGSM-AT candidate seed{seed}", "fgsm", CONFIG["fgsm_epochs_effective"], train_loader_fgsm, CONFIG["fgsm_steps_per_epoch"], CONFIG["fgsm_lr_scale"],
                         CONFIG["fgsm_warmup_steps"], fgsm_adv_cfg(FGSM_EPS, CONFIG["fgsm_eps_ramp_epochs"], CONFIG["fgsm_steps_per_epoch"], CONFIG["adv_random_eps"], CONFIG["alp_weight"]),
                         make_fgsm_eval_fn(True), make_fgsm_select_fn_v2(True), CONFIG["ema_decay_fgsm"], fgsm_clip_cfg(), best_path=P["fgsm"], soft_path=P["soft"], resume_path=P["rf"],
                         early_stop={"min_epochs": CONFIG["early_stop_min_epochs"], "bad_epochs": CONFIG["early_stop_bad_epochs"], "no_gain": CONFIG["early_stop_no_gain"]},
                         extra_meta={"train_eps": FGSM_EPS, "random_eps": overrides.get("adv_random_eps", CONFIG["adv_random_eps"]), "seed": seed, "candidate": variant, "overrides": overrides})
    passed = fb["epoch"] is not None
    use = P["fgsm"] if passed else P["soft"]
    prof = val_profile(_load_model(use))
    return {"variant": variant, "seed": seed, "gate_passed": bool(passed), "selected_epoch": fb["epoch"] if passed else fb.get("soft_epoch"), "ckpt": os.path.basename(use), "ckpt_path": use,
            "ckpt_sha256": sha256_file(use), "epochs_run": len(fh), "val_profile": {k: v for k, v in prof.items()}}


CAND_ROWS = []
if BEST_PILOT is not None and CANDIDATE_REASON == "promoted" and CONFIG["v2_candidate_train"]:
    vname = BEST_PILOT.variant
    ov = dict(next(o for n, o, _ in PILOT_PLAN_V2 if n == vname))
    seeds_c = [CONFIG["seed"]] + list(CONFIG["v2_candidate_extra_seeds"])
    for sd in seeds_c:
        rid = f"candidate_{vname}_seed{sd}"
        cl = SEED_CKPTS.get(sd, {}).get("clean")
        cfgs = experiment_config_snapshot({"experiment": "candidate full run", "variant": vname, "overrides": ov, "training_seed": sd})
        if cl is None:
            RUNNER.mark_pending(rid, cfgs, f"no clean checkpoint for seed {sd}"); continue
        st = RUNNER.status(rid)
        est_h = CONFIG["fgsm_epochs_effective"] * (CONFIG["fgsm_steps_per_epoch"] * 0.55 + 70) / 3600 + 0.15
        if not (st and st.get("status") == "completed") and _elapsed_left_h() < est_h:
            RUNNER.mark_pending(rid, cfgs, f"time guard: {_elapsed_left_h():.2f} h left < {est_h:.2f} h", est_hours=est_h); print(f"[runner] {rid}: SKIPPED by the time guard."); continue
        st = RUNNER.run(rid, cfgs, lambda d, vname=vname, ov=ov, sd=sd, cl=cl: run_candidate(vname, ov, sd, cl, str(d)))
        if st and st.get("status") == "completed":
            CAND_ROWS.append(st["summary"])
    CANDIDATE = {"status": "completed" if CAND_ROWS else "pending", "reason": "" if CAND_ROWS else "time guard / failure"}

if CAND_ROWS:
    # ---- freeze thresholds on VALIDATION (candidate), then evaluate on test, next to the v1 FGSM-trained model of the same seed ----
    cand_frozen = {}
    for r in CAND_ROWS:
        v = r["val_profile"]
        cand_frozen[str(r["seed"])] = {"T": v["T"], "thr_clean_val_EER": v["thr_A"], "thr_attack_aware_v2": v["thr_B"], "checkpoint": r["ckpt"], "checkpoint_sha256": r["ckpt_sha256"]}
    cpath = os.path.join(OUT_DIR, "candidate_frozen_thresholds_v2.json")
    atomic_json_save({"variant": CAND_ROWS[0]["variant"], "frozen_at_unix": time.time(), "seeds": cand_frozen}, cpath)
    print(f"Candidate thresholds frozen on validation: {cpath} (md5 {hashlib.md5(open(cpath, 'rb').read()).hexdigest()[:12]})")
    cand_frozen = json.load(open(cpath))["seeds"]
    trows = []
    for r in CAND_ROWS:
        fr = cand_frozen[str(r["seed"])]
        assert sha256_file(r["ckpt_path"]) == fr["checkpoint_sha256"]
        labs, zd = fgsm_sweep_logits(_load_model(r["ckpt_path"]), make_test_attack_loader(), EPS_V2, desc=f"test candidate seed{r['seed']}")
        for tname, thr in (("A: clean-val EER", fr["thr_clean_val_EER"]), ("B: attack-aware (val)", fr["thr_attack_aware_v2"])):
            t = eps_diag_table("FGSM-trained (v2 candidate)", labs, zd, fr["T"], thr, tname, EPS_V2)
            t.insert(0, "seed", r["seed"]); trows.append(t)
    CAND_TEST = pd.concat(trows, ignore_index=True).rename(columns={"recall_fake": "fake_recall", "recall_real": "real_specificity"})
    CAND_TEST["eps_255"] = CAND_TEST.eps_255.round(6)
    v1 = THR_DF[(THR_DF.model == "FGSM-trained") & (THR_DF.label_source == "true") & THR_DF.threshold_type.isin(["A: clean-val EER", "B: attack-aware (val)"]) & THR_DF.seed.isin(CAND_TEST.seed.unique())].copy()
    v1["model"] = "FGSM-trained (v1 recipe)"
    CMP = pd.concat([v1[CAND_TEST.columns.intersection(v1.columns)], CAND_TEST], ignore_index=True)
    CMP.to_csv(os.path.join(OUT_DIR, "candidate_vs_v1_test_v2.csv"), index=False)
    print("\nCANDIDATE (v2) versus v1 FGSM-trained model -- TEST, thresholds frozen on validation:")
    print(CMP[CMP.eps_255.isin([0, 1, 2])][["seed", "model", "threshold_type", "eps_255", "accuracy", "roc_auc_independent", "macro_f1", "balanced_accuracy", "fake_recall", "real_specificity"]]
          .sort_values(["seed", "threshold_type", "eps_255", "model"]).round(3).to_string(index=False))
else:
    print(f"\nNo candidate run: {CANDIDATE_REASON}. Nothing is claimed about improved clean performance.")
print("\nEXPERIMENT REGISTRY (pilots and candidate runs):")
print(RUNNER.table().round(2).to_string(index=False))


In [ ]:
# ============================================================
# CELL 21b (v2): PRIORITY 4 -- Grad-CAM stability: why explanations change under FGSM, on >= 100 test videos (diagnostic only; nothing is tuned on it)
# ============================================================
# The corrected v10 implementation (class-specific target, shared display scale, RAW maps, inverse normalisation, crop alignment, projection onto the original
# frame) is kept unchanged and re-used from Cell 15c (VideoGradCAMv2, deletion_faithfulness, explanation_stability, heat_to_original, gradcam_v2_figure).
# Three DIFFERENT properties are measured and reported separately:
#   STABILITY     does the map move when the input is perturbed?                  (Spearman, top-20% IoU, channel-weight cosine, adjacent-frame consistency)
#   FAITHFULNESS  does deleting the attributed region change the explained score? (top / random / least-attributed deletion; a proxy, not a proof)
#   LOCALISATION  does the map fall on the manipulated region?                    -> NOT evaluated: no ground-truth masks exist in these datasets (nothing is fabricated)
# Explained class = the class the model assigns to the CLEAN video at its frozen threshold A, kept fixed for the perturbed inputs, so that a change of the map
# is not confounded with a change of the explained class (the class assigned under attack is recorded separately).
G2_EPS = [float(e) for e in CONFIG["gradcam_v2_eps"]]
G2_FRACS = [float(f) for f in CONFIG["gradcam_v2_delete_fracs"]]
G2_N = int(CONFIG["gradcam_v2_n_videos"])
_g2_est_h_per_video = 0.0035
G2_SEED = CONFIG["seed"]


def explain_full(model, video, target, layer=None):
    """Like VideoGradCAMv2.explain but also returns the per-frame channel weights [T,C] and the channel-mean activation map [T,h,w] (decomposition of the map)."""
    lay = layer if layer is not None else _default_target_layer(model)
    store = {}

    def fwd_hook(mod, inp, out):
        store["act"] = out
        if out.requires_grad:
            out.register_hook(lambda g: store.__setitem__("grad", g))

    h = lay.register_forward_hook(fwd_hook)
    prev = _prepare_model_for_attack(model)
    try:
        x = video.detach().clone().requires_grad_(True)
        logit = model(x).float().reshape(())
        score = logit if target == 1 else -logit
        model.zero_grad(set_to_none=True)
        score.backward()
        act, grad = store["act"].detach(), store["grad"]
        w = grad.mean(dim=(2, 3))                                            # [T,C]
        cams = Fn.relu((w[:, :, None, None] * act).sum(dim=1)).float().cpu().numpy()
        amap = Fn.relu(act).mean(dim=1).float().cpu().numpy()
        zlogit = float(logit.detach())
    finally:
        h.remove(); _restore_model_modes(model, prev); model.zero_grad(set_to_none=True)
    return {"raw": cams, "w": w.float().cpu().numpy(), "act": amap, "logit": zlogit}


def _spearman(a, b):
    from scipy.stats import spearmanr
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    return float(spearmanr(a, b)[0]) if a.std() > 0 and b.std() > 0 else float("nan")


def _topk_iou(a, b, frac=0.2):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    k = max(1, int(round(frac * a.size)))
    ta, tb = set(np.argsort(-a, kind="stable")[:k]), set(np.argsort(-b, kind="stable")[:k])
    return len(ta & tb) / len(ta | tb)


def _cos(a, b):
    a, b = np.asarray(a, np.float64).ravel(), np.asarray(b, np.float64).ravel()
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d > 0 else float("nan")


def pair_stability(ea, eb):
    """Stability metrics between two explanations of the same video (mean over frames), plus the decomposition into activation vs channel-weight change."""
    T = ea["raw"].shape[0]
    sp = [_spearman(ea["raw"][t], eb["raw"][t]) for t in range(T)]
    iou = [_topk_iou(ea["raw"][t], eb["raw"][t]) for t in range(T)]
    wc = [_cos(ea["w"][t], eb["w"][t]) for t in range(T)]
    asp = [_spearman(ea["act"][t], eb["act"][t]) for t in range(T)]
    adj_a = [_spearman(ea["raw"][t], ea["raw"][t + 1]) for t in range(T - 1)]
    adj_b = [_spearman(eb["raw"][t], eb["raw"][t + 1]) for t in range(T - 1)]
    fm = lambda v: float(np.nanmean(v)) if np.any(np.isfinite(v)) else float("nan")
    return {"spearman": fm(sp), "iou_top20": fm(iou), "weight_cosine": fm(wc), "activation_spearman": fm(asp),
            "adjacent_spearman_ref": fm(adj_a), "adjacent_spearman_pert": fm(adj_b), "adjacent_change": fm(adj_b) - fm(adj_a),
            "mass_ratio": float(eb["raw"].sum() / max(ea["raw"].sum(), 1e-12))}


# ---------------- models (seed-42 pair with FROZEN thresholds; optional v2 candidate) ----------------
G2_MODELS = []
_fr = FROZEN[CONFIG["seed"]]
G2_MODELS.append({"name": "Clean-trained", "model": clean_model, "T": _fr["Clean-trained"]["T"], "thr": _fr["Clean-trained"]["thr_clean_val_EER"]})
G2_MODELS.append({"name": "FGSM-trained", "model": fgsm_eval_model, "T": _fr["FGSM-trained"]["T"], "thr": _fr["FGSM-trained"]["thr_clean_val_EER"]})
_cand_json = os.path.join(OUT_DIR, "candidate_frozen_thresholds_v2.json")
try:
    if CAND_ROWS and os.path.exists(_cand_json):
        _cr = next(r for r in CAND_ROWS if r["seed"] == CONFIG["seed"])
        _cf = json.load(open(_cand_json))["seeds"][str(CONFIG["seed"])]
        G2_MODELS.append({"name": "FGSM-trained (v2 candidate)", "model": _load_model(_cr["ckpt_path"]), "T": _cf["T"], "thr": _cf["thr_clean_val_EER"]})
except Exception as _e:
    print("candidate model not added to Grad-CAM:", _e)


def _pred(m, v):
    with torch.no_grad():
        p = float(torch.sigmoid(m["model"](v).float() / m["T"]))
    return int(p >= m["thr"]), p


# ---------------- video selection: stratified by dataset / class / method, with misclassified videos over-represented ----------------
def select_g2_videos(n):
    rng = np.random.RandomState(G2_SEED)
    idx_all = np.arange(len(test_records))
    pool = rng.permutation(idx_all)[: max(n * 3, 300)]
    info = {}
    for i in pool:
        folder, label = test_records[int(i)]
        v, _ = test_dataset[int(i)]
        v = v.unsqueeze(0).to(CONFIG["device"])
        wrong = any(_pred(m, v)[0] != label for m in G2_MODELS[:2])
        ri = record_info.get(folder, {})
        info[int(i)] = {"label": label, "dataset": ri.get("dataset", "?"), "method": ri.get("method", "?"), "wrong": bool(wrong)}
    chosen = [i for i, d in info.items() if d["wrong"]][: max(1, n // 4)]
    groups = {}
    for i, d in info.items():
        if i not in chosen:
            groups.setdefault((d["dataset"], d["label"], d["method"]), []).append(i)
    keys = sorted(groups, key=str)
    while len(chosen) < n and any(groups.values()):
        for k in keys:
            if groups[k] and len(chosen) < n:
                chosen.append(groups[k].pop(0))
    return chosen, info


G2_SEL, G2_INFO = select_g2_videos(G2_N)
print(f"Grad-CAM v2: {len(G2_SEL)} test videos selected (target >= {G2_N}); "
      f"{sum(G2_INFO[i]['label'] == 0 for i in G2_SEL)} real / {sum(G2_INFO[i]['label'] == 1 for i in G2_SEL)} fake; "
      f"{sum(G2_INFO[i]['wrong'] for i in G2_SEL)} misclassified by >=1 model at clean input; datasets: "
      f"{dict(pd.Series([G2_INFO[i]['dataset'] for i in G2_SEL]).value_counts())}")
if len(G2_SEL) < 100:
    print(f"WARNING: only {len(G2_SEL)} videos are available in the test split (< 100 requested): the analysis is reported with that n.")

G2_ROWS, G2_DEL = [], []
_t0 = time.time()
for n_done, i in enumerate(tqdm(G2_SEL, desc="Grad-CAM v2", leave=False)):
    if _elapsed_left_h() < 0.15:
        print(f"time guard: Grad-CAM stopped after {n_done} videos ({_elapsed_left_h():.2f} h left)."); break
    folder, label = test_records[i]
    v, _ = test_dataset[i]
    v = v.unsqueeze(0).to(CONFIG["device"]); y = torch.tensor([label], device=CONFIG["device"])
    meta = G2_INFO[i]
    for m in G2_MODELS:
        mdl = m["model"]
        pred0, p0 = _pred(m, v)
        e0 = explain_full(mdl, v, pred0)
        for k, d in enumerate(deletion_faithfulness(mdl, v, e0["raw"], pred0, fracs=G2_FRACS, repeats=CONFIG["gradcam_random_repeats"], seed=i)):
            G2_DEL.append({"model": m["name"], "video": i, "label": label, "dataset": meta["dataset"], "method": meta["method"], "input": "clean", "eps_255": 0.0, **d})
        for e255 in G2_EPS:
            eps = e255 / 255.0
            conds = {"FGSM (own, white-box)": fgsm_attack(mdl, v, y, eps),
                     "random sign (same L_inf)": v + _eps_normalized(eps, v.device) * torch.sign(torch.randn_like(v))}
            if m["name"] != "Clean-trained":
                conds["FGSM transferred from clean-trained"] = fgsm_attack(G2_MODELS[0]["model"], v, y, eps)
            for cname, xa in conds.items():
                pa, pra = _pred(m, xa)
                ea = explain_full(mdl, xa, pred0)                 # SAME explained class as for the clean input
                row = {"model": m["name"], "video": i, "label": label, "dataset": meta["dataset"], "method": meta["method"], "eps_255": e255, "perturbation": cname,
                       "clean_pred": pred0, "clean_correct": pred0 == label, "pert_pred": pa, "pert_correct": pa == label, "class_flipped": pa != pred0,
                       "logit_clean": e0["logit"], "logit_pert": ea["logit"], **pair_stability(e0, ea)}
                G2_ROWS.append(row)
                if cname.startswith("FGSM (own") and abs(e255 - 1.0) < 1e-9:       # faithfulness of the map computed on the ATTACKED input (mid epsilon only: cost)
                    for d in deletion_faithfulness(mdl, xa, ea["raw"], pred0, fracs=G2_FRACS, repeats=CONFIG["gradcam_random_repeats"], seed=i):
                        G2_DEL.append({"model": m["name"], "video": i, "label": label, "dataset": meta["dataset"], "method": meta["method"], "input": "FGSM own", "eps_255": e255, **d})

G2 = pd.DataFrame(G2_ROWS); G2D = pd.DataFrame(G2_DEL)
G2.to_csv(os.path.join(OUT_DIR, "gradcam_v2_stability_per_video.csv"), index=False)
G2D.to_csv(os.path.join(OUT_DIR, "gradcam_v2_deletion_per_video.csv"), index=False)
print(f"{G2.video.nunique() if len(G2) else 0} videos explained in {(time.time() - _t0) / 60:.1f} min.")

# ---------------- summaries ----------------
G2_N_VIDEOS = int(G2.video.nunique()) if len(G2) else 0
if len(G2):
    _mets = ["spearman", "iou_top20", "weight_cosine", "activation_spearman", "adjacent_change", "mass_ratio"]
    S_MAIN = G2.groupby(["model", "perturbation", "eps_255"])[_mets + ["class_flipped"]].mean().reset_index()
    S_MAIN.to_csv(os.path.join(OUT_DIR, "gradcam_v2_stability_summary.csv"), index=False)
    print("\nSTABILITY of the explanation (mean over videos; Spearman / IoU compare the map of the CLEAN input with the map of the perturbed input, same explained class):")
    print(S_MAIN.round(3).to_string(index=False))
    for key in ("label", "dataset", "clean_correct"):
        g = G2[G2.perturbation == "FGSM (own, white-box)"].groupby(["model", key, "eps_255"])[["spearman", "iou_top20"]].agg(["mean", "count"]).round(3)
        g.to_csv(os.path.join(OUT_DIR, f"gradcam_v2_stability_by_{key}.csv"))
        print(f"\nFGSM(own) stability by {key}:"); print(g.to_string())
    # attack effect beyond random noise: paired difference (random-sign minus FGSM) per video
    piv = G2.pivot_table(index=["model", "video", "eps_255"], columns="perturbation", values="spearman").reset_index()
    if "FGSM (own, white-box)" in piv and "random sign (same L_inf)" in piv:
        from scipy.stats import wilcoxon
        pr = []
        for (mn, e), g in piv.groupby(["model", "eps_255"]):
            d = (g["random sign (same L_inf)"] - g["FGSM (own, white-box)"]).dropna()
            try:
                pv = float(wilcoxon(d, alternative="greater").pvalue) if len(d) > 5 else float("nan")
            except Exception:
                pv = float("nan")
            pr.append({"model": mn, "eps_255": e, "n": len(d), "mean_spearman_random": g["random sign (same L_inf)"].mean(), "mean_spearman_fgsm": g["FGSM (own, white-box)"].mean(), "p_fgsm_less_stable": pv})
        G2_PAIRED = pd.DataFrame(pr); G2_PAIRED.to_csv(os.path.join(OUT_DIR, "gradcam_v2_fgsm_vs_random.csv"), index=False)
        print("\nIs the instability specific to FGSM (adversarial direction) or just L_inf noise? (paired Wilcoxon, random-sign Spearman > FGSM Spearman):"); print(G2_PAIRED.round(4).to_string(index=False))
    # target-class consistency + score collapse
    tc = G2[G2.perturbation == "FGSM (own, white-box)"].groupby(["model", "eps_255"]).agg(class_flip_rate=("class_flipped", "mean"), mean_logit_shift=("logit_pert", "mean"), mean_logit_clean=("logit_clean", "mean")).reset_index()
    tc.to_csv(os.path.join(OUT_DIR, "gradcam_v2_target_consistency.csv"), index=False)
    print("\nTarget-class consistency (an explanation that 'moves' because the predicted class flipped is a different phenomenon from one that moves at a constant prediction):"); print(tc.round(3).to_string(index=False))
    # stability restricted to videos whose prediction did NOT flip (isolates attribution drift at constant prediction)
    nf = G2[(G2.perturbation == "FGSM (own, white-box)") & (~G2.class_flipped)].groupby(["model", "eps_255"])[["spearman", "iou_top20", "weight_cosine", "activation_spearman"]].agg(["mean", "count"]).round(3)
    nf.to_csv(os.path.join(OUT_DIR, "gradcam_v2_stability_no_flip.csv"))
    print("\nFGSM(own) stability among videos whose predicted class did NOT change:"); print(nf.to_string())

if len(G2D):
    DS = G2D.groupby(["model", "input", "frac"])[["drop_top", "drop_random", "drop_bottom"]].mean().reset_index()
    DS["n_videos"] = G2D.groupby(["model", "input", "frac"]).video.nunique().values
    from scipy.stats import wilcoxon
    pt, pb = [], []
    for k, g in G2D.groupby(["model", "input", "frac"]):
        for lst, a, b in ((pt, "drop_top", "drop_random"), (pb, "drop_random", "drop_bottom")):
            try:
                lst.append(float(wilcoxon(g[a] - g[b], alternative="greater").pvalue))
            except Exception:
                lst.append(float("nan"))
    DS["p_top_gt_random"], DS["p_random_gt_bottom"] = pt, pb
    DS.to_csv(os.path.join(OUT_DIR, "gradcam_v2_deletion_summary.csv"), index=False)
    print("\nDELETION FAITHFULNESS (drop of the explained-class score; clean input vs the attacked input at 1/255):"); print(DS.round(3).to_string(index=False))
    for key in ("label", "dataset"):
        D2 = G2D[G2D.input == "clean"].groupby(["model", key, "frac"])[["drop_top", "drop_random", "drop_bottom"]].mean().round(3)
        D2.to_csv(os.path.join(OUT_DIR, f"gradcam_v2_deletion_by_{key}.csv"))

# ---------------- layer ablation (does the choice of target layer explain the instability?) ----------------
LAYER_ROWS = []
_nl = min(int(CONFIG["gradcam_v2_layer_ablation_n"]), len(G2_SEL))
try:
    feats = G2_MODELS[1]["model"].feature_extractor[0]
    _layers = {"last block (7x7, default)": -1, "second-last block (14x14)": -2}
    for i in tqdm(G2_SEL[:_nl], desc="layer ablation", leave=False):
        if _elapsed_left_h() < 0.1:
            break
        folder, label = test_records[i]
        v, _ = test_dataset[i]; v = v.unsqueeze(0).to(CONFIG["device"]); y = torch.tensor([label], device=CONFIG["device"])
        for m in G2_MODELS[:2]:
            pred0, _p = _pred(m, v)
            xa = fgsm_attack(m["model"], v, y, 1 / 255.0)
            for ln, li in _layers.items():
                lay = m["model"].feature_extractor[0][li]
                ea, eb = explain_full(m["model"], v, pred0, lay), explain_full(m["model"], xa, pred0, lay)
                LAYER_ROWS.append({"model": m["name"], "layer": ln, "video": i, **pair_stability(ea, eb)})
except Exception as _e:
    print("layer ablation could not be completed:", repr(_e))
if LAYER_ROWS:
    LAY = pd.DataFrame(LAYER_ROWS).groupby(["model", "layer"])[["spearman", "iou_top20", "weight_cosine"]].mean().reset_index()
    LAY.to_csv(os.path.join(OUT_DIR, "gradcam_v2_layer_ablation.csv"), index=False)
    print("\nTARGET-LAYER ABLATION (FGSM 1/255, own): Spearman / IoU between clean-input and attacked-input maps"); print(LAY.round(3).to_string(index=False))

# ---------------- plots ----------------
if len(G2):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for (mn, pn), g in G2.groupby(["model", "perturbation"]):
        gg = g.groupby("eps_255")[["spearman", "iou_top20", "weight_cosine"]].mean()
        ls = "-" if "FGSM (own" in pn else ("--" if "random" in pn else ":")
        for a, c in zip(ax, ["spearman", "iou_top20", "weight_cosine"]):
            a.plot(gg.index, gg[c], ls, marker="o", label=f"{mn} | {pn}")
    for a, t in zip(ax, ["Spearman (clean map vs perturbed map)", "top-20% IoU", "channel-weight cosine"]):
        a.set_title(t, fontsize=10); a.set_xlabel("epsilon (/255)"); a.set_ylim(-0.1, 1.05); a.grid(alpha=0.3)
    ax[0].legend(fontsize=6)
    fig.suptitle(f"Explanation stability vs epsilon (n={G2_N_VIDEOS} test videos, same explained class)", fontsize=11)
    save_fig(fig, "fig_v2_gradcam_stability_vs_eps.png"); plt.show()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    g = G2[(G2.perturbation == "FGSM (own, white-box)") & (G2.eps_255 == 1.0)]
    for a, key in zip(ax, ["label", "dataset"]):
        g2 = g.assign(grp=g[key].map({0: "real", 1: "fake"}) if key == "label" else g[key])
        groups = sorted(g2.grp.unique()); models = list(g2.model.unique()); wd = 0.8 / max(len(models), 1)
        for mi, mn in enumerate(models):
            a.boxplot([g2[(g2.grp == gr) & (g2.model == mn)].spearman.dropna() for gr in groups], positions=np.arange(len(groups)) + mi * wd, widths=wd * 0.9, patch_artist=True,
                      boxprops=dict(facecolor=f"C{mi}", alpha=0.5))
            a.plot([], [], color=f"C{mi}", lw=6, alpha=0.5, label=mn)
        a.set_xticks(np.arange(len(groups)) + wd * (len(models) - 1) / 2); a.set_xticklabels(groups); a.set_title(f"Spearman, FGSM 1/255, by {key}", fontsize=10); a.legend(fontsize=7); a.grid(alpha=0.3)
    save_fig(fig, "fig_v2_gradcam_stability_by_group.png"); plt.show()
if len(G2D):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for a, inp in zip(ax, ["clean", "FGSM own"]):
        for mi, (mn, g) in enumerate(G2D[G2D.input == inp].groupby("model")):
            gg = g.groupby("frac")[["drop_top", "drop_random", "drop_bottom"]].mean()
            a.plot(gg.index, gg.drop_top, "-o", color=f"C{mi}", label=f"{mn} top"); a.plot(gg.index, gg.drop_random, "--s", color=f"C{mi}", label=f"{mn} random")
            a.plot(gg.index, gg.drop_bottom, ":^", color=f"C{mi}", label=f"{mn} least")
        a.set_title(f"deletion on {inp} input"); a.set_xlabel("fraction of cells deleted"); a.grid(alpha=0.3)
    ax[0].set_ylabel("drop of explained-class score"); ax[0].legend(fontsize=6)
    save_fig(fig, "fig_v2_gradcam_deletion.png"); plt.show()

# ---------------- examples: one correct real, one correct fake, one misclassified (uses the Cell 15c figure with a shared scale) ----------------
G2_EXAMPLES = []
try:
    wanted = [next((i for i in G2_SEL if G2_INFO[i]["label"] == 1 and not G2_INFO[i]["wrong"]), None), next((i for i in G2_SEL if G2_INFO[i]["label"] == 0 and not G2_INFO[i]["wrong"]), None),
              next((i for i in G2_SEL if G2_INFO[i]["wrong"]), None)]
    for w in [w for w in wanted if w is not None]:
        G2_EXAMPLES.append(gradcam_v2_figure(w, eps=1 / 255.0, show_legacy=False))
except Exception as _e:
    print("example figures not (all) produced:", repr(_e))

G2_VERDICT = []
if len(G2):
    for mn in [m["name"] for m in G2_MODELS]:
        a = G2[(G2.model == mn) & (G2.perturbation == "FGSM (own, white-box)") & (G2.eps_255 == 1.0)]
        r = G2[(G2.model == mn) & (G2.perturbation == "random sign (same L_inf)") & (G2.eps_255 == 1.0)]
        if len(a):
            G2_VERDICT.append(f"{mn}: Spearman clean-vs-FGSM(1/255) = {a.spearman.mean():.2f} (random-sign {r.spearman.mean():.2f}); top-20% IoU {a.iou_top20.mean():.2f}; "
                              f"channel-weight cosine {a.weight_cosine.mean():.2f}; activation-map Spearman {a.activation_spearman.mean():.2f}; class flips {a.class_flipped.mean():.0%}")
print("\nGRAD-CAM v2 SUMMARY (stability only; NOT localisation; faithfulness is the deletion table above):")
for l in G2_VERDICT:
    print("  -", l)
print("  - Reading: low Spearman with high weight-cosine => the channel weights are stable and the activations moved; low weight-cosine => the gradient (attribution weights) moved. "
      "If random-sign noise at the same L_inf barely changes the map but FGSM does, the instability is a property of the adversarial direction, not of the explanation method's noise.")
print("  - Localisation accuracy: NOT evaluated (no ground-truth masks in the datasets).")


In [ ]:
# ============================================================
# CELL 26 (v2): FINAL REPORT -- generated ONLY from artifacts that exist in this run; anything not executed is written as PENDING
# ============================================================
def _ex(n):
    return os.path.exists(os.path.join(OUT_DIR, n))


def _csv(n):
    return pd.read_csv(os.path.join(OUT_DIR, n)) if _ex(n) else None


def _tbl(df, cols=None, n=None):
    if df is None or not len(df):
        return "(none)"
    d = df[cols] if cols else df
    return "```\n" + (d.head(n) if n else d).round(3).to_string(index=False) + "\n```"


R = []
R.append("# FINAL REPORT (v2) -- generated from the artifacts present in this run\n")
R.append(f"Split fingerprint `{SPLIT_FINGERPRINT}`. Strict scope: video only, EfficientNet-B1 + LSTM, FGSM only (no PGD/BIM/AutoAttack or any other attack was implemented or run).\n")
_reg = RUNNER.table()

# ---- A ----
R.append("## A. Threshold collapse")
_res = _csv("threshold_collapse_resolution_v2.csv")
if _res is not None:
    R.append(f"- Resolved by the predefined criterion (fake recall AND real specificity >= {COLLAPSE_RESOLVED_MIN} at 0.5/1/2 /255 with the FROZEN attack-aware threshold, clean balanced accuracy within {CLEAN_BALACC_MAX_DROP} of the clean-EER threshold, every valid seed): **{'YES' if THR_RESOLVED else 'NO / PARTLY'}**")
    R.append(_tbl(_res))
    for l in DIAG_LINES:
        R.append(f"  - {l}")
    _t = _csv("threshold_summary_mean_std_v2.csv")
    if _t is not None:
        R.append("Mean +- std over valid seeds (test, frozen thresholds):")
        R.append(_tbl(_t[(_t.eps_255.isin([0, 1, 2])) & _t.threshold_type.isin(["A: clean-val EER", "B: attack-aware (val)"])],
                      ["model", "eps_255", "threshold_type", "n_valid_seeds", "threshold_mean", "accuracy_mean", "accuracy_std", "roc_auc_independent_mean", "fake_recall_mean", "real_specificity_mean"]))
else:
    R.append("- PENDING: the threshold study (Cell 24c) did not run.")
R.append("- Thresholds are frozen per seed on validation data in `frozen_thresholds_v2.json` before any test image is scored. Threshold choice does not create robustness; only FGSM was evaluated.\n")

# ---- B ----
R.append("## B. Cross-dataset generalisation (leave-one-dataset-out)")
_rg = _csv("cross_dataset_registry_v2.csv")
R.append(_tbl(_rg) if _rg is not None else "- PENDING: Cell 25a did not run.")
_lo = _csv("lodo_heldout_metrics_v2.csv")
if _lo is not None and len(_lo):
    R.append("Held-out results (never used for training, validation, checkpointing, thresholds or hyper-parameters):")
    R.append(_tbl(_lo[_lo.threshold_type == "A: clean-val EER"], ["holdout", "model", "eps_255", "accuracy", "roc_auc_independent", "macro_f1", "balanced_accuracy", "fake_recall", "real_specificity"]))
    _mv = _csv("mixed_vs_heldout_v2.csv")
    R.append("Compared with the mixed-dataset results (reported side by side, never merged):\n" + _tbl(_mv))
else:
    R.append("- No held-out fold was executed in this run: **no cross-dataset claim is made**. Folds with status `pending` in the registry above were not run.")
R.append("- UADFV has 10 real / 10 fake test videos: exploratory only; never part of a headline claim.\n")

# ---- C ----
R.append("## C. Clean performance and FGSM robustness")
_pc = _csv("pilot_comparison_v2.csv")
if _pc is not None and len(_pc):
    R.append(_tbl(_pc, ["variant", "acc_0_A", "auc_0", "acc_1_B", "auc_1", "acc_2_B", "auc_2", "S", "eligible"]))
    R.append(f"- Best eligible configuration by the predefined validation score S: **{BEST_PILOT.variant if BEST_PILOT is not None else 'none'}**; candidate run: {CANDIDATE['status']} ({CANDIDATE['reason'] or 'ok'}).")
else:
    R.append("- Controlled pilots PENDING (not executed in this run; v1 had none executed either). No claim of improved clean performance is made.")
_cv = _csv("candidate_vs_v1_test_v2.csv")
if _cv is not None:
    R.append("Candidate vs v1 recipe on TEST (thresholds frozen on validation):\n" + _tbl(_cv[_cv.eps_255.isin([0, 1, 2])], ["seed", "model", "threshold_type", "eps_255", "accuracy", "roc_auc_independent", "fake_recall", "real_specificity"]))
R.append("- Known limits carried from v1: the 1.5-point clean-accuracy gap between the clean- and FGSM-trained models is not significant (McNemar p = 0.2153); FGSM accuracy > clean accuracy and the label-free FGSM being stronger indicate gradient masking / label leaking, so FGSM numbers are an FGSM-only upper bound.\n")

# ---- D ----
R.append("## D. Grad-CAM stability")
_gs = _csv("gradcam_v2_stability_summary.csv")
if _gs is not None:
    R.append(f"- {G2_N_VIDEOS} test videos explained (target >= {G2_N}). Stability, faithfulness and localisation are reported separately; localisation is NOT evaluated (no masks).")
    for l in G2_VERDICT:
        R.append(f"  - {l}")
    R.append(_tbl(_gs[_gs.eps_255 == 1.0]))
    R.append(_tbl(_csv("gradcam_v2_deletion_summary.csv")))
    _pr = _csv("gradcam_v2_fgsm_vs_random.csv")
    R.append("FGSM vs same-L_inf random noise (paired):\n" + _tbl(_pr))
else:
    R.append("- PENDING: Cell 21b did not run.")
R.append("")

# ---- E ----
R.append("## E. Reproducibility")
_ss = _csv("seed_gate_reconciliation_v2.csv")
R.append(f"- Valid independent seeds: {VALID_SEEDS} (n={len(VALID_SEEDS)}); invalid/unknown: {INVALID_SEEDS}. Re-evaluations of one checkpoint are never counted as seeds. 95% CI uses a t-interval with n-1 d.o.f. (very wide for n<=3).")
if _ss is not None:
    R.append(_tbl(_ss.drop(columns=["final_evaluated_sha256"], errors="ignore")))
R.append("- Checkpoint provenance table: `checkpoint_provenance_v2.csv` (sha256 per file; clean/FGSM identity verified behaviourally).")
R.append(_tbl(_csv("checkpoint_provenance_v2.csv")))
R.append("- Non-determinism: GPU kernels (cuDNN, atomics), AMP and dataloader worker order are not bit-reproducible; reported differences below the seed-to-seed spread are not interpretable.\n")

# ---- F ----
R.append("## F. Final conclusion")
_done = lambda pat: int(((_reg.run_id.str.contains(pat)) & (_reg.status == "completed")).sum()) if len(_reg) else 0
_tot = lambda pat: int(_reg.run_id.str.contains(pat).sum()) if len(_reg) else 0
R.append(f"1. Threshold collapse: {'resolved' if THR_RESOLVED else 'not (fully) resolved'} by the predefined criterion; main cause = clean-calibrated threshold applied to compressed/shifted attacked scores (see A).")
R.append(f"2. Cross-dataset: {_done('lodo_')}/{_tot('lodo_')} folds executed in this run.")
R.append(f"3. Clean performance: {_done('pilot_')}/{_tot('pilot_')} pilots and {_done('candidate_')}/{_tot('candidate_')} candidate runs executed.")
R.append(f"4. Grad-CAM: {G2_N_VIDEOS} videos; localisation not evaluated.")
R.append(f"5. Reproducibility: {len(VALID_SEEDS)} valid seed(s).")
R.append("6. Not claimed: general adversarial robustness (FGSM only), cross-dataset generalisation without executed folds, localisation accuracy.")
R.append("7. Pending items are listed with status and reason in the experiment registry (`experiment_registry` / `cross_dataset_registry_v2.csv`).\n")
R.append("### Experiment registry\n" + _tbl(_reg))
R.append("\nStrict scope: video only, EfficientNet-B1 + LSTM, FGSM only. No separate image pipeline and no additional adversarial attacks.")
_txt = "\n".join(R) + "\n"
open(os.path.join(OUT_DIR, "final_report_v2.md"), "w").write(_txt)
print(_txt)
